# Chapter 5, Unit B: Break, repair and transfer durable memory

> **Learn with Prof Rod** — *Build Your Always-On AI Agent From Scratch*.
> **Read the full book and get the latest learning materials:** [https://profrod.ai/book](https://profrod.ai/book).
> **Join the Prof Rod learner community:** [https://profrod.ai/community](https://profrod.ai/community)
> — bring your questions, compare experiments and share what you build.
> **Original source and updates:** [profrodai/sovereign-agent](https://github.com/profrodai/sovereign-agent).

**Student edition · 90 minutes of dedicated work · 2026-09-09**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/sovereign-agent/blob/main/book/exercises/ch05/profrod-sovereign-agent-ch05-b-memory-repair-transfer-exercise.ipynb) Runs on Google Colab as it ships today, or on any local Python 3.12+ kernel.

This is one of two practical units for Chapter 5. Unit A constructs and connects the
mechanism; Unit B investigates a controlled failure, repairs it and transfers the invariant.
Each is a complete ninety-minute session, with its own setup and required conceptual introductions.
Basic Python variables, conditions, loops, functions, lists and dictionaries are the starting
knowledge. Libraries and specialized concepts used here are introduced below before the main task.

By the end you should be able to:

1. Explain the chapter's mechanism using a prediction and an observed intermediate result.
2. Repair the failure: Putting the new preference first does not remove contradictory old guidance. Remember, correct, close and reopen SQLite; invoke context and inspect the actual system message seen by the model.
3. Solve **retrieve the newest eligible memory under a limit** using changed inputs and an independent expectation.
4. Retain your implementation, failed/corrected observations, causal explanation and limits.

| Minutes | Dedicated activity | Evidence you produce |
|---|---|---|
| 0–5 | State the problem and make a prediction | Initial prediction in your own words |
| 5–25 | Foundations and library examples | Values, explanations, revised predictions |
| 25–35 | Trace setup and the main interface | Input → learner function → observation |
| 35–60 | Reproduce, diagnose and repair | Source, visible checks and runtime evidence |
| 60–80 | Implement and challenge the transfer task | Function and a new counterexample |
| 80–90 | Retrieve, explain and save | Exit ticket and retained submission |

Installation is preparation time. These are planning estimates, not measured completion times.
Use the reference primers when a term is unfamiliar; in Unit B retrieve an explanation before
re-reading it. Run All checks that the artifact executes. Unfinished student functions deliberately
produce NEEDS_WORK. Keep your first attempt before opening answers.


This notebook belongs to the nineteen-chapter edition. Its supplied teaching runtime is embedded, so it can run without the textbook or another notebook. Where code uses `REFERENCE_LESSON`, that is the frozen runtime exercise identifier; the reader-facing chapter and saved unit identifiers use the current edition. Building against a supplied runtime is not proof that you have constructed all of its dependencies.


## Run the self-contained setup

Open this notebook in **Google Colab** with the badge above, or use a local
**Python 3.12+** Jupyter kernel, with **Pydantic 2**. If needed, run
`%pip install "pydantic==2.13.4"` once in a separate cell and restart the kernel.
Package installation needs internet; the lesson itself needs no repository download, API key
or prior notebook. The closing extension can use an OpenAI key from Colab's Secrets pane to let a
live model manage Lucy's memory; without a key it replays a recorded transcript and says so.

The collapsed cell contains 91 frozen teaching files. Base85 represents compressed bytes
as text; `zlib` decompresses them; SHA-256 checks that the decoded files match this edition.
These are supplied packaging operations, not learner algorithms. `tempfile` creates an isolated
working copy; `Path` handles file locations; `sys.path` tells Python where the supplied modules
live. The code is available for inspection below and performs no package installation itself.
The subsequent lesson teaches the libraries used by the mechanisms you will implement.

Run setup on every fresh kernel. It writes scratch runtime files separately from your retained
`practical-work/ch05-b` folder. Rerunning setup restores the frozen support files and keeps
your saved work. Restarting a kernel clears variables, not saved submission files. Source basis:
Sovereign Agent `444c5f6`. Some tasks use reviewed local subprocesses; they are not an OS sandbox.

<details><summary>Supplied offline setup and teaching files</summary>


In [ ]:
import base64
import hashlib
import json
import os
import sys
import tempfile
import zlib
from pathlib import Path

minimum_python = (3, 12)
if sys.version_info[:2] < minimum_python:
    raise RuntimeError(
        "This edition needs Python 3.12 or newer, which is what Google Colab runs today."
    )
try:
    import pydantic
except ImportError as error:
    raise RuntimeError('Run %pip install "pydantic==2.13.4", then restart the kernel.') from error
if pydantic.__version__.split(".")[0] != "2":
    raise RuntimeError("Use Pydantic 2; the tested version is 2.13.4.")

# Frozen, reviewed course files: data until explicitly loaded by the lesson.
COURSE_ARCHIVE = (
    "c-ri}3v=5>)*$*<@Oi4P$%LXwJuP}Udlgxk_{Oocl9KF{q+)>tC}B(z3_)7fxb)xed3B?^(Ev$V_SjRJZ#Nc!Mn6"
    "xV_v6#Eewx025sq%dyR4T^UJP#H!TV{NOy(K<-D)?dch5S(vrGEF!<%qAk7q%<6Qq+kh(At8$sm~r<1~s#!7To35"
    "ohy8aGT6;f^ZU~!{I1_cC$E}rjslVhRMhIViq^Kx68@pBpFZB**pkm*VAy8#rnY@o!;pmH(_=&O8WZuzp@lQ9nR8"
    "mu$YbDO;d&rq`@!e=O?Fel+5D6{FiVNjpA7&I91c@FOpHzOQ-Q9p5YwMemOn~y1}Q*NdW)5d^T8Q^K=aFVBwd~-Y"
    "y1rm(Lpc%h@cR;3ItbkDpGAchgxKEe7)p-(B!S{NJa`XW9D&e{lBe?;C#{ynFNZ?E!oU4<_L_=2w4&lVmgs`Efew"
    "-N3~0%_jby#px`HXFWO@eERafQ9i=qFUN<+Zx7Co+%XPs(m^^3=W$__wttx2s$tH~PY?e5>FD(IpYAwk^I3S?k7u"
    "*H;y~VUcGNgm@b5jI%H<@Ahru|`vhX_2YS~RXt#`Q1ECwi=*juds<fE9SBbiEem(Am`KBc_LAf3!(9{uv!pR;5h2"
    "Vnri%n+VulVoxo02~b72mM(BGa06{0AZmSAmrLz{SL65PDXdZ{3Z^v#T2eZ9Ld+i5Fnx%yo0?76o3T%c$D5^>tq("
    "lUIcvUeBHlJgGo9Mri<ACt^!PRayP%h`N5P1qc9mmPvM+y9Q71`Ofu-nTC;PG`Ti_2g!G2JA$Y@@(PBL1jc)`QY)"
    "kKbe3y04XN$P5Pb)vEX?QnE!${f_M(AEX`|CEIY+XKcPR=;4%V*RS2c<t6)+;%EdTT*~)hwIO;&6<Oehx=jY`ufC"
    "pT@Hg(GE6f+5Lwfd-L>tJfRUbcXq9}&?#&K>~k+3!s!d7OaSxgoi!j8T1Pzsm0FYz7Gu~+wd!O@=a*#3B%6nmL9A"
    "Oef+!j2cC<rb0w?@OI9kL<vspT;;kk^bqj;XAlOUda0LYof!5DB9KoOmJ|6hRSV2`<z!A+VB;tUX)8l-t0&#@bRp"
    "&J%wB`q4k2;irV&^3xD>S;arpDrN&%9$cjFlLKb8E|mq%nha{eO|P#s^=p6Lx4^XWG%yFl4LhMJ|XBH=Z0{QPT`P"
    "N-OMP7XjJYD{+x}&(dbUkRv=DKSH_mdcxOJ{m+qysG8Q0Dq?dV`j(P(a4-cgHdKM2CSvbmXPRVk@U<wFeqIMmy4r"
    "8m^FhlYTfMINZGIU2FEFilVW?6!hpD)`&-OxH~=&E?&^Z4T&_Rk(w<C)UU!_NTc%xAPS_!6Hq;XyL3E!h#gpX2HL"
    "PBx-2B|$;hVSafDRrd&0FQGKG>4i|it44r-Ytjb40iV=b{MeyzSHXtUx?ayaMcOhA;206j#*9NROnw6T2ISivh2w"
    "q%N8SnYS=0-dm{>SLMY5B|l7R34iRki<(vO;S0mJo2@fcCZg^35#aRMX@36@L<o&`VB6qw>)`BqX`Lf!}t+}?}ef"
    "BE+e-kQUx=`y%Xcqw}Mdia|e5=nRu&;Yk%0GGm`%QGgDO!DNlqMY(j=xk))KAg4gnuhM!gj(%iGn@i%8R6(DTorg"
    "0EG9TQ9c}<X1{bCyn@By{psD&Xz;hB#VMFJ=B=Ui$@W;*UCIlK*TSmN_z`p{it<~$zoA_gt08yPI45-nFKeonE<0"
    "C2r#=IHD@f82yMh<CzrdlJtv}P`U*Kw2kX+%7<W*gK?i==)?;#+}aoE6ebO*xqppZCm(COrH3g*l5W#Bb(Ad|#Ge"
    "6Knz(Dx6#+Lze(@B3M%L2BKLS@RfcP21Af{ZW#Tzh|tF*lygRa4d;uD*^YNdN3YL%KOY>P|9<eccY5^4o1;H12Ji"
    "A&Gz*8c>2&GMH^WhQP5l&6gS8h9SEX?xouq*xeILoW0Y92$AZr2nij@eK)jR^3+IT)_e$|srtYz_N*uX7q1VaE|1"
    "XsYW*>xkhfgOxzz@tp!TM+)19G@C9AbJ`{uLAflogfgQVR8*zVFXN7L>MRFw!#4@ARHwhVj|Qv1X(rS-+}|UR)hE"
    "Pd?O|P5^3N(8OP}YHWl$x7|hdoI06tEjWz}&6a}sq;S9)T95(|cNpGp{V0=QAz`5b{-lHgpm;(f<Nk&2~&Ohk0i~"
    "*pt05J}x764rZ7Eb{Y!(dklyEXGbZ|3u9=fw+{N;tYnvw5f8-rm}N@u3ZZemH-T6Z6ahtSFp^T?41kHfS7TG?b(1"
    "0tz@}62EbNHx)4>MC3c5eR@Ii`!gj&$ORy!fhn5TTp$8`g>PzF&mjBV-D=eXxQI&4V1dLf-8z2J#Oqhv-RbRY?d7"
    "}&^0?5h9{fF^SG(09rIi`+$3YxNLR<!Xp3eat+<_oDN`Pj%EC=l&BD;*nV-P$3`RBK9-W|b>K07&ncXnhcA{58os"
    "g!jcX1Jp{;eyDkS=Wx+iz_u7%wWUU(h{CUbAfay*bL^2enyl#(@Fq()9w^RTkfl$qPl1nuO$`pr&^!EBk<sjAnd1"
    "bGmvYsx&bLXWkR1p`wuu^JPDWwXciV~O3e3~%P@5lz<6tJl+;(qS1DCi^I&#O2>yiL)S@^W0Q-s;pxf&K_V4xT=B"
    "Ldtih7}Z18}ynftUw55@r~>bgHwsHwPN7L96Ur5vm)AUc<#`ZiwRiA+$zDPrj#ru!l@BgGAM!0W$-rB2zEm`pa`9"
    "vrNaY@^=?nqQBzZ$eR3{>EP_nxL%3|LPH15;>DzI@p%w~R3ErE#ki%!C(uQ&Vi=4?;N6|hgWB;KUF(K%#Z8JLyAy"
    "v*OiYUZYZva;Z=?go6$HaD839BEXY&y7_Ay+Iamc*v0^SlY8jnacW1Yt<kIsiEHMP}BSNLka0BQ&iQMQJy%mJD>k"
    "&Gw-fTsz{nz>I1G~T}X<0$y;`1R4-KyTzT&u(8nll|7sf}h{~-}B#3k8;?fE_u%{H2ZOtV^;l2_>U`dYh^xvhOBC"
    "%Ca*|awg^Y+NR{-~K~OGqCnyv~>h(}W==FjwO0vBkUd>)l3$lDuFDKB+5`FikSKqw}JQYx~h@_eTMfWTK3^U`iM9"
    "p3@*%-&;6i{8Mj@wdcvkF^&%oY=TMGEa{I7ePyX|hk?=cQ71PwyhY$jLyx{0T${0=q_VNbJn(WB_pfa}tjt{ZWP!"
    "jg#w1k2wW`xE>KC>E;~Pa|m!-@&Rcoq9lWv4Q|voujMx?){j2MgT)-V73|5-sMETd@ZTP;ss{9+hU=;IdPBG%fd8"
    "6E^y%nGKSkCC0Izx-4jKC(#By@_=J4o@=%TGYM~z*pQB-+%f~}T*ZtAGvgS{P#PkomvES=T#ZYm7GJH?D544-dt4"
    "LIjn2q+cA&3+OgY@%vnFlYJ79N>iI-+UBfL@iytUm$yp#Dk8mHU{7YuH<@t(`}PdLH}yCtlnSamCO5f2U7$5wLq|"
    "$-*u47)T3O_yRb{67<+BG;$M$|+!6w{@)}NJmfmUwM8}Z{;a!~sarncp3And-FZF`dQ23zu!4fV%S}<>^73@J0#k"
    "6AK<dvL3GEA*!byv{v5Rdi44={VOX%LS`3TRgD@X|f%9e!abc<${!aZ7p)y(_HgQHyLQbCCFJrc&t2nz<-1YKW#C"
    "tps6iI18l~QGGh<3EsG0<2$u&i^=;*daK)OyzUoAMJQzZj~cQ-1=T#6F&r%hcN-*c`0OvGmbARhVv@|^24?dfF#Z"
    "GLO&J|rs2N<@Tw1<kM2-sEg}dL?JCyg5x8MUurRBQkT|5+Z@es1PU+<Y%z^uQClHm|JRYJ<FbO8AbS$$%eus`d_{"
    "#vASR90(HDc~MhXjiSczVxtv_<?#XoQayCh6DAW(2@k;{N(H{`GF^V;W^~1B3K!kkk_XNKcCySEp_fpnJx9DO`K|"
    "O$7t#35nAv--zkWTwDUsKiHD0~;0VSjav>!aXpvPVaaNQg42<u*p>a)Ct+DdF^Lyr>b*Nw*BvP&T#OdQ^Bl9&NH7"
    "4A;fHu82Z5Mjnxl5-Hn?7C6p3=FxZ-E&ZEJ&&g`5LowK!Xl2g}@@tQ`vpsf@5UHl-|~J1~#E|UeXYrbzE7}d(QFH"
    "2+Tk(8=)3B>cR#6cRT^+oL)4Mogrk8zi^YOdA(jDI4YPm$*v)(uo1b-&6>&NL;60hDUPj{L|swTArpclcO94xlvt"
    "Q`Ki$_Y&i$fhl5vIU6v4Z)i8yu#Z$BWaf|m;0U8iZ}&Mx0cd!8a7=y}>eU{18kVZuZ|yUPA%t;^6=;q(gnZzafvE"
    "6*I7KI*)28q1~*WBO+P*aFX%SK!&|5Z(js0Fc`#1)+y-AX1?4j`WgB4VIyWE+p|iAWF|{^9n9L>xw{cj}A`X9i6T"
    "uSHSGj(TBKJN6iMF%|-hvc!Bsn26;1{sg<L!0X?I(4lXqINQ><TG8jNSzoZM2KU6JAKtbiUh+*^(QF5yf-NNZ6Z%"
    "!T2&vQUF4Gn)*H~PKPjTYdZT`B3ScYb{Q_RRVk*=oiwJlN6M*}*?Zuo7}<*&Zq#XdM0DFY#zfzA*sBF>biSs?deW"
    "njm+~vN@|ravWu=_Aa0O40{)cAZaRIo*+-??B4_6$fyJ35^{_9V~DzYc7;)w&Fx$~X_%;JfyNx68?>6i*<w5nXUV"
    "_WR6uwfAZjs!wieNJdOrw&4E0HSF-XU-^oZ>OQ3}@t$EEuNly@+T;mzcly64HvZ)WM@`bG~}m^tkssYzkp(=-c5W"
    "CST8d`fv)a;D<+Q+lE&73atdshEHQIm5F#l55<VaGWkCb7v}7ws8ot2my0Psk-t14i@v96b&*tZkeMs17`+P4~9T"
    ")U*QG;Z=8x(K&G4zqj->HXuB{lj}irXy#!gzo|u)8oLfQxK5o)eIDQbsfWlJ~vk&E9A$=%sq(&gjzz>bw{DTrZ{4"
    "rh3#k<HH%+`P!48|n>$cC?`8eKIIZROcKoxYiivB@h)<oLuiR4DE>{&*8EGIj{zLwtw6Oqwl0;|h8^(eSl$7YL1p"
    "YUIDt^|2!x?bR5PM&0BZ^PcV1$v8y|fF8N{isUfOc5Hm`8Zt=W^HFtw(+9CAp1$gh&sj?LWtZZ5b(*W$US&1)5}C"
    "j_T|sA^;DeZscExgI*o?O6(JL{dZtk{PjCB#qkm6gRJ-G4pqCX;ULi_<$L4&x)_`B{$IN>C_B~cr>_M0SASLT&ia"
    "<9=I36nsF=WxPEJw{eIoToF5aMTH8xeZSNFOXl<6QZt+e+?HCvg*LIGB+CKewIvx_%yXUq}7hdAv7xYFKYxBieDF"
    "<RN4mu@PWwPh2c6=KBTR2lBXAnq+GEF8IEZ=kJ0i0dnXeR8uU|=Az);~9-4Qk$&jN@p^s#3>3hUr3DZ<$jQt3eoI"
    "$Nl=ya%)d_cWXy@aJ6;;5}F)bYrRFn^koArslH)D~-Rpu)YWQAdUVrhex)&Uk1kQ;g@i8qM4>L$Z+F5QU?f7Om6t"
    "Yx7<H?MBdUvo4+VbLMTX6%gMpwBGTwzWOgQnU;M>E*dn^WkWABx-u)%4Ya&aV?XNLit00r8vInvq2bbRq;(h2c`h"
    "EH`8L-&007mjcVdCjV1x*Pr7x0wcy0}&bC7>TsA#YUMAl{LsaAJPhJ;&Ksn^h`hRrRAvh=eYi}^ZWw5?Ak%Lt+Q5"
    "x;E(<twb(Fr7O&I6GrgsB|TE(4P<9yfuAQY4`IpFW58<8Qwvn4M41DnCNcS?Yr4xYpJm-jD-%5e>-`5bbj<2=0)~"
    "nV`A-9bpgy8o{<!HgqCc+gez$Ww!PYc?G8}l89IpI?xB0u3_WxJ>>D&qASHlfMZf`RjQ!_;sxF8RB^3X0zHStdPh"
    "TLv(8(V~{}pVgvv?RIzKj96Ujy^;Z=&W|lWuF1ES~DSV-8<B1G!nm`wV2n9vU^+rM8!Yeu?dr{>wUreB!J*DEUD7"
    "-SlLa0=*RzcG%<1ZJ?+vE1~N$j>rA-Y3N>W)7g97C5DbPf_U=RB3{5TvL8`N7w84nC+Al&0L$-{t<1R&#1I9Kg&L"
    "W{phsdGn`T)pu_<h6xR>a6LsT{N)Ww5FO4M0dp8Mhbt;g2p>paP*cZW;xPs28qsux9DOuoRTo@x0Up9t)V1eP>C3"
    "R^VKV+74`cK4b*4Aa?N4R2NQu`8xacIntaZZpj2J8Euu2tkKkLNPV{zD}w}z6jZ0qXfh445z0)e4wsR4Lhitz$+N"
    "07WFNt;QI|v8)8>wqGi~DPyv;_XFguy*I>Wlyi^b#e)U3t=tbMI9Y+1;V3fji>MPC@z!V^j=&gZ>3e(FcP}i|ObW"
    "T9W5E)7@$OES!&5<ZURLq@g(Q)Wth@@y13&$fKLb^Z2ls|kELmXg|qW-RY8#k?OwC9$SMVj2|`?TithZwjt=fCGS"
    "vp5d=ixFUq%roCDYrcZ|rN?v>ZOlo#Z1=B}fuZBmPEyY8gfI-z66uGK-Ud<{+)^{lg$N9_qVKTr>*IG4rAi9qFng"
    "bs?0(bki^yQbQCgQuyzRP*j26DDQg}8A>lBQArUhN#87{pCXp~=ung*Y6F6er?agK4duNCzD3|o`V;k8KG@UhK#b"
    "$HF*xmJu+TrZ-HbTVNeZ^mTvMOUt~qqj$g=P=v?CiFu+`1$nsx4>pX;DDOL7<eb3oHYb`y!Y+mVvdz}y{Oe^k-p`"
    "$4Dq<~L-ZuRt%T@R)!P;{DYnDnT^#YgT!2F)Wrvk^RZOWPzMF!c{f6CP9q!ab-4;p$DJMmNVtsc!Wg#<(f+<NlJg"
    "`l&ET)GqoufvvuArJHnz=-XMBP~12EqIu=EFASk$3rO7A`C^-SUm-HvAElh3HcYf0gxSDEkC%Y>bF-=)s38iPRn*"
    "QEnfzhec)-(G9b^tKb!1FDqcEDnz$lms)T)4#bw>Q8QtQ@eVPOP0z_mQHcNP?eXESC`W76f~7n#G0vd~UNf{pN_f"
    "YZ<&z@5Q6Igw6bhX+Gzv?#3J=pPlyP<6y2fG0t8v&t2U!ZI7PWiKa=-`v3;miX1%vWI5T+1KrGq$HE#=7fpkJu&U"
    "G@g86Eei%!5J#!2Czd3Nbqrwc00?r@r&y%1nH~8>SY+&CW>LD(@?rF=5HIyD;gU1RM@6^h(MtGsDCA0#gkyYn&z6"
    "KLoo)T8yK{{#!>5?2Qoz-G--20UT&xuTDZE@PvDh(%ToWtB|a9j)!cK=&vBFDEdUyTKLnX>5boIc&~DDt@yJ}8Ml"
    "Z+{<~Nk4%>(*m<N<}R_G4H~9OS5rXeWgMSO@@j24-iwE#qa)#D-~XwaS6OUDONKLAv8zjRR{I@HQCpbQ#IO+Ojz7"
    "5+1$iY`G>jnS!$M@z1)~8tLNTym6Iph>bTrMnu@-;zK<aIiprK_<)N1bmD2Fwx>(;^)yhySv}sPTrwVBKKq2vX!Q"
    "F|3*d+i)toya-^_;BU}?C#<eu>zzi?E_F?qSBPnXYfKqGf@Ily~A17)#=8p!2LD2c=A(a*o1A-7a_g_|PlWdMUm("
    "9Qd&?bO#mW+?nKzCuyF5m?WeO|xb6wRL*V%?v#XHt(H;DJU>{kfiv?A}a8F<LSIyz-yNYB5?jmg_4N-^=X1W*)C4"
    "^Ud<41-Sq;_m_!;SEb*wzwK~5gz+wSQt8N{;CBi)!Er8qLlkrI?9yz>vT}8=yrRC%vDA~glT+6>Mk(U+buGN=H&R"
    "W)7Mf6!q9DN)?P=eFD57n&k_XXWrahfN}Wt3*fpM91<T8dhUM=|)!j_X=<|1jZIb`noY?OGS!d_75Z*DI;+b^?$$"
    "$weOa2uZhLTZH}2I9b+$tm8O&okUaVQGx{k&J0zY^SgD$)--#PVE-E=SiixZv1d=KyTpLcnlMsqp+dTsJJNgw`Pd"
    "&a9*^SdSvc;Y=f@--=}2>V-cY~}*D3}Il#WQ!J^^HQavpRa%K<%?-WOAZof?M)4>4>lAezx_Z*EoPrsVI)Gq4jFU"
    "we{NdXlPupawJL7tgp48WmAulJ)wWB8%J<?~CEFH{sm>>0Bq(5e$>((lPU1ew=f-Ebgf%5vUahFBkczRN?t>KqQ9"
    "FVvdSXKv$Jc^}JI>w|aRF9m!wL`*b)27Ai9wQT3~g73qcNNELeME#E`vKy8C~dK%sx=t!+RJLWeS1cpFgcimX=6_"
    "bYRc>cRg@R`q(L*}U@J?ssmf9Yw__404xl!r(@{t!=KKQY#N6i1jx3_qqD&_~-=3`>B+9$}1soQVQ6id9fM(tocE"
    "VQ#Ny=ggVbi#{yQCi<3Ky3fVHgL4jj&>12JD7{f6$2g@0_7@!eatG_nN<z)mO_J-j-)aRvvL-;S$xIH1(3gax<li"
    "F4DDR;%3CE*jk;Ic;p{DaK=}aoM00j*|rS!i{^4C(o<eFr;qNu$n*B8`0T*2xZ6;~(ls7eOg>oV#asRn^L$Sro7!"
    "di}Jt4Xva@|JYj6c#Xlu(J47lztqWbi*fnH;R&=@rd&Q%2c{aQET%VHA@!J6F2u&aUO_vbuUGB#NOLhz2QW_KAIZ"
    "f%ZPc8YH?J#KIt3P#E8Y>JEScRYpSq?@2b*a6QVSb=yGMYaZ~%03pD2%Y8S#`wOd<?wc6j`uPeR)p0~{(n|1Tva|"
    "Xqf1Ybj|0=(J1Nx34jK%@dkFoj0`Wl=?egabaIIEPK2l1oUGlF}rrV6J$>k1ZWN%WljOF(2DetWgK%YGQXwbFZ)b"
    "YKnU$5N>G3axJAEv?$T*z1C@R;CJ{w*YIVSr%bQAY==EsEohN5aKNdx8bQ0YedV5+lUxV$479F_$#EprK#A(Zll@"
    "PWAxY$OFbe+s<>>S%;H%$#etLB9`k&8(<I~qir@>GE<PcoAc`zul8>Nb9GboF~mu=6EDe73XZ%4zB5xTH;SY~n4x"
    "ErsZMi#@$qh7xJ>Vrkk;FuH%YEp#`$R~~-mQS+nnPlFzcsG0fB;fiFsFAuHGz)?x-9f!#c|};Z@u3jAe8Xw)8Xn)"
    "L;tCA?kXPO4Xraorjy&u2&fT}A(odCxj=fm^z`I(nPVZ3XfM+?G&SBHW+8(AJ^~D}1LV{AIlj^W_S<CER{7~%VjF"
    "<etUOorc9V~$14lzRyvLTab&7ln%f}2LNheJrD$tT8UD<Ck1uC$)d?-hBE;}jGLzrXwS-SMCAJjrs-j@}irA?0Lf"
    "5GT`lg+h7a#i+J=83kDAgYoJ6WD<3s>sdF2)1WPSZsO6UD(F-4=ES6ma6Wpf;>XQDzCq2n_FPvx+YQ>T61p6Ol>+"
    "-K`IQ)2s*MfBv{=C6_4%-JI*+DZ$`=vbfmDim31T54&b@$qATFLH6|{jF43(&mbayLHbe5*XBB7qEi)?8_O+!bW&"
    "E1J0nU7!P#xcB(=X$|wPN32kr(y01`UbMX234Zs8f=ao@U31xfp5c&M>HaA`%z4;-vP%p2uZ43)&O~!HQrWrLRpX"
    "lBy1>#+G}4wm0K*!T`xDNx)K&NnI&T)(4`EUTrn*`*>Rf)Tb_U|d-ld<$>$pyAZNg#0#}viAM<^{4yRc&o_t7VfT"
    "B2N{p|P;K%Z~^@ve9955U;Hb3j-BI6e5ScYgfq(K}}U^J5x_G_<sv<CCM)gY)ClGu=ZogT2G14Y$Aw&rJ%dpg2y$"
    "G?Oe!uHlP1-DE_9^c?e{w>T67e<ESyY1K>j&CO94&*=maNdUV|91`3V61vkYAt!?gK=BNyw4%Z}Ccyy-<;VuO$F9"
    ">+BbTVITaDHzul!=Au|VJA3@xG|9Otl8Di*IIo+=viF1}Uzo=Uk>RmO=jPv+!znEx*CN1<IZU4=9Fg^b{0rqT@c0"
    "U0B;#Tn)S4XCINBW<!oY=!1nbShBJ1~|DOjK;})&IJfCw_t*WR?xXN%P7WI^@fS(F;TK|>mu@uggQ9CF|#D&EEJP"
    "395GIkf;XBfYc2d8t-!jhCI||n(*57T!MoS4SZL>3@(YQ<+c&?xIS)2l7J;l8+5n0XKz>n3UqfrSnCZYfXDRbCC;"
    "x^pki2pe?`r7+B_c<B_J<s>fRdZ63qp$e)3yz$&ipa0#atdL80OoJf2kHI9Zj$6YohHvpDv#<CAZ9gK~ZPekrD5|"
    "tb4x*5lQTVP>*C=jJL0CwoFv5#g6VWImnv0{>%>%eK^hcg+VY6sxm*x_w@pl%MRPgc`wXZ$g6ebmMYOo&_EJ(|2N"
    "(R$$iO(dex{e5nKf6>PE&{Ihmuqbv1aYx${frEvk_&=Kb_zk5jbdjx_nEdcVpBsj>$v91lB|0pG_eBmnqVu-_RLg"
    "CiuXQ|zPsp9+;Fd{J2gg8)LnuxL}!gwreV7@O3xz^kSZVLCI`Y06tHUC412H1zb|12<hQ8GAs^)4k**az4}XlisN"
    "*S0h>em-jf@^LiZZ<sL^|X6Rq3F2I66<7twm=px9}Ol8!?D{*_1P;s&lqeUYsg7P@-M%kh-+(G{6F7tRJsRzK-e_"
    "7Y0)q2pC4Go{66d`Suuem}NYP=G9tUAdC;iM7VqCIXccb^Qy&laN?GYbxYEn7^~ldl-}qVh%cSf`k$tmu5nglCVJ"
    "=}cUPO7onpn&7OPCpe?K^*E`-)=!02l@8653T^d#W%*_)()~I4$?D$ZW88oJKcW2+oz8<aUk1^1l|81ggw1kX3Ia"
    "zmidNEoeV4gWZo~I+m3^6{<wA_q<9Pk=dIHn?w#R=iyptWjQgr_=1Q`yejtE^KxoqNpYV}|v*xEPL<auJp{DU8@C"
    "W?r7!|^3^jGZvS*j9EYq>Hcn@9MwsHq4QC9rf6Fm37-K?`jCxfLTn&<2XvtDS!YeXZ4GcXHn(<`^oEr^P|A_+{H}"
    "6XGiA%=U6bYYv;KU?LE>{l6)rVZP(6-SPWLE6(f5p><?sG!%cTY5M;`{IWP3)m85SFs|2nF4?>O?M}RJU0Uuv&Yg"
    "^XVimrs9)3>``<C=U_&&?>S9SxbJY`VG9DU~}BNFCZ0EMBnfO-VBnJufJt%gjsOL<8XnC>mCQOUPCY7?9O~G(K?v"
    "(!!g4#n%cqYRiCy8(?3kOsi-<QmQ`RFX}s_&-{anyy}W@KkVsL_{8N9tu7=<{)Kk#%oJnfIE7T@A9Z~KOH+@U4Sg"
    "Y0DlJxdjE=D*IlEnRaN~*y1@JCT-i2Kf1f)wzf4nhDZ`~X<)(Y!IFBFt2>^M5(UQr|CR;kABTu65itq2?3_`z_YK"
    "d*p^@{_$Pb5XE_+_JNLOq`fZ@ZO;k8*!o3f9k$e{zmT?-RH&{p8LS1>^Q#sG~4S;b3Muz&S26m*>h#$Woe(5xPKV"
    "7prGI-@jkxmu#i*qErWNBT%BW>)uirPfn$RYu;nKNrt@FqO8)cg{OGsS<KuJweg50Y>o=#8z3kxKKjg2&-oe|qm("
    "T9KE*TXjRAX`C*Y6=%t^trrdD4F=(LNil-9~TU{>BZvZar4dzW<GKt8);o5;YL988*C(bFRQMU0X^O(e6k&WR!i3"
    "F2@V>CQ1MYsIr;LhDzE>h7Rf!u0}xCUM|lO>m+PufG^pdx`vvmc|xV3?L}Q=cbUK`s>kqfHpY@C&enxULbUu(LS6"
    "i4?vQ|j3I1Jo3zWghb&b<2l=+IVO0_xJHz!BMM)7R6q!HTos1i`&<8U_bP2$_0IO4gDN<FBS3r5A91W_DDD6uF(W"
    "SmaYIS4`nz*av7wf&YSNA=5k!3R_)Wi_9(MzYZD|H=ntpKabEvbNSsOcVJqt1^R*F|i>Ds>O8VvnlwhQgbhiX46@"
    "6olLsVxh94N^o~LIJDbmay_jgS_!g5l!P0W&cTSQSY2|4ZwIBL@$3Iucgrm<L&mbGc@wC=zwl<wfC^sn7tLF}l-U"
    "%5zLL=~Hu)vC{R5VmpB@Xn6aWrNy-Do^O?F3$26h|>61w{F_XRb*p@a;No$QIBj#i9c%xspc6FD|Etrl{N@Ssal|"
    "%5{kSgY=3GU-=McHCNu_Dplr|FkgGXeC_R6%$I%m7s|FJ2kqIP=MoX00YG7>TIcd|vJn(Y`R+*pkjq7@6>hcnnma"
    "8j^Z~8UQM1@7ym?*;4T?$pJy*zy1MU)5(UQ6~jbf33LrdjLXFi2a{7lq3!l?yXT!Vxz!%`lr8l4oJ@9}@ObE{`Vm"
    "$0t>Km*xpf_K`smecDOck5ikwbQ&PK4;D~^UnCsv(#|B#E5Legce0R!E$lCv8*>%LCINRpVgw;(*ANKxY?;E{LKG"
    "32Ct36JNQ~sNO(f4*V`73yjni>W!te{KmKiSu;!AU?knC`z4I9AEADnd)jSDDg_vNBa9fq{QVqFOE(x*@CtKhub5"
    "6M-%rF;frhc%T-eN*Hr6y(8x;svnk}}HRpjA}z$0_;*S4@+lYU__YAD{fHVpF6WwvG)DLn5jH_Ne$qjqXdrb9R1w"
    "azc@eRH79pTU3+$NkJ;wGjA}8!^!B*)g0TN>0j25;icD);bn(wI0oSqb^&B{ILg@vkRUP@qSg@c0N9um16_iU3V2"
    "~jV4zAA$_>}8=)_?8EU<{P^%$-S65nfhE69^^H~~yIdt8qO9l*YtGlCM#<?A!~eF*;nA$OS{f)GIUG1`Lxv7V$r*"
    "Y6r~LYR$}>2yPCQpRW$!%DjUi8>_sfB&z00BB@ZThWtqW*J^*{Rksikz{n#;xIj?P!#n<_2T4qTuB+>$Rmzg!gXf"
    "b7<<BeRy%FahZ`Ge%L*@1pF#0co{w;IJ-xNOBdFafc293CwY>3nQ8DuR=#O{5zkO@Bar7c~8|8Oz$2M1LC`^+iBN"
    "VR93NwXc0RYX!7yePy5an{Tcz4?ad+U0h0`m*qsPq~G!I=7v4?|c5W`tu+kR^)TCGjY7u0Ql~GoMd8FJ81a_nI)R"
    "W}BUtc`o?h9%U}5*ZFGb0C<NlsHF2n8n2q)hCncUbaPD_UPZr4$I>Yu;vt$bYwR<j%duON=69-VncWF4meg!+W=b"
    "KVc|;{2-vQBU)t>UBkQ}#|U_=X+S{U8s6;(7*qhnOF&0}9I^hFi}KY~NcK~24C7>$f<DUYKNEJ<Hmtj8dL`?BM8G"
    "bqdYq-8uuRfr_Sc6wL+Z%|n~C)+-Yo2ZWqN6gA0#iB1c?7?Syg6ni0BZ?B_&(7cTJ1cFg$~qJ!Al}StsVZ4f_h}?"
    "m?vV$m0Jwj@2<*By>q#`~D`*28Ay@~`b{*31nW3EqlaBJf-UtkLu<j;Rh19+BjMckVt5vPYp@EAtS#eyyX2;_Pa7"
    "ab!BDKka9Xei(oI)M`^U`Tlm`kdV^QjV!fF#1-?JVU>!5K6R4;)%^cPji>OxQ5%ntnh(2D`1*g}ufq$}`U{k%yAv"
    "wq=1|e2>as>dmY8UM{7ytXDKuyRMw^%3Ip!QQofXpc;>LQsw*S-M5PUT4A3YsPxFwye;~s`}$Dte0Hogwrs_G%j*"
    ")iYUn?W(IOcc2$G`jFaz#t2t$FrLijoH$<aVgvMng9FM))=1&8dSg`6PykOs-*U*a}E7FU+8uV?7r&bu55|I5Juz"
    "$jq|fK6oA^hhZ4N+=<01vfJxX6UjYpdIy)`5kpz1q!EjtiIG5QhuVsSIVw@xii!dQ~w?>bSa*D*k%1&a3D2dL58}"
    "y%M$ddmXhn@M~Ly(=yi&2aT8!KQNe{UoQ30<z~!3#B)$x^K86nsgAA`=C2n)ClH0TneH5)>)z<8zHfN)TCR&k=EV"
    "*Rm=4s^-w*VHt1~aX*UGUlI(I3Z>53T*VmNK;><qpgIUcoLV4C4~(kXXN3E145d@kA-AE*`}po~>3!@?`QF7LHr4"
    "l|1ATs6l1V#TBl`aFsG!0?8$BSLCslqMKD&t2NlF`opoUUbI@D{H{L|fBr4c<y*Ls=z35&%XH4UILnZ0!I^kP{8}"
    "L-F2;*dNSP~8FHP(DYNBF=7&NyteU{x%Y_^I3RCz(aDfx%>Uu@D*fLXIuUC04&9%2vm*LV}(FLOI6aW?Q!51jx_X"
    "=sJIN?Ht2@%>wpUwJ(vB*PTeDJqZ$jSR}DLkV%99*fRd2Zx78CzKJ@``rV^$9`qF0!IYQ2bTaQpshw<k9^v*^p@h"
    "yN=`f1nK4EIuGDw#VNuy^xZdL!hUY#0#N2`mCbDtfj|~qs^-=q(Ziic9AA3#)hHymkXIEIRM{+l^5qSeK-Fz8RU&"
    "o8lzkzx&z%sVP`OI<V&Xe#eRG<4gVoJ({+|V=1x-M>(<XT3~eTL80D|BCG3NjVHpk}Efn0t>obE@weYY-@9ASF12"
    "!ZL7IgQ<=}`9apd)EC24cX(rI*9_)VCU+Ik8L4(1(>9|m5EPxM+tgQ1e!x>)#YJG>>mslZloB&FN$OrY3|?wTdF2"
    "smr|Be%C6!eaBW%W#JYQeYIpE13;0yW#e1;#u<JtcWIOwTuDEa`H)_>JX8MdW@fJtb4&Hu|>18l<j6@3Ks>;Q;1$"
    "Qy4%U2Lm^Ctrfk_a!iqq{b0Z5>a>kc#w|cK#uM)oeC`2`fv0uP+Sq3#1)!ZDH!E5IIvuK*VZXV3k;YTpqW|3u#44"
    "v49bz{x9exHU*>0Us{9KShq#7w!M@J=?FJ2+N%}t9yTHR7e`cS^ljU=C_(?W;KR~Ixj!%M-Ij_EA*!<^u0JI7o07"
    "}(@yJ)8J<App}o#p}qryuc3@8+m$3GmWsQGnOk!TmCad`BrDi(QlmDzk#1LK2FJ*S=9*OuJZ%3z;smAE``fREcY*"
    "!VQ&SDZQOQr#H!zFFsBMB$f;VhZEN_`nNzFtAUK8078nu=_nk;<g|otOd(*bX^17hbaLZ%%fPbDFK5H9(jGk+{vJ"
    "<(w8|7pbx5k3&qFO>Ulv)(^-;A|V1k2Wl!To7tl`_khRo}mxrnOt-)G&#()%sF7w)OK=FHZ&a6u9HtRyDQ*b1#Wv"
    "VFh4jv7U?2;YbrrQPzXQQEDJ!c~BykiA7^3gW&7;=)NVBrX#J@kjE4l02hSlFgxc5MpHa<7JBk@gB(27tqN3M^r7"
    "JnMLw@Q`F=IXeGS(L7E%9dMHe>w@;1WSe3xZx-WGNzDj9an#<i?mP)|5lqwG+4O_G2YI@i4mf&=fjrhg$5+87#k6"
    "pAxM%G{I2rk}YZn=QNW$dx+K}>*R_5GDEtJwsPcmur~^^E58QCBTMrG_F1aH|(-aP7M40GXYci=Qi6J&J6!Ok!C@"
    "J}IGDk0WoGaz<`gdL%JJUN2w1YUr;u<XgSY)o5xs88BFf*PM^1iqX@Jo3}H7FV^ooG)wlB>%OLxRuz8K>i#U*YBQ"
    ";^-dSH$m9icG1pxB4$>ZvE%lwh|3)xd_E`PdNv&*t$=i4RZx8~nrBAfcI=2)?2#HW$_(MF0XM(d*S%kgDfslED`5"
    "t=;ERIonAB#*f!lvV25OeEUTjtPR&jl23u<;2B=;~Jt^uc9;7;yZYbv6_qd6!F-7-71<r;96y}>uR)0;~w{CerR`"
    "@E4uRzJ)TwhRV))c8dd^fG#gSKT?q!FS|PHBDtlC?N^51u5UmtPWSpEz#*0&Ma*un}y9GKU$3RS6wipb6l|fko!-"
    "oN*7~K|%5PXu4dqk>2%zoYzcNW}ZnINLT;b@WFbPb0$m$Bw5<L}aU@$IR~n%XhSbxY+sDcSu4rdWV~et3V2f0^4`"
    "vPe_%rc_g^QiV7^1CXMARJjzBTsDiU0<2*82C>Gu_BNe2$M1n%S4XKODls}eB$Mq+mC5!Z2VfP(rBgK9L0+n=@E2"
    "9SmZ(}S9jC2xeUPg4Z^1F9`qC-GwD&JPAsq~5=vNWp;y*)n2|UonB>WI2BUI@$|C_SJ@n$SfaOO!_P-Q?=7B>ebf"
    "r6zYK=O4z!QIIDLVtGzi6x(wnjg&@)KuByoHkcqJ(F5_Cbvq<M8SgWA?J>cCa3Co!EDQFhzaiz3=ET?V!g}MbLRI"
    "A`P!GNVza1fN@sRD-J;R1#3lh>=5<tT8-o}&xrNDR`P8&ZJ7kC?^EBOv!`bLg^4k~F#o-mi`t%+h7A+p#a(b*@sP"
    "R9|vp4_v6~v5s*(bjJ+Nzgpv*;r`k;jHTR59mwXGwRq5!lr`A2Uy;QqmI-diM%YCbS=*fE}fACgxyb%Mp}@VK(0Y"
    "90wFN;<AOL0dFpaN>c*)yFN|L1786X0mAN1LH}%&TcX6WD+aXKJc3sQAYP_x5&cT+qGibP0G9_!dW}_l+ITC<QsR"
    "l>K$>{3k(2XPm-|{<{cXrKtnKsb!@aroR*UzL$*~r8vk|z5q?he<l5?0rF|C;RHk4fSl`jtduT+4Vr;EW&jc{i#e"
    "c$~#gj>2wLZEV-v9lh|XsJ$%Us;=DplF^<{i2FZFIU5xy<*11XLcnCTSRME^3~N>wfp6&qf%#FPd;&d;tFFKYJ)3"
    "Z$lz54Sbr}*wI#-n&x+_M%e-fT%jd^e83oXF8u9`uC}Q+_5v{&?{r1S+Hc{7lwN3+ae?x@@8$vLd7IPIU5h<BGdS"
    "Ri0Ox}lB)qA4BQXn_wbe&!$mdeBlD2MbVE>2iMS0%{S<*iCJVLd96Y=4;>A-$#L)FoDzt`oHV4RcVYMwsxy+pbEV"
    "pr3j_#q7phu}W$^b%Iv8-mNr+osUkGvX4H`uhlXBQtRSl7d$d0wY+-vk}q<!zNBGZb?>ulOj8O?o~p6;hJC|m(BR"
    "hk#=F^e9#*`F#j^@Cty#V(H_;aAgCe`)XpV@zP`HG3NUR!b1iqGZC3Hh(JnW*A&wgbKEl*lmZaLgQrh&mC_9ee^N"
    ")Erpm^Ib~>V9<C8b>WiSM956j-u}L?8ZhUlOPTUH*mSq5mkgi49npSoO;=VSFWto%c@MHr)u(8@{a%pP{oZ=mxpy"
    "9X^3nwR)Z7}M&%$1D)Xu!P9nuZbylY;>tmbl+kVr1yA#}`NRd!QrHg!K4PD>)x5E?8&Yu$AphBPUAJiJc*z9)nV7"
    "q;8Yky4h`ny!&Ju+c@JzIPH5xAHhmhH<e=CF1B^kYMBAHQD9KK{@Mh83$cQVyPTc+d#WVQz;2-6h{<yck-7xk1yr"
    "`As@0`G~3*Vh&!cQ=`5*(kew95)b*8<FoTOZ;yJfj}L$U?daWkUOUY|3H`+gMTkj9hhlNL79hN%WK8)+f;ZO_)aj"
    "y5Le(ICF0Zqu&ai*^?048Q8*yO5yZ+)1tL$)ym(Kezy2FCwNygRC;a{aeM%bTNMu|fZb6^`VERQy}qsGJk`#GHrF"
    "yAQjN^Cj|sBRzj{3;j&dx}{r;>i%3s#+OyK~#V#2tEd;vrLKABQN$$5a+jln6-hC_KdlnrCG_6>R0+JS6<mgI<sQ"
    "EI|Wd+Bm!Cq9LGXQ?ugd92I@B7DLx_Ve!6#+B6g3!FomP^+UQW8^u~kf%3XO1B4Fn#j?c@byYrWVoW86m`4pt~Wv"
    "A$&V1{LO5CYw7%yqx+7!C)Ct$B{<lS0MN@JQ-@PZNFTxmB0{xhQl6qHTdgGCt08peq|@9NER82~~*pPOL3JWY~O0"
    "8b&G`-PEQtzC@=nV7zR8O8L4C!(Ok4qEW9WAfBZ#dTcHcpEvn2nmg!^Z|07O;>ql9+39#B2|h2sYQVl6obKGX8;q"
    "xW`EwgIdABe@dA^JH^|g{<6c%7E_F{v0L8}h!*AireUlHfeuUOZV*;W{>s<^3bPIYRWKFG2v9$B_cRcKAe#I?3@H"
    "DqOe<`P_MNcB1NZ`1Lb8*}Hw?rs|zG{o$sgcvaRxj4G>mC51RZ`jNb^f1G}0nzzXUZ_rAw0Y4;c(IAgfZY2L?lMb"
    "6!3yCr1O*#}!8LYkYNG%Z*&HKR(i=>BLf}{iETIZ6%CZNvGXvzs7>dCgMy>m`tplG?yC#eO@>0?o`P@{PNv<3hOj"
    "^8I4BJLOIU=_<Y$asAk~OB#oM*-2wW^zap<s|cb-~bNvFPA!rM-06b~?-#$z#a?#iakAX1X9ZpEmA>jC7Cl`Q%|$"
    "@WJ6P$A`yn56+MBM`nJU`?95PUP-Shxzdq>0y~J&^@gu1Nen<Kdz1|kl?3<2G6cT}-#m2~5E9$<SQonED__M_+0p"
    "sUEL~jRuw)AYuX3W*x(wNndU5|Ax6X{C*!CBiX%*viN!5^(xQ;*a<M_-J+dIL#<MZJ3_jj)XtOab6NXdaPA(cQ#%"
    "A#p9!DQ0OnDE4k*)sDY5PH7&JwR~o3j-(QmKQubc=!6J<Nw=x^BerOE*&5ak|ITswo)%=ho=YUhriem5Kpez>)UY"
    "3xsFNw<aF~I%blI}@LJk*mt|pe>*c>{w6=ztV?)OHs#QbVn_DX-6SUa^rCICbjG<y4H>3cdcD^F54LVba+1Qf9?F"
    "~{~c!4W0ltPU(yS$uWf|IUBYBlmqdj2ArOc(PPsNNutm*@5R1sA@1eg)sSHW}b<3l4|UyfY#Hwu|SoP0tNfM=5-Y"
    "ePQW1814Py`Bm56$9m)WB8i@N*sXYhZpHV27oI(oS+HfDnus(Fiz|@ihedusfUqn7*8Qd4Nd|MgafE8=3Phe7#PH"
    "Pxqv<P7IlvDLD&x?+!{gsh-X1w|d&O)-#eS4iFIVj)KT(1v5QI0k5bzdk2;1G++1mEdf<pmDRt70-PAU`hkZ0AC_"
    "z!6;N4OOFH&!k>BsnjZmI~0zaUjTjbwEf!w@QZ}`Q(c3gc!Pv9vt5?#uZ(KGaTqZhuB#Ux%Mt26Zdm*&u%q${CuF"
    "wBgN|~p4j=D-;R23-~9IGTs%kemQ>egG1W~6Di@M@xat7jOwiSul2)?Bj&!p)Xv5q8S0<`HV)YfP6}310iulb=5a"
    "X4==w2L~;&m`xMn-lXm~hY^B^0um&NFd^i^X$G_I(9SJg^a{V)tw2i=l7nctouqQ$2q?9AdcC*-bkAjVlMQqFU#@"
    "A%}VH=J4R`=&YePZKYn4F*$6H%~-tfL##9+%aroUF<a^O(Zsg0n*Epa^ON5gC3!XOx%TU4K#Y_5pTI3(E)m;+w5E"
    "G|QDJw4wypUM>fe$K?b>lO_-S#6Y$QH(HF1Ap2?55hgS(o0boB?#isppQ$kTXMt2gzB?k5%Nz@JSKa&+(uumGf0r"
    "|Ov`Q!HjF;)6!O?NN0me>2GeQu;ogWW^O9P-V_<XT6TMv48dGCJ{JWuuuOppSt3+u{E9-);`%p#sJuK)=we;Q{iP"
    "gD=1tm_Dcyhle=1p9A4DGuBzv@HKao|aYIQeK^;Zo!4ge4RykL4S}+cyxM{_Md}#%zs?}bl+ARB46{x@y!%YA1^<"
    "TH~WNYb%ig7dCLmI(NJerdB8Tl+076b@{OmXt{#Bh{YS@}vcB|DQBNj7YJ)hIxwbb|%1798++p$JG#_(gOt06-z~"
    "0^9=}J3|{8;1J?y3RhMvD*_ZdNaOm@RSRE4TB)L!tp=_aKG&aeI>L}VV?0du>CyiOJP;LWT%~LElyxDrHmwvQ#w6"
    "oWCTUMYuNJu#d&hg-2&kUS#z1zLco2PqHfFrTD)D|MWsURs>JUQYV!E1e*9?T~+kI;}^s%+SI0+w`!!1#*?PY73$"
    "J_JL?f>J=(VxB3qyO{!o6}s<(Zw-wEvP<1g%mA`rT#%WLr`by4eiX*@l;fp5~H0U8!T)!A@>?~`$1j=KBFHfKqq)"
    "oc)%MU+D+u+blO4kMeH*7B76!Zq#@rkX84pxFpX_RZRiU`_tuu~wU57Mb@)zoNj)XkYo;79d+`u}2XV~=eXF0<ZM"
    "7DCLrBdEQuJ_-hRh?k!2m9qXoHsxkVXGC>Nt^U90F?J)Hp8V+6w6N!+LS&7PBH;UsMHEu3Hf??-V$}7mSXZyTS|c"
    "B6pB-J92tD(4Zl{S<5hleOGjQG%Deq^(f##F1+K)l*>paPNH2f(S+`5Od@sG<D9iI&;@m?*4e9{q`N?Mv%%d2(tT"
    "&?Uae`3%f*NdxTNW@$|yG_YNitd7=i-(6Eg9`U>L_>r;e&9*oySk%x=QXon0!H$+xS2H;;kH)tfi*N6D^Ex)A=mN"
    "}H1RAve?o5YBt#1D&EAe-?5yE8QYlIminbHWK-YP^f##ThS>?Q7n}R4duWzcO+G^H+JTspLFTlUqdk8vIO%JWCF1"
    "=SwxsIxL&}&NL_$$QP?HY3D|+@l3W{{q<b{#hlBS;dHj17zdeF8SKR4k6+&>S<V+v+vPn42Zc>VW372%Wrk5M0>z"
    "->TJdSzq(ZluK;%rF=O;XvR0*g`!kx%{rIBL!|iV@=jl?p^yB=+nIwbNHR3`Vf03z!kJ<iTGHAk6c-b)3B(AcvNJ"
    "Y~6IN|Bq3Hpx7I9&qXo2$t;)ZOE^qzhFss#<#9#)WgaVYeb5DA%_#C|vquK!^ZCqK-9+LRQ?staNq$wUsGI4Ko|C"
    "SJNBA@9&g`Ps^e6{h1-x|KYgq7v5@*<s$0UZUJx9p|ej~wmsf(2~dwBWm6E?ix_$1xlTTHJ*Kk?oX;Z)Jh#lHO+X"
    "pHy~))o|B{b;7Canfw$ZQ#qxiA;5QH`1`YCvyvzcW1UP?-K<&N*c7ZKjXC2M@3%}qvt|2sAJX;3B{laWbHDTh6<C"
    "aFCx&u|Laz&GLq6XX(~<0Hx-puwE0wCz(#>{hZotTOfXf=<>d1()UB!dNOLVcW;F3@hxIJ~T%Ul@L_Fdi9yXASm}"
    "1Fl_42PR4z_ZV!#IwRN$QziT0$9LASQd!pC$2dqg`1!h#?MnWcvQsIGze+qR0`T6U32?!}qZy0ZB8|eP=NemI3Of"
    "CWC}(bySX$4FGTR7{}sOzCrLA0d~z)GUageDvy@u8jxgs)it=whq+(wXDOsg*d#L#J)+agXXMak+(q-6d+lsBHxw"
    "=B=xSyy;T#Ji$)#ajMo7s#nH555_b<6!&)@uU;24SP#l>t^FGit~B`k^2aX*YGz<Q-MZdR|cu#?bvm8!1L-e!Q&b"
    "rW5<T370Na|dmYt+n7ic|AiH4FG8+p%e>;IkPLcV3PO}2O8HjJova`vW?1v+*<7*>E<rEp+@i5X}ZR7ZmoHo*t;("
    "+u?KXuk5<U1pm8tBxo$@$!^EX^yN`428d~V=Ix@P_2GMXHAU%Hj_ULu*r-Q>^eakuuXXJ`25-v<w4xLp}g-$&Hu)"
    "XR{CVPSCJ8|J0`}Ywz|A`2rwu6z0_5}4SF4Vy4%C5t&xLTm$DJwPqG^Y#9gzy=L=cPpUOvLMnYrEIn!*E#-Rg<>M"
    "UXEsu>l+ar<Cr^{rVQ5cQ<t*lFgFC3lKPsZG5%hPAqCj1o&GF(u^R1IQt(d^&E>{d3$jM>>Nk@B+}+^{>_XiaZ{p"
    "ec!CRHVMK#fSFptv{I^<k0B$owGRR^)=dQJ{wbw?~t>Y|o0HyUILue0HcZlRY1TzT?j)Jl<;%MI3ifMPZ>*36c!j"
    "Ej&5HK1pVS``M86&5Sh+$s@Qd%P(h$E&)fdG(5k4$lnT6%@3&|F>}3S4)fWK`)hdCQblXFf`f5wEN|ZfoNS7M}eM"
    ">Mb_Yfu2xE;;ka7)Ie~&A*l^~NOQIG1qlAjl(!=KUDDA^{Kd1zz^}5w7bKI+D?k`!BXbRbD_I)ymy3fl5zULOpT9"
    "%{4s;)xQ>jpa>enGz%L%EWN0#e^hwAn{@<WjAo8=K#?4HZs=>DSQaJOnAIk-jc!$c<&h!wxL0m@Q@>;Atki;4@*U"
    "S=+fa9y9WO+p;ShaF(@!CKr^(C!7s#1o5K_E9aISAPkH~xm`iU<Qqw_iYA7#So{Bc+Iv@5{%FDZsN)0>oUR_J52M"
    "G|mOG(fi#HS0U#S|zGUG#FSdgsptd$a;dQjS%zZR+fTTQD$q2cQ!!lwgo6cXZ#pQn+2;=wubDcwxXOhT(DBxd>qP"
    "UyqMrs58)?5z6rmRvqueBs+|^{p63-S<0J_4TsLb8GR;3zNAzg_`eZO!^;ACU9E@e)fBP)?3(aij37=dzfvzK`DE"
    "Uo5#6(v&C?jd;|p6RAuoj%`@BS1C)hlaa4%B;b4wvc`?CaQ9+tQNohZ0)rEfl9#C!CQcj&su&NTCBvd9HndHcmRF"
    "Z=jFtdL}#o`C;5>?o$TzxSR1KD{Cpz3BBp>F=C1<>Z%6f(HfOR`meZW)*yS$$1|PDx;~SiB3=T?zpD0!SYeKAOE>"
    "G$dZ9p-UDK$Ciga<pHA@z_?0OD9+~^)x~5a?8ATAo`_O~!-KRN0sOz6Vp`5xyH)-Qo^LiVe?#%zHhu%Iecx@ipwG"
    "<~e`&ogI<^WV6U8Ag7MfMVn6E;#Ca<>3|CD6i&7~vU6XMJACwY=m6`)fycFx1*ABZmx&rs!IyJOcrV!_PJW_Th@_"
    "j;VA8jnJvLWx!DxB=e%7>z?@<rtvbDy3mFdyqVP3MmxI*oBht)Q*DV$6R8R2C-w~Pi9zqO^Uk_lSbx=VvTRNq>Qe"
    "%vDI>G`q8mLyHyTwyIoZuQgI_<C!3MGrVP-~Q|hWefVaqCUZ~F_c0!WS?ya8W@r#Tnik`AmwNC3|+H7lPYufo_5&"
    "oAe9nfZ(y{ot#{&xO7T5cDT`VW6#K%KAjgYs@%DRFd3MFgcUM0}hMhq`Eb0sMt66#7!*>xwGgrjY-_7n)V{d}CEE"
    "F_Nkm^YBwH{!Z|r_>?m7H-ZnV<lt9Hz#lIQzmrH(G#ASYc~|7*UnwvDLvr(r^%#+8shzSaLBEZbDmp+QsaL#bknp"
    "weB^H$=`;D^v%1d(hubkw+D$)N+xT%y=-&xA2m&ppt8Teu)aFzCdDHHEt0o9Y4th8RQv>kZLrNJh!kwCN_OtV#qa"
    "1wpX)ce&k!E%$uI$4%&X0$Z1`RZjpmnsem=8n_f-@SW-sAVYyFpUdJ4??8bRk^T#@RD35-^^wTP5$hSEL&jtE*3A"
    "1KLI@Ryg9+@2yYlcYq`Q2&=uOpq6nEs6KQ4@HrUycTgpug{;OW5WiYh~s*aIY?DpO771GbE&zuDGK2w~CE`~Kk#O"
    "m_GAgk}ll8ihDzn}kdeER16pS`p5<C8KJv)tynF5G`HM6&W7(D&f*;N9U7n)}zVpMHf+S)LN!O%+w?tVIH~xqXet"
    "*}vAev}ccnMArwRGGtn@7kR!V!=RzE7i3x!tE?Y<gF6){vP5=yKwd5zKS}gh(OA4>EQwkng@<JHVnqrW|5P#eVk1"
    "kpf^AVbwAxsB=+~6Uske~iZJa!%f&e@D?l&tv9t~+U&CF!{+hrb%t4&;mkSz&AE9=J8-}-C-<$Bzu+oM3GQ&RR}$"
    "wD=h6GKbK%-Sx>GDz^i+CS1Z;dZ`~AjF(U;95BiJposp>McW1%IH@x`Kqz6B*8koK(#WB%Q{7Fo=MW2S&Q>4vx>="
    "*l&WbJg-p9{XrgRYT&vmkXG$^!jP{Pli|%v5&jV!l%lvZJ*PTBCmr+R+%DD-TKKO`%AXrRDxRc5Q5*}`JvNBZl7t"
    "wV*e>9--X@{MMD}tL}5!`jvz)m{@1$NzXC|;FdNwTMuN<6wXxD{#<cTM($%pZASmFsoG4Jcj!9HdaRTbO&tM$H~`"
    "qsZ3xmtQfn=&HRAr|8$J%kW%ZV+dmvia=|pio5_0f;z0heDb4NYS34^(Yp~w_x6UYRJ}KMp~@F9Kr=Jbj(L!?0|8"
    "~t)b2wL2eo>kUtf<rn9oy4nFYU7yWrPIJVtEVy3T3_a%lKCGIuIbn2hdJ4oJ@3*nN&jvebF4nD~!QjAqN`KD2jk?"
    "ox<l{^rq%R@vs7D>8QD?=3jyz_%rTZ&g(r^wF|*Rr0*gUspJaGhi7ouY|FHdZShU(R%4y47Lh<jOgXF<tP6?o}*u"
    "7KU$Ua{1^~sb`v5=?*sQpg|G*7dYn>`_*<?A0hg4R-ra^NPy`}N)GIRG8qRQh%T>O7CjqWDI>UF#98LuTsiSz1P>"
    "e!gBq16C|3Ua0aSx>*8YUkpQ^#nq7)iCW5@*WQTmr?nO4VdgPEY7E)8_RkH>kEss>r4mwpg$3QMltzA-K~~#fyRv"
    "9SjjN5PgqQSYx34G=uOu#1IVG8#CWJYzpGfHPaSQk@DQ9kPA0{X}>}SxSlH3<aqHZFHpEjPO(<0P0^DMl*b--V>s"
    "+Rj$31c)$w@>(nl#m<aqt*EC&ufLPsH?Oe=mz5(0chQs^?rMZA>BK)V{E>+)iX*}uXO=1qZHi;HOh+n>xywfm*&7"
    "uhKk(b>wWi)PlbXR%i2VdH!ZUD>DRDs_9!sC>6ywv6Hy)^rK-z%E_ddALWG6nR8x#^$|?OlXPo7W+~r!eJLL32~}"
    "w=DLb{-50quWq%mS5!E2REKoIUyn#!mG=>n?G=)3WjnCx<S?73?WH;jQ>`_<T0FRz`oXZZX+hy{B-eO%GR>OpN|5"
    "JB7jI&taivX#|^DggflLwgJdAirhgdSAyRlfEzpRQo97vfX=E>-x-(j^sQxhqY+Un)@&B_K>^-TWM-BsTxk2&B@t"
    "BNVqcTPvC8#P3s`Ltcc}flw7{(9j9WyBcCq={TF{&;m!A-*uA_D6-iV6vxt>x~$Tb@4A{__mF}@7fwWxdR?9bY8A"
    "HvV_vF5zlYGzK!=aWVc<)-3Gq|{#M!qq^!}sTwLxC%)9+!X_W?CmwL1Jm;&br-1g*5Sx93p{mz!4vcGaCS!}c^ZG"
    "z1Ge&{_Az{2m^icp?s6A0{4DyqalHJ=EnhpjZiVDH{ipp|kvgxcMy*ondb6?8aG%GG(xSq{NvT)A}R2n$N=9emo;"
    "*!~7bALwGw$Rd#>3ghwOTu7Ad+7sh_{O?Nj(Ml^ROyk!^fKpjER`z$A_o)}=;Uz1GRfxI<f72XtdK*;=5UCCVIQ0"
    "$DHAaOK*R3dR$6oZK>Q7NN}`mAECfqddnMMtAn;@{v$Hq;e&+mi^Un%YByrpame@lds{>oM>U0QrU-f>A~z7kP9-"
    "jxvn&_?n>hL&llKAJX?q!S#?~MBZVJ#u0u97Sx?w2P>(v0wCn({a*+kn$9!jD8A_^F-M^>3;qP3^-x9}ahYN7K-h"
    ">92Z(}?O@|iYrZN(&7L?0S1Ze3-*Gdn$=91~AZGZ5$K>M;!Z|<@L#b~Qi6mq9nv=_w&WUA94sTu_GqFg1}^v=Xwk"
    "l&-#StV69)647xFO?_V>;o4)Z{0V_Ixz}1x!trk*XYM25rvMH_H*yu2*J)toR#i`kW(5lr*|E~c|59!)NEQaN$$P"
    "Z;Zu>NLXWy6mD6RhBmZJpLZ$3KdO32fXe@1i-?6k}hK0S=y1(~DvKtXcI96FcCM{(wiC<p@-HT?#LKV;t&b)Q`Ec"
    "hYV-ChrOd`)L{tsNrHI=GD~RcDFg`U9wI9#vMhODhRQ{)wp0OYnkeN2a_OM_#8WxCsjo?FWQi6>R&ODU#xM3xV;b"
    "=xU;2baS`Ws=f{7%;IP6)`hjElA|f1>d649UM0y*qK4mou8D4$Cy7Q;z=0mUeS>wxUV9J4L_SzH>D|w7PJi>Xb#|"
    "21;ayj!h_KoOQUZ*EXS3~s2U1u`C&j^Z3qAeNs&}x;eqKn5=$!JRXe>{pBC>uoiQ|YaQ{7AvO@kZdpnRDFIJ6FkR"
    "11~=zQT||c(%*lz4U)~wF>$Zl@2WhKx9ZLZ)>v=l<_^B_f+L<C}u$v^mlyrrEnh_=(R#Wo~GzwpDy5GUd`GkM%j)"
    "QBpGff^yzw9D7{=*9SB$9GPAxvdPOMOLsV&E>FDXo6b~|^uB=Zh>MvGGII~<mggsk2CQo`yvB;=hs@eKzR94U8F*"
    "&PEN`3e6{W2-2LMYkvMsNG+8d}&`_*LAUmt2j|N~(^%2do-X>WWVkSKcFgST84d?ULq{L~H~fd_HaDc0wR}76<Pq"
    "?utP4azxq*w!Mvqu&|pRdk%JT?T4?a3-;A40W0%*i;3`4CA(`-DZhVWGR(Cs)!|3EBEeTqbZK0DZYBltI4jo7(7|"
    "t#pjk|UoZ6x0eEov#v#poSX<49^f4IvE<E_S#uHlF?6HM=#RCth5X3S^RGa|(4bfX36H8Tw0F3vgUkV-M2Yx8tG<"
    "tz;Y8?9M4u6t~4{5_fh;Y)1|G%&Pm+ly@WffK(XJ%}l(cj1);k!CLdfL=5~eE#sFpG;n8E8zxN&-6aq6*16%3i_V"
    "SyXj;@D&uh4a+)2Nl99|6XY+J=77x1nt-OU@yy8q*29#FmRh}w5H)@j7Q2Dw?QpiL3rW?4<Esb#uBpX9QP4yM|5Z"
    "JU`5TD&|8FF;SnB-UeEx8CNyD?IKrNjWJ%Q2yyKg@8G_hQy7f7=3DRqdP|IA!(&^M!pRyDa*4I2&jp-Ye}_o!6}@"
    "Ec6dC2f=P43Wn7g%5ky&maMXXy}j%6qHg~RjcNUiI9&V1db%}{ygBaWvsc2*kami*#jvNDLh>a&7HL7rIdC<ND8M"
    "?Txy3Eh^`k+$W`7#A#$z<Hff#iOw8u=Swd;3(+3jG-YLxFsI7f7B0X7q^;vJ(yQ&FdH5yAlOD*P$<D$YK)jHZ!v%"
    "khl?1O-AtnXAlyV}>qu+c_bY>&dcilCC#-h6(oOUa22}*~xWJRZEj|a-Of$8HqfIT>&4F4p^PkAe=OUTNKGx>qkM"
    "LDkdtw4@u%&)niB1RpBz?<z6;l*GCBd3$y?(pM3rn%l>AM;{Y_rqo0N29&)7<tjWipiN+VeD<+K*`2(FCzkPG~&("
    "$1!+M8SG-_r<;u&nMDrsKJs9RBj=?d#sryFYT}^!xa(6R_4|qSH*ocMX14m23hNOvHDenPP$j@qeEGa{P{a051GK"
    "XN%G735@$Z88H*t(6ypp<~2<1JT-Pk0%<PXBn2{iV)<1ccJWRcvF5MHSrQ1)QdQW>{0b!VAtH0b0z;iprDUgU?av"
    "`{7=*(fRydcYAzGRmT5A3U%oGg81QrIQkUMM4cj>$M_EeR(`;1`=t0&om{A|e~2a|@57uj5UdkVT40FY<yMsPuep"
    "HfwwOKg9XK?!zqLn<!V)rERJEa%<p(TvRrQhK|^6f-ze@ysjdRmQGm_}o}OsvlLIqWKM`?ZrFqlzin^%p6?D^WS+"
    "8%^o<q19KvtIW)QCM~%iQ6#Ic|Cv}Q?Tv(pmb=9|Z;~q(~niU%ZSXSw%v3`8Vs$=%a<jr`3GKe=bik=A@hbFFQpN"
    "_?9#C_1Hc=eO4*H7nKe!nl|^at!X_kW6|T3J&79Nq_#w2Ge-DS<|`*!-nxc<*MEWIxN?>CxOU%`<A$vxehinP(qE"
    "FSBk`fObCWZdN<`DJ2{CScqZ*&Ugy+InJp?QJH8u0UEw<Ht{xmKUK&i#v+OpgGrK4JXI4YYD9m3_v^dkKi?_cBl*"
    "zXt>8weHFe^3_pp#auX+=+?;}bw>OB!8Y*1>N=6pQW1xLAMN$SK;h~98H2yOgVgR5kTXCFK^cG@<4>3z{|xoR*~F"
    "w8qSE31@2hc;W)R7k=SL99T9I#nq`bHB(BEH>kVDW*!!<`nbzev;l!kYE56h^>mygHa;a<_B$fagd3Tj|ezt<6tk"
    "G{7<mJjWII?4!<ORQ0mWoDEejtC%_rvC9RVwToNI7vmFgU+ya8p<IElITVH-8eF@CKMNHbHxvE_7RGR~rTQ<j5f~"
    "gJCS|-H-uc1KO&lPBfOsJipZ6%sJkm2I3Lkk|y#ajo0cNJ89p07908E(Tk9{0uMYMOFXEoc69#KrVflNTI6D&>_v"
    "uWMbfk2;oe$cZPpo8Us#{Zva)S;Yu-HoGYsA|p*2($OF<+_a1vuEP|OnG66YT8v_^xyh?5&s~arZ)MKDS7pXDPvc"
    "=S%K(rUaX0dsI8cWRjPa<adHvn(f+fOrP?nWOio(tl;N1&=WO#h(FUa`R)*2N3B^huXBiNAmaX0_2!KGu|{D`E)E"
    "(?n_Z|L)8Nd{ByzeqTLdV75MYtfN3N4-?1o=u#(57DyK-7sxBVRsEhi{@u7L`GT7DB35t^qj?a4JPp3WY&uiquAS"
    "arCGP}1wJH?Gm^%+cGvty9P*j3-rwI}Vy`zKk`i9vS@iKl4VfOeb6^bW9=AwerG$szYtu=wdKSK0QQ;!<I3QMCTc"
    ";GRN}OkM1<R6kBHQ`yTuRyC2_YboXyrKyDkt%N!6}&CXpExEp)a(Ev~EHk=x@&AOHn#G!GNDtSS_3&@?yVLwzSA2"
    "FFu1w_o%`-Rf5{Ach*gxr%$*lK~zP89uH8IH&*;f*NKMoW`;4vl!|sML14-ED-?aILf;-3HLJmR{0hwaJLHUHdg$"
    "O}Fj2r&haJSAMcto9R(zJ0c#!>2>N&QG!`vsBOGjL$xS%1@0J$X%H3hw1rCPi%^%QhGh}%0Q-U&tL-gbLMCqGp1@"
    "Qhumb$akKA_8Oc@|hF6mK8GoeJf5Y#fT%752IU^W!igdbFyAE;jNQoX1ndXQYD#Yc75mUzo?#uoMlHfSi(jca?2O"
    "aX)J<n*~_BzTUyq4$$Pb=4avz>nwCtUqI|we$X-98^3{Jdr*WB>UIGKFbDJ#yPL#YqHn4E_3kfp~J|PNuo<z^D?&"
    "WWGbA#XBpkG8P(v9Gjc(NFi?YFiR%X}U?x%tJ|0UX%2B-C8pVS}XOSvrC*Mt5BAb47x>VkTTOXjb;I5O7lzo4HQV"
    "x0up=BUcnZEUP(ok?V)bI*|rCaDK^rN;{J67ivM~gR6?Z#K~l7XJ!3-S6GHLxMf;rR_r>EnVUjK;kX}#9qvPW^Z^"
    "eb;OoheUKX2ScF<6t`;sMY`r}O_I}I{-Se^y>Tj7b?zFUu?UDRXemKy)lJbUwxUjb3nz3qMbv{f&ST!k;}+|nENA"
    "WP5NWQ+c;Pv9cAJX}#u$ktBTX`4<g9Mr2-zl@Ss^kQeloU1}AqWhw}W>bKYD0=FBWhL0(f^$rZ#M+|(wILZNN<uY"
    "bZVf})7KgyvP2$_!64NYLWoLK2aGlf^8u-EwVKPFL-(w5P3N|XeQpHWLle}^nfOPe0zBo7#l5GeM<;26pb9!$`yH"
    "y$}#veHe*j2yF3exf{R-7`|bn5!QFG-gyL(->9vLyT3tdJzRJpF<udtxxwaiQ|wOvpeEK-D+xs|}>CbcGIzy>eF6"
    "%2oT_7lV#Im*DV=9u^%YxVGs|b#P8@?p55UNtg2Mj)sl6G$to3rZ-l4mOR%FqBz6G%$$6_fHKtzYdq-Gs}&p%jFZ"
    "Hifc~v2mXlV=h;S}Hc;1uWy_v|nL`on@hQcwOO;{;C4_?mQR%;0ZCt22A_sdeNlTID39mXY>vJp8cvFAqIB|N#W?"
    "k*E&!1C1Ts!X>Ton0E;qkR1HrmgZw+a95{tT!fy(WQYkzUj;Cl~XvQ2s_t2M;oHoscujb$C)`%oS7TLnH3{AGX@d"
    "0n;b>@9}D1gw?~o19?_Ca`?*K56>S$RClLi3bRFv;lQc}tf@ID+CfS++Fq$n17SWw?ND01e6<(HfwXMPs?7U}H->"
    "URVWjQ3WLCP*)rI`1?@S2ZclV)Q~t20jqL4T1Y6O2#-qV@_Y7*?;Ybf0eoWr_1IpDAwhT%33Vs_}ta+yvSbASnx`"
    "H{`+#!q7+ut)iE5u$Uq%#F?R+UYzfbj|&&0IXMhnlugU_>MRwtb=Y~Jo%TxGE+&#==HKWwy0kg4UL%SV5^sk<TqN"
    "SC>LYawnpp=fKCAdZj!MZ*&sJ0h`2A#zut-#!8MLCldLPTtD!N$#Qst@lK;|aD#DZK{f|B|6b_6Z>T{ptXg3HYG4"
    "1XaG!NpIVpj`jn7Iad1z^;!sF+zB&k$!H2<`p#OALmYuXoVB%WnU`vp#0uSJn>Ys1}Rl1sJft<R+pDWFCC5zQX`Y"
    "&XBT5;yv=-@{>BrG13(}=UW33PkRllj0SeEEXJ7QHBrS4&NfiqoNPIr4InZAmH>1Mn?GIx`lBdEo(OAv%X=wSYUR"
    "jo7g~EI5*6<L!XJs7^v&|}Xy_Uw<QXCY@swyU`c_1EzC-kNk#eA7CWWLvn(gEBD^V4P+MLp$(ShIAq0lFr4Px91j"
    "Wnbi~*cG~7+DMq3Wv$5H(Kk!<;zw0Ee@Oq}r)Ind{%rC(tpH+loi^SzxM8kN(C_3=nrgwJ{mVr^?|+f#XSE@`rAD"
    "8|)8_a+bV~`-yRCE&tLE^uGgcx8G^JkpzDo}9X5#Mfu4Z<m*In=>6>n-gzs9=7*+JH?sLHM;<dkLiOy#qqB;#lSS"
    "38}(n4xlO9B<@N+S$g3Her7Zn<N$ynZlcR9R7i>%Isy$X%K7&Yq8&}H<V5H5YPw{;mc?Hn^Au&+TPlY2D|(Ho!$N"
    "II3D!3!YGXU(aX)fz1DDNe{-j$yD;WtaAcZmrMLwQ=d_b8pzHPghKF=kJp$Z2iDv+3txW<}<D;C0Uq0JvzidaFd#"
    "(Mw-JRin2!Hllo7;odaO>q@XBh79^tblHmz(=r`|aKS%e}$Nms{=j{>yl0cWcnviDRo{u6Ds48`ztmPD$dy|8xTu"
    "m^#+Cq~t7oN<eHk<RV#cF1yx7%+kJ$Y{Z^6*!#Jm0hdD$8al?bB2IMOu)OudO7kGnrCG&E+bf?*PMU@kg<bls=!a"
    "B5r4{Pe5VZ{SCgx`7F9Y$VpMFG_;1%xsrQCQrfep%0uz?Tj*wdWLUeh&~Zku?K@ulrl$V~yiUOIk-sw>N#n<+3Ql"
    "Vit_OWO{6i3CZ3Tz6OJqQ)00wCmDPHj+)NbII^@Jxco^WD$|nm>HY%697;o4Z!PgL&E>22fvXr=u#lFqa2}WP_<|"
    "IuTv${s^yxdA-n94>QeCs#s^j|aWg?TlS+SjZvIDgu`Ct$<=SbtFAL!_t{Fnu1<9(7^EJbyoa|@T)>#?QP!_HTWH"
    "ml?qghdNUiaqU{&BaZc1B`P@~NQ!a5)(lLo9HJCFocVO=k`nU(iPjV0?P>G#zF5^Qy*=&we>Rq2D^689&z<{;4e{"
    "GD8@hrIYIot@pglvzo!5(G1Y6Xs<4v)Moeq{rxVHsQRuG{@3icvxnhDmlKv|tk@+zluj#n$<ih#S%4oYU8F1S6<6"
    "Hjmu5rmo-l>xzBP7&CoxE<Q+j3*2&AQ`A?W2L6+3qPp))slJw!UAn%sLko599UR_JPOB;7N6VjTn$sJ8%yf!Inib"
    "Pta|#FLHrq90Ii!R<{v(M4X+`3+iS$;W1Jb`xWHE6O9z>^Wkl44%TEz=#I(n{dvZG~F#3g%ce(28T;r4#s3B5i9$"
    "%ImT`ALa@vXg-u+;2sFDU8AWXsn5O8&v!PIiW>Sisrg@CR_hLHL%CgyF6bHF8%PZ2Pe?X;PGM5X68`9p}X>D*FCt"
    "6H{5N@c|UiRWhS2k44!yzSQ$|pBZL2muPwan(YvB)MjgA<U?lL78^GPnsw;T?(4m=Gpovu-j$COJ3=;3ki7o>(0Q"
    "GuPslp>zI3SYmU1yc4vjII?HS`NfSRz<NO{e*5HZev_JUZ5S&Dhnj?=S2_P2oc*7-KoO~qA?)2G1->QGvDdReMgF"
    "4)Bk2uAH2^2Wx&kfSE4cJCm|JTym|Im0X50pja%(#p4uNAFhW)+0aQJd>Gv02!9LDW1j$iJ!qIeL+`@neaZnc0N?"
    "e7DVIc#l)t(V)aQU()XI?YgN`eRp|uA4q5&N3#`{JiC7FgLfK7|bUI^NGQHVle-s3}$yN_HxT$FU_PM3WAE6H#qT"
    "zk$1yQyusWih0{>TH}jhYq_cfR@w8oTkMqT3H{6;hQL8ujCFSbP8Qm)G*Vi=Xh^0TVByp35#NhBNNwPyd`s>D4aI"
    "t_?r@)z_4;pc%vnH#78Z@X;q?=!wWhpgdYc|Lq(0ez_IXChZJp5w3uqV;_j-80Zu8!WeUYh$uwDGs&*GF%AKOelY"
    "3Lvk;QF|uT`l1tZ81vlYFR$+^*<p}TMRSLQ?S^wCUzjo=OQmEvJe35B8FZAs4~DZiUc!qbXDwZcDQ2emO{~HzX@;"
    "Pc+8W@B(`kmR?Z4w$ioAJ3hLpT?S)kY(VV#kk|Bzx`I=0qOMF!x}DJ=tF#RO%EJ{y$C*^yUBoMZ*FeY%(gcj;mlU"
    "@`|P!h1((LRjP5l=o12f{h{$JY-+W4SW{lXr&1~=(3bzQOEnD%*2$H>o!IyCF70CzFQW1+sk5Sv%w38&3aKtYF5Q"
    ";zubNq?}USJxWBc#JHY=p+}+=gcB9R`{Z`oC+T9L^t#*5BYqz!ka%Z@=xjWo`ISh9<H+R}E2iyCl%r;^sJMHuT4i"
    "4VE#$y8Es@o=NdFfbMrD>zyC}YAs-LBJ<LTLM~Cr15=QGa68pBVN3B%{9M@~MnGIt3^vh-nz!O&O_#^TaXM5=p75"
    "%C)HnCNLR%@QZe?VP2=Zy!4&h;im2|z0JB#`Wdt1a$XwKFXtcjco#)A98T_PGh){{Sq*aQ-LA7j_9iBOKlY}&<TN"
    "Eo{6V_8`1HOryLiuj$L~3J0z5}NqgSK{pkHjerHS9${a<}Cmb+{tK1UB-(dUW^Y4NPfa{nMEko^U-i4z1~NqfY_v"
    "MJt@*i?L<Eik_q8g>ncUc&kw>V!v<EnGCu%vP8goZ&=>Kg7evgwB}f5$F-}%E|l{T)P=@=RDC{@;T>#ohXgj!-%!"
    "A93vW{?iWZ`fMr3h7fBQr=IUsbd7`{ffF{!=F~l^_vxv5f)Xa;FlD=?Ildvw$;0>dTJ5f2qKEn8b&9QZor6URuCe"
    "<^stRWQ~|310fo$rjhMI%a#>;o*asZ1{QW|F~v;tLUC>ZKkF&;gniN*opTcc0kPC-(G-J$+(NzaRD#6Nbh;!iWu)"
    "@-btt{8l;Dfz#{Kb$G$f1BgE7>6jc6@TKWd{aiaCVEO}rF!=wMlMn1`NAKGDDewZoxX__y^y-jotCE{{1jJFbR0z"
    "9!!63&|6(4tGB&pSNx#ntBczjcQW-m^$v&JMFdSVUaJr>Zm%Mr2xebV{lT-FZKg8PQd3w6!F1azGvHrg|zskB^Ol"
    "~L2jvHyfQ<O1Plg)F%}8^!B%=YEbH2-GOKPB6w;F+n}5$ugO1HxOE)eoaUP5Q>^CC76U~G-v+m%G8Z3K!MKGCtr-"
    "Mu3F9Ku+~fZU6H*8)P=4M#GXZHGa>ye+`k2EOt6^faBs{c8}<_*DG91PfrT80cUrwXy3;&3-c-20_z5MNTtUmspp"
    "NV|dn(*u=}+M9uyry>=FmIV3Q^U41xA~}5p4j@=`LreiKyizqdV*rLH-c5kqS-ZUvR-JYLeG!WEC+2Z50E46eY}k"
    "57KEuA_h<v)PU2KCVg`Xf{I$<<XY*P<#q>hSy?lUy7oKeTtog(xkreab;j&Apc<7({yM=bHXP>vopR>8D|6;<J&7"
    "vDe0Ogz-r3yT-rw2Z-U7CJdvoXIU~@m*jt5(Vo%Ys#s}GNM_WN;vxF2oyx8t3y_+@K5ZiV|>`|+@pF-PdfjO|UhJ"
    "w(zeNjowU<OiOkdm;uU%sF;Kl~Dao9$Qb$`4e;g#GF4d=ieN2URKhFs`&USnOLPvXpz)rmZoE2$&0l|r1XeX9x2o"
    "Wxu7NI1v`+FYq#2_ZkV1iPdnp*_?K#p81_<^=D<g)HpivZ<0$(?VS8dkuV?9koJsgY%~!f?@%=r*LK0<Nm9fsu1l"
    "=g)04e2;X}})5K|fX+3gda9W7a$a6Gz#3)OGGGg>^ffl#Q0@0@@J$#TaFKVFmvRYhn2IAxv{&jCSQ1?d2Tw<>g|t"
    ")!qk|n5bVEp<55IbpnOVq=h-BBy8Lb#o!Z@uz|oPw;d}^j}}p!t-v<ZS}++~JOZ(h^yB?lSVOFOk_8z>{053$3Es"
    "R0p879#U{K0k@wMQYAY?NSj#_PKrz8J)FzjHFEil{`P}dNcfmf`1W~H+BxHgOT0pgo8@F{<3_JhGW0I+<BjWky1W"
    "rCGBR`8_8WQqY_Ug4?f2{~yj@b{cTXCp?fu$;w&)!1Ys$e3$K4jE-(WZ&;zVK$x{*5R|;c|TSaJHK1bdhY=`ZxY)"
    "o`0D=F&VIbL74PrF;r8B6e|vv(e`{yBx4kup_M+W*XMb~R6Zq+uJFUIJV7uSif7#j%`|X{*&Apw?xLo0lwwxYUeV"
    "}YTm-!<S;e#Io)P_%|u+2XP?RxNoZs%50U=2$N>%GD_tM{vL+2(HhiDiFc*`HYUCzkyi)~A~nSJj3=B!pVoi7uog"
    "I$qa}qg>~k7rVhE^o7zkZJISZM|LleUzPGRjY)djZEx+@>uy%&?fO+I3ru--r<B=_9lSf-bwj_T-(Q6NZMf88`3R"
    "caI3xTk?Pp(;!M3-}&9KzAQronue&s*x;?~C2mod-F81!}KSov$UIc`@o_y+;8$hROZ1cO;jz7X1@nv+0L`5feUf"
    "o~0?ARUr(G#9&?^INh^Ws5O-f><Jhc3V*%-AlOspO8NlyHrMc?K$(}GfDaSQSRV&s~H?Cb~wXP&?5xqCISwlalSO"
    "{ldVD`!@>Jedi{!7dF8r78Fr`~7dw-TaEapt(}x8~rGET`{79I`BP>I8Q9)FIRiM)e@V#O50hf;I?q+b9j;-QEx^"
    "i<6PeEXbSf!5g)CgZIL_<=@9eKF0Sb}>Gn=+$4$MDYY*C+es12}j|d|JW7w+8LO%V>8$+}Z9U*B)>0yxiS+IUEeb"
    "o!xLNgs1JDXmj)B_TF~fhnCTHdpp|hxBL73&HjF+(+bz3;3%0X<`2_RB*<ACZ=D>B(jbiF$o~#7HmOxOw^X&v^Kt"
    "AyvE@%}`4e0I#Fqaj*>ah}$^#e1g+3ULt;OJDsxc}cNi7tV8|9&J8@3yTPxCDIl%@-wa1Pqw1QRNdhbYBBb?mmJC"
    "}*eZ?ot4-k$l6T2rf|Q48IOfTpk8V)9DFFu#%piHT>kIgK8b&;T_UMui9UZ`F7{3Tt6)4w3w9GssCk9`=MHJ&s^q"
    "qLSx@loSdJlO{FZoNS2HjBRKUqU1V~nu^im(4Qx5$B!E8jI|gV3h76VEbRkyel+Tinx-1T+tyfS;`>8Tj4lzU!eJ"
    "5`F@eCcyu_5pjR7p+37xN5gxxa`2c@P6D3Ns^I)(np5DRLt>A+p68MISP^gX>0ynnE(8l)X+6SEG-17K|5jc7I{c"
    "n(jeH`&}93NP<Z2`yGD-+^6@FxNMM05E5uMyaVhff@xDG%I`(zB%v>qJGgRL7yJ#pGMxf^0zF}3k!zA_kOF_Nob~"
    "?q#aXMosNKAD5zWRVnuXGdmqzhKicGJU@zbT<FBVDE#Q$v9C>8G7gD8Bs^%M~K#EL($;!mvjH?1MJFQ059g0o73P"
    "Wky)$bMqv(tV+l^WxEqU9p&6UA)34T+PKAjxQVKt<`m18j{1E%DX6dehfLo2FxzsS%E!G8@K^LrC+F^bR_8fuHX="
    "WgkahL6ul6;3sxL)TH8xit;MR>r~uGg_Qu>5v*?YH3z__o+$uMh)%DeidU~}~OaPojtWzeml%pWE4Hp~jE9Dw`aC"
    "mrha(?t$yo4rhP2{dYU6)>IyX4NYOI>xRO%ztnU#Rgcg!iyhDDK^3orqsp9Hj%aNXdz?tSCQN9<&N27#1Y@i<`)F"
    "F}fExagmUXrq+7CoG9s=X~CDtS+8qKxqSBfyI<cO|M||TD`5cWvoae><vu*r@>8~|&Bs}le*Vx!O%|ikD)Gu83Vn"
    "CESZ39Q3Ffz$?S^ybkh>8rNayU43_HOB@bL^Ik5}*mlduMK_QG^J7&Z`uB)h$NO#&xo{8>x`O0I*Mc9Ltd=j0W&V"
    "9v-N4S^^rp;qU#41VKGn9>Os{R>H&q-rvYQN+yE9Q`%}t_`fJI$-Au@<AgxHz#yr0K*O1c=H;m4UpS8P@y}CgdWU"
    "|ygQ-JKm;-_<_HTU9gndeRg&_-kdqgovlqo+M8p%1RQw7>cSr0pW|YfO<~f#j^?~O5h61y~h_7;C+i-rvh<%Z^X|"
    "6Eu+Z&Ly`x$8z5+Lpr)2;$V%#C0lG@WYom*cmuR}i&10>zN9M<D>yP0UG~RP=~|-7n|oC;58S7qqe<p5DY*o_T~v"
    "KSje}z?FZ_IjUam5y!`i$Sy)82MBRGD7!!<>;_peb^RgEfT~4*7AnvLNC+~Hg2j|n09<THppqCzE{n6Gtlw^ZPmX"
    "X><F*-3)4>g<rWbp1j~<js@bDFDX1&7q_LCj#N%(#ezMq8e@5Mb$DnFx9Du)qDPl-s7FVJ$=G@K9i+SM?jIC%#ms"
    "Mjvrji_c7@@pH6a5+ka1RXf&WIT?e1nV-@Ip?1`cLmoaknsD->x1*7zz%MorK8~N=se)zMNXq1jXGcZCZgLW4&KW"
    "aL%5FASyM;2vLIx{wA0s)Wl%O#i{W&@Fm@iuqNk;|(TJYzNT#H~8uh-O$M#U7`PHn5y0x++mL)k2M=?{p9<7d3GR"
    "emrZG&tc7lWg!r=R8Umv{Cxo$`vS^jZepl|I<?=NqnI{PT>~Ugjw$Mi`EF!B~qK*PK)U!w^pL5#TNq9&{nhd20K`"
    "Det+J;>S6`DBhzHJLSz$JouO7Z(<u)#<CXA0IoNO$up}V!Wc%(Ae*1wP7sX(UNX6SINyw_ug6g{IGCUc0%RyER;O"
    "`=9t%qoAm0pF52J8GEhk|ma*V18$$1V(lRe>x8!GS)Ql?RD$_)2TCL9_hwKj!Wk7C3QE5^$ht4eTTDam?6*p3^v!"
    "r7KV`bIW{Ff)j1xEdC}u?sCvP%;TLg62XFa_n4-R2fcTTDKIBbSJ`ShSo<+AB!yC_i9FLw^xr|$2`j_$3UeKaWme"
    "0xgQO;2V2Aa_SUdH819E{Od7SjH;6Dy{pI#f81C$BZtb*pw_E#r@$TMmv$Z#jx1s%Zm3tl~ua#Pbv=6v`<T4eXhJ"
    "t)kvUpVGmNGl?=a|~rdy+YxWR549<4NZD_M+HLRID+DPXE#ne;)Lf$AVU6-Ecx79+vya%U`XHZQiA-I%~$)Y0|y8"
    "l8YoEx;0%ItDZaM#gitpRCM@=&)FHy*_TgdA}TX6lxhAjQb^J{;P84^y7>mxF6<4e`LkR#`09j_T0mglZDYX=d0)"
    "TsB#^4-Rmr;fY*g7Xwd1PR7*M+bZ$Lw5_wvllC^A+RZ?C=tZXp_>VM5nUz&Iw{{CdMzE$j;y3-eFwp<+R4Z%f$z6"
    "FF3@h7lm$CH|hnx<koLgf*uF=@{76Y>G@7*-B8RRebkU%ZextQ1;Pf)xr(}K561$0*oBb0JYOI;>RiR6NU){*J-L"
    "4KUGcZCPn=^>C4f|fxI_o^~dmU3jEe{$5gWbb*g576+53~ihDmmQ-|>t^~QX^8%-`&6tpNP*S{(b{(MIHudLn?uq"
    "EzY75~!kOa^llvt+`~q)aY7p^{>`y@J-FtWa2HOflFh`Kzo2OQmK8_s<!3W!I&C`FG3DZ<g}&CAj<r;qn)fh7V7("
    "g0&y+^>^C+-NELdwY9lB+}#a#wubRA+}n>^dprI1=Jw|1PQTykzuX?YjCS@n_g+SO@ZasNmxHar-fk&tUnAyWeuL"
    "5+s%kbQhU!P{f?ewSSMsUfjc@PCi|>i4e`4yNnEL-bQ(t}Qb9}5W;o-sA(HRUV`BEQ%JC27##DizR?@);#3~PsL)"
    "Wz7_+JGqU06~;SLiiofz3X<ZfE1kumHArixvM5sbBAbfqShMz)2Lun6|2}A6MHPRy{L0eh8l|B)0Iw-O4Qycb>{T"
    "YQz<ulthsaZs0vIc(2~Abf5g0pyl;&BmS7&Eul(tk)5RBd(5D+OS-P*O%()^P>oUIFH>09);6YO=O+4#yXbPirik"
    "X{+A&4}<1CnU~Ib)d~n!!+$jJOE6#0YrR5wdXxYl%5rRRULrCghw^R7FkmQeOX5RpeE7NSK@cEDqmi;@BiQ`Qc*1"
    "%751Et7?GEDCWDQe66IygarHm19Y2_{#u=lo-T<7NP>Wq7fm|u!JS0T%U1MboF<US2yF!?C|;-`xY)4NPz!VRc?L"
    "Xle7k}YUuCzVUBugk!iZAi8v#B+I7TBEXZ{LDgN4rYB`SEBG)iQ)!XD%Vu@9hSLam+(lVohyp-i_lmB`p-l5ya>O"
    "avm7#J*qdf9pG}5<$tZvq?Aw=*0}(NZ!$HNC9P)7N3l;B_$&Eo=OcrvHDM}{u8VJ-%=vN#+2AMtcYx%swkIRVS!m"
    "t6_u{TY~Z#@jfon5{v)~Ms3H?}qgAok#Pc5w&Yf=RkI&bsIPp7R#((^D;;S@a)Rd4Y9ILJb24l#*Mpp#(vf%nc_1"
    "rnfS9ikNg{sOEk5qNSwOSN9w7r`iR4gOkf``oR3%Wc3#~me<*;uVBD!M*#EwUcM4@-I%85{8tqS-g2$Q87&zO*Bg"
    "Z|+{1#hQOHj3Wm)GhpN49#19nD#%Amrlmm2jV@Vzx{h?EW@8uO;D`hCqRLem6b2E+D0}tEZ61C4QmmJ*3oLsq3Eq"
    "ei<dMmt-yA(poW&I><~Or+aeX6sa?xukFVk^w^2VtDtoIg~Z5DsjB?AfL9c(36ItbV#mIa*QD}t$Gf-XSg$sk8au"
    "qkrkIA5^#5=)1<2!|y<mQ{cbjfqLtunEnD?x0Mj^Z&E=F5GP#S(@Nq$yU`K0df)|C0Vwif@^4svN@JW4oSPFmf}E"
    "3AVFpcuSNorSQ`HKz0Y`MWCEn*-JU(J?jsXGWMssRh#U9$UCtx1WWHLV{z|GIduu>9H2`MUIitR@3`Pa(vqgc2IC"
    ")htsu_8(o>I0VrA9CUu@a74WRkoRgnGRolm_Apxe2kF5t=l%mahNL)vKP})2lK>2<o+7m3;KW55OP4|Nh&{;ow>J"
    "{Mq*4>34%?&n~}zGI~1tVf!-6zWpKpVfg*`&$GePAD%vYI{5C%vnPY?;rH2d&~s|lD*^@Sfpj&+E8?bewIV;`H^~"
    "p}<niNVXS+_dk_Ut5s*hN7gg<qyKXt7?b*(>jt^eX~70ZzI5FRj(!_}CPS4Bc~=p#}<8r1G>W{$m`od)*?@>3_U="
    "G^`6LLzVs4Jr-&p9BF6Ey#T}D2I<!fAvbpnJA#`KTM*g*;O*2-KkJwV}Q!#j_<zVK>FsLrZ<0kj~Ca}&cI3yn3>B"
    "Ph~tMiVU>1<t)oUMsOyE?-DHp#W9+afHG_5t5^Fw9<Q_&5l&eu74q1UY#Tb&FPsI&Dq5d))4tDu>zL<$vM04brA("
    "Ymx*b!hwIWsTl&$u7xAr0b(!;S77soR#Kq&mpQW1Dc`<`t00Rtm<g4xEe50c|Lf9!D(p_;ry)J1Hi^CQr7v=|wP5"
    "rHM-Xa$4zUqCh7(RaXT<gikUSjWNYRy#fmqIu&g9|6C#UJ|&`&K@@|$Njm*u`-h$Hx1W9YbTE4QWGCDC;c|QQZ2R"
    ")^yQkk~Ab{@tFuWX$wzKTJ@3N=Qh9Il%JjqAj4~L_{chF#ar&c=MG_5S1BM|ejw_~reou=Y+ckOq(sb6Jo-+%w7g"
    "!`w2`=^Bar-b`o*qb2STu&dG$yW0etwqQW(&ReN#>8xbq-5sGTikxLv}85S_V^*}$(F6=?}9TWqo9`ZFbT}egn4L"
    "W=;er+l}60e3f0o#ux}f8`A?^o25x1P6={b3Y&9$d7|G43b_5o#gj#UUA0E^{OhOKaAG7Gz&*4j|FGt>7wQ)DTdb"
    "0w}POS)jVWJC<!q5VwHextppu{kbg#~9J!=oa*&aZDu#<!usnQIKuzsz&AQ=;v)7>8BPqNZaO=G<Z6Ng^K?+bIdB"
    "#oW0Z=uS3FBg+xRI>~_`<T3%1(Kx%(MBY*=PS9!SBuAKcuCTU?{KHI;g%lu%;%Nx!h-giw^JKVyck3pEXR$EHNN|"
    "x1xKUHYjZ~t*IU#@!2fgsAM1<dKr8&|z6M!L>8N8qx&GZEk9e$(*V6k;Y<pBkHiE;xFPnTSMNDkss7G;NFb?$@z1"
    "M)PCb0TTOjQb<vw9JO}4H`S81v_gsROtcNi;A3(76Ssf@JZy1m$TWDzQde~RBD62r<`9P_mXRkpk%(G5dQq0mSO>"
    "BnqEnkPh)LpjJJ87eVcvvWUw>Z&M$Yqzr1|%!_IIt96tZ{`Lpky!Lb`W`Qho)!Sio-zT0{J>@xpuwEYDBmv3KY-_"
    "}ax6ncjX-b-2pke%A_dxQX7u0U{T(PdWF#^XDk)Ru_;;m-=ue@fGTO4ENz)Bn}&#UWcZ_&&k+rou)0kHFf|;u2)N"
    "@QWMtcAow4LlBK04o;3;T#9DIO6KYDNO-rj1xLQepJ5X-CL3NYauTNH{UV#w2Gp5=#C9d+eT}b~y5-d^6YYmWh1e"
    "x+M#S+dxJ>rFv4R;*bBNS$`ol#12)&{*cbs-#5P~0naYfCjeeb4m_NFuj7f3Ew3l4YZOAk|7(FT5XO=j`AMnsr!c"
    "vBgJ%1_{E=JT8+bvmLbg7bdaNXSPS5`)%PgypC^jY7d42v7_P5xJbL;WAo|i^#+Hf4CSx0}l2m9dwFqk>RoZo#j-"
    "Nh%t4Qa6zfKG|T)JQ3kasfdYk5O&$172Ty01-RRJeI5KpB#2WUohz77b;1iSi3uGBND*hOQgYxqTIB&_JD3>>Ixm"
    "%#nk3jP+E<{Js-woQt>Bnr2)BtsCxbm=9IZ|Iyi1-`}+-RXyPb6Suj&20i)(U<cc-~@ANCxwi$VT`_Hl??sM0oKh"
    "NSbnxPqRfaBTk|)rPxA4f(BB?ys{|&l@0y)pNqFs_%Ga)AvSwHTmhkGi^m`!<J$Mf*=5=PumdUsjQqh{`oF!|{D!"
    "y666-6%s%Ued!H`rR0;uJU#CIO6Ka~AJ`wSsx9T2Au@VzN*a7-K@!(Ic;6iz4HqZ#XdChiIILk25C&Qgh#^12t7#"
    "ga(H6ono!*omN;IFAH~%?qV_qr^mR82@<41ruSNmP|8z&(W!yzKsLq>BB(fqDE~O)C2PwuRrr)R+>M{>lK_%`w^t"
    "^%=u_m-hOps|5jPMK_!8gl3XPMb-$18`~7wYSj}t<vQxW*Y-hTBJKfto+kJiXGc<M3)LZoQW3=AzGH%)GKIIlHK2"
    "7Wa_LmJgVz=Es=Nm#=p)GBS|3_<wMdh)d*V$!P<VO(8`+YdU%YMIwI7+ZdW>}$K^xJmVL0LzbC4>xyQBwNgV)l`i"
    "82?Ktgggh3zNyNJ)sYEHUV98XoUIlEqjccQ+c0JGu5(qXFM{U`EOgK9$0db*07fKxqQg3M%j=>8tO3pk5D&}6umB"
    "6<+kf=DVW>8K(e*Lm7a45X<afMkD1^2}(R_63WNyrsoRp+_F-6(Oo+rCvDAf5AwY+8{1#cw!-QfDOP`tbSWO%zZ0"
    "Oqh&?HvhjG(PWjqWGESw9+}cd_PF^xkm;vcm#2#WNWA`GO@lfNMDO7ZlHAyDFW2*upKNDANCHCVF_YSFnau-w*hs"
    "gX|<iDTVpfeCZJ99<h}d}>kHtuU&1Eo_vbeQVAH@6NUePE2$(rgupRVO*$<=u>b940tUoU-Bl%Ck5!Ef&r{Bo}6?"
    "@Rl3h>^=cASgKwt`!KF~oSnb7DZsYxqR;QJ7!Mxhr>}P?9cnkUUZMM8jLh#gZDi$D0hx@YE#x)Y?uxTCiw-k!&ST"
    "od-lWB#_j%4?6#X?bw2I^SMaANp@~Qo~Bov7Z<mnhTy(0Y&8#hSxWU=5l@F&f-=fuz$i;lD4YpXzGt7OU70SWfz-"
    "~{wqtu;Dw3?vmeiJa;ZIzQa~B>HB&m<#MENmdayuxf&`^})DPvsE>3+iL!Q}I~1%CHH`{BHy2+ag|A!Nz~!V!d+2"
    ")|=OI2(?JE&iI8M~gJs@yX2sjNW9{#p6<)<%QGb$`v<D2~a__{$=ikv)~3G=$i~gx1lY52O`iFiy$I^+;v$&`?;Q"
    "l>ls}*GGLWt&IaMO<%|w@-lB~wh;5u7I`Dk~nyW6ZtUJGx*M0r+9)qq_VD!%vgLtM`^5G^*o7J=cZAnWzC}r-vxI"
    "8*<<*G}(n#%gl(I=87k5ZHTtA;_xy7!@2%oymW=jzG$zxUnbgZ~Xq=Y5)dz~eqEJMwe~20QUMjw}ln^oC9!9Pgg}"
    "!aV%+pQmU0Z%&Sm&iM1$o8y-UC;a#8-NT>duf6{6>(>ZYW_rAhjv4mD@juUgIXdJwT;N_=^7ghoobFv`aDPb-5zv"
    "*&9_FM2qI3gHWcd-@X9Oi(7XLQliEe1tZE@yQcmYdpcKqv~uB)Me+{6)d7HCxsud|xVcQK)>m%C}Z1&$^YR72pl0"
    "1;d4VhGfR#M=4^iDsrvH%|M)t3yQUS<jK!JI9##65R3fmf(6Fd>r=7Y=YllC3?<k1izNc;cSH!rtDh}j`ypL^2K6"
    "JBRm6?msNQfxMhEuf9!)&M5%T&Nt(jiOb_O>viRf+Jq{DXY_rICQyCJQoM}Wz7&ySFiMLR0#aW>tHK5)wcb?%Z6u"
    ")#fE~k_^b3NnU_Q_H^494>pF~NuqX!L&7{H{SHrYJv9{H_+W)x4AZ%85QXe@?^xkOh)rNtr36T9KPIBYVkk3qvwV"
    "gI71E0>2p>Oy!8j2;Qv<d(aF;(D_wMr#>%+si>44Dg6;pG^afFN>|zeWd^ptU<tpCuOL<Ya!wB#h_&B*I_%+H=Ic"
    ")UwqU5{LHNl&m?o&6^zan(Pfkyc*`YFw>TdO0J6bW1Rj)TsB*e01?aF>^*()tg4yMEW6a5|XC`1}@=t8(7=oF0+v"
    "1g-&^(r5_7ql~Y^oR_W?P_xaSm6YoR9hiTq|;jmZqL^H{Dxe!Y~G|hE5AwQV=*;7>%G;EG6)vYtG6O-bG%p4BE}b"
    "N__Od)Y!GLuXlOUwfm#i#z3?*P&$9U<4PgIB1Kea`1BWcw%B6YOR4Ib(^zX<Ese~WnMDCuRq6=cA8vze1eE(|q;5"
    "Aknt3J$%jk^CM`dHl~>OF|4Ep?CBz<`)p!xcI4_Adj&Mx|G}QrC3KTHq#q-o{lyMSSIt0fUJfa5mFN#Mq)D>yc_6"
    "Xh1IuG>xdBqd_r4c2%%yvPCDl#sDLVB9GY>Lb2jkE@CL$calP#N=uiui1cn7g_F>%ac}<}9-Z}1fBWgp!5I~{yRG"
    "A*{669e!GR#b|FiQTgKkzBnjM(*;_YPOGmXXtWmI+j@K&z$BRhP9ew<nLfzMH>o@c|}xQ~3;Cm!i7G5!z67V_8)h"
    "7sZEmk2CedDQb+rKEqh_5P~k&9W&PSk&p+$!~jSzn$#&f8Tw5@Unllzq|L#{z?C2|M!Faf53nK$8QHGxOs33^5U;"
    "$VrJyTknFy$G{hRO;(tR|14wO@fsDMe6jeH-gbQcxFA@_bB$X!6tuUn;F@qu~%uam7o;P;QhW3-dU}?RzQ5TAO7a"
    "sG@+*)Skdt*jNFA9+xCx<B`Hl4^P`^%HvS7$O%W8!vG@8dyYE=H?nzDQBF6^Ofi1iCGKha8ACO5>Y33do3ATy3WX"
    "DKwMF1QKQjN{d(%`)D+j(Cfkk@@XiceDrN11Q0x;xQ?4HY*8#(Vk#$+6LyHtNTJ;#FH?g%oEbY1#d2yaCMoWSH^y"
    "J{(G@L#CTfH8WLg>yljN9GZwjwq#uH+fv2z)fMUc6Y<C`J29vGxyJ_a^v>F1x0IUy=vY_W$r1C`@?lEXutkU|)>;"
    "fGrKxkagQ!ERwrkx4n?Sq)H3AFamjnskyq?ly7u03xKB(v23rJfWad!EW2*eU&_=QiuLXhedWZg^N$?+)Q<bmUdL"
    "~Q-{ALV9Zm6ea67FEpmDL9bXqynLh%$V3{2hJ9$v1xH28o6<PtY8v;$p=eYPcsd&8+Jplt=bxFuoDhrK(zIgy-FK"
    "Ez<EK+nO0y}Lk`EaKpI0c!nV6$f985Y&ltvht~EDPNX|DXiJ%AO=EGyb%JmR4wP<ba(Z=5bO8u?}Z;Pg4m}B7r)!"
    "OFmZRhM}g<YPpC-!LQ{aFINjBYBI?uaDp3oh~WR>IU&A2uyB6p`O~yQ1~O`5B|qWDHa>MMRa+o-g;ZTK!A{QTF?V"
    "l_XMz-YF2l;k4_+F=J0Yt<$^L8D1Mdl2a0~8|d<r5THu#9^g?n(E%~Q6*C$zq>Tc`O)4283smg}Ml^PsS{X;wi5_"
    "gKJ54ph8E1GJEecYZFxO=x|R&;YEnim3leEmjq!^eQ174wlHuKxKxT3>K%PIBUe|*W*sqZan43tL0K_zEZ%rp{JB"
    "HH32ACv-(3WMQc$rMfn&Zw69jc87q_a4}FG472r#l{Q7T7H_b7Ny3CjCs4r#Dh^r7*dWsR(K5&Y2%7h)GQ9b)qOj"
    "eWT0CyrZS0qmH9q0x_Tjl%wMyR@^J1930le%*xXh*oJLmjm35a{KHoPxv=ECmH;^f44yQ_-mxB;%C2gtce=6^)5j"
    "VzlBUTE+Te>QjV7PS}(9-`Pk)+Dm*VXqpuHj}~`R%y(1H@NF$;h@Dc0EBwwlEyyuLCu+zSewHnw*0TZ7g-U77v&#"
    "YzZZ;Dt%s~KQadHWVoS6Y1?0gLB8*(Wg4M_;VrHiGrYNebqW&vA9s#Hm3D9H=Jxc9ld0wbQ@U<$!%h}*`-AUfyWW"
    "JnB$dYDwe06uS)q=7A*B7m9YQ!M&$a7z`W$lQgy#|Ne~0`mnW5yJpsJvObkyma!zNg%V-Und6$g)M5+gu=|IGc=K"
    "6Z|<BWR7n?`6C$F#f$F6wX9mIGh3VGBZ5i(%FpC#jP%m6ky(i3y$~wXWLpWp!*|Yd!5LX5E|KCrK4&BAwBuj>n=7"
    "w9?m4NQ$TO=NA$w0Zr3-Y^V0W-hhdLK)M*B#QF-0yg|N&yP!N3eIv4+A@xg&T|nG4dS6KaPvCo}28`QJ|&ecS>Tu"
    "1J5MoFNQPC8u}m!O(F2qCM@K6Eeq+-XX9}n3+oN**oYQZ;rq*n9IMI~qili^fh-YWl@-07NYu~Ryss5_nz+gVo_G"
    "cwK(8~75Tne;2`ziZk=N#)VKyL}Y&EB^E~%c9shOC{R-;X3)(Bb3YLZP&Q6RadB^=c3SDwz}0fvWK@dDmvv?MlVi"
    "M*Q)>4wx(S7Nvvx%h`7s=)~Zf;2Fk-^$zqp3%e&%rU+|q7$WW0Tt7IbU9nTP+5gGr%8fSJO?S`LdpOolPyC*H{dd"
    "RxF4x1B@1u4E-+EDG%Kr84$N;xKyp%*hp+28Hgn)eO<dqo7&fQ#rFNYidHxcCv9rO_s6k6mrc)}|<1yjBB!VQ@`F"
    "M_k9SMbaIhOp!>*5L#FyPW2p}=EH;(WZGjq@0BqLA3xfE@9XAd%`eBpOrL-7<NP&}^jm3Yidde0h!Bg|%*73ybh="
    "`4owF&W1v_E#K4C>ZA^{5fFdbam-5KNah8}QD+FD1Ck$sd2={1?Xf9}EtnYsdmQ=BVpCr8T`jMHzTej|nN#!|ug1"
    ";X<xN2b`;RaU6s$n|<5H#|*_DHQy|O~PmS02xzcD^iOy?`S9$@MuxUJ;WfE;l-mgG?QnFGg%u&MqvyDXKpC<EdwA"
    "CW?acoQzril9sZ6|Fu#OLF0K_EF9yX`*o7%;&kp$HK76H9SHm#A<ed#+ED0LR~70aJ3TNFGDS%3Im7XlM9t6`GJZ"
    "yWrM|x;Nj=onj9r%vk*d4!_x+*a>l!jhA-@<#J(4tgKCygSt2lZ%JivW4toYRJ_nVL9;b6@D3ofd7rl8l5cEc^N~"
    "L6nktuQK<5lSm(}bdv8sTh>rgM+vXAXpm$@z?<1wTiz@@9NlLbD&w&^~fmvO$V@F_V}=H)tJj2}=j&Mz%L;9u#Ft"
    "ariu4ln@6M`OrE06yI&GU<8x|ojj*_@YR%p`>zSO2kkk!CQCA?(6rpN0$!j^IKcO{96&Q0YnZ@3y;So!8W_Rp1k0"
    ">RhTDLKJIGOf1J*cMNg%7>E^Vf{n51q@&YqfOB_Ei>9XMG6nLDMZp;gQgsm0oW*?yWb{=!-xfD-wFqx4rxP-t=zr"
    "ujFPNl3wGYbfVpywD<YTr^seo<&d<+tX0*9i4C!!#L(tM9DfK`k8!F-2M}fTbNWXr=CNJCNo1Vb4%$&NQIZJ%|=p"
    "c!qFIbzMP|Fli8ak4~8~%MGvx6Nnr^Nzb)Z*oT9N{j9$hYn~a1I#Ph%d;WC$B(Ta(?$yNtvC68!Ix+hEG$jO}~z!"
    "#*Gu9mD*O4J}`3oMG+Lf3wvIth+|nT{D4eKOCgX&5&)YG%BYjW$t~SZJGW(Sa9WXST}#EyV1NhO!YzXsPv)N*X7d"
    "(y`1YCvAHC$R<;9#2M8!lx!s%Qd7fqcBl3{6{Wcq4TVisNste;X;%JrJ%gc)slFV_IzICcR)6urMYi0i&y=ghhl0"
    "wzXfzt?)&yBGM%HO2)x>8`ETDqbn?(udA+r#YFN)EP!ccG?Hpi1lLYdv86tPM+O5S0fgikrjjEe=wn^YJ_T^GjBV"
    "z#O>BI7W#abOjPaGJQR4zf0iyYdj?ntUn5qc4Rogb>ao*i&we+|m*l%Az~fqUSuc_i#YfW<!^=H&u<2YLrc^7AG{"
    "PxG$p^y0z%VBm$>&fcb?}GN39V1r4jwQo>#}#wJ^cT&GJVZZ#}94^j{fDP=pW?~LQsGYcbzB4C!B6>YRxq8g_nB2"
    "&p0gX=9_A=+&jreZX)5axXnu}!bd@Fs(mAr)2g07r}g&K6Xnfo5o%u_&|%&y_c!up1ynj_$A(!3V<YsbUj*nOOc~"
    "$q>^gCi7*pD8mM~QSO*3;S>{Zn%ZL`);x0=kZ?|_6)Tl)EcJ#gXQ!6sun(ZtzsrT`XJd^KsN*yzjxWq$#TW$t^}3"
    "CzA1!I{)Ei2iGsIQV<7dns%KmMs)>MupqmH{aJF>zt94TJ@ghVcsq{$0F{Aw!f>EBL~UF-)>aI(lgx;|P~vrrO3i"
    "7UqlpC-BGfF&RHndI=6p~~yTOdMw_zEGKt4gyChYAxOBB=sPH|Hatq7?FWS*NOw6!bsjv5hqBjvr}757v}yUE5>w"
    "e=qfKtR@_l>SqR&y)o<twv)8G-F6PEqUG;6Jj#ZZzih{RcL*SO5qM<<D>%iv&ym)s_l+|;7+z=N679_#o@pT#xEj"
    "2l@hz?x-S3EvWq2Fwnl9PB^_;r8v7k|)lhS{-K(iY+&)$#*(&4*gA*(Q==+JplB*qqWA`hkqM=#0jvB6ZN@>^sq8"
    "_bJ&Mgo7ABfg1J4K>s)Ji8D&U$p+}{rMMgs{@F3-7Npxe4#dsTUMT(v?L1RTPJI4*@VQGCHiO|cDW<MiORr8XxGX"
    "6)><1m@7!8D+#`<U6xDt`PKg@6+Irz2>eBl*peMU0a<GOFm>a^|yU!Y+nOTMB+crtbDOyI6lnh%aMDPOk0Aj!#c("
    "7cT6M&@yVdWtO@D0vx-PDL{SVHi}iA<(NVP!w3$SAJZe;lP%;VDKW311*ZH>slq}6>JW4bV}$1N+xFRhHvq2;!l<"
    "G%P*XoeHe+O7!HQymQXL$SAhqdy!IVKPN3o3_;01~-v(?Zi0WXS+>8yk5m(WWeZ-z7H0A~h>u!SnM400i6)KcFkv"
    "1CvhqIp{6JCI~0qK3-6zH&xfL8Td5825$25w5o0SabNt_rS5;sd~$^!YJJN-MMmL;uTbkkY3t(%!j9V4w_nrHJad"
    "&K`@aXjK4Gpdnxqs!pL!gjE$qAP^J_uk4EnqggMtI1Vp@#AD|j3}nJ#LhOC(np;oSdnTbER3nL_5C%)9#){@WS^Y"
    "@DF^UgCi5eq<nyM?bEtazhGPkj<iX)72nUhEqErH^3rhR2UaE+CiXh;48lCT^TNJ`fA%bXz@9M`f&Igo1^?k4R)>"
    "*WMqWoLRC7qv!`(Y<O4@E|wQnm(JB^l@#pbM|2Rd=gF}-0pOp+tm?~L}@;~vQA|_;uNCA#XetnD`=1&g(9v&M!PL"
    "PW^+;)9k2De*fLdLWIu6L9*H|4&(ds~r_udoVi;IEvO0U_)HQRqxf9ArrF2MWf>S%woi(U{LAor)7F^|3D-E(nqI"
    "H1iGzSRnXll8HBfLa=6-o+Yj<mX3NED4$Ho8v*9iQoN-Gv%n^OI;4*1*ufXZFVgH6)1eQ@ZItf^1C%0oF2>u%?p_"
    ">u*cQ^5s&AS`y%S(06diiuGS6tNBNex>Mip3>$vT79wl>q$qDvN@atvU6-y`3P+lW-manPP09~NnK=<<JPDFN+~8"
    "3ms~Bw2AV3Bd&!!v7T!OzwZ7jGb4X=@dyn<VFksyjt$)?<4zMx^1WZ@96C9c+5gNdl?_97Qom_ud5@%2DjQd3<l`"
    "BqCU*jQgqV}@;HR*<lM!co=MCfNB-N)v3$MogVRR2{Az3}~8kqP1>831<caA6*z$uIJzk5uq?5MIGwp47<*}snnM"
    "%oTZfAg$_)kl9x_|_<c$c_@`VfVs+>INFnPSK-Xs1?unS#NX%ldt1;G;tBKm~Hq&X;UdPdAj#7tL$&zBwI5^LkT)"
    "B3JsvsObohrhxF_DSw3zHERWEjM#6jIq!0t3bX7qNR&fYAy~WAoMJ7}P1JV79g2iWV}z;c$tniL;o(hG<|7KjF>2"
    "nzKGaK0)hzW*V?KL#8#{xnx^(jj^yvCYdotV>P`Bn6S&me9W)Mv^dCk21QjK3L{xFwj)wF4_d6uB>h|wg0molDCZ"
    "TJJAa8uA=r8)FynQc<yh8@&uJMV2^SL(GGCYokU-$8+M#)^UJ$48;fl?wnOGBx0r|pldWozY8+A}DkQHMG+c_t78"
    "ND(V_}$lADQALg{Tu>-!a2E<iT48!Q0YbjQc)j5MJDGcy)i0|a_&?#8B3h?`=;1;@*FedGQOpPdKcoup-BwY3%Rb"
    "0HP_(A-2;=^=SmeA&l>i$YdmF*YL0__$af@)$k!-$8gn=L95=VKnDHqDATcO(H<Qk+JG~nLE4-0otv*{uC`{Gd!U"
    "*2D{2Hxe-ObCJ7y3RI8B%V1vS^l$n;K(&a0g)?m}dqa;@)gB#_Vx0QUJ%8DtaURAP!~43!QMl6uUCiz$-$IjM)`l"
    "$Pi<|>4vr-C(XPlG17$0I07iT-lY3#wiuWNCSwn~%W#<uJfq_bhL;SGyHpSd1G&b*+i*SJocE1+YZOIK+;18BiRI"
    "1|dzu04W7&~5fE4}YLAxOa4CF4JB*KgoxlXr&HJcMG`8xlyF~XSGHrCLLL(P$3U9YosaEUY1L7AlWG~O;AvWe<G!"
    "#(8TVKPeQ%F7F_WlOIsSe;^Zaos~|G9|=L(7<6=qm>EnXxOo%g1DIla3iELE+iZ3NX@v8K~iF$k`hB;M+8h1EVM0"
    "ShE%RbHy2%lCD+yxcZU<?BJhC(=6K>`F4`$j9yD5FJISd*@W|cDV<Em{2caRIYt5nyqRdme*qPRhPUd^b(cUtN*%"
    "&<GK{*oB>-Ra0qTg3sMe^pc#m)hI%*y3v;Ex0^??5<yT7Fz)^RENp=*+&yFY3U*lt<k{^cw6(8WU_!1EeP=7oQs#"
    "!PkhA1v}_$_w?7(I)unrPHO}j(&nprxR9sx^cg(dvM<$4TiJsm@nqHq&2~23hpBQHDr2{l`V9jLRVgLE5Jao1ykl"
    "4S&JdI<;1ED_)820xWCp8y#$ZoYz^KaiHCUER>y42tQ2-~fJ3mo+>cQhEiV}WH(q_5re9Xr0TlfcT3`uY{Xb0^o;"
    "FJR~G9^S;eDswpz{`BpCOVTkpr|!wrsu98bP=igO>aO<JDSa=5Mf4^p_`Pk2}w>A<aG(Z9&^B_9YH`nTnBtD0?hl"
    "Xq>uD@Ms{{VyMkR)h|`1r|A-q7X5&6bY}cR+8W2ziNCth;AXt@z$rSD?1vjHNDS99S2GSex&*?1LN_LVij{&;s^S"
    "2e8L>sK9h&c98$Zv`=K}R%#EaB`4ZsSA339N>|4Xj^J%KBfr$J}HTplfbgRF;pnE>6JiW96(J5D9aH3F5)~s2>3k"
    "1@1iVs&hFT-thhlFo71~jGYHy=Ty%_<$3=BJ|zk>;_bA0B++`Jnb1Q}b;pEdULj%tFJh7OdgeNkom@vm1MK&W0$8"
    "T*0f->0m@Uo_fZi_tl4DL%6NM{demWrldeh_7hGFHTDcYYmkP7IZgOh0XmGrnCRj{U^d)ziAuYV>nJgbkIF~}tqK"
    "nkU}eGkwG&-Ho&7f$fXZ*a;${|=mvCYlC^pf(a2iI23XkjnqB0gDQgsX~jcpEu3*Cm%hO1q%bTX%g=3O)|(g!Im_"
    "TLYqK`iuc#Wmh6GLdUSZWzegZE737h-BIAm^pHY!k!0i~n(tuaGk4FkjT(koOBGSWjyabE5c7!Cwa$8R{tRE!kcq"
    "hAur?2)2#nO!RGeQBCQb`W4$y;x9W%j{PlA<@Gz)1Y%xdm9cz)fS&w=hKdo~J3B8uicv-!O(Zuh6ohm}2Y`2n*iU"
    "8qNl*IDBXoDaJ!dv&JY_7_CC3iS*wLzts#?Ug^~59A*rLkZT@TGZ@p&hNB69+RUnZ-(NK1vHgB<dhpZh{XXuD0J_"
    "M5xZ>Rn|Fqpb?^M6h#YAyxu|ymvmhsYrFVEf+hwpnE8ucw_L%SQ<*&&W<=8PqN(5e!UcBBAWvb$;|p`f+dZx%MKq"
    "V>d|R64i$@P(va_jtI0shhj+;dPQ;@YZmLw^{04H`tne9z0i_fOd0Uee=b-`D`FR3gfXcZZL9!+0$?-4bR6a9A=^"
    "8!cF5?tBhV-vpK-QXh)r~71hM$Nylgqs^Jk!m)Nf!53aYLc*d=lMyn_jx1uCgln@2}$qOgMvXn?nHB!o3JC!mZ=r"
    "U_G1CiBdfKg3sNb6*m1GWBOUINAzNh@?ik*Op%mHEt9(ggF0&5v(5yySQ>`*efoUA7HbN4U}M^v0OF=89D%zx@Q)"
    "KE)uW2og0au3i=hGV=<7(^A|#66;Gl2zxpG<><I-cxtM@fJ-6PCfF+Yys-)$pB(J%pVF1*)#*bZvv0Qv=o-G=`{i"
    "iw==JW|K7Rjn+kAft62m`!+CMq@Cq3}px3}D@iNJ$?*{)K)UtvKFTB;60Md2T4i)p*Y7#tc)RP;Dj?Q)G*hqGM`P"
    "j4Br>0TvYd7q<`;JucXE#SwC3?w{w(Pc3djA{5mm45qzV}7xHi}OZ(56lFCN>PZWHEK<YX`l19dpnq3^T~YL-u8N"
    "biQakJcla`Wfr3$?-)Cvwa<%Mjqn`(R)o$s!D#~S_+&-;-aBLkL)#pc#7MS$)_5p`&;&Mq51zw*;&eg<d!*Cr1{s"
    "DOBHDSPBv~PWk@P4YxR#Hi^mM&X>Mtk8f9PtIVK!7*%Eim@+s_Xz^wSJ@a<nUAGrN8Dk%1OmnrELwDtRgDP?(3S@"
    "c;(S_Oe}s-MHMobd!h&2x|~FOBLF!+9S6U!&`dd)!D4X9IcUab>s;Tea?h=$2gPe}X-1DENY|dBg6)BSmB`!P-B-"
    "{jxPgiLkDl~dp|^K;nf8@3H-3~j_GOlXW0QJz=__Xp%Y^C-+qH}M=n?hg(Od^)*4EU}bXz$EB`Xb+M?Apu$O2yY+"
    "oT2=(yZryGuMPvw^a7Rhu_D#6GEf<OUy*YPtJy->_79ZNy+Kb&2w%zpKJ%o<LH{3?ED<=8&Kx)UsH<GCRcO`yc8J"
    "=*Z*^LIg&#wwvIZCQXCBv#M-<nvc2n)|6IjD2Zn?uIMrJydnDG%A&qJ5d>4JjK$w;05kt!kUl%AW^kv$iI&&LBl)"
    "jxZZlkgO`)k^Aw~J-V3<OJFU<-a}!sv&QvxDwT_pVDFCeQECp@su+n3;L0lUK`U5%?>@Va~;QnO1JS)I3r59=E!r"
    "WnuhM3(p{>&R+;Rl7^d6C|htDrY<YAbM#zS_-fqZU-Rnc;=CK)=TEk^RDwCT8wmlVNTRd32PFDm=h@c3cenoU?X4"
    "gBTNmH_)i6rum?K)>IrOea*byO2mQ#IMH%a(<lxp@>D%QU~smfG2g@?#-NDTAc+j-u9`u(#i=&o$}P|j_A#$P&22"
    "?mUvF_tTd>-f}dyW()NU6S&MU-75)nv=;cE7|LGna3>A;wssB@_q8RUS*Q_-A;e|+4uc#zk9xkrIy1tmyAwIL0be"
    "rCLso)US{Y`pIoA&&m^`OK?jxbR;N(q*JT~UKBpB@?#3QT%FfeO61mMTijmxij_5Jb3NSMI3*SVJF^;n9(EaOGyy"
    "82?=^OQO%HImg&0%~Wr^S-VkHRgqnF@sN7;0hFXYY~E)2)V#h_XlpY6$x8Lsxz2!oyjwR-`aT5uFD}!|>YBKxCee"
    "lQUc8E>3xhK$pA6JY+-m%^SUm2q8OuDkbM#g!{3^zPiB}*xqJh4f|?aT|?98;JP(6_uQcpjRUg{gLTT{i2TzsL<y"
    "^}NKUI5?btp@JqU;FPqS%{rdsvGBZra>to0}PF@9Vl)7gZg(gjSWKSs}nN6A<xY{-23KrrsROpn{<dML=W<xkb3;"
    "OaLi<=RT>`0%Hr(`W&ejgK71mC}4v<#7P8TZnV+s62Nz7vS?1bAhvP*r&N?s*T&t>P{`-`AuLk4z%yM2L4BA;kBA"
    "LB{@}V!g$=9gh5a1tWn0Xv*=y9fim7$;-FS3zk_1FiE@5d1-(*}fMnAOsaQ!z>VyjV*HF?oR@A?yvM%MG$feBtV<"
    "T<7ew80RqV6h-v5Drsk@imAuhZVwY4F~HJJ<wXm{&#D+FZqpD|OV=RXR7(=8P`K%d~+mN9)-5ny;$D>6<P*Lypy3"
    "T9&>>LFuu<qyz!cC>5SMVV`NQ^*KDftIy%-SN1tfD4aANhgwWF^E14i;&s(m<!ha-KXAFo{q%OKDl-b=roVHBp~3"
    "Hi;80SknSr>qmi+M0T?`5aieeHk=ciGC0+(E;q_a7Scp9(bP;RY_jNX3+`=x8lR!dkD)wFHxRmzA1wG(dW{9GrKC"
    "Y??P3umfJjuAs(((hq!sWpAuW$XgG8d0D~*CVe80u~uGF|K03r$k0){r>#<wi+bF!wqF9e8ss4O4NeOex6r_$cWu"
    "cocUav4V92$nAWx0z)n|ZuQS#;qmkjv9rdH@3e~<g)V__p<hJ_NNEAM!=P@z7b{COKT}VIPR$Td@XZH_{r0lhf$*"
    "91hg~V{Tc5wA@CcTh_o00cE-?r1XoFM}M-K*67p{NeCJ-Dtz=r*vM<n4oB^6@-*4T6kGCG_uNVxVOwc}4Rfn%EI`"
    "64yiTfOx2M%DKcWcKn!Zvr6vT)wqR*RxJ^L*cLwVJAcW}XwD}!-aPjWaFQ^6U^2-tos_X0t(m0sZ>FI4UCgt*q?{"
    "hCE0|9O1<I*i!6E2`PpbiNJQz1wkuHmH<U~WtWl-o%M0cuQUp2{P{+ePnfTk5aZdgZ_xHRWq*6@@vb&xWg>=X>;V"
    "k18J8qH6oZ~SDA$UN?Q{>5?@{bA?1_%NyFFMK^3&ocbs`L@GxKdCw^Pd$Co|L*zs4Lo(Hzy0Kyz7xf@Xsc)a?H}a"
    "*m{>PK(TIk}BAdgY?X(!@-*2$9z3pcd%u(!fOrV}V!>Qn9q#Pb;@RoRK<Y!C4*smV3Vhubp#QU#ME3F-+ON)3VVA"
    "L0;H!bA%Y~W~Xz7tXo_yyZd{su%Qu50RuDwQ@c!}_WBi)_rsP#a?xGsmJx)G0~^!ja%f-hyot`zK$}=<YBK0oNHS"
    "fJ@ylh$!@;*yalfRogK6+a{x%V)Wrv=F^FVCGFH7=q87LHk+RibJaiu(q~*pApt2lU+C4<rE4V4E~2cW?yOf%uTi"
    "+<XSMR6sEQTKTFy&Trzl2h#zu|`-a_J;Ab{x}rLj$BuqLx<F-Qg79k0T9Z=6johgnx0n7V6aBlh$USV!iRDiAt&l"
    "fA&Sq{C?|V-cLw9uJ42#JOADsJ`oI#ew}qM)Lvb27gfLJ4C}V%#Gmk&^S)vW0sfUW-c*_3|BUl1?4);RpXM~5Wxe"
    "fGttx?Vd@MpmGPx!cxEIFEauMaZ#s8(_xN(IU7l8rxlksD^(>E$scPl^DTiBoJt@6^?3q<q)8E;_oBjUlgEt3fzD"
    "tI^BS2@n?>g!-sk){dIriM8pFDBV_Uykx5C6Vw9_HTB;o1KGJd2E!RZf(1dWfIblFt2mL0=iUGm<S?%ytlTt=-0W"
    "YDmPRU-u6qV~bky&5kyQBFZa&^tX+7nb(@i&4}I{z1)A@-#a>uk&La}I*MM?F#LzyZT(Fp+LlJplR9jg($lJr)n^"
    "`dyR;)Jk8GgGFIvCdaGNdN{gKA#Z17gSXVy1p9Vv`%HT%1n_=CmJda{4eiPdWw8~eLIyr1#UZQRz?rueO)%$ji=9"
    "CAh7-%fXb-uJgX_w`Ne4bFgvq!x+-?rF007%|Ff+GR0x+!AmH;Aatp@@dB{utz_wnZ(Kd@%}C_(*EA=>(~8*%JMj"
    "u(l_??KU#h=t8(->hd6p2W3xDB&cdReUDlz;LQ8@ZQPk+tuF8p)Xz{T`PNKtJ-yFZ*rwb*Oj#yJ8XSVx=N&KQJ$)"
    "<z-H^*oHgnQ}u^*<}DrtOD5Ru{R-Bb#o-JKl;LpR+rhH7oViutH-4b8b)Z_(wdl)wRBZ8O!S1IHCsQl-aBO%`9Jz"
    "uI^m#JhQ6vcfGmFk4_A4Dm;Fc(AKO<3H3L`f*YcCpvSTreS_b^8i0zkXmJy-Shn6!+=rE^Z48jyF=_Sw@S-&;x4g"
    "`8YO>w*?ZMS@3j6zDDQF&1&B(DTldoIM!%~|p|8$+L5bX`~NyuWN+0m>?V9_#!@S)eY>LZ-V2I#DywFUZP63bw9s"
    "g12LzM05^NQRxQ54Af~%!<B4CEnCAHX2uG5Q3iuj<V-5gKBKoZa$UrkDZE}9}v#w*(S4blWXFOD+WJ|ftVKKah5z"
    "MNJt9B$v$4@i^WayjJ|pyuuV^%Z*Q|*676su*l#Wpp5wo=C-|4Mvt#%CvQOh^nhUnd5xkh{gY)KoVd_Dmr;R|P(T"
    "~H3W$QsPNcsfY|L4bi`V<K#h=`;gS6GDT)||#El@5BUamyEf5YJGzPtSY1FMOCr@rWZj@g>5EV-?;$Az62>m&<wI"
    "c~V?y{#;_isElBq-C&YlETxkuWSloluSy#s4h3o`p{gG0*_c`|0jqB3MV?J;m`vpzKx2_BIUJ<X9tf|?Cj_2Ex!~"
    "PoyP9X3XK>dc&&)?7f<43IIGuS&^WN1K<Nm^?dzQYfSd29;Rrm2@ItGXU*WKB9_Vn4~4?9#E^Ra9;Lh*^Q4>Wo8o"
    "U2*Z%j(*ZXRbl^0Rb7A%^u(HTW7|@mCZS52hj)_8Rxb^@zRZ6%XEw;&E)8?p#^34C)?XG;j4P%vDptfh$%#(VNsP"
    "pwGa6iVb;9#QuA)Y$_OsA>sc|tv&*nqJ<|mt+0bv4*6(~xExoM%IKqhO8cm0!oS^C;-$qmZYvzw3owk?_Js&sH+6"
    "fVk5533(iE&Jtrp38Aq_H`fHY}?3`Dmb@RbrbylUH!gx0CCMy(OeX?pf1cxq_K*i))y;gK9>8@ev`P>2g-4ZeeeG"
    "!HeB-rOEk4d)?ZZu7cGw-=q5X^vPz^P`p1J#TM7)SS)Nk>Bt)5NRe@%cm7_27Dld}vv>QVNj#Bu4%pGTdaN(j$|}"
    "wgi}vF<xi4aDs2%rYV_zJyBD8MgEux@QmfR>Vwe3-cpe-)nX!JDcwEB3U?TI~_SO9Ve!<@|pR6RnGRSG!;zbuxPE"
    "veaT)TtP9qHV2Z$5?YJcVa-&O^+i7^DDvNUBV07QO(B*CeR78C$MU3RV$m=($d6@yS2f#2XKx)Rikltm&-+Q336i"
    "G5Lb7j#G`JUno8!DiK$gA*DHphsKjk2Ev@urCY-t+$*z8i9z5StV(UL^$tWXJMJ*~DaXt}U4DLq{e&Vm1U+KFn<l"
    "A$#KL{iVn;^Q?b5~4PjxAY3&>c2nJ2b~Ph~}h+WNsKbZB7Gv`r-<>__$nVkE=tS*cbYSk&8Cs!3D}&&^<>}^ms_;"
    "CYC&407~L|qfU!DG9lX%1$gf2dd`<#G^g{u4dA_KH<?@{zFCJw^K^^=t(e;REKPQHIA@(AgI#9cs;J!0q4l?s))&"
    "#ZLn_EoJ{BXz7UkR{DC2TA8DkKlAlVTXF^S~V0<}LW8A3nq;1>oL31?IZF7-n}y@iF56&A9q)Id4t^4I{DPi72S8"
    "kBQd6#0k>xDK5_BQ~XUk}$vs2tnZbQhfo@bTIX;%U~6zxE`aamVN~)gYwFi1F4$8lwLsK#L^$1sVl+_Mx35(N-+3"
    "!kxUf9MBz+tr?86j5*yG|6T)<j^=`fQ6%4jdb%7`>3D@E)#3j8VnKQY(!TzY;0HVfH*(HlpL=QhK&A1t}+ny4Mqr"
    "$=QdkSEj4AN?h7RdqD%8}?Rlrl?=+fAkVk{g4h=wiassI07kW5q3AE(VVmSfm`^5A?purp3P*iS@BsD!drm<81IA"
    "87>2(cQ);d(97dJo~l6r&5JNr1ka?M^F9ZG1y<zk>dE%bvxL$+6ZR{n$*~}7K+i*jcO{71djidR5xwPqjzO$@x?$"
    "vQ)TCk2?fHq9^0}rtS3uOpC!%@OwB;)8IvrxvefqNY0fEWZ(G!qOUv%%9htSm}?Wg%ixMo1<Tn_rv*~gj(NJ=r!o"
    "Wc({13%Cuz}*NCHljK5Hqd{-$+AX}q=<O$+YO>>_OkMNe`*aN*0MK|(pLzzOu+6bV^5bb6Xa-8ZhD%~8#`*2!VZz"
    "8WD06dk8@Y0RyqbILnY{mfp!waR@Scy5scoQ{`RKDFX1`0-yxP2c0fqmNNviGE9vZO%@+L`vFYJH$(Q3B364R&NA"
    "u_;r&vOsLnI}P>V=JsWEg0PUNX~qhr!QRR5TIR{vD(D19wNRGIN*_z4#K#)^nXVu7;qH(8#eRR>{@|EtbjFR`L!J"
    "xBFTnATQq~#K^p8)p0R>kDYRrTPprZi2EGegDoahP=GL$1Yyv>@u<<q408b{Tm-h`uDi}wQo?1Z&LTM5jco}|I|B"
    "^!zgZJgoL54@P&k*!Z?PR6Yv-*%^iACEzhT^KzF6KknYv|P7l<PyldT^K1)>{=19Yd15pQ+p_F#%^X12IFrhQ}_K"
    ">i7q+$N89EEjlRIhj#GG+7DPb!&LpHC&U%?^u&H1iGi3gwm8)wmaM3bt1lZL1LaH%RIyH0?PMxo?x!NWh1?3CgYz"
    "3j^Sm;LGSmxy6}^Q9PEhUzZIHh4-QZFPtKB~lLXz&yL<b|!Qt7Fkxp>b20$_SefRZm`=?3k?`iV)cEx(CXgicT=X"
    "y8w`+gr`1p57UJ02|hMI$bB;aeeW94P{+u6b&fI`vmLK27ug>*fnGJuKBzZ{pVZ<TSngL#9u>3J3u5bJ@bH&^<7<"
    "Q;SRrG#}_{OSA*rlRMk}ZB#$}iK*eTYgLK24{GP;$^dX~NPj>%s#8kZX`0j!Yr{K}iloJ_O*nEO&SEa8j8pCyWR-"
    "4VSuFERvOiZgMERXlu7rc2;w07BI>|fPPVeZBd*|(>cYKlMi!EG`8@x;O5-g^AHNbU6wYSh#0<U?8gy>86@8G@~&"
    "pyI^jWMHU3M?#R(K(P}#(5^tD^%l0>M=en181Jlva4*Up8v&}-Csx)AfVixox}*v$z(NN7F$RoEl}6GLIHqvx)l{"
    "U`(-vB;(wpCy9wU1f1zURODd8#q`D2jAfXG4_(UayI2wh?ew=zM<4hdMjXlYD6B=(P5w>b{zQJO~(y;XAHP@<9*w"
    "2$KWVx`o<2-3e{#}`DZ|!W$o{^}4)YhI$rKYVdO#7jrzGw|$@h^c8?H4qry`xvJ_V@i7n&mQEXzEIqygYywC;NwI"
    "x+pqB_a|@oBy^Ed2XsX8J+vifOD#Ho$z?Ioz>l3N$}DZ-=?V*ZW09i)^f%C=A=8)puXcZXeb(RGJ==YK^s{EOxrf"
    "VUw5e=qe`~oJyR_MQU;plN*o)As_bhcq24@@hwDPPzwc%)kd<QdjUQB2Bzvk1M3=5!)MtL5a!jqj~3eUgYj>iw^W"
    "OlY|XY<^dP1_emlyS(=UBLuhZFDO%E@!ax@B)-)_yHjcxAnN|DVDs<5>T#I7%N)g33%+A&%7hbtU35;cnI*Qm~xD"
    ">k47)p9S%7UBSUm}s>13XOwGK?JOq69lXRu!L(zD!gw{CUdM6wA9TzXf@<N2*JVuQIF~V=;;?SZWrpg(rL(5zSCR"
    "PsRr7pMmj;BKvc;OghqFBC#4Y!~n*Gt0+&Fz$#1N$X`8F9H2Op9QPIu*IZZYDTFt%--I<*OwZ=%M0t1=@G0ewOGO"
    "!R=G9q9W@Gk00*;LsnNQ&NmygQ=JI+^*xSRXH%53F<Y!a16mZHP=zupg_pBl$yD9QYMgjCV!nGujNBoNxoxt9%Lj"
    "r6p?4bUc$jpH{0~3P`4(?SmAS2Qc{A4XKYDK%J&H%KMZj3e#=U@PBvUR`M?)1}T$I)#_Yp2;U2;2p++f-ChNS=^Z"
    "g^XAQL(XcW0rcmo8!>+B!J*RV0;l-G~kbqFalHZP~$a@SLlahAmlQY8eXm@3OfpZA=w3un<&2k^<MDOs1d-s$*5!"
    ";5Ly-s>k@Gh5Y>@0yn)T7j9RW-MBe#gj#1`;;_Knv5DROA=J%u-A|*3B1TCs%0|PQ%Hem2%;-)R#afDAAEN^s>&}"
    "0l3-oP+c8lP|5=xO}aG{%vO<AMVri;XhO&-nbU8QVG~FE+x^t+Ihy@zr%fQY9(`^>hgT*Qjn7aZi8oZ0hyIKT=m@"
    ")hg=$uTrV0osKNtX1noOe+u!IQSG%d#M}1WA|>yv0!H5W|9?eU>|GqC;!z-a&K(($N8Lv)rsWHNYWGTg5N8opLjy"
    "HQeQX%_1id#aN)k6`N8n`eUYJ_F!mA^xwoV)v7bip$yk7lh*HTgxgc9EubWQuDgvB~Nc)8C)#AK2W3#un+-@r~nB"
    "<9ovThp*m=E4__$z?|>H1d9@*nLUHW-is4=Ri(!lNp>Gk=FQUi~(3iE+%EnW^Ao?LPlKOmlas14Yus*{_Fj{v$fn"
    "Od3AF1##xMiP$fRRuzJbgaU&9^b_10=qkK8Qx~V~yAz^ei)HM+|Pn5s@CIDM-g>|C*4$Igs(PtzoDG%6=yY4AFG0"
    "G8+HzL)8;BdquQ)%bV)Oa&%lewGbitY*9g@S#mtD<)LY*vuzquM}{<aIFwX^x8}`!(!i*ig|Qe=E_2ZZgh|i@Avi"
    ")?CrUqu#2Zd?j1X_qz2e0V9SAKiVGeN6(L#jYJSgSfSh2lcjSA{O)SzIDO)DII}{!`kDE0;G2^IgBvmU?BjL5$Tg"
    "CSIg(%GdSkku<Ze7d6SdWtQ%`FXjZ$5qzShLjUe>P+U2DaVN=l+HRIVpKFOC>{&<ehcG=dg<4Ft<q411=RVDj%lQ"
    "|URUw<G4V3YAz%K=hkzoL7HcT~e7dSkGQn7%EZG2BN{!AGzu+#~g;F`&-YWm=AF$X?8f)(X286r?hWR{K=J|Z*ff"
    "b*-srA=U-UO|3Q^9H2Pf@W4eA>vm{%l{>r%Fm$|xQ!_Hv8u1pkp@;<*IXDbDbv#<8(^-FG)V__lEv_uCByK+{ji^"
    "|4r_p|J>Tf90LP-jNk-)ta3P#i7|m&6nuqhXJDyINBvU#{fU<g1*$uC~-Xi3)lYg05KH5zc<P`(_{K%DU>iiahr&"
    "X_1+{jhDPtDs68Ji@;2{1f`#9l@D`@f2R6%Bs1X8V(XRFU~7`!m#aNnGk9s2@bZQbKajr`!x#8HSHfFhI6^=$Bo2"
    "C|Swz8_=Rl}zA?BNmzef>mkuzK$O+gDqSEiYba6Vil+>Q;gkVd28lNua&oHEk_UDEIKGVDw2Sn7(l58<w|_8^A2C"
    "IgGzB$Xo|X6pGE(d%US*s+0=XEPj@z$u1#KySO1d!vR83!jx4-FoNPWsDKA>%niwFL%%O9TDnu|IA_8e~*Y%Mkot"
    "VCO6kazOMQq4ByK2$)k!)U*n7<75D9f|KzMUeZJOr2b<qnLFQ1E!`S7jt-5Xa$kD-|ZTr#fjy1t842DW=p>_&fE6"
    "PDJC*T3s6$fl#Hrl#`BVy!zIZ1rprecEAh$b~Sp+L|{-r3#sPV(@}<hR3v|MA<t+gEE+TnS>&PMak)HEiPvTVhNg"
    "#f210Xa52Z$&FV}nHXv)14Ih{$Y&j=h6iYnEpXk9soQ6=ZXbr<rpe`s;&WD4+}9$%T3~T>ic-1E1{8jh$vL8kH1s"
    "OEeX*RyLYc&kbq)vu2v(hJQV9u2F!^f@a#VKd_QUpad<762Pfd!rEQ4z`7h`ny)&Qe=hOm$esdY#z18m6ciK@0^4"
    "q|e(%0Q)C<}L%c<B1ZhCmklc(swyZ+yiE9OJ^OoANh(=I`Ak_iH_Drp6<|gRZYc%gVHhQrZTk8H5Mv8#j&4*V;yZ"
    "{B++hCZ%}eY&2EhV8htDf#t0<W|0?>Ja5T=MXXX~2zOsBGf?2xyN-06SwiEV8$)_$uu0-^p26}EH@-Bj^^qH!l<+"
    "bB07|sZ;mG@ui9jit5k>Hs-t|>uq#@AW3jCr8$)?l-MJ0IZBFZWN9pZ<yOK!S+_k<9^^@FcV`F2Z2GK^}uZdI#{^"
    "@^(K(P<9Ib;wvihs(ARmd2>W-MAc)tke-XW-RBafXZNi8b<LA(r{j0fxJ{BB+wamuH5ODj>bpjyOI0vHvx?&HykG"
    "UachHfJwS>d$x1`e@YZ<zpy{Cd7<2`uMeE##^Pt~W}b%#wB+ripL@vkomS<P{S6}af$?`WXF7B!BkY)42uo;Fx_B"
    "oxoVMgI>E!9~DE#3{%`uFei4YusvT8&7NRX!rI0>E3>;5`mj0+wElcRKjg53?P`EV*r%t*=<ZcXrbq|0as!OITk!"
    "qk1zfM#gvR896>0#A_WpeIE?XLP#`j6Y~%gEd-^L9)jIEO!x84Ou=r3+JITS)h)<xNnqnVa5fmwIncec7^P*{vpq"
    "IVF6l^RDLSufHWLGExE?pO|_~wm=7CI8&J)$uz(fh1%!Z?9T3Nx0F|0PLJT#%lz<?$~(ZK(u6^9{iZ*Gcwf2&Fs2"
    "gf?{AYe1MS>cB@Yl`v+dhnNvzl{nDYO<mv^uz7Z4<l_lW2j;HUaF8I2E1W3@|3qVq{TNx#U%;E<&F=r~pX{HW9qs"
    "+v2LbvFt8{+5Z4W^ojc2)7MQR)obQ0jd-j;buPe^Ds8cIhQrp9CvSB=s<IVki{<N2A7S0xjcwyxbmQ}#6^l=Wi?h"
    "o=5a1l#0Yr~$r1I|}L$JQB8Dm9{$JbV$h?_%~Gq8qr`WUB`}-J@pu}ryWE*2X{RCU)32<<pWIUUbHx6SeMd~8Y#%"
    "zcOdYc7qxbJy|_Z<2Hn(1n-#0xhMfhLKp#gLIp;u8#G@R&oqGA6TF2N+S7{MV9q(M<q!u9lMQ$R@9A}YH%=XN8X("
    "YTpRr9EvQ#wqxHk)&EN*)w254LsLzD%M6E8Q3;M1x}HnIR}Dfu3@J$T<IyIet4lT-=(V15ouGP^K|(PAWBptcDfb"
    "AV0%IEi%Vd3ZQCQUaO*fdh}-B9jd|=3?-mQn|OgMT60p2jf@ql0k=6CUO}w57Ng=wfXsRLXci+QA)Nk?FuaB4(+d"
    "AtEXaf{Lod)=HT{>Q7-JsM7_XxeSUJCJgqg8uB2YW0@KuWQ6^|%v2qd40#B!A3MUKfLhSn#mDPDaP!NzLFc#ILga"
    "et_xuP!fN49xsA04<S}b22Jk=F5+W`pQKmaw!r*p;nm16OYCr3#7L%!%&yF--mrQvUfO}8?SNGQzud%2R7SknrSZ"
    "jP<X25@O+#tVDDUKzB~Z4f~6Re)3(~9$q3t`jIgA}*#o-WY8oCJ+I{9Em+sN&#Eg#{czq3%&pM(+fhcER2}|mHUt"
    "*mmf?!420lhfzrSmDk;>TOTc7=D;e?K}n#9m;>CP(mBha5p!2fo7R`Qpg<+rN7=YI!T3Dif^UCbcf*rmZoeIAA67"
    "XYXCN7M#Ih9<im}r<IWKHJi$Hc|%OiB?VhB;Ro1;l(#m`zi6+ZGnT>>D4HRgRyU1_?8~Ont>Qco(W33Ie!*mWc0`"
    "Fp;R`eP2{ga~pZr?u@Hu$pLwe)G_AOJ?yfH@2f6Nx|)hom+$^bZcti?aBi$>y9<)nYn5=r87aH1n#+_tT`G{_Z|d"
    "y^AvWOjyQqT?rhlzOyWf*R>8I9g37ONR6{PnWPy>AHpq+iXeo4(d?G`DlrsP5v&eFkEVtoqkvTn{O>DEI7_)W7a1"
    "gUc>^0D5z}Ukp?<~3pqAM;&}UjQA}JbjkK*c-Fz)L@QJC1)`W0DI4>X0nhx<HzxCn8ZFF@6Q^i@M>T#Mzuc<p9p-"
    ";xjzamu`#Y0@BniG^^@XsC351*Ti!-n|OM2`w`MQp^_B8{hphv8Im)q0`8Ro<8L3}HX?hH<tKJ?3p7HHE39A^0zM"
    "IEFM5XTi@O>Lg%wlW|6Y=thvCtYT(^O+0YdJYY=<9D0Z0^5h_6_DnxqXCFA^e52#Zw2MYzDOaKUj+!x!g&qGK3wb"
    "U@PC{bW>=O}T?zL0o*~nfI9n^?NjoW0H560Od54eat9NPq<g%MoN$jpG0O`9DJznG5kw3rXTJyeam1K+TUn%EYS&"
    "(g~+I|<1sBts!16()Ur*0<i)F?a5hV%jFEE!s4Nu){ClA$SDP5XWX7el2YBx)J(@%Fq5{b`$Fh@D{mE)xZpI;C6M"
    "4qwQ3IH5HWAPS{6m>wLJOuCA4xx}j;*OQoX7Tlgn*Bw^?po+Qxt5Zli5hC!|jxZDa%_^Cj$*l*fSa4p(QS=|JcmJ"
    "s|1GcQ}*kRf%!nq_i)1BZ3WkSvz<aFH0aZ>L(tWoV)`rd&%)%#~uP#oQK!u93)=+3bC?nj3)Gm?~)|Sr?nR=14Gw"
    "Iq!s`@=QOUD_t8sqCRG`{xq<rDH@3NY~xhNSurWU=w6js00GVLQH#*Dm7VCJh*Iyzq+*}_0%F1-Vq`>MIn1j;QqZ"
    "2FhLsmbxy?_4+vphY9sBkqyOBxd%+Dy93ItAMV^kU7%Xp6=G_EXAHZo{3$dplvTRexm-%<wZscM@w6mB%q@^W**="
    "er93PMEf{pcqFLJp!?a?aaJ=z0%CAG3l6fdF=WrI>XE4A#emdW0YWpGKrGsVdLq*@e8~b*~_#VYApQQlC8BS5KLB"
    "UXT25OU#Xgi`qH`n?;sf-&H~>CX$ET-OQVsq;lEZTnRpb~K%i%ki)kzrk?o5(RTy$#qd$AjXo-Jzq$*i$PpUePc&"
    "JrlW6x&K>GDyP_lbQ|(ntB7cM2%_T8`H6{5p<R*%fAMa!)_%s5jp&y6i{{5CW;6TjH4Hq53Qx-fH5e4Zevj|5@lt"
    ")IfR=%}R7<F`^OPb|%FYw21sGU3%q&tlBjx489C#wtxsfR(K0#x4`#UE1azr&5l=><6_`qm|o{&&}=ySsI&pi>@W"
    "yD&FYLU^;qCVhjQhMgn<i@%9y5&0Yh0Lc+Mt8Jx~WTQsEs>B9o<7a`k+SmI{t%${TJJJfN0zHSQ@5&WfTz1@5XDX"
    ")iv(f0wzWNZP4{lv?hX(x$fA^0}lGB}a$*UUrzu!uuE<py~|pM9hjF=7_qOb66IURed(_(_}tdBJL;4cBzdkw|gM"
    "9QD5vgfv1DCwJdq})Beu~hsnX4H~TLScF*?T;ikQ!mpU0_#@kYM$hVd)sFb=wah7-y=k8RI@8aaG8;8<7Tz~;hsV"
    "C(-bP)NOaIk|YqJom$=2<CDF>za=Adl~&U&(Qb_*fcB^O5Y^vt;L)Oy#4kT4AERbQumD!U9Kn$Kkhu@sm<WjEl4V"
    "j&N~UA}`;0nGrZNPJA#izlLoYpB}H^3*!oOyz|Qfg%}4-A;T+|;2kkecamqA2oHz)azT08L(eH`5LVESHAI0Di~d"
    "w~9KVRcT`~q4!kMzwuQo{?^e!>2FhQ4;kgv`Jo$MaIB(j?iLj8&lsKysWj(FD@`@r%H6kIByVo|}#6!DWOi?SBUO"
    "LR>Fj~+R4g+2>6uEIu&HNYHdOe(^<M`#wnXr|k+aK~Q~YfXD~aB_N<y8c(<^J5aY%8HxR1krik*IvZ;KdAHY2ZZz"
    "c4evhex{mMHFnu2pe|@^xMxv9rz?$<Z^sjFsnH7;8RuS*kY^iFljtd-lxc~F+*}?DoU5(QP;iarbPyr{qT-_KX;C"
    "m3^k4W=qzzwL6k;eaR(Jon^<R5R)P#Z0*>cWZb%J^$5GF5_$Jbh~2GZNEY-5Bd0tf&=T=OnFrIEZ@f2FJDj@!R-M"
    "_W(^mf0fDFZ03p^?NaZ!O6CL|R?m&rM_3c}Xg>UFN3$*;!Y2fI8Gp!!58Fu$KtsH}zBxYLtYtGvH>hT@VG?~TcJ8"
    "jq{iZioZ7HX_Z1g*A?mKU2#kcpLsN)S@h~K(MzB?yw-1#_#CV%g4t$24f%oVA=>b8=)yMb`Z>64UrWpO1)><oJ&D"
    "U{r%lxF3L6fR;0t0yzLo6uoK5__=>J6y~#Fq%@o1UQXF;a9(!K}7}Eu%Fd*3x<*}IEhZu)QmiB90p%}7m5hV*I)x"
    "pQ$0{AquEmmcACUtdhFWQVDQ}IMm3@kj5wYmQAv}it@9ZQ#qo9<T!N8OWHl{KvU;Z|XQ0kZvSq7w5_ddGcgObe2;"
    "il-=`zQQ+!)Mvr;JTa)*>dk;~#zz6<R*raMrJnKES-TluZcKvKhS4AFgbDfueO^yq5~}wy$FGZEy$eYSVfPm@gO9"
    "TX^?wPR_|>Ym(GF-l50ep_!nZi%$;s8C+H6eTyt)i>rCIKuTz#g_}~km}kpttl=O(9K%P`ed7rA2`~34jl`GNlr8"
    "WRJ-DwU=R0f8Lpru#3Lm;QlghWj1$js`R}9k8d3ScD#@I3af`ZXij|$Dz&(A20+*>Cb0^^o$fSGP>EoL*?u2^)dh"
    "jhb4-BGq0FX<Ds!78VU#<#kJ@i<+<7yPMg4H0aE0<R_{kEEvLn`S3T5A+~E<;HE*{B9&e(*%ZJ+ELJda7rZ+SK0t"
    "2pozN>_-on0**f7z_>~NpOCafJx-I}M3R@#L1EKdhl79fN><lm4x1O4A9`^QE@3Wt37kyMt`L>sA$1OWe?y?XguR"
    "a|GURTMitIs4gQ~8Xn<{`WW1EUlgm%o%x_=A}${ooA)el|0;v!sKL#ck1W2dseBL)nj65q7_v-jvIHvj3@Aw&+*^"
    "Bk17jpYL=paIbF0+objo0gs|3+T(F2IUyfq>~qJ+pz?Y)XD^OGR3{_sdZ;yw&{x3BD=eYIs5Npd@5%f?fkzG(uQ;"
    "=?JP~1M%!k#g&;l)RzR7sUKkQC#(&RKBkIXpcQVmC2yuzG4DK4zWtl45SrrKc-lk{{r+cmWjH+fiVR~cq*oEoR{%"
    "O3nwTdoFgW-K)*wV!NB8I*#61jn;+F}V3kb$1$S>DY46r??6BaEr7YTBx|=!o9-@C%U(;^4@lujPu^kHsKbwBZV-"
    "U8cS)9VD(z~8F;j0{<m$#52@PO=NB-;G?hPapZH>&)NdA0upjxeP9%mm=E1<42j|x}WicQCCOjr?_zIrMRZ!8=DL"
    "LYj_hmjV@c16tQx>B$#q1L8u#$;>CmWGHVoFe~APK_U)$+Rc<X+^YJjuRu>^C4Lj}GGxaeOWLR`G%`eQVxow>P1#"
    "^fY9;P*N^qft&7LByt!kjkxafdZJT>$)6D(Bt~nh!C1dZ71*EzW<{#xoY$bP{dU`Z^@*_SW4L`|FE?)4em<WKuKl"
    "G2K@k>f3aj?N&a<u2_SO#<pPxRt{i~HOn3>=4DF>IVn~XpNcP6sLiBMW~znG5TdYksBN~6jl20+R2zFL-XVWFLQo"
    "!n-Tt$(2@#<*3laDluh?7F%6@w4fuxPnUv1v+N2tq`VuD8(l<OW5bdy6*G=@#(x@Tl#qWURkz}5|c2+j9mxWzbt4"
    "Mt5UZF86aQw*lfsS>0s^y8SL84c~RBUMm2Ltb&K>Q185+e3JYiBhLi8<Z!fuFjo?Pzv>vtB&eq6n9L2JI(TuuHwk"
    "*W+X8<A)Vz5U_WI)}60|m{cYo@dDxaXX&8H}3H_^~wVy0IQ(D0+z=b?1H%iHm@3!+1ApjZ@Jf3tjAxr=uB~qj>|U"
    "ls9msT=`BGjAD{TGtD6|1W4VlWjUkKYe9u$)4N~c*kD|CMyv6dptYc1m!IIim52T3s3J0rpjr^Q-Mso*$5@5bNxs"
    "Zb$}v+}cV_K=_o>$7@rc?s&E~MRmNjI5`#^>5$J(-nl=`zNY|UvrLG4?QIKd;D*d?hz=NC<l?P6nF{nYS)ZUb!2f"
    "7dKRbz5WM$^yX%dApiY0+a(ZnJ;hZ58++H6RJhH@56LhcCMV>0F85zd_BhCSjeJ6PL-gHbVdTCi26cO$p>dZNX##"
    "ly&p1)y8)UVog@c8A0C|uR(aEPWR2eNe=&T(;BE>SQVn`py_aqCn=+>JJ0%kqh}a2Rdxy9VVV&b`w#^xMYsbCe!i"
    "Ah$l4gbo(n&&TQL=~^nh6G?_^IH+JB;PLP-6LnX2*P}f*j#J`S2!<Vu+HI`c50SvPp32?fAR8?ygW<Efs)z;4`R}"
    "4xE$tJW+y}j{NLj5HJZefp(Blz@j6xC|d5Ee7JT8Lyh~({L`{Ui|Kd0IU6&WkU|r4TibBR!kolmxPmq_$dQk9ImK"
    "4omt7T$c~mPQZ=h~)Y01xzQYXuigCE5RO1ghHumz;*&tS^>Y6I?@v=SY;J2iN7CXG@28T^h8*3?8LvD|>9+SZ`Rq"
    "OyQSVuyYzsRUa-4TdeP^<e$sQnohQOwQI}IT@ewl_yAZ5}tId3fX!x8n!YlMkBlqY`Ue{qct}HS@M$VTSa_vsAUA"
    "15m%*mST%}Oz0b}>x|ROOQzX_sDy4yIrg(@pn}w2Y`nu%wrVYeUx1BH8gu|!`m;OtuQl_*G#`d>l=Uzgn(5@;x*3"
    "dsDKB^cRjNC&AK$CgBc({T5c?U8!2#Wp?o1WT4-VL-3olr_{CI4EWD_wp^>_fDrY_woHjv5npg}axCRRaWqNw|&#"
    "1x1*e1!)vi<1<T%@3t$Vr^G>F6Qc5GWu%27LaGZ^@FB&wb6e>t{@!!1zl%_v+gzoK_gC#I-j00ac|L7uZg3Lx4gy"
    "l_?(OX#6T%42{bc{Y1CPf~x9jBW=)px1!_>vn%^(Bt#A!9wcRX9EOH;cjOj1}|*FmVe3B7==M#@!hLXQDRX0pS&1"
    "H+)<zaGw56Z3Z41!tOL#GG#eI5cHCU2_ur-p1J!WHUH)1QXK@gVr`0{9Ps>fjZV~sUU5YHjcZn8;hl5{h&+7&&Qx"
    "kBY<*|=u5}txZbrHh6@@HT}ElG2O8$dSTkzfY=XT;^HP(#KQXbTbBt|;t(cj)+%E#0JEB+G6HIPTdwADP$hGs*l-"
    "wq_77))ZM)d(5>-Q9`7um-?Zq{ycd9%!Q6Enu1D{0?d1pohB!d4YnW{CIYljl$RJI|gW`xF0br@YRdeEZzC>Rjia"
    "1XGi4At6QM?WzuEYXY+1n&OFrpFR{h0<q#-nAkR!1HbBdj*x)gG7P#Iq=88wb#<-I`)DJZO-&vzHVkxD9788$U|*"
    "=dR^PO_3QP1eWnHv9cjDWfEwvW~VddOklhXC>6Sf0uQ7pVlgmZ*GYZA_jFtU!mI+vS-H=$5pP*d4hCp0s*hYc+@X"
    "2+_UXHu6Dq{<=SY8>_1es)4@h`(Xd2a6^~uF%-D)W5`ZVpcu8wR9lDi8vAw&{aU&M)V3Q$b<j1u)-5P>fC?W;_37"
    "4?KN_-#C5n49f~*0jud?}X17$uWUJ}KZ5>jHz)U1%&ex-VQ{j9){!gg-_v(*XRV(^jz33M9zS41?yP}KIok2wjkm"
    "oZd<l$3X)WN;#It7R`WvjjRhO?`Q0zJ*a-qLE8rx!LajTTf^SLuzvNK~YezBc3M0=@BtnzK)&Tuq#nS@_~Bd@?!i"
    "Ft10E-)%;~oMnXzzdWo@GDa*8!$Q}K2lwz2RD|c)=J5w||ASPoT!<9strRRIu7w(dmG$_A(6}t{XBIwP5%O;GBXo"
    "Kj1Y^h^G)-9Z$M<oS=s;(!uDQU`Yj`t0xye#*pp6-=oKQoy6zZXdj?s-x#o!2YM5rg_8oCDBVgoups`VD#`rDgti"
    "xDw0>6c!v%pshz!F4orLbHhNrJ8^x`1oX!3@wLxSH-eaqftq&*SVMu#;c(t9n?DK>g^fsoSgiyNOcFeIGkL9I%Rp"
    "bSYp|mvd7nCna?wpfO{=8MACKIwb8on|BS)k=`qL}*o;hu4BC-zZPhYs;~Bc;u*yJ$QpUwmlbp48bolDv<PE{2dh"
    "9NmNAz<4H3$<Y8@9HVkgoSB%sZl;C-o%^3}qxK+Huyj0uazTBaQ3D*^KBaRFD9dp^{Vr{Xs*Rt;+EYr8$jPB_e|<"
    "*r!MH`yOjoTMkF+u$n|9eB0b19>@I_!zlS8=f&;UgY1de-JG^2fG*nDZzzhpJrj;Df4@T!6FKS@OKY4;Kw{{R=2%"
    "j=NioR`p><Zpu!H}68xV_rINW`+e|o&Tx8MKm<h3I>Z|SWG>>&AOjRHnG?7d5T+3kWO#zZpqyQgJNDnyFqUgyhH("
    "#BFg2Px|csZy?*u=@6a>qSo<hBJGZK0M&KThIyfh3n_nL+}tDeE9O{aG(A--9J2g7^@XtHOjba5Koufn9PyK9r>x"
    "IkS%qZJ5gCWqZdw4gNq-|HBZFfRd530YF?9`$7S}C6@!mOD&@w*5Bv+=+~wWMsdgHP5EQZg-#;_D65x#~(!8sB4^"
    "Q@gKl*k5<wMwCA7=0KVQ(kR=4hFP)r18r#=Y$gQlonfiP4TsvnDgTA^}z1AvJ=Kxm$ZlR2B%X)9GL~v;e7+k7%Oi"
    "?<+P$S2DKbcU;Ce^H348?kQv}cpNd^&diNX8!6*A7!JeIU{eab7oVg5*REgldNcuNeH$9B;U~dDHna$=kX83Pa3P"
    "rX+bQuQ>95`4KU!_iTG{X#C2&Q<b$+O-whbLNvpl2KRe}lbfYoqU@uXI(5R~{{(rT6;<Hr1myhl2F({qGP-0~z@f"
    "-ZLEKsAmc;*J)xb42ELV>a;JKx3MC?cY<GEd6ySZo8J#wnR#ZXhj&7YnN^<LdPaV{3`scHayTtv8eE!KaS@_Mc#("
    "?Y+<}BOK5r^unXPqEaM$Y8I3I2q4&qL&eU9>UjZ74b~TS4{;ZRvP?qn7uq+S%s>x6!c>dL-O2NmrML4P3`!v;#N#"
    "W#=w!+w&pPNCL?k(JxnlYq%cn7Aojq^D=OpyD$KG-{JA>zZ!qeSS6)U1m2I#j+ldfO=(Tjf*!iOep;j`L+&?SmkP"
    "<&g>IRaeH@ahjy2SLe%!CLoSQMrGkA*%I8|ynOj|4Id@7&XrjAb`SUVU%y6~b%5a~<1z8kt9&ZP`JVmro1F+PKp@"
    "K(y~E#LzfRe7TG9vIXMCF-940Lgj(1=F6ZEE&-wqEC4u6KfetLbhhvIVE!$|$I8ETaboTRFEy@$UY{(5-ykHd$N4"
    "Z#C7Iy4RSc>nMv5Ajs~y2}t7udn*MYHeV6wO__|Lui?Tt*-cnY%$3>7}zbWjW94)q2oZRGP@}a1|k;)AV)4|&>`r"
    "FRwpJ86KQoFx6?f=>CJ#?SVx>RJL(G^+!<+%*0)T4PmUD;T`@1OX&B{d;R5A}+Fn+Tb8Ku6Aa9#ZOZD(i@|udtDJ"
    "#Y|2=W75MOwyL?_idV^Ky{4rmOL|7_}}7_|NpJuU3bL58G;is3bSg8EleE<iY>wRjm%;W6l!aR@^1Vo8X3Nw9!MN"
    "R9<iN=lvk7MyS!4ATLIInYHQ8+j7I~M&OoRkIm!9$@kBnnIuee$;x_Av{`g)Gz1l<-3gI6t#o%YLunIZ_G6Ig+1q"
    "ZnzXZ9cUev<fV}=6rckzcv*h=g`@#jB8ROUU5i~tGHr#q*|1S5)kQzuT4HpR^?#jJ`2&vDAjO<z4hAYXesw#U#lF"
    "ts}2f|jg9<4Ac)JHhItiASQl$r4p+{0n42BE~fT1hiSfg`ItB=?7gES3zul;RRF;T*uc-z(qBPJrDvM-**_}cac_"
    "A)ccDDaXHJ671=;xh)0h|)JN)tNTf;g<eWvQnU;qo<&R@{Sq(+hS?K^e3pH(4rtvv$W<|^iM3<}4PNH#<8=7H*Se"
    "dSptij%EdyfWG)<uRkQauWwCdRO?AHf}m(X)awy=$J=jLUTnn~5-6LIt~GnYm5BK;F_=Gdy`5YhYMa;fh9gj1Qv3"
    "`LuyTH3Zoh?<yO9K&fd6TKi<?0m2+jb1He8AsF6lG=h5<<hemU%@zgX#OI4aQHto5l*Ok+yGkZG=C>8)<nG`v>{E"
    "rw<Ehj(zn$%+=1V@cu!zfS11zixSjuWSn|R<=|G*&jmpRS_ZveKb_HDZq3lt&euIUjM9(a3zi86s31%6MJf_;jjg"
    "=N5_Adl2`2y0zTy+IOeqY?F>pDPUIT=cKuuB_>%7#f@wh7zo6SI)-XyAJe=M&OtUY!?iVQ#>BLU~ypZ{Uw6R-xEA"
    "6`q*+d8=J|T!H1Oo-Q#c&C)O7hN;hE(k*~V68KOnrPc%J`y!S4vc!BjIhlJWgD6~jO)YUFgC(Xx*$_#Smbuoks(W"
    "`(^#o+2-3UtfC;F@v<rflWblkF$Z)8vOV`4)EdzcKl~2YX$ko3tb#Iz^z1<S|wYv~>c5R-MWFVX<K3@&4?+n_|Ff"
    "K6Dc-t(!_!fO0_~Q+<3_*=w1OJ+IF+Uy@S-H(uF;)%tfnMWO$*6Ws)CGDB(Y?SotQQ8-y7LS$n?yb7K^IQ)Gd1cQ"
    "@*_Ro%f-9HSTCG8HF6zv56K?;*~;+xEK8a-<EH(I5UUv~aAE2aeOg|@T~UJLnck3Yc1LkITl+Xw8;g72INY$=Y>Y"
    "8!|g4N2;`JJO-+xqroAqgHjy9ABRe1s7-x;>NfnF%ho`_zM1Rhu4+#1zQv3;;=XVR{BF=v!>qsYA_}l8FR0tppky"
    "pa>t2XvhcnWJW9Sf(+{6tqe(mUknIc1mo3|ehJUs@Q>tEW9v99WK*r+_7UUsh<1IJ-8ZqTkzb7iUf;?c{XXM<n_k"
    "7!+gZ6NHQ~au(Iu#u16VF-p;9z@1oX+|qOw8};?NNQdv%9^4*zd>L<Z_rLeG1U(`Zra}NCv#d*LgNxUeoIPi7DTE"
    "2T8lcQQ@TZqN?P=z=Uv8_AF`M8(l#G>E&CST}nohF<pNY5rt>Sp#VNY%!W7XuX#S_q6P@oa)qlu&9T0yJ92miq^b"
    "o!u}76ofYTS_ai%c&VRIv#@iPY+g6tQLsy>I?l^^+CUVt&*a35VSljZ8NY#JC%Wr7h(3$4rtv$=%Dova<8R^!EB>"
    "S>9FFuZBX<O3`?*GsA$=m|fDUr128)7teF_~J=RkAhv+)L%H0Nnxazrj6`$Q{8eD*N!pV41Ul}DhlGBiSLwkdSRB"
    "hvI3gDqTRL8>-a>ygP!TszYkPi>jlzSgcfGk<89Z^QvUCyyYoo^L*$(B8<qAQLCw5DyVEn4ESHw+>};=p_R>6?d}"
    "GDomUi*Pn45@45k`$8KSo!5voT36n~gi%XrMY>79CAzhN3KL5|!<d6oAcyaXyO@H4w^?hCYU91nISQI>5lUJBAqM"
    "4GlXu@=`b9W$$`6Kykt`Y=ipZXT{TTEykdU_e7!ZQb4WO$QkA~4im;2>{A*5`cQi!T}N)Pw+}w}<X}K0E-4FdAFl"
    "><BmCzI4J<LznDz1AO&4~b6h1F1V8lF!0M_)D?tQj@(?xyj-0;26_U~KI0*zh4B}B0BwhNFI^vl~I*k6umd{puyD"
    "8IASVvzT~lf#XDBbSih+m$httH~roGWR}X>I2Du@Ek=NxB{T*AQ%<<ei$|bgHTwubh&g`Fwe-N-Q=_R`?g{pvP)1"
    "c9%|(U=tCrp4x1BhNnIryG>3K1v3M7!{iPLauOVpl9j+FNK!t;BAu8!-uC(ZUb-VTBXFk&2x3RjXe5|^9r}66OR)"
    "^&&^9iA{_CD*EAUAgO+rCUPR+i5KAqRlL1N0cReDe!0+r~B$(n4L^$ZI%Q4^b^#V&>Qsi#-ikdSP9a0x!g5LKdeq"
    "S(RGF8bs4E++D+~d~siJTn599<J{IF-2Rl6`5PY09swY}EYP$=sExG-lRE;$vU!bYEDyX<O9;LC41Q7VeLQZ^_xr"
    "#S<|y!*7t3c1EZiqM9BS@o1a10uy0?3_`}*i-%NQ`q``6#Le&0Pjc>Q|U6X)>5lQj7r{{7Z)U1`7f%hBG^>)o?`l"
    "{1Ck<HsF%2-Nkb+u=i}XD7S=_-X&-<e&W1>2i^Mgo}oiu^m$HpM3W%KIH*>I#`}9vp6cPiT{Ka3Z`XO3`8F2AGka"
    "4r^|wX$9+_R0EpMVBZ3`TzF!4Uu*FoNU}4}6Fzil%O8Rn<jh41=|5e((M$QCKr%Q!8?OUu<G~kpu==gYPdDvEDzj"
    "3Mr@R>PabUC<Lk{P*u7ddb~@&&WgkZ>xbMuFl|Q_5z(B)EIec5|0hG83YMw_VCzV--epFwe#mp;D5@Bb&1O6<vQ?"
    "p14H=h*j>oQ<0I4Dt$Cn<<h!BG@Q2&NKuE6OqpnbATAq+j)B6I8i23A-QNCsg{r=Y#EvB;_E0Oto7Xrb5AwaPs@i"
    "sAB^c_ux}Ti2@F;%U{wje_!TdL@7FVCX@DO|0lER-4f%U;5-;PeY<<HEm;U<rhJumRh8Eyght+^&D2)>U@DW3`=p"
    "1n%C7Q!0A5jeV~)$h8jnI54E7Cu;mhwRGL3Ib~%wkL$a+7OaWs=IhmBn1`cDNHn6VFTgtPj?5JfRR>XTnUQ36wgQ"
    "33JiGGG2!M9|7z-gJ#5!NW+pRnbH%%F=o%|3w0kroa^lz2R_eGBUAxD}Cr7_y?3js@N=>L$`p`rL!*ElskqxhBO)"
    "=hm0IyQWWLkgkJ$#5KgNE0`s6Pc>p>Jx=uhx#z!d&&o^CFt97L=gSAS7U{a5EBq0qimiC#b%BJ2lB&&Mv?wwvw=N"
    "8Ytu1R?%v_W}-`8)Q(=oXf!NO?HMv)+E{3$8;BqEcqf{QEcJ&_c&17vxG^Zdm5=de$PIEsqDf%3T1s&<_&_mGsLp"
    "}hE<O6d4R<fMBn~3lLV7U=(XL5LXW~}!yKhOP<ky_5O_^R}$P-~5bXgSUt0-M<QL<{RWIA6hX2{|1l@y3>QmM$qM"
    "8WLG^T_>wQR$NrCI&Y=yq|wM_QrMVZm%JVoU*rk@w0hvvGCWUfSgt<$oqEqzAKNn+cw);65M+|Lz#BU1&sXtORsT"
    "qVX3t;P}<4Z!adFFqv2@IMvi9DHiwU6wqXlA2k=yohCk$qSCxw@9C(}jKUcG*3;P&|?I?is%pBM5tw)cjzsi!@c="
    "@hRgGx&$^0?ACdBnrrB=yFRrg(X>`|2#djaWVRZfakdY*iz#b$>(-%!zS^q5o(w!$RJQ&5Uf!HxTf$+r=b^jto>>"
    "VII}YL}y0^zt~}=Ra5r{)3RKlGO5?I+52QQPcj`BaZCjRfLY;k7$pZzY(lG29lgJ~f*%dy%e{tA$3pL^$_)0%`v("
    ")?(?iOgp}L%&nJx#JAS^jM$sxj~nCkImbzVO234WwkMkaOP8KA5P+M+F)s*BrkZsQMRQKgMSzMW#qJGkwL!AuIaX"
    "tl(#Q%cP>F}_@A2!zyM3nC%VQktNZ16H(L4F>SgXf+n%X|QDZLL#-N*<xYR)d^c{0rZ{Zu){r-nw<Q8P&JydEdYm"
    "n8m*=S!Q?qt4Kk39WL7wRouJ<UyK;UNa8a@&QRFAy#OIoiFM1|V*sT>p_}}NGy}>^hC~JMbwI0pLD0~4%zQs#$hT"
    "#u{e4uVR%Hc<n(BN!SGf2S<;Ag!3T{R#qi>U{a>Lvd<SF(ML`15>J0zX|$s5W~afU&N9Jz=|Gu{PV+Il3m`TW={z"
    "y|rc3S+9iWsHMD}VGUF7JA0o5KX}(i`wmE}qG;E^o9<x1IvS?&7R0}I6h$^f5!R8#@q?wM;XKe&O+QSE#$JS4e~W"
    "bClEjKrvXe-nx)bUSpL=&tjkCe~GEuBO=LUn9Bmd{I1?e9r=(QTG(7}O0g2GN@h7_Mc-wWD|vfB=^#&LE-mFL|4P"
    "~H(Hr98>rpr^n7_QGCLDC>{LsFSlBNa`b~&E0WLV|VAU%y`yY;nuYHb2wz-7Zst%B6gs;pM{GDFMZzGHRr$>536s"
    "ZR0PVKZt=S`*Uw$Eadw%H{mggj4z3LJ8)W66D7<n@W?&um{bS(MR`Qwpx$P#Obr`qgSx=L<tL^RV>1Ww)k00aazw"
    "PMf{|i5Mx1S7eap5~=$>}5tewZ}HnfA>oM@gF}rASb1hiJ~&&akJh4uktOqu`HqgOCOBTyfOBXShCMzs%|NY)%Gu"
    "?=sv)37VuHbWh2RN2O#3Io|du-*{3-dp3eeO|MpXD<F#H0@({C<EgxPfLJITeg`{W0vk}O>9SK+g-1Rd<uxon-YW"
    "!khoNRcDMjj&@~TiQu(mG}%f>S-b%gpnYrLc46Z(?qH#$71ApCN6cFfc2xbt?^EX}oWQNr`_$dPGkqztEpxQXk2e"
    "&jYSCOK&8Qbx>uOZKS`Kg?Il{&MynHaK|^)Ju*Mjd3Y;T}E+)B`bYoFsfgPG{6M2lJUIpzgU(Oy>c_A7;YaoGi;f"
    "WZx;<Nt%Ci*X+r+sd<w!J#)BG(i}b~bbN5pgJVyq#ZH-ws4aplLunqU(_OEP~KU~S<oCepuh`yC2j*CiqmixkPwB"
    "WU&sN^!W)K%A_fC3ZzhLTJ#P>vF<vs?B)-}+fC60GCCp8|Gv!W4CK_d0S&H-su+O+I)cDs8C1iKOMqE>)wMnO)rV"
    "0COJ6eYo|xxUJ1y7iu&qB_;sN(x~3AVk|MP{UL2?@2cE!GhqGGRr_Fl+lb+CqmI?0STl*+^#B(JV8$Tp*)vGnq8-"
    "Pl2u+_`5!2^4)q#SbwFw=o_zPSLGKPy6Jz(^2d(BylIt1t_%*U(J)VB1EKNL_?vUp^iNiS;5;L}->4LPMZ8#|&Js"
    "mPs%98hXX+-v5XLRzU=65L-{t(bHU_k>ru6nD9fKVd-UV73|$>6!TgL4G(fn4W*3mo0ro%5#qLZQ1d}R3oxsIQUS"
    "jJcC#Se;e1w#VwgU;8w=39i;<*p+6FmvT4>s^6StPE-JF3JNy%Gwdjei0@*}F82$EeW=#`tGYxZ|(1wvLCPB@OW;"
    "r8E;)FzX5D_rAC80=+E)`5vh(nrz7sdkTVhgkJS@!rtDm~YgcOJ~fW46+R@OWurT#%XLU~MK*Mp9pHix!ut@UIP?"
    "xkJ6@z#QZmWYW+}84r1}gLCuA@=sWdFz~mfL0)%Vq7(kxCy`M4evP5}C8smt_kVmv=uIPpI40@q2L-WxOz2i$0RX"
    "Ak4XE1LLjk{1A|PCh&Z%$3RIvYQI}iO|%y41p014|GaK&Z}m!TMc8N@X*I~2C&U9-*4&o5GHjQT4FT6fJ;^-d8!s"
    "ILkNjiCzfAFuPNz(t#gL5s>pnCC9)Cawdy9A;q(6q&xi1Up{3U^kiop!NQ4-LDFI#*t5yrqn5h0eZYs?2m3DascC"
    "PM4bn2M@!fWtj;C8?7a^=Z<M)+ldftS!%_~qtSsx|2&!mV;<BJ7B&ka?!S1NKitrkhHfnvjKm(IxDw&t|MS7uv(-"
    "COKMGP&Nc4N4O1w5o79H^2J?c1q%v}EPnpq+vAUx(q;6Prm`B?D`0Ho-M{(DDr!bpkQtLP~r?fZt66Q}QzeVNGCD"
    "1Q_iTLbZop?V1((OL8!!+7SaPXq#PKE%Gb!dngtPC|%hHDY&X=@um<u-cp@-&((dbj>bEB$R%{6>4*y?TGH}2NR7"
    "AL!(%~dTgD|dpd>m(!Z0~Ngg;1($lC|E)q~W;0U}(W{@hf~jFJa~^$80xI0jDL18+Z8@S``}7G!k=HeHMh!Q{;_*"
    "m=qDr-OWfe_=5TGKe5^8L;U;ITn2=DV8r{{bH6?20|dtB42RD%YO+J^c8l0d6SIzLP4X-$4(St&e>HGGWB2QqR@l"
    "UBhdQ*30pRpdX|$7_gWa+qqz)<dvzBA*oR{1KEYv%&(=ReoFIU=kDy|4R+Z90K^hHS7yoRBA-PtFH*9n&#{_<&xX"
    "*n1&Q{C8Y!aHn<8e=+P56j<Ao%`=#gG*=`oQZ0U!?K{--Sjm>x6Eha9Q2Uw;Ouc{Xy)~{g;hRje&0RSnfo3X!18-"
    "U+~!*K|YR*fw$IX+{znN)GVMr5TQ35dpOy%@dwPNE3KsfakEMjz1pcU3mjb%>+obh<7H8JUkV9;t-zX$ylsnCy~%"
    "Q5qP{G^3<&j0PJbLab%U)8WPUK4Fv0`FbvwzY`A1GKMWeZ?0YifC9oJCuGJA0!gMS6k5oDI=@P9gVsdUtU2_o${;"
    "~&4!6OG_3^e_d?&|kMc1OkS`*K0eZe|1Ia@)ZyQ*6TWP0x(fWPq~|N<3GEB3Ke~h^2^JvD6jFZ@2nW<WwZc|k~cv"
    "_-=dh+zW6aXGQig`)1=eskl(P~sBG~Ra2Wgta6O()%K>`8TgTry<Y5Y`B+&w42L>enS&JZ!DP6P2FX%{iB}l0|+o"
    "?Cvp8LT^Nf=3hWJRxu1J)e8e7$c<8TrD{Jpw5o-Q!BDkE29$&w`kT2OKz?Ofn=3aYZsQfNbQ{#Y?hbHeaGBHl+Zd"
    "jE~AznU5uD9OH2!w+SX7Pjhs*bjT@FV!RPXC%}VOi)?~W<HWgW23#G$Z~3_D>8Plh3E!3+VOh)z_6%)MB5+;6tSJ"
    ">ZU<#SA9+SCQfmoj7*kO^q3Be&=wkFvp`lI)x%IVPqL{F0I=P8kdK4V2Ax`>q*SMK6_D5SN@D8F*&qVeY^bw9JpS"
    "zU;_-Zs!ORl}6s#9}MEB^uOB<*2<#AYFrod6z_|d*MTG3Ii+Htl5XGAdcN9our{!frnQqohY2^nujpE0QiU|LWr`"
    "kRxPaut7aTJ-AL`t6?{j4*cC8VggaddU@42r6z)eBc9haccpkBX8u-8#Tz3mt#_151tYF$CUv6;D3{F$?K@%f%$7"
    "If{9MdsSRE7xl;HWc=3U^e?-~?niw)JVs49P(|6fa7Rx8t1dP3HO}Hdmk~)ympvrrfw;o*PdF3*a!WEU?!SMeJV)"
    "%d!}d3c--UI9vv~<?>}xs^b2WWx;sOWLdtTmRt&Nqq*&|%+0d17WH`nmp`-txx*9~Ea7w(<1zUTshu@P+<RKA0&I"
    "3Ia9eL;X+7I+w7Yl|sKvu~6hl%}Iw<{_dZ88iIOHkU(I>)1Q0wmAXP7Dgfy$%jnUQtLX`3c;P)zP-T|Y));@K$fr"
    "mRsC*ZU$9S*IzfWg$?On&nb`WK+G#-oY56Pu8D1Nv=B^TBYy?;&J{*?;+V69wx63-W)&=)vQP(>82Z`44IMRu`#;"
    "2Q++S)NLR0zqnQS#$kf%Wdd}bNAj}KH{wk@MCI_y^d{h$Rf`qTNqctI<6xmF#(@-@l+8?vWio;Ajr`2?Zmni1E<f"
    "D=1mI<^p{5ChjaA2-PRa!f`N5WB{iNxj^F^?jqm4Kp|Nn_y$<Yw{1=m%$MeC`&R_MIp#KQ74$L85#c7ZJ6+kqjwu"
    "8ZzK>4&=C~5ga1{#%75qn`uH0T{oL4fYqQtmr5#=jnOhCn|jsDfM3<1Bgp1ADQ`i-lHw*;V30Q=6m!Vl#%aEHFnA"
    "la*e%Z$8B~;Q1pAYBMqR|x=elt=XA9mv9jrDjtm5dbeXu#1ur6b);*+g?MiVmDm!z={%<e`m&D)TzIYGx{P^S>4k"
    "tC6%emW2%EV>n_>0+;Dl3dikqu!|8n2NPoN|tSPEP-0&lC&z1vV<36d}+wO+9(Ul=---ezV^JfBlt@|cPCNHrIyX"
    "tV$#c}@M^Xu#I?Ow2YCL-U0e;DPm@PZzp<?p9a-|flXaP8PPYw->l%RUaa&cN6ictFKRKqN^|Xr^DY}iqBQvtTTp"
    "%rv-}1Xd;j<psz1Dc-;=AjF@x>E38N=u{t!15|a9Y>SR>Co@?N{}jjf&;r6G8dBdSix@PNEAO7E-iT^=hmgYijM?"
    "*6*qH*>#aMmr`4f1S`*ZdCQ1}DJZNM^cIj{;&?&4vl)D6XcBtk(BLk1_cT_JFYgKbrNLR<cordC8%kTc!+Yw|xcf"
    "N>({ST5ei~jBG)l@-tb4b%v4ub42TP_Y&I+gQ2j)QXoxe$SR`FPLT+|4GKGlMTWVJYYbNqUr>tBa+q8yOAs@!g_Y"
    "CIC%*|GBCYI5qmlT;&7<#oBTJW=oZ4GJ4XGL&iB80b)a-MakIFi#Yj!<SPinm_Agv(N(9rp@vzWI-D*c1<DKSY@q"
    "O!F=^g9PV3mP&dnwS?3#%ypyUd35*1(UA4u+wN*SbMjE!lR-8>^;bwkJwvE?aHcoqe$#BPPcsiKP$q?v9yCaL4i>"
    "()HfU&ik!zwB%=uu<Rdh3x1YN;_GZKUEjrX}0om)`239jdk=g=u56lA)yN47KChgPjrspQ767A_D{mkl{@l`6UPV"
    "_{Iveu;w+4sfXRjZ?lD9S#vyyEHzRP^K9HVdYN_-io^!IO$X<!%390P-ERnxp86X{G~e4aXw868A9u4B4|BDnj2u"
    "Lb0_p`}@b&j{)eIN}v(||W8(Bp3z&&~-BTH+y`K{;CzjIK{*9fdxJGixhnSV)k{lH0sO^N9Z{||fb!rr!#tO@@Wj"
    "C1xAnSiw9I8KHRXGW12M-y4nk>q4@Y%K(lkc2TsatP9j#-soKb=~@g1_;XX?0L@aoZVQ&rO}t_>gwvM_r0B~Ejyl"
    "dS$j)BTfymIW|rk}EybON`=;TRrh=clUfI2SvMpRw5rMCdOtzKy@KQb$DRkrmMXwas`K-LBw?pKb#JYx^|0k#0`("
    "f|@{i=8tYq6+$nJd;ZuFN<(Ji$L4g#8BNQ5uWDI8=`TWT{_FA>VvtMYCO|`Pd!^T2eU#6Z@a^pOPRQ3_iZ=hM<%m"
    "+)X+PjZd|*6|rcKo>=1rv^{>#3<5ssNNPS+D58r@nY)->4k_yC`%L^;svmF%%E3ew^U92OE&y#!5Ev-Ba>SP=&cn"
    "zqi|%LLSgzy2&fzbE<KL2BxA#t~g1R2-)M(}O{_mmTyfAvUO$(#mv<3t`4^46lRChrrtpDx`u37phunJw#xs=Zo2"
    "Z!AqOeF*K{yCh*(Tewy$V29`&f#=G2OpC7BFMZ_NwTy3G(>d%af1|a|A#Y3efV`3XMK(QQVRsGY4#}Tdjh>oXcZ{"
    "_2PBF*Q1YJ{QC{F;_9ig(?;Zm@9p1SD7e(RWp=a>;IO{)i(9jY6&_gxOsWv^S)~faDG=tdCIYErf00d=u0Sf~Y7?"
    "2uM27;cn3>qUiH?(Srlg?SOn#K992bRjsv++{Pd!I_2)WA{mazeb(x_$g<eUywa<Su9$(SEnzdKl$r`s)bUh4*a4"
    "E?5buG{E_)uU0<6!+K30S7bzMjqe*!rR4?eJaw;-?5JJzAKL_p<fRWxN@ufnf7-XHB>M@v;yfgBlPI}S^Y8*5-`a"
    "E@ff~YPgpP7<xF@*Sk}s)w&U5t#wgFhr<UijWB0Gco6p+#zL=QLA-!e>j0jn`dVWpZ7WH?{(r<`YFj?}^j2_P<hk"
    ";Q&3bN#s_76d%r@{IKE#d36&E!=xs7z*r>S-MU_kVIJfx_JRs5@-*uosBv~?m;`B^?Th4klk{otMf8Sztz4lYMl+"
    "GXFUu*zbh9P>XXWHLl3*Scv7t0jjO9^Kh3Crn`H*Iz-*Sym@$ti>jI9q#jB|&H65ajkf@%fo(aH7Knv-#kToK;DM"
    "S`RheBk<>ttDw{KxIecr(xN<}b4{E<qF0Z66U6eDA(f*M3;1ca_}M_ETOK-|^tDs+F!QkYB0m6(Vc)yM_~G)TmV@"
    "AGaIgk!R!l3bEvn9*DYEOT|k^X3(;Qg>pGBvNCQn?z&iL+QHDp=5dRo?w9c`pU3-cH&rU4QS{fty#rrnZUgKOhX>"
    "KE+Dr6DT?wxz$r^R3%?XQ`Pv*|@woa&HugO7)JWL$qs+r-q8|yLAfh!-h^jU((m@D51o^q~w$<{~ZhY{Yh#o8p+3"
    "Ou&8Z5-5<v7_nQm8n8*%pM7@L^Ts4$SA+WT@36%ZpeH_QMsDjWF8)<WvHmtt7SEl#l=k&7+&Y&5oZ|52Bw>2O8B#"
    "-FVxl%cL^q1i*gT-A6VM_$G~z(=fmrohoqyuSOK$C#v#4vWQSEfQIkf*dLl_jtW`pI5};c9rV*t%c9ElTDQt~>{b"
    "BD0Xi`VN>dRplVRMEV<dq33m==$Itd&HpCAl=s;<DBW%D21h8v-8ve*q>M6u4N%{u8ECxUZaeBcg6J!9<o@MQ48@"
    "jW5*(>{+Q7$djrl5-GtS_hpwYJ)liq$}Vd_s73i5{QO8;Fa^K@O$Z=pGL>TtbW9SF?hT?7i+>Zn&=028Vo-?Y>li"
    "FKL9^!8U%1>=O|4dXq3UBHW!Va?d^31+M(dmw*$M(2rP>_s8qRpzIDT_*K*TkYzShS4fYzuhV)s1)S*Q<bO9(i&O"
    "G0a_pTG)gfm_t$Ei_K>Ym(Tf@gg>2tM+jiV8BGP_06AOZN=m#>~HKE^GE~}w%N28>EgQ^75D{?aT_4)p{;(rf=;a"
    "I<mDRj<P|3?ed9iaLA=9W2^)v_b0F|~#8v+uAYK#DQ@rhLorO{Hdut<KGA`|kgox@$M~u}$d%Ck#o8QUdJK{M?@S"
    "WN>Ol(g%gbLEo+%T$m4Z6EkL`)W{D1D$$AsE6Q0~s&~*`sW?m!QTqAEE?WN4KbxZzxlZa8n)yRKi5c+dOQa;ebvr"
    "bjD3s1?*IHx^j4Q1pCK(=Sa>mL2D1>N_Lpl_`dn`F(4PTf8kRQl<XzZi%MAqBBYz3E7HgvD}6&P=L{{gW6-7Ptpj"
    "g&RWtV-Z;9kOk;lXeF6Z|^HbKKCg5ixz<;_%Wu&0xJ*zz7z4^(ZjO-T%^Q7$X$c;90<6&r0ksE8r2<II6x=ghCHr"
    "FBUISHA!a$}6B?#cSEjNkJNdwsQr6W0)Ns7v*f{gz}@#G2uYhb9M>7SJqz?eJCAaVC^e1Bs`wQl?f`1v^ZG6-20S"
    "lZ{%{Lu+BFvof=L~as9-hY{smH7tp#@rUd#}ZpppJ8;=5r{>8A!`{()qF`&IiIQd*xh!Sqg%yi*T^vm9fN1dpRN*"
    "0|<!}oxQeIqd5jkjy0WRP(Jinh2b7THZroTpyIdv-#TIb$LTA*5t?xVklY)xS-b#sx@Nd`u2;vAmNQg8~o)^)W_R"
    "{_caBC{_cBlpuCnV(C!7fU|Jg1EUy~KW1&2`<75Obeoxg`2O?2ZC1nmn|#uG0i>5AgdxKzv|qQY)Z8yemA7~Lb~!"
    "6tb&XOv96MwEM2=vJhEwL62^y=rslJs}2{2Bb8mGUk&QtMqQz_6F?7QT6@Skr6C!uKHhaltBjoFF}w6#EW)&SJOD"
    "9irAoBjP*fQ8^kv|=k<2|Q4-2U%f4*M}(0psPk@u!<ES--wn1E4<4iQlb!1yG?jmY_f{@MvMXJm-$kua^!S_zXJ4"
    "@xbXJMA9eZE3KKwIDud>ri8ppa=u1yG$^%eZNE)mj{b<!wuE&w1gyGgETw|4LKPuMleVJ&}JRGot+4=}L;)i*W3%"
    "U0(A<{wsn8bZC=n>HtVL*z;KoH_VwKx^Hja^Qz<|*+~ls;sPwqyx<1U`tJ7E*N&$jTpeZHMr!oxi5}N$U^Psx5%<"
    "QCs!Fn!Q_l$c(PcDrmdZsdknI%u6RtgXeN}I>r~#4jqAj*^(o~{1T$+(h|;t>PFlJBHC4E@y*#Jg@IfYZ_ENicnQ"
    "Y6ZJntJBBK6w1ENy^^%J91Kn_2;Q8D)%wG~Fixx}~?r#zb>Btq`8UiaDMeOKuM_Pl7|)ztv0tdB|{K#0vNjnW(IQ"
    "7(ii5a*;iQQg1Hk+TnBP)vksJE#{|r(UmXr##S?3R^<!C?t@rK`NZIC;Vlfg*cTA=8G(gH4;XaMMcI)sSXXJoPB3"
    "iO|A&Wku$V5W|87CP3onLpkt~wQKY_0ya-{-@9u;;8)*6L_v>_7XkX!4(aj*ztK41NZ%%(YJl;F~EditAQ{$Fzrc"
    "T82uB&-+<i5#}%2{pY;PIjt-K3KxVYQje#Tc!PqotaE$nkd$+f@Djxf%#qtAUOY@IvvQP6rYOC{IRx3i|F15Ga2G"
    "r;NpLo?oa5{BDvC)g)4dAQ?xxK!5CFzRc+4EvE~RFr+qYJ#m#3AT6CgMfn8g(-h*UG0F-d*f|A+0XBfX3Oz{7<^#"
    ")QI+jsTH?AON7Px#QXAD|;a`-FUvs`qlRp}(sprHt?W|ab2KRJBx==kvH@MLhJBvs<0l#)W_rye`HQ$jU?v)MhcY"
    "fn@ItwKz5zEbk8<o+u>gj}$LI#OT00USL;5DE=*#uWEh$^xTR9q!|m4%S7hL9ixp%tuy1s(FCA2zrWQbw1?TEy3l"
    "hKMHsRdiF>u-eEpd5~7Aq$vj4*A54`E1M%RK==7)UQ{rU2P2WTG_yW#wqUEfs=K1yV29Wue6Sxi{a!x0|Nin@Wk3"
    "p`$RZqWgAkJwB&JiXMnhx=1N`(v;<GUEZ_sMo*Yl^~mFI4xWlRK&r6i5hJH1a6ED2RWInfvR5+MS_5&p^H{v1WO~"
    ">O#{`wqFmZX^ZRmba{ob{O^>2PnO_JJ@><}vAlD&wS#Gp7eHjFG)Iba1ItrV29300M<&=#It_>BogZ5Bnvco3o}d"
    "lRnbH8%q8btC@eX)M-~fXxPl5GSWu-E;?M{Dcb@@}>+nBRYf&Qw0-%XV9w;X5P=!YpRlb|Qm&J?Oj+&QXMGtD-sC"
    "UHx{j-RNBK4VfHoFHnD_04m&Dvs|Go*(BlVbVFkPst;=j^8~c8GK3-2ibS|qzxnaDM=HQt&lDNDMX$>!VHQgxeD?"
    "ITV2Q0h9ehu$7!F$xPnY2te;{DWrpP`0ktM4L>cZT+KtjCNFI;_g=>Y^6A*`l#37uUfXs06&T)3BwhrUEv;_U^ww"
    "Blv$}2sdj;p?6*I3@SrMluZY?M52X6F=vGTu#2ettuF_CLSroDQ~03h3O9!Ox`M6DZ!KgTp@w40{hpJJ#t8c+lXD"
    "V>t;9BvWKrEqemA)TQo&)q@q!O*&RK^$i#b6GLFD*NMC!g}Z<d_$vyi!Q2DO^5{l;5ZJ6{Zt)%s^i)$&cZFv+8cj"
    "0_@C6V{wMrqr5!Di+&ZAkbmLAkLKfeKkX$q*7=u91|il}8aA71m}8m%3-0Mq$*@a$pg5*i&!b?f$s4*^;Lf(b%nz"
    "{6q7P<R}>7I>c3|J}SuF7vT#X_Nb$MX?n9;`HF{94vc;I)pO2!GzI5eE{P3V7calH|VknbrH6$^d1E2S@kSC9FDV"
    "&5=bn;sP14&scVkT(mOaD_tGw`9syOHAd(cu-6=8Zzk9M=Tz0-w#-^J|h$Z~><nUlOg9tq|H^Fb0d3I-AhhBO;Y6"
    "cd1iBdJ-6KdCp#omYexFi`64F#t86!Cxe#F)RxkO@VE#6Vs=U<iK9e|M%LWsR>M7FyJ>HKKYA91f2U1HG^s8EHL&"
    "WSHiNmIPWYvJ=?F4YQUm3Qh$V?FSBmd^Y;1%lpGzGjZZ{)ll{7r_#I8WC^IYp}~QY83I%mS0cV++sy_J%`viww7>"
    "&K<frsQ<_-zBMD*O1x@6Q>V6yOr$Wy_<^WH~@N_n}jJH`vv(3V?kKAkQ|zk7)zYB5p+?HE;%F@i6&&C=Ua&vo@K)"
    "u2hhWj3DpV#^LBT3{r^&sjDj`qeCr{+`as_u*lssXY)J0w(DG>6bL!;iPko1m>y;P_&q4m0nD_kDcuenM*pNMLk{"
    ")$<wLnqgWH8yVx`cvrdHRnbQ3(ma#qDv`qBh`k&Cdmn4NW^#p61cs#--v4MR~qH9+Ko^Yy39+U**sNIL{(<eF%z("
    "h9Mt8{SV;XE&(M`~GgPm*Db)q0VC051@aGP^x)QYCfs&Ty(T6<B0&Yk}}E+K^)riecT2qi1pSTqi@a>ToPUk+`zf"
    "SWdv<Ea%9=ZjW1FIuTO)g=tZg<-)EYCM{aht1ehFXSLGmAV4qzBJkI^Yk&{{IdJxIjmY%pUiioxEpSxW9gcBAU?+"
    "ADj|Ruvr-#S(nLjMh2?nWoogCYqqEp!I!L|u~MCv7~7;ETmq!z<*ik?BG&2F+(ZQMxHq3?zy*DeotE2ls0o$!tsO"
    "jK|s=4w)@js7)lXeRU=Zc46UGkcBhyaLRvi_9+Xa;a=5wa!6u4fu1>myPDeShXs;&9e7h+XF=DFbS#X<h|N|_dS8"
    "v9bieP-m$gR|5yENaWJvBg2tLI@IxTK;=VAQ8x-nbVW42m#g?_g<|8d<Uqr#RtSgQWGCIXRwAK^XoekPJdaCNy)w"
    "_asPw)cl#qm$;`_RcsFxBQrjhy>dM@hE4Nme56{hl~os#m~%)2#ze>ULm{rzgDzV2zxPiX%NatUmh%H@VS|d&eiI"
    "Jv6G>^@3h$W#JmX%>)b>eO7RT(T&Rswfc9|LUIE+`CTI?w0YJmm8;9xLtYwq5-_NMB{-^k=P<2857PwmN-t?%ca4"
    "Yl$m>bwu6fy8(mj_<^}=o_z0q$_j3Fag*~6Zv5~4~L;H2B$Io<nZ(1YIfd{9uD<A+i!uj?35d%Rnv8gMuq|5F#mp"
    "WQm8d1Ns#dG4CXcxx=rQpAoWEv1mb4H^plu*R(v6QnR$b+ihQOB1x9stN{ivG;&VIzj)1DsOse5I$N3onA!WwmoX"
    "&|7S4$KZEK2I)h2aNw}UT_-O2V*M!}ay|(@6Eg+Jk!v_CAj80s95UCtm1Ala>6JXKm_szO+<}_Ifg%dFP^X=xDM2"
    "cWSfli(BpHW?vk1FGnonBH#9=Y4Nf39IKC!a<e29Jl0VFyH{R+9`<B!9JEu8FS>OOH9X^wSvwyqK3tY!W3?C2Xgt"
    "ciXVqPNx4zd(p}MkDHrY;E4x!c1)`Vxw)gu^d^@x76>plD>xz4sebnypj{oW)2yRA(z#aM<p^7{U{oLbU%oF%=>P"
    "4;(<7NV$^QW}r`Z`U2L$-oQ+w7)T`|liYST02bA34Aj|w?=F+oriMn*@HjBjJghW<Q)-d08fnl!8ck@ARCB{-Hxy"
    "qK}CdVY~F=HOGBs5Z`3FO1X4)e^#wCYPzQUDbAI0{*>d9~gA;91RYQ@Cg5oj$>00_TnX>Vo`dSfbmf3&MZqY#{J>"
    "PJ&Xn)#>|=<kSwLiX5f(aAhaF>H7ho^>JFY(Kx?f5p5?(jKQ;(fi2_01KY5>z#}q`Xu09TgRsazTuFyu|FS=t}lr"
    "JEVfRu5#lWN*9kY7CL;gt&bI<4Pdkliu6zAKQcj)Tx8Vo820`#e*f3rW-9l4a;4pdzLCqKO9E7W!QJFD9>`?dkmI"
    "jOcWADi>0&hTn7^rHFNtT4uuWT_crZ@=1gl!;f7hxNb!d!bSWtMOV(yzEDDkiUcRC#>&#*bweO1krN+IU!~LPg@!"
    "-A$UufAUU9Os4IvSFnSxPqQnjY4_aGbu9TFo|@OV07??=>9M%*v(hcXE_<LEo}$2Xhp7~;TA;PMt;{zhH?_M3Lwd"
    "j5s=&~rIH5yPCS@I+bOdPM8L{+;u6D7x*HC8A<b&udHj+AU4)e64<Z1uMO@xJwu`i^`p6S98HBfU1<MGqEXo7_Y)"
    "A!@lnnW^O>0UPJNUU?=}k{rA~7)TcXec@zJiYR1>fP;Uocw4VN2KT=uG>nU&qf&qd4{pPg{h(|m0$P4#69uNwM<p"
    "fR>i}Vh%e6J?BLlz)tCjLW$M!jj2KtYH@;l(mrs(po<>+<f&DOh^6fWjA2h~r(%DTFMNX^Uqv<t--7UZq^rG&M&i"
    "(gbq0UFP#bP3ru2nwRQnK8ynlR`ns{m{7G|+)krg@LZ%LFasoHBa;GELx<T2)?Ns^B85tp1JsgXq_Vl{8JzO!#}p"
    "5VbMq;2?y>CtGn-HSDR!NWM`T=*+@Bh0+O&l1z+8T=#P5$XxesVU$JGKq2KKT!Bcm~oOz6XZL&5wuj=qK6ruFIHP"
    "+Cx?w0xU4wm`njSXeGAv9?+;7Co#O`l&Dr(o^SWWzSU(#?TyH9aj^jNmJ-oKJzoW6@A}X3d<qDQ~-Su9_za>g=B$"
    "RPKOpAqUoW)sC5WS`%15I9%uqS?=6GQ0+@hkLQA`(@f;|63VAc@=lAAKdhpBw>)@!@mW?8JiDv?LUV=0({bydXHA"
    "=p$nmdRX$t3l-@JLJz*d{$PQKEklIDX&GQ@{swIc&<<Z1=qHQWILk+ik@*-F=S@s8aFOS*)_r$H{Bsvhj!D&u`!c"
    "@Wp-{8vd%G^nP)Vph$2Zf3ARS8zBVIP(F!iBoGfVRjNd3ITN)w%N99B@0ZZrsI`!%PJgKCa9W)92{L<pKl+Hy4zj"
    "-RPtGKm&hA>HESr(R=D15}AntVsLDQr}-Bx;NMPpI1A`1dIs4$)kRuIGl#R4c^27+uZvdD_QwzBE*u`pMsRO&50g"
    "pCAYp<lsv>OA(#{AxMZ44)uzx`}V89AUk~9+^UZiRpy41|uyvo7&Lr6q%Nco(F@X#}u7X5S!a#F>8m?!emF(J<Y)"
    "IRy)02jJ;ITvNcn?k^FZcdRY$dI%X@`0eT#@c#P!9UMK|xQk>c9Z^7QDr5|sj!-2HQKa`1{1?evz%IDrluW6g!=("
    "b)p{KZ-n@nw7{0fj~NOzuzzys4gz-6(d8%IqTmz~&1|&0!eV9<}Pe-L5IE{n=Wnecf2u=_CPXeXM)maeC`ssAIRM"
    "Pn1Ben)X)R@V|TV5gcvn_tPa|F5=8a>&TtwH8S*tTI41pO#6xSj#`;j1sPi1Xu$JJZK@byA0z8Ah>%qqDV@(Yf@a"
    "H4inoP?oUBh{&Zv$FMJ=W(J!Ay88jN85ZCk&puET1s^@g-*t(Qt@rVU>AXbS5}(<Fqe=mu9|>ePs^13*#c2@AYok"
    "z(2um}<wWmjkeuQH}Z}n5OIi6Q;)SbMQqByn|Z4FO_;*!e%6y6s@{faT5hqbd2-bk7>x0R`_cr<P$JUo<>{A=H{m"
    "S|3C`=3W$nQ-hI?}P2pCMS;z;DAgBl%9m4Q9y3~z<;<9l%D}r@2S*{h}RA7gM<&CllYKieajVUld@<y>imTgp0)c"
    "i35s}5IueFnl_1DGr~{x|pm{j3WQ8g8V3%oy+-?m0w5Z(HPheBCfpw?owD90%U71`UxTs*#n$U6H-MiXJ=9DBF<i"
    "Tg!-ny+DbJ)!8Lb>npkkj#M8|CaNm&!?vplYUi{|BF?P4*05egY%WB5?=~(|4qGYRq!km*b%Y&NkMdSsk9-o9HLh"
    "2e=E!VR!^sqBlM~7#SzS9D9_}IBn)dX)X~eE+gYt#>HP>ykAonNZ^dciL0m^IWQYLau^MRmr`D`;C{_EYl3-lHKS"
    "^WV&-kj|6U!&=GoX+tEXge1l0lu<VQQ`1q*xc9D2oLc{QK~LQj0w|_IVO1j3(qzFH3NAs{`{|Z%gxR7>x!zi+YgW"
    "+O?i%x1#4gvIQRs(I=8-Thsg&khq(&n+|wkRl?~8lzzxo<B9Y?~3*>_JkY|vr)LSc6(S5)gnBLYf+SdR<S@0YLqp"
    "`wlPe<u&LCrJ72<>i?fWAqRc-8x)mU@u1lr9!?F<NvLV)V4MSC%u+Ql(S?XeztB<dpPMndOI87k?kubrg&Zf)T?Q"
    "t=M5eSiv}7=*RI_?eoS>!x{GctG-GU+>8jxctl_KumO7kjZH6f5sj{B)p`gBlqE44>(1(5-wOso1y|1&+nVUhUn!"
    "LayG4|~g-25NwrxJ@m(9i6cOCRs>viXPZ3!Wq6sw@n?fUXqA82Wk3-#@|FcdMQ?`t1-0d-c)K+<Lohz?y^Id@B#J"
    "gZ))0qj}*E@U$h#DayfgLI^F1~`(ylb4WWH(rj^wgjRJRG0l@nHM}EAFMMLbG0O3SdJ~lvNNlDvl(7e&9cxAqX0p"
    "Ip6n?#HWerYz-~55;>?H8Zr2bnDP3}tLRFtR2dyRxBOp{hFXJzJgI^J`Z0{HWuu5<{hV!Hp{3;+xP}W<VFXME=h#"
    "nN@Ku3Yg35+_fv;-o;*DnCSPz?%z^>Fx78QD(BZkAHOLQiW`uv;-aVmkS|pgT=3Ic$o_guMW&+45qX52<$L5M<Tq"
    "+V<CQ3($p-Yz&cWd}0isQt$~5+F(*nY!vtlO+(zQyL)GV^n^kbpCUX;X8sk+8<rBIEY99qkCcU`0t~Ej{;xM@i>8"
    "y-ZFZ3@z^PhTFk|c+aUU}^*(Pr*LDI0xmX3(rLmjN8IB<ytM*{U!UELUgG5T*QxFMS9FfSz~*Li_}Ab3hQV@W`ab"
    "A2MOs+RkpV6gYkxnDh}%*PM}te9h~{iYaEDUfE3oTk8g1pZBU;julQo~~QrfDgyMGizH&<lkp^M%2}hDotpAUp)q"
    "QgK)fhb_Bc0nGeAy99LNA0ncCwJ`*-cs-bz^2{V+f8Z_VK&8r3wL~1LVD{$YX%(9PI-8-o5OIk=|)f~B&6o)0E^i"
    ")_nYb4V?!Cglv-OeIaLV9C|ylO~Z4Ea84n~(5h;suM1vZq6N@iau_Y4CLhEj8qai|l0%VyMQ^B+EvH;O*dQ(T(<}"
    "5ZDN#&hQf!!DX2=P!Cx`OBTm)_q;F$FR}pv?48IK%<dta7!aPP)*<u<rJx}Scrlfg6WGT9<A?J};2}v|4~4gUynO"
    "fMBTUW>D0<`UzDK{zH`Fy@qo}fIuvu@lXrDF4*TT>nbb%F4aQ5l#)Z!YCl+wR|HbIt9!T87;;0AWjBm3wjEjV_D!"
    "6_aT@?0!0Fq8|zjpfo^Q5~h+C_4Ew68y#u1D4l~93IY+;W)Rx`FiJQ2QM^^HC{onTc}xbG5sB-z(K3Qgx7U=d8@g"
    "2k5NK(X9YQV@!Yj3E$@ru>+hbo8wF6GQ2QiU=?1ZkO%U~P4o&E>gy6;kQupQw#Bx$~oKPsEUc{QA`4i&Uh`v;{h7"
    "qKo)gI~=XdQB5MBMW|siOZO6L{zLtY{~)qRasi_=v5qy_?v%hnK=x+e&&-p#Pn?6fI-{BjBx?hwRvd3c;46b0aam"
    "byElGhP(KA7^%AoHJ{>r?}ISmlMnfPis?1|kM|{_J&32lW`wBnm~;#8ck{^yWpT8u@5Rx_`?lSMq$Pu{$!0tTP;R"
    "0+p^~{i9Z4(=?zsvlm3E9vpBfk$$Cqe3li`g4@xgL!6%{)6#G<NACo70dPa{{VI6W$zUll!np8K-J*>GG`IS<a9G"
    "sE%`Ji^x_Rcec-5JC)wVv^3(>*y(ELLmDL%Q{l*f4pz6f^vmw;+t>ji07;s_ad_bK(lYaO5iAK&{w6HuthM=jHT4"
    "*;|ixZbbpsaSE}6RPq^(p-4tT)Q1(K_JE$!CJX~G;Rq-UNQT;6cu6Wwgu=)+_+G-GUhLAEwLidz7um;3TQ8_&Fm<"
    "*7z*TDBM7t77?DI1OpN<I<Y|2XP^1~U;9+OR@cPIN3QqfapQw!K4?{kd*Rw}Mg)^$X1`fW)F!J<TU-yY&<#5_2Od"
    "MPLq*!5qx5>i4pu#K}|{=?l!RwFs$-up;tQLmZNIcw(h4;wgds-_*`k1_~rVU>c@v=N<~2hyl@}otz2(iET?hMF>"
    "(3u?0Ms@kKg(u#H=r);BeVFOXHOmBsh?sd=V9^d~Vcy|l-Qb^kmt@lNvaDM&A-@kq2fe-x`45d}<wqlIk!kOV$h1"
    "&b7c4=7VuBYa3+O>oCaG5U<)xR}e50&jP3-jDJ*Q!G=EM@$66$@D${gevtnr{8Lb0nVf5<mr}Aa~6FCsnL}GEHfz"
    "uOeH(Kgr^|F=qi)ySsUMa2Z^i3b0Dl&bQCZJbA4KLApx#M+0p9PVX|{P*gjPjE&dq%ZD(Kp{`L6qVE?x`+MK@7bw"
    "{I9-Mt)PF{W$;kG|l5Y}HFydSeH#<ApAOD`zd$ns({d<#<_K+firmk>YMLY{>%vuQ+YBUFAaU@H{H;0o!R}IT|pO"
    "l_a*BYfq<d@^5zO*q0jTjc_))C2DRyNkxu_Xe@6hVt*WcH6|_7%V<W`UY!auN0hF{#C9A%<A*$u=a<<M(0KbZ@~m"
    "=)G07eJ&E#4g(;a$AnvmS7uU`?^3c}G??r}bm;8g<AnDwGyXIeNQ2GrFy?E%3fGzk&WU9?4=!y1da1W~p&4EjE~`"
    "b2S#HUcLx(0wq^6ZH5D7cEALj5q!R9grJs?&GS5xS#R4{k_+Fs%B+cTX#rmnc-Jty-4pFcpt!d%s%APWr5(2+}21"
    "tO50Wjl#;;{%l+wc4w56y3f*$)0%M6r?|Vy+2_V+cDmv@SnYX-V_3Z_$=u@e&%*?)It=L6E+$27h@TV8(Z$68BP8"
    "|KXHD~EL`-1zZcFxj+rjO}-QujEYP}%}+78r=i1?>mDV=s9|9m$&4b|88CRnFM90Wn3A;C^Jb|KJ+c@hRXK#V)&-"
    "2t-a3qRebb4oVm0#2`bkCoCxD_|<3$=_l=C?bVSoxMoV2kE?vK;z+9c=t(3@$heZ4;1~_Yhi>4${0x=KAr}lqrA5"
    "DX8*F#qA<#r8KWy*(92OqLCfh!B`D7X~OjutzIo$^0)Jih)UIk0L$1!*jq#Cj{r%hS1#!+LF^ToPDMRVL#mc8H$h"
    "O-2*mMgz+&V*>Ph6~fPzo&;U_@Ge)Nr_v<JIky@#C0Pqs!M6yDV3wJqWA4i1VCUP3AncN0PqZQ$01;0p*gVxbcVt"
    "tG?Wc1K4o~w9$r)Uc;{d!=k`K3$!@I;s2Cq+97+lL7V-?GU1&)4n=CDsh^m324m2kWaMPhD|FKkyqYY==`0U8IbD"
    "b?UAeK+A(Ag!#tKhW6(*>dmgACp2`<JT!N7Lm(92ba0znH2GE@BCEKZzNkbYk=<kTg5nYNQTd8M_|{EL&&|TgD}T"
    "3+Ad*|B0&^8rW?AXIIlj&IoJPbzHhqJjql0n0ZS$Ai>YB`Y4}tFO~p*fvBmN{)5LKkm!E_ebjc@gdwvSwkpw|B04"
    "4sQnkAnuC$`tnC{MI>Ly!UPe%xLhtG1bJ6(1>naCYa5BJeS1kBk(mK&>QN_i%3ubHm$2so9j<P$rxp5n>;4Yq%hV"
    "pY$lrL}5n>fjsDV7j|)S9|l6BCbB(w+-Gp3h*!5CZ%$pZ9e;^v-zU4^_>H-==Fdj9_91_ohB7rlS-fNdEbr0hZ3{"
    "&A|LA<mX8#mL4V)hhZh%|@_UoZDZe3#nDeL|ke3g3b<~F8zB>9vSs<5=6hNQbqk}bj3Qqiz@YMH>ZqnNa1eBZBg^"
    "*_g!KCcEuqYFmb7PUE!)s<1#zPl#exq}XyDLgZZ|#YWP+S?>3TN|FjVVH3uFOm&(tvEPCE`9_9Y{aQ=^*sTdDV6;"
    "tP^W`zcM?{mzq|kDk;z6vP3~NQuFzaQ~G0AE+QxG4X04`bg!3Li{JSnQftN?LCorfiXQ3N`S))>a_$|)oot@P)>M"
    "EiTW9xn1PZT=XYDn{G^kzj0yhO!@^kMf{+U`-GVH-8Q1|(4$a=54Nu!aoA!!dJTx;J4D2D?+BKtcKic3{Hqfde}6"
    "pHVB{p`i&GtB&;P-v>e8!J2)((@+7Mxv3S?%gP{>aSAm=(GH{;bVc58rOkK?<v~=U7r#PffmwbB-t$_|49M{Op<7"
    "QO7=`s1CSjGlPmTDcxIqS>qgjZ9l{@#b0y6T%j7$d5{ZdWpeN}#^HB^z3kMNg5b#spesj7L>x=(>0K-p45;W00CN"
    "9MtmzM2|uJ+wp6j5M0o&4v14uQe!{ep;X4%4P!QyslKN^>MIP)pd7ql$}b7}?9*1#~h_Pv1wZoiLVPR0$^L8>BuM"
    "O2}K7F%l+kp%BpLc7j`w$V37N7AohGNQyZ8N1tk>cj_`sD*e>cr_pyW43cxmb^GLp!K=Lk{Q;jdX-Rnr7v;m?U}5"
    "a}<@Y?_xSUgKR`4x2nyE+v^mtXF+sll~b#>qkdR1;B7W(S=@Xb;5!*7x7>%;uCl*jn!Rp4S9R)R&hA2oNt52xu^*"
    ">qV8qNu^w2wi_5eyY8LsI{?kc<|%i@$13vM!c~**x&m_4bARGJK%;tg;yzyBO2_hvNof^!EQU=R10V6;RavcJ~}!"
    "+`~^RJq6*qOc%}Y&bMW)Q;jag4Rv_^3%1qQS5%mQyuPxT<WDPt?`D4WlpdN#i+;9u&FzpGjszwGR2TO)SPeA2sgU"
    "aFB^5RbpOail_JI<Nnnqf(9ND^tOj*dWDCoIX`Vy_BE3WXxs*r||gJRUJ3J!(m3!XW?p!zYQA1%Q(6+ziF;SIEDz"
    "9a@|{l1Q9*12Lubh^f}oy8y|0OI?oZkEc*^v!s<!K~RG!%Quia!S-;Y6mZ>p`6y;B#|#BQJsXa}p?2Cj;drpU`<q"
    "e&$8Qb}NFgvm^hUd)jKkyI!7(UL#&nOBol0kX&o*ml&d=-#Hl%s=`Fjf4uuYGVE{iqbIzgSL<3t%dc@ZnKL`}<4M"
    "Z?2uNwZFhb_XXr>-UtJ8aW`kXKO5l)&0Sx-Bm7+q;rZ|;eqIdNj6pu`FRj+EPy~<?R9(qYs0Ln@|~JCdg>NA?blN"
    "wH8uuS6#2I{r1T9uuO>0*Z|oi(3^uA(0&Nss<`baXveczjK{*9!IXsn^&EmvZlVR5L54I>Ki{k;GAO|2EXzF8ESY"
    "p_wqr*xfK*<@t-iji?yb0Q^THNlN?b=VmVJW=$R!?`eMaRwqWSCabMuYugjJz~20q+*h0Kqer&m&=%*R@zGwLAxV"
    "9|<X7T5yHf%%BQIU%5Bg@}Qf#Il8Qs_190{8&qQ<MNkcxXRtvv<prvnh@`^o(o?OX?xhD-)L$3gydnCPduRx8nCm"
    "JwY8t|F&@$9Q>&0OHp)ZPT6*6pHvKe`Eg`}RyI!7cx?EB)q9&yXC*{n>S^lF}Ej<BM!@b(!((rng$%IDj}wH=}A="
    "`4!u4nPf@$a?>hMN?Hp--Wh;LzSrN5Hj#d;NX#CGy=)piRG5wa?bZ;)0d>^5fisS7xv=t2U3*jtzit#17-RZQvj?"
    "^-8?!g<+jxF9X7@&re8LwFLAoYvQvU_(AF*`-2vd29vU`{x&e#mwbh;YCg*kYXH~7iGU_Ue7r>XZIVWf1t{`2G@&"
    "$rt185xcKUeTVfm?K`Qzn;w$|N)gZeg>WZswgtkK5{Y8@D-_z)dOSvQfZOT$~eS1}S?(!N3DFXyqhz!0jXvClDNX"
    "TqaL--gQ+ly?5FShEZY~Ocw&ZkxXy$1><7eq`$)k1?*+kuNv7T>d*ln7Sr|ygE}see&^qn&-kBodxk5Z7o!5TCc@"
    "8BdhippG|AJ-#wPvHi<Yh~f3;mDMzW?S|5#2JS&{hgY<SaogEkpU@wb))MX6S`6S!+Wi$@q!QZbo5q0$-ur2IC^="
    "n7_=qXm$j-6}up?{Or-*wL=e{uDh)a;rLd*Qj79()=<f{qPT*D0|R{@1E>0hj$x=aV189Oa-;j9daLPHm$cCmI}q"
    "pRhvImB05+zx>Ln3`$Q}Nomddb?C&A#$4A=CKYbc)ZEoIo`RD(FU%ky|9(SuMa2xW=kVaKQx`Q-}ilv#xMe8SY>H"
    "13b2M$Sb^ZJgxVwp#lYY1l2y_;2WX>=R072KvdcD%MZ+`=rM6BHJ#O2ojSx-8io`4v0f{_(V;QpOPUEi%<rmciy%"
    ";8(Cb^)yzGYCS@xI)9DrQRlwcui**}VFV$=+IeSZ`(S4PL22RcIM%T!dT&CS#;Ky*yZ4YNyq`O^E2XDzS_tO6xAj"
    "*B%Q`|-^uB%MMq#OV=yn!I*#yFNlC&7+xedFnCv4cLQcdkdA1PciqC|2=s)pTw+JhiybtEa~B++ooc*uNfLg%Nef"
    ">3DSahWp?*@u)1iThNlN#qAjUb4}sAz*<x4Jpw~mK->jV&V{NtDc&RsZ#kMmaLYQ^jN7<qLcLlThtOiB1?j*5Wbp"
    "e>NO!S`1$f-7NV*TTpE}yBppuRu@Y^)@KnR0vvg?boX(+fg^ip<zY5%^(dw@8Z?5+mNe+BKnS<Kewl=@Q5$jv4<v"
    "!EKq^Jm%06v)E%_f}8=vcUbSilyW<g2pkPmWjsaf7?JTfm7c=BmV)<uc7B&O7(P62ug7y1CRPs>26DSv&%Z8+4{+"
    "c-@Q3i#ABSe=aF2vno0^9!|zFyvVjbU#FZr`W__VAQ}*&z>s;`^{5N=f=WmD6xNN|I@AT&I5!fL*YSo&yt8~fb<c"
    "H?9UHE85KGx`_(rWaz$Uh}6<h4R^W_-+`AIbv{@rRLO&@US6(A#p>fz|zJL5dxpu;<IzsF_+d`>K;Kv=4(#E-q3%"
    "*N>uGy8fZrogWbmG8hTgXptpG{w#JczUINP@V@I8%JMncHq1ZpH}5St02JRVwsN@$c#Ik0shsH$c6z41uNb<m&4T"
    "aI6AjOkI&moEBJ0gtYUylYI+qmy1Khc&4yD;pk^oBlf^W8KbhXXo1_=2qp;h4Q1u!WoS~^n4RwiXJBNqpbP@rFpq"
    "hV&QaMwi3K?-zovHziC{n}&tV&b>r<3aDID*{Ld8{vt?>YnzN)cK^B}lCz%gUV1k&rPz$NDPci(iwJ8Cc>LN+*o("
    "v>?Ffy_*2<14{ss3Bky6)a%YQLUu-5&-j7t_iR4QF%5v)i~>L$sP87Y-b)vxh)^;u2(mz4?4Tv8W*DTqVHm9H=R;"
    "*?kCt;}cT9o#FkPtm2}{i#Ov*@^FVVRhihdrsXVvc}Tis}zn5{YpLlG;3Ls@ZH=sEU(agHjfb(3GQgK+Cb8^(c%s"
    "9?#$nLC`0mp2o}TT{I=8L8TjFo)Ng954c^g~NPCVZf%)Z=!UJWZ+mDY<@1rYQ<*jmzcN#Z8X1FGLhypRrd+U)RWa"
    "jHSRKe`Z=W|VQ1})fLH~yxQG-J$NN$#unwa0fiLND3G}Mqe*EF^^ruKYrF!w4s2R@H%9sw(IDI!E!mku^)|hdOqY"
    "wl^2fBwtMG$3doL^lpqH}T~pGUwVHDND@Dou?TW_H6_S250K0PDom*@#?Y^_MWHlZt^=Me7$<J4lu+Eaz(Mm|24R"
    "7BK1x9Qck5Zx{bba@AdotGBa#g5X-v?|Zw0*GGq^gM-uPcyP48{Trm}0Xl*V=E?T!0gRZG5-o0l(*+WMbAX6bLo0"
    "H$fs^6rc<=T0@o&-3gWrh7ov^wh>INtZYFb2?rvYD3`-hWLzReQ~1d{!KYOw$vqBLBthd7}jhO#I`_1qp1{+=o$_"
    "OFS&sLtU8C#<@tdWsldOr1tVt#o{w-W6)q2y`IS;dt=lo0GkRSJC!?8V-PL-#*>@VSf-E?d>0)_9EcB##k2JhtuJ"
    "3Im=N|C;=<Rkgk+MQM-FT{;0NQYBc0ou9{n48Fd*2VOGx(xm$3{>HM{b&W}m-v6qMg4bB*>A^7#DLseZ%cR(>MU%"
    "7BqF!Vz`)&>B0eBDtbsQc7J2ChX_WKQY>=<QIhkc*a}sqpidP?Imf3AjKFqBJ~&O~P|Hc;oC6j)&E~(ZiW*i2&*A"
    "Mz~Fnl-rF^?(#5?z+>fs=Og_zIGAvOZU8}a1jBTUHlh=3we6TvYG`?+W%a?Sgh5O=WSZ<Sf=ojzFb+?vS#E7?BU0"
    "B%x^ON6^}Aa8-Dl}_jM^G!?k%y^F?JA{wym2b<v^*$CK@e)`qT7jx!_1njWI5*+)ryJ6ibn@)%MtFa$IU@9B81IV"
    "1q}JG}rhjQfric<i;1YbQLsN=(;({EVJdUz1p9V<wn~LLIfb<k&p7(Kg{n#d(&-MosoJ`ZJ4Ly4~SX^7<#|wl+$d"
    "9U8oG6%aP$%e`=eM-O)v~LG*6Xm$Z|iF53u{1yM0HMkE?!r;$=h)yo-#5OR~;y13gJOJc14k~jO1;>Eyud<hbMV>"
    "_fr#Kd~mTr53p1j%{cT=w4Zg{@R0bIW_PER@!}<(uByrD;j15qkHiaVyLaR09_XsDLwJ;|g^W*XWbDR-p*8K$W&U"
    ";cXih4aa24io49vr^kjT+wtJ0Q2RA_0AA^eio}Tp8#pjeBQj6hYGKZlvFlp^xDjI8j!XdHx>bTdvGP%2Ph~bokzT"
    "2cvO@34`9fLU#Z&liB0FlZ349I84T<^uQ_U(h8#^-PQ7N#&fq~UvQ{&DAh;Sx^0+mIOWnwv6;G;_CjhC>mQ)}1~t"
    "lh=!v|~l%YNL602gbOSmUb~^#%!>cJ2|<gH5oHp7!u--HB6lBhKidP$yP8hc3lR<FhW*iwhfQ85LHvF$%0YEy-}X"
    "Rp`|32fFV_KyQVBdBNsXUubNG+hE+s2tByJ4w&t+QLt`_&vFe*vXH-8L=%r7sXuCOR7%gMny~!fGnya~kA(9|@Ss"
    ")*s*J~c@;@uPGkjCs`e4$;zn1zdZ`PLWF3G6+!0U=6;7hj08=Y+9gIsxJ`HO=Q>XO;`14ICovg1R6FtM+Adlzz|A"
    "A9D#{ngXjHDhS!UY2yiM4^)L1b1Cn2aEBmuzP+A?n`Vp3Q(qvS2J9@-C844y5#u_utrGH18BHyztGJs3%O+TmQTD"
    "PgEdcaqfhz}E5ykW-lf%^wLSOHOP!rfr>U&15F=g99;i;IYQeM~YHQ)wz?1|v3N?!Hbem6r2CEH?UDS@&1Lq9N41"
    "GB77{ws`G&#=W6-?3r!v|t;T)(Tm8EPq{nYTLGvSrp@f<M7Bei1ve64`>F|v3<IBUT7ad=y0`ri9q_KGKA5KR}yv"
    "vAsXMkjMO5W&2OPM#lfAfs;X^4(9#wG2*Dg@HpU=Ah3RaII$%{-KA!dOmiq?WkDkSk#+{5y&L+roo?oHS!?S*)Sk"
    "VUr;<T<xs_%!PTFI}b^Si#;QII4;V$S6_uSv0G97#$dtOEL!`Di0eKDzB=yR=$`bh*}Hn!9<6%@2E1`<Yi+yK0ZI"
    ")^3;VdQmKvD=@S9)uBJSp`wn}H$eNVxtO_ryu2to^5{S%SSRhZW5-;2G5wHLxaZK-H<^Ztna*d|DFg!Y27#7G-*p"
    "#V+~E|z#7Qsvp%Q$oKwhmOQJ%|%<F>8wV^>L`JtbSA`5Ml>a4SZygA~wjZ$A&B!!58Ng$0Y~`{e{q(k6cjeZ9#dW"
    "fmx18(5*r8^ju}&_h^ov@2RE{5#0J;Yz2Vg&{EQrS_QD?K;dd=y(s=^9$xq5y{z03QI>#S;}&n7;^~0bTf{IGM6X"
    "Q&U6MDv2`t6e|@lJWuvC#Hw78C+*|ZKpAfS4#RUVRoH#ZDJX_ghR9)Ya9_OJ<=p(`RTm|j{22=x3Rf2)>r{z>F_|"
    "1>?TTe=GU$NWs9aOz0JGzj6s@HUn7j{tfn(zBuA6&1&9`HU4UdF2tPjKJWx{fO`(8-qbu)2bt&oJ2WB`;x3KL`AR"
    "wcT3w1lqs~mMhSnj_Q-;tZ_qy;JD><L<nf<8N_0t(!llgPEIqu1vW;NJ@FV(>U{kVTW-pWi|qJ<j$z9)@2(<*BGf"
    "Qu#*R{P17|SOgjE-uCr4Kpj%FNlN=CB|8M;dSUQN)wh#^!L*%+Jv^9&<~;NX7-feTAD<QMtXl3+2F*0VaA0x}?pg"
    "QBt4j=h01=Gk&S1B`}NDZ~R#yPu?ftPQk?n8Uy+s_dDuvTq7Jx1>PiHd#irkFg_jIMfAjawzxVIvan;3fZMdKn(>"
    "fSWz#28@f0E=kIXU$AQG-O)C84=Qjl$sZrS0Y?>o=3&WDh@F0w*5`<!r&0nh3ghNTFylXgWof7$a8lC^DhVhkZ^E"
    "pNB4i{Fm2=w1A2GuO&ME&6>sf1_I1cpM&L6J$gt#x=3`b_hX1m$Fbu|LOC2$eZOF|2sq2_cd~tZNmo;Rwm|(1W`i"
    "R{;(Df`S*oNoEg(GeRJvH|YdupOt!>!jZZ92i)pyCnBQ@#gWA5d~KIj!#*)U>J4bgIiw$6P0pX51Mvd=fH<UcC=P"
    "M9QzoXGGJt$^3sEMq@IaZ`7Z@QirAlxJvg%h=v7`O%gM-0wvb}S9c%1C*V({Kg3T^ESr^|VPC}h7L9{=n=Bp<aNG"
    "v_(5z+kKj%*2)&pq?`#G3%8c4=ieSdk8`*{zv0N1DPf&EK)qm_+sQk<|c4-gg$!NCZNGfS`d_iyFzZk3e|UR0I>*"
    "Kdvwm#=EE28KSq))t%})sa|~O&{ohFF5dg%Q<WWm0DTHLqAn*MawTc`{EDIfbiGzn<lK77TTB}-lE+I(-9z!WofI"
    "P02R(+O3+4W%EU#DQ$vnXC<*~G+3!#*sBw08oeN@p1DH~5<xW7PnSSBDs?15w`4$+VcJ3u$@a*ae7Mu=-s9i3S%Q"
    "_=3F7R2yqF@9{Z8*`X(43><JZUM&rXUh#ejJ28Hhs-aZOrTMygfOqYV{sBo;dw_PhaCAZwu?2yNQ9D^sG77Qb&jK"
    "vF?pZ#`9*LX*ryr)Q!pCY22|hzdGeW|ZwZQ<G4dwjHx8)Fj_@2`wEYyWkL^Ff#7`SN-Jq4FJ{gnHYLB^mbDHh9qg"
    "b<<e0b5&lNm?jR%yW`R3na>!3N*0@5Y{Zuv_TNIst4(x*?bDZqpc2BZ`yCU9HXkG`zd5t+)VLc0+gj;P~jdo8+)n"
    "1+s!kH+IB5ATSr;;xhbe`{<O_+2=MH~79JI_UIj*O?gGcwu!MbN0nfc0A51`pWwCpBG&tT?D%m#T$B;%phno1RJ$"
    "7+;=pj$D$1Wl#0y$@Dqp#){;=Ln&h^eryS7W#?)J=(=VJ_jH9rLP_#vub#sm~%`Xp*T8L?=htOzm|>$~ZtTRW?x*"
    "VyG@F<BIzGp_pAaNL1orc6~0~Ee~LT-`$%ml$h;9s5km6<0#1{|5#=~xS>(xmGMx?Zu0LZA}Qm4-9p2wqdCapxy-"
    "Y<6URZ^z0RfgOZ%rpfiPTzp@_h|5G{{NyW}(ET+<I2(g+(S7dL>`OvY8CfM9LN@M@tZizzfrDIB<qm(-BY^H~cog"
    "66`D2zI-<Y2{w1yLbmvRZWv)X(-_GFPm`@2!69!oQ&H^D33WPAGzCP`syrsS8`h1fwAeTb|uDnWk3X{`K(POp^bq"
    "kkN$eNcObPcqFHpPesxv1^mF*HAS~e(qTEn_*(U(@?z#a2k?uYV>KZ+>y64F0Ej9Fkh1YJYg&O7QKJgXMg3Htw%y"
    "Qb_!e9C54<YCv=mKjBRiOy2?he}{!kA3bKd5%OOVG=VyogSB2p7g$5G!9c>=v~+(Rf?)ro9R+39`1hQoRy?Uce<P"
    "4<qGzL2wKqOi=qdv5!_f#}jPjb9(-jpF}&TW$TJK3e4MvG=L?=VRi;;X@}!nJmRIr6VE<ziFpzxBD7k-A{!<oT;n"
    "X~&C%}m>7Y(BP6nrDg2XAnLa2%2$V)|(k}X8_K{^2()UfQ-+4WU8LVk;CU=!tkf9ku7N>#K&V8}Jtvzx;dF(xbo>"
    "MO&zSilGT`CpU<z9A@-bH-cQ@D_k)v0(Fzibb@IWLF?YnJ6{>zi@b3<%%1eMP=8$m@=@FP$K6Aj(Zl2Z&|Flj6Mv"
    "2YrPq>*-W#Nc2yfj6k_)}?8|D<q{|}6$DY1)p>Sz;(J-B~N|eRwwO(|ocCQN;M;|Ge^PA<3rWM=VG#teihaClfTH"
    "varx^7(*i-|<F8N8zFZeBppn(B0Q7i~O%vTpHr+v1teG4mGx+={;MOTFMv0B5K*=cs%}fu&iAE*&u8#MFa+Dcf-w"
    "XXzSGz?!1=S>RrS^bWd)Vhmer(Bw0i490fSIC=Rf0LO-om0nY}-&k8{`h$)c(3lNazuvKi{WDPmi(uBEKQyxyw54"
    "u~s+s`HN%|qpiSILDBKY4WkBWdWlyJH}o3#+M5u%nolIZ4X5K|PU{&2TJx={RXr9~7dgHDzhm)b7L+<<=j%GSTtF"
    "<A_>EPjKEPBgt8R|cJs!$nu9_tDgCBL@tD-bYiz`|3~!d+O9#+7jzB&oGua^*m$<CPT?aRz-L+0-^yC)Y{-E-I~Y"
    "~bXy|W4ncSp)xjFzuV3A^TC3lGxW@F?YN=NBRJ_D%Mv{)0<IK*xSsgX!`=4rL7`(AN=`4lp4`nA2nlF5Ifg<=a8f"
    "S0>fsS+b*U^;henxwT=KN-C<WlXeS8Nhm=HWOhX@_<_Dc!^3ZY%;gw-sG|;;>#cj%8i-%5A6)auIQIUs@?;u4?L8"
    "8m#h#3M{@A;WJu1$?}mC1Sq7u@PIdtA8f<kNsPtv*DloN_r2rY=l2@>vKfoY@nP>N(+#SJ>PLfz5kEX7E{Ss%l-O"
    "A9zB-yj1Q-$LIWGE)0WrQ)k4$G6jQP{)=@G2Fn$ZIi>~li;y2uC^1JX5ccg|>K1h_$J@WF@v$`Vn!7~IDpR*Sj0U"
    "I2{NqMUZ^_|nUC;@6ACtV@heQjs52-J!k56lMLB>bNoN;^5i|KKT%malF2qkAY4ZTcp#d=i@?+)IxX7>~568iM(A"
    "O1bd339mI0l%>k3+N62ZZ%o&Gv5q$)fq5F%tS8)5onc0jR-O`^C>y%N<@snyjH{Wg{JFYSsl@wjT{6;6Wx?$sViS"
    "Hbk-k_$fn|zX}339c#?r-5fO8?T#68~7H=p2&W6}@=1Q2JEOEqHKK!`<c!3qQU3g~YrgGs3Ob48lg|6V;mkjqF|M"
    "%l{8;R!Ul6<x6V){tvyQq%IZQzMV0i0+f;*h1^VN#DSFtwq!|SDR4l-0f?9pp5D_j1Z+(~kf+OqY)|{q*Rdd%I_C"
    "{)$(s(ZrDNEK<2wq7onuby93qG<`DnY&OF*x65l@y6&q3yVb7wl2C$H6LA?O<)j(>djWb4_t-A(m>TZCHG>+v5r2"
    "JpN?666-A_KI`lW2(3hoSC<9;9PMZ!Dy;(Ue41iJY%dZT<v;IJE+!Ao!U0=PZVl#7$cIpR@T<RfUA3DOq~4=$VfM"
    "lTQUD;2?2IyzSEu4krCFht!1cgcv~6`mpQxVSr_1jZ|G?4$=kGZx!L*S*~hP+-Q$jt9&fixU54%p6h_;5QF;uYh`"
    "x_D*XlwcmC_w3)dsgRLg@P4#Sd_2(O3JCzKR#@smXT`@zkH5_9R+do9XwIUvf;SFRi~}9j<3=wBudPL=HVt&1a<1"
    "+8trFDrZ&o=_R4GI@b(w(0%ivkP)!hw%b9{GG2^4!J}be!BtyM<lOc+6;$u_1s@iExL!32QW9>qRcXLfRtlQF&Qg"
    "G$>j&xQc8GqibE<jdQjjnr@_dR$NSIObZ=DlaiaXoV(>+)AR=qbsA!LGHOHNgD`479jG`DJ)h;ge<Q@)R-@sHS#l"
    "0j=Vxwu<oMSm;VeEwbX&9^V&Dh?JI$}V$ns5Ri(=4Mb(4o|es^Cv1mP)!L8ejb4kV!m#gkG=Cg%@$|}7D){*rlY$"
    "uL5y{lPH)FJPd-Jh$QQk;F|Z&})6tE;FpcD$Q6(bPIHVF#UXC<!wR%l?JdCC|GzqE-)JngyJUN?NQ9q_R#YOrr2s"
    "#<AF&Z`@wP{oXn>E5+(uBk|pzhp&O-le)tXaAmYe<AeX2HGnY3faBTSCBmian`|uTy!_U6pE29{8W5`W|1sAu)8N"
    "ZYy;$#UZ*1uiuZJZ-&XG%dE?vrY$AU|G%AL1Q6JKwpgtXK^rKO_Wg3!3Mqj`)7sHzs9DSuw!cI1w(IGddNWFQa_H"
    "Np(DoXvs)7B0#uw*fwnm-H=a}G_U|mDT=#ep;VDFCxq!-9GQq@inqB9YRA@DMxXV(#J>FAD144R<V)A2}`kWl_+V"
    "RopS5;r%!p@!8IHP68DSE@xIb`Au26V5w>U#El6`g2g--^TX^4lTh5yU5i4TiinAItKj`?$CtuQ%#HlbT_zkFgo%"
    "5D4)0JM?s!%%4kof?``jh!vrds;(QUj3)gjAn@7jnuU>CQf1fS^YB~WwP5;;JedDvJM~d4y9)KhJbQ^#o_kN5H4o"
    "{=O-}X*UPZ*5ZNOp8O_}i%sNg2Daa9T~`b5PL1oBjQm<<`C0_Mn>|UQ1p}T1pS?9h?qc4UQeu%$=VGJ3qsz=T_UR"
    "t`0tOt4lVu6tzx$CE(=nINE!4aCkfrcmc8&4UE8_<4wX50M}y|{`0)m?p|lV$L&w<j=8u^p+iaowszxambSD)C7%"
    "|1L0e(xsJ(KKHd?Tr4|$x}8K+8Xcj?J?z(}&Hq#hDm_nWqzcub4RRl>Z}tnw)bRa|E2{7UH}rO&IX0-ho`OG5B>T"
    "4o_!V4k700PRx_@Mt=q2~;fr(W(!3{mQ_B;T=~4EHBc!!To+!b>``<%ZuP{Vgo%@zyXjbz!p4K`4ky`E#JtCgGOH"
    "SQufi4%Z}g{qQs}S?RGnC`||sR-<a@mGQM>SX~11bHp;N*;Mrm8hxzn<HaW^?8ON^gXc;=rln@Q75Hv&twsri=5P"
    "i3p$Rv@9n9x)#rkHlhgd!yY(5lnfM3qe$otA<-1#6o!H#>rsy<tK1fqh{u51vtgCA{4_gXNJ5w{mygdWsI!wX8IK"
    "Ok#3f@Qw}$Cw;@Y$)pCWLU)2~_=q$xvUh;gq0KW8&rCK3aB1u8-fs{eyC-(T&|yB#uF~P1Yf)f};X)}82ae?X&X&"
    "cmr(Q((F&1V{PYUjV5X-`!PU3o8Q3!8{poq;<8FF?=MNu9Rw&AVruro@s3mgrbhV&zONq0R4@wBM#v=7(>?@mUM{"
    "~ZjgX$Jxs?NJ&{lUIXNZ)Jd)H<|uJX~=``HlLNlc|_>p3%38VN?DK+R=ca%8_0GQ>QK{fS{~6beh-E{;p*jyV-;?"
    "XRpIBGe~dx=30{LX-t2rv=WL>r7w_9uu@~jWSUG{~L)(ZUM>&GT)S<)MjFm_Puv?p(b@(<`Mq%LC9ID7UESsOyRm"
    "zrEyPae)ohD<o^k?h$mi3+Riy&FyJ0GiYv%dV&mBm^}Fq;UiB7DX?%!7m!XK%f<%gwX4pR8FGQgI(Qib4zZnvn!n"
    "XJ!@rOv<y7qBb5)!W1m2I8==K8A8i-b=n{LF6-=ivS^^u>Sk`JN2?dy$y$?@mR>y^9D!&&7p2has@2(7tYtaV*db"
    "_hI34ud?;xcnlS{C)s7Fd`#L?l&<17+#tQ8o-u(b*Qqqqds|Kh9n#I$Vn(%M|1XLwn_Fl{dSd+;3o@kOtCTGwRZ$"
    "$qaKo?VJEnvU*V#~=+08q;9Zwx~WeHE-3y(mI|*_!#>UtKWe+ruq=wk}$b!YY)0n+EzOF_X>dm+`0S*Ul};JYYPi"
    "B<gHzmGfKi-S;mUvhH|+TMYh+I3-QUCf-B0!Sp7<HX*Wd-u*27|7)r{>+6HL<gdYeU=;)eGu7}BUeD_fKu-COw9+"
    "gCT^7aMOSfDw5{O?y=?uMrR1CEt%%cd+b>({ULg>oL0>3%D9sa-Vye41SUk(%3q47RE+!Ep+vorM-0cVAeV|IQW>"
    "{|GI$>?knPr-`Y#>91Zq;Sx((J-VS3UY#CsWy&w@vrInavNMVrEmd{uh+_>yGerB1`#S!R6Z>_hI4oMNy3r{uVon"
    "rdXd6Ob0H<0=gOG!5<oe#EqJToGb`|flyTT_UZ7NsIWJ*5RT&C)FowR?a8b8>IZLs-#EE86D_h+?b$xdzHJ@Aa&E"
    "o0;=wFQQz$c%6DhUGTU#{ux<`v-5sq|^FQ@Lb;t$m&%F8o1{=h<GVu5_*shv#Bk|gTJaV1lDsp{6uIfhM?Wv*%=("
    "+J^vuz{khsic_895%aD`3+x7%VhBJ0piK$cDt8=t4y-`j$Llx(H<i<{bvD7)g_Tw44#W>4mt>E!y`O~4Z+z@CoLd"
    "(g)=QXBkO1&bC`6uKsCB>L~Jey_dyy#ZU75j|ldbt=)ZznAZ#P6Q`^yV}<-u?Bsz4~1QR%<nsa(|M_0LeCzgZcYb"
    "ynd_Y#TtW)NUaxgdS$=ZHQ}@07_urn0Jjg})o*(oDOv^Jr+*UsM3D8xel7t*3aBAJc?tOL<Ep3UNvWqoy(lpOfGp"
    "YakC`xn8M;R@E4tg9?0JNDTBEEO&U0iw?kCA;I!uzb{c1NIjS@)^RUO;uNLC5-EJXYEG4w!hPwfS>stBaymDlOWE"
    "W1pX<3<0wZ(qDv@iK9H+8<w4VKrAeL``Y}`r?QXI{ShayZ|%e$%>+2cIrYpIy8Xmf)Nm=Et@L}(^GMs#*m_2Rev}"
    "I1)G}+!GE9_eFkU8kI3^H%PID*!)V#j%w^Tnr}xV@Vs8@BgZQU)E2#76(crDvo)FxJ^tk@%d>GRj`O66Z;9vpgkd"
    "Ohn1xZ5Qq$Gi<lqA}ZL^I3eYI)f=PbHtj$<ETg&1m}y(V>ue4wf(k9RodbF`d3gf9-!|^J!;rGaFCuvdn?6r0lFc"
    "%;(XBHM4a1UVRyGbKvkBqU<BEkBhrvk=?|S1BN)SlWc(*x(L0)jK6E<W_>d4q{SjR$!-&!ja3iMx(#rmv7r9N5hl"
    ">WJUG1k6_WcNiqQH_4Xt_yl2u{E)qrnxpLaK@N0LM-WW4-Vz$URC+LC$q#3{|XO!c&`NT--9eiO_ct4h{Y?k%`*R"
    "wsTWua5X_1Bg;|em=WfTmw5psi)4*ebhR?D~IDe(m-`N_8e^hPf*SVzvqh<O^uqlL7=d1)d=M_yUNu%3R#EgNZ?4"
    "#{8chN0R^jUg6Q<{_5MR4_7>9{NW!bj;j3sq&L~aWs;t8V%8Dan2dpE;fT9SW+8zA3{bv7^U=UA?oPBg7m&nGR&G"
    "X4HpQYmtqVGE9G&pj`nfILc1kk4E0EUcDh00)dbBJs1sj56d4y`PiYGJ0GVK!rr?bSTRQ#ZK6kP@mQKtHaNcjaU$"
    "*dh<_x1v4tRBdC(YVq=x%k2S>N^S+B8q?o7lC&VQDAu2R7l~>ICJ{AiNePs=qAl{=W3&bqPO35qA!v3c8=!HRB#8"
    "N2NV~|;YSbQ8&RmJn)vET$bC25|fcVO8*A^#$Dh5!YRh5?KYAZd;h-VTQiH56cUg~PVm`#M4z^(q71@J_+A?Mk)y"
    "jYwm=D|xijtP;$w98bM2&lpqO-#_O@YB^Q_ZBuL$_anXiWH9MQm_1RH%_;NHDxwIn5tzIV~bOVm1_4J*|v5mpzRZ"
    "?xK$s6s)XuNdH}P%`l7?xH-iDV$VZ3Oz+JqOM`Z{v7nhyyjQ^AI$UPk66pzM{@{YWy(u=Wr&UTx95$Wlrws+%vcn"
    "6MChVKTjd(?msS}p<HxEjD>6}wbOUr=(jkX=*nM2MOOYeYJ~OGawzG-MFS2xIfIqN}>5APDmDol`HMZe!+&SU7JG"
    "ahMt8XY?Mk3!h^{yWOtdnzYaD;t+Zo@!TUsyIlahF+~Rw5slPdYL1Lhi|N$z6A*HJ-#<Xa(F_J7_0>Ss;6-TU;TA"
    "ndQ~9hN1URHg%q5yR9V%DP<330m6fRO=CO8U0=u5sl@E85d0d&QZ|0eVnhHRuB^Cf!J{^xXKJD7sQMk1qa;4c>po"
    "9Ae?l?ne%AcwldK{Cf9la88sO0-dupDNKEKea)o@I{CSCSc{ux!Mg9O!5S1Lmp~-4yUubFp@<8_?hkB@tBE}yXl3"
    "-gaBYKgukMvKzU)Dwq>8nrD{lu#t6%=DZDz47={A>84wFpun6c+4P>T~pe;md_`kpCDz;Q1|I8-hD58jS!~iN>ED"
    "F`?LLd#kW8WHl%DJ)iLR=bN9?FYN9qbMZ1!aQCTVFp*zJ2kX#%7-_A%^UIHZjE}o6nxhHy7zR2|o2a+597aUm1}3"
    "4e}mC^<1EeO}kmdHRC>ju$wkO#0<bL7$u@AuwJPV%(kaM>%ixI_`u2sArXId!qNnsk5?)2&XEN@rZ(xuhl5m{ik3"
    "SeSk3Zds*+o)E2-hzy&5vZPgVWOc#ItXPX71R-({3V&?E>fUmdc870b{t31y2~vL6R-n+XtyvnY(LhaPaGI9z^7*"
    "elZ}0nZ!<mNCh#VRZ1wrNxah4tdnDBWciz%2D(BPucFRisOgsUR~X@RyKt1er(+NjRv6MAOtLm@B3>aX#krOcJUb"
    "Xu1<lI>7y}jR;g`O#xW)*_Jk1zjtDUuhpq5Qamy$?tOaR#QO}^k&#TQ@Sn~(9ewErTDPk8f$ZSGj_Xe$RSs(53a>"
    "rU&uU<WCC5vlY;H6tq&-Qp_3#LB~$J62ao?hPHzCDX+`8pFg*_)}FQNW7MZ;;iJ+3)DjW65D)5!|XK953K~u=*|7"
    "VZf;s^nBx&ifh<O09@Y8dj~Qwpkd?m=3<og+-%R1Ad$`WbC%5_q6I00;Z?@wrJOo5!!uHp@ew-^-4g%1+5la^(>i"
    "zOl=+X2$TNCPr-D2M)OP2lKpr+s2&EXzZnQnW$QQ6t!!$2s`k7o6d|)_XHd7^#9mcO6Bg)T(cEl5<8-H<e4K^uk;"
    "1(Ie))2p%5>uvo$`@TcxE;efM$1<^N@oj6!2ZB2zVi&+uTa3PYI+ByiO}<*umD>UPxik2p_DX9XG<Sls_BZp8w}="
    "{-E~K50N{_72z^`3lyo%Caq#iGt#<Sk7b|+w#T8qkp+hrS|A46r$VI;^y`v0jV=5I{1}WSb!7O9^l$E3ia~Kxf;<"
    "{z&o7mPh42%_)Eo*0C?r~94N^EsTY_}q|8WD3FV%-F%j=Z7au4|GK2yt3G#l2WEy4ng3iXE<CYTb1|uWNRDosTnT"
    "T)vl)v(>LtDQi}j$sVXDc|hJhIXOK%N{$Er`DSo}F(4kR1xw4H`dWroly|!KdXQk!iHq-qPB9Q4qpZ7p(IaYrloK"
    "n+<E=fi{KlEdxrD?N?hkkGP+!fR!vjdi4b;@k(>~fLT)4Zw%O?YCf7D?y^8p8>*zd~_$E8j@{CRK?s9QPm5XQ(^E"
    "&{<V_WeM-m7dR9B?NaJe{#jn*%(za#_JZnKHMGbCp(8Hp)NO=3}yuj6=`m_N<n_ohJIZmJPXvCRA2Q=0LoDRK0VZ"
    "xq@N(&gFVNMu!h_=EWUv|0+OONbw}o2;9&r0bY!ltrK{TVltf(PAc&(=`YQioY?D<QgNE|Q?Y;fas4rv}3J<$;F9"
    "frXSQF((;i~SL74_IV_+@*4Z#Q{!vi)k{PC6><tI!v0CS!D7gRFJ)5uw_ni8G<sP)(ne6rFkvlP%zHST*5yoA%Y&"
    "qzfCYc|krVytSec$Acq)BG^rKw)gjwz0&Yl3;I{)^M5khY+7DV!usiOKcwWL&dV9Z2xG~X&yW^3><Iw`XqPwjrl-"
    "Oc2uwdqqrX1dA5a`47eo=)7#C_^&?H^xuGIw$ULT$Qrk28^{ohL3sqOI<lt#9!mg^SbH5cN>lWUC$_85zYElwr<s"
    "bP7n1)m8KpvQtx*{wBDt<8BMRHG7}oo>?_KnIaLMdKQ{>_tYY3{I>mfia=Qo=nmSF_j0|-h;AJu_D4mHKSAMGB=z"
    "X<J^s0x&c{)Pz+4`Ud}JmAz?@(W$6-OOmn1rh0~@EY-z&uKi$s)13fJa?lY6*%FjD<i*{LXP`wHidM#k*unSx_$0"
    "0INmSb#PqM&5^3S*1dmu{;=o6I4He2zA6k3-E^J_mq^TKfo~%KM*<!DV-oFd=~^YUJxN!iGaQD^lvrrZ#sY?Y>)t"
    "+M;7r4^pMk;cN`INV!%cO{g(BjR6ki5oc1L#lTtg5?Dqh9=ds%_7ERlUV@rDxt`9rbJQ9&N&qCWd4V$A2B}-sZ>Y"
    "9qs*gY(z(F<D)@6ed5u+Q@;7#DsiR#*m>F=>VsHWSEi!2cy$Waq<RK-Xd6chWVp(CDxeugR3qJT?vq1|QHVo~;Ra"
    "W!@pZMyrL({DAV_vk_uE^^$$k3EGr2$Vy@)zT}H9+lSUf-gxZ4wzNs%mn+$G8upa7W5qcp%&w*viT<;^10Gvbi8?"
    "T_{-pUu=na9Q6}BNX(F_^CyZnbLhz*BC17n>K%Stf0OdIVOdL<pFKLJF$&7qLNBD$I{414@`9KP{9~9PVU5MdqcB"
    "*zM`H+gVpEI|ty-U071XUID#k^(sld&lDO{uP0*+sM{!GD>?8Q6zDxJlYK{86oOyj6SUNA~w$CqHbT43am;`!p1Q"
    "PTqU^RAY<xwzi&s{ru?%$l;71P&zh)ivGuKHu?I2#cUt#B|i^-(>MpQZ{ehBrBlNXRLE+w^<7{|djR}LscYrfzQe"
    "H^K0*iDr@{Bjr#u?Bv`=psz`_Q@e)8t%Xn${T>>)bnGMvM&fO1AQtS~gaL7vs79wXLp!-_1O53duVo_3$2X)vTnU"
    "~?kp+k_A<_`YNS3ad!zGI|jL*OdvX#v^nG0LwdiAu8X`C&f%{(mEk%ag7543taGtdzWL<#?$p2!R-FEo)7pqNO^?"
    "r#DOzydb)hwsX#%ClY{julFj!L3SYF7WAsbBwymosPPPwre>nVGviBNy<l#Api&x{KPIlA+u=A6Eq3Jb=>k83mYb"
    "%P~!yRBs|LO4L)UBROOx39{c#EYnwp;G<j`QYbI8%e9)`aO5kA}GrOE>oU-?rXOpobjHT+AtSvi<fB_D=V<_xJvb"
    "Hp8%2nl1#m7Z>?^9dt(t&!w3L!^=1O<r(0)n?+4DMpI3r0KbCPV0pzEvffd8&Hdr<%>g%b0kA^UR3YIpbl6Uoy1v"
    "iXxJQ)nt6NT({N>3Vn$I=}|0fu}{INuH9`FM;4$Fgj95#b^K=z=X_oSiJCB2HRzt<clO{A~J6erdf(VOB@C1zR7R"
    "#%_?spsLv&`D#)Qh)>G?@v_A=NA~IS%C0o=}nO65VE1^Hf60pd+{vUdj3tjqFFpBVM;HnfvVt`v$b-i1Qu&GIp{>"
    "N#S@2O1p79OjbBdy`8_M|W^V!me}@}DDsmE>dXwrb<Ntg!coWDYHvC>vbW<`kGn|2&Twx$RryfNH-z!KC;CwXtJL"
    "IAf{~71PiPK#cHFJSn%?8Qcrj7k7`Z|a<w&toFU?6L-K>LpVvO!Uj6|uECJRF}WFy0{Y9%M%~4>Aty_zd5J`Uqc>"
    "JvU5jX_x!8fbhzLURfp4qa6-x1r74h4Y?2WriZ=9p;nj9Yz&`rKPb^vlN-%id#%;wIGa%z&Tg2|6Z8>&66g`;Lf_"
    "OwK7$h6F&lJ3h;6DG97TIImicWHYll_CeaYuf6N74>K=6FkVj@Fcp>}zI935(uao=cVGN=3W54bV3Gp3KWS~JC4n"
    "nT`rsD1l;<IJGLKcuq`bzw!*K2x(Qn*`3G4igPmik&S@_$*0NNLp?QZL8EKTTH5CPIIwyu5gK%U2=qlDs`2N+S2$"
    "d>;m=<UXd~9@MC%+Y&bAY)vEOCa#GcUhAG}|VuOnA?wtUt_->%0c2~W7g3dVb56Dymb-zfJY8vTD7g!K5KFO)7u1"
    "U$+#ZOmh7_JJhpXfurQ}*rU1C5?4A%!FK5$2p#&pml&m7aHG`DQww_?dG0Ara8PEef2)B7~fn(r~aikEZxyo@bY3"
    "E@8^UA1!&c*FUvEVqK}RNc?37PLEaf+$4Ru>OCBsZ6yqw;M9v(&Al3v@nq+x!S0*=pw&pnb&Uo|ZE#c%{1dzrSnQ"
    "kAoxn4|daP_OrEU4^!)ZFsieVP=#{{N#6WKFQed=ug`}E?`<~gO7u4nNv{*C$;y)TxSf892<Ah%f~=jbP3)q4vGb"
    "iXfLWJNv`-K?7H(l@p0iO=N#Zf&f#fPobja;_HHO@T<*lEZCu^KbzWQr>3ILgpP^bk~E*%11GWCvtTOa@`RWgVj7"
    "#TTa-H?+#9O7_Rx*rtb>2YIhcB9;xVmWBYhROYG><TK1rh5_K&HpfEV?ANj3kxTg>t^}8pZ+AC%MjdANz=CZ1i>#"
    "mV($nXMH=>*Z`diH8*-Q))!WzTCn69RW>p+yXXK+d3JLWTesXFzY-cel9IE7%0Hl7vFrN)~Bt4g|Vvl;GtzH`yo$"
    "mjL=`9t&QqL<o;l4$wkLRQnspgYDhlHsbU{nq$y=l3uEHvA-FVODdE=?xcVp_QCPHyHWGP_Q5V>eYSYoKpm32YCa"
    "eM)^=qui{;;n+x3{>0XJE|qQh{Uk_l&tQKcI$yoS*P=_(Nb+$@-J_^D?0Gao#6<P~cp;X3#PR;-zeuk7A?@lEp0*"
    "WZ?-j)A=-`cs6rU##}23Z@@KR%W9jdhtB^IsYN-vJUh;ow7GEQA66GS`*mFh^j$88y|Ye`_37m_lV#ku+NHr`QNW"
    "I>OB&l3dPf|l!FDN*8AhXhPGfp%;p%EkE}05SHNtJKnwUvBRIqBW_P=*7HjghuQVgEs)Y0?9l=nKZw_{T+CF$?Ib"
    "uqk&#5gn7A`%pzID5^eXujw->>`lJiEzmF1*QKy^#PM62aIg1#@+!unK=s!13@tyNf-08~tS+3FjbJM=LD#)ghx*"
    "%~duSomPeVTAD!*wW_P!pzD7;Jbra}dOA3$?s}jzU(TvJ-rUAB#Io^C_pz4#K}@?{qwry;QPBZXG(QzCd)@u!(e|"
    "5@;AWE|EO-LZ7nw)L5WlP80Al;GjsVNaMNI^xlk8T#qA?Y^jW(4TwBrHiH~gOEAlD_WQ;d2SY||SnWUfd*WPzO>1"
    "aX0<o1hFoWX_r<vRb^)$I3`#A|8J$hQ;@FMO%_u$SzKNx#&~z;OLvF9Y7(z2K?c9IZLG@>^Quha7`B2wXiFa`n$a"
    "FK#BlMC>RK)-6k-=bTeYk(CtP-N2o4HiX~N(OK0WK69nx_i4MbVi@ALE)(G-hKzezHkJA%b$jWgq*XgrwUf?mDZ#"
    "Gm(C-!1TefG)~P`n<N0C<*qe~k(2x`a-*Ux^O{kRGvBP+%8+#RL!~YM*ZKg~~I0Y6psj#u1}ykp2m}YwZ*Kz|5h="
    "mKq;Cf3hF=G{cV@?A>xFln?nG6lveX7npN?In-hOvYeQ{44tPX%52KUeF`YVb2Pe6hRW)bHno+Yv$p6m{~hAeozG"
    "Odi)j&Q>O<mxTMfqoIyDrKRNNI^10UM%g8ho9rNC^fX_PP$HWXG!r<DCrfLy~Sd((9>ok0YW1le~7AG~Z1@CFt66"
    "-<{Fh3R)s_FnBBoW_y)c{(_L?Ps#^UHS)Jjadq#FOwdAxQofaem<3|&dSKDQ?+e=b-evL!F0#~+tdZdFH7ip3k#-"
    "fAN)jJprIdX2mhNaZ?btlbmEP{ok<W!*hggaL9nroPNv?U&bxV`^lHBF^O~TNNM;39kx!^XV8h8jbq1I&wcP*}Lf"
    "UQY6DhqW@Yx=7wUb%SF*ePDqo90aWG#Tf%ZmiZuh)bE!$i9}eO<v%@fcwMYXl&y0PX)AHChp*;MJ&t=Lw?QD?=An"
    "dJ{iaNQ0vTOr~4k5}Nh;o=y&`KVqd~Irakdk7F8H;YM(ebqif;-?ss?V_95R5RudgM{v40{QzwT^4Zo_MHyfv<&)"
    "5dmuf)faE=2Ba}E9Rj!&Kf?7nQ*dKameqP>HtwejZQ=Yzvv4>sbBlfl96-odMlb`3)$Qgs9gfs{<?%J*tzxNfF6D"
    "K%c4pbmNn%UM0+l$$>k`X35?$%V25I`p^XU@+J{vEt+J*OHkpqV0+59rpEE5tj&GLNHeE>AYA}{|2ZEJfx^75!Mo"
    "N!ARC$fuBf~b*YSvZge<TYdoNzz}Xu3MAN(IIvbC=l|*u~M6J^Lo>Px3eD!iHD^LAUtW&(LJ{!<#)#^S+2R3+a)j"
    "`R+M<1d8xzRcezOj~$b&qwtK%fTiFfa~=WA#^%oK}c`4cQn;+#|^rO!0;?b)hE7`|$0R!CNkhgCEEhTCi6O@9G6R"
    "uI-~nLb{G{HWkYCg|Vp+hRM*@)lXImYZEd0BHEu`0bniwcU)w6PDidk^%gL|0Nm=$j3AgKZ+1A(Fj6cXhZuOsn)w"
    "_yOQ8PMmWg0~FtSH%Z6HE@XB7m<H~QySfMESid);QjGJ2}k&|kM9yB+f{7677*ekv)vwspp$g8IDJjH72?ekpI_P"
    "EXenSMjYz?xn+{BkK^N)CS4)&nGaz{bz<F2AD?OZ2m3Jq?UJ)YL)%MaD_)yJb;K{g7^Vc?@bHs4w)9Rc?%H9kP)W"
    "9EFUm`QYCY}DsrGoG9gnC64cP&Ee9{191(FO3BGUXTS_pM;V|pJdvdYN$0KYH*(q7HzZ~8vTP1)hz@@n<v>}2Vy9"
    "Pstpo7lvrui^abtz}t<)IY2wQ>MW+C;#W>O{$xOSEG=OLx5(&`=^Eg<O{1vpTpeP-x9fk?l&K@EVDZ5*`!k6#HmC"
    "a37CsOV{NzG4O(%^p$SOsaQD;lh2?+wWD*Se{<%h$tTT+*&3Oq=Fq47itTZ)xNG^b!w&=<v1jq>8To_)xRoZDfFn"
    "mud?a!d{b#w~su-{MbkcpxIUil63V=1snt|T11$ne=b}G_W*z*y(>QrBj$GF8RAUc;0^wLoWY};taPL9a?)T&m#U"
    "%D+k`T0%fm+gbS{r&BgT|`+)VMev*t7yn1Gs&kh<94SPYMw4t6N$+8&FM~Ld92LKjmLt`9dA70P#iAtR|yg;nor@"
    "&=t^dFRp63=x{#6qvSQ7J0i;19{5?rDpXaSJ*k+4pZ<jo^+Akd0)ob#3^-<(;T1W2RXLoD9p%1avdW#Qb!qT&~;^"
    "madvDWz@ofWS86373m>RzthKn$u{^DRT=ves)7*H|J4bh`EH7n`f5C1QGXW>ST68P7J8&6TBS#E-g?Ruv-g{S}mr"
    "FP^Wg2C*-WmY$MXr4!96yVl52<`=5thY~#!M9IV6y&29roNWKlA~-d_qYbVHm7;S)JCGatDH4FLgAOH&)|InWFA1"
    "I&W&n~70gyP0t{LtLW?%%Zc)5o`RZJcQ(R|Ar#(>GMm=*md{hj_O<D;CFm-_5?V86*1(cyTU-lS$9Qe+FrL3kPY#"
    "UKR`fihp_*?3g3>Z|t?exqnH@N}R4ZHCmwArUrO+dwKi)5lD-M&gnh@tOdhVho0fjm?5*Q+9CCgRt7J)8ZNl_Etf"
    ")saL~LY%WD#GYXW4V7ATfMyeBYUE~jGkuikV4g%2Zsx5XL{fGlNc^p)=435P$bQVDpCU}wd=Y*nvg5evwK5dr<g?"
    "P34RwY~L{OaSz=7J0h0H7Q$Y>x*72M824wh4lGAlj_26zczen|A(r2LIXY{4wdAecX&+e0~2PTBWeq)W5&o`7iZ$"
    "7hd0rUp%*72OD|5q)~LsxXCBUIGbE8uKQat!p`!q@3#K<OnXA;^qKMX83CN$7jCgjv^<D^$SG&*Mca0LaKPs1qNj"
    "%nra+-v7n_cXiL!$KE~d-*kbZbDbTm^Ag#eJKg|hNIe932|!l_=sDGPWUhqQml3&gwZ+7Rt7s|CgyDA8!?5KRhL;"
    "$Tv$q8%N;Q2=<~0knRuiE6CJ_<eQzo2_T1yYMNvhvYQ=n^YgT{Y-A#{=YzQ0fXsV{ex->9S0NG)R%!m)(O&<G`1!"
    "Y+WLqSn&=;I+Ip1FIH|E5cunDeePkBb<6`q#(eb7SDb4u+QcHwfRUJ!CXs89@4@V#8)7!V3XLd4`ymP-gfP<@J_w"
    "g{DrAjRCB0SH_MlV(O%z<nUhLRj3NKHIJcT1RSys8&VG46KXL8v?T)n>3Gq1pXEJKc!a!W?14<-EY8ss{DiJ2)8}"
    "pGJEJr-$Ln=?~&`qS-`~A?7Txb}0(~-IHIo_umXoqSl{d{=afsP3=UD?abPCIL8&-xe)k=t?jDSVWFe|pi1=v)*7"
    "-Z;*VHQvGrZk`YsEAN6!iW`=~_-BO#?NCTddRCG9wQzpnRH*}Lf9F1JjSwGK;fpsIb>G}}URSeY&1muiwll>I&&F"
    "GpE49mBc#%}qK7nh;#K$<`6BFwGT0Iv(M*EMkS?DftamsdL8%1RS8nOVaF|Rjsyx5LifYb*g&s6@+^RFXot>DAUU"
    "*<)wnDXo-_C9V1h^kbaENz9iLN=Jr@eS2mbJy8}SJURO7+Ii(*ulhvX=MJNCsUHvh+(vHkr-3m?R!-Ht&@ZiV&y`"
    "59DmhB!!tkFPsNm+gQ<EyRe>0ULP?a2z~h7~gsM1QJ`EA-tonuPb3m18S50ZOf&LrTErcQ!{#9DRuvRDQF(5kk6W"
    "o6O6D7MROhBJ8Xjf3XFa`20HjsaCaSh5&f@b8uV-7JisW%=&`$?G9B-F4e?s3A(C2Hk;lSZi;dsJ$cR4WIHXna)q"
    "dy8)bo8VM#O>E1S!fS_C~{_xKd}JBuM6Um)L-vOB0l|3B=|7Ck}IbQXQ3Y8PP^5?yD9{aHKI=k8ghQsk;5pggs&e"
    "4i<6oC0;#g<jO^xHX9*zJO$<?e^Oq3(c8rucsppX7|W}Z3Y%nd&fYc1eg^@p01YTR2en-Sef2OhbMb~i!R6M6@))"
    "sE|nA(cQ@mF@}3rhANPJdjNl-+@a^vz@c<_Lc&pu1oaytl@1}ePs7hJ7sZtoOe3%^X9v<xf7X2$yYoFR*&0Es~=K"
    "5iBaQNfl{{G>wFhMul!3Pd7`HvnD0w+6)mf{a)l#H&vhRLmzq=I#5d-;&+k;@T2WZuNpzBU_{sfkAPjZ4_*w>UQz"
    "+CP~(@ykx+R=Vh(Bzq^vgI6snrCTWduGT`NtgtT@`9Fa&X6r@r^>@#Mn0t>fG}Rft_+5!u;3x<Vf`FdM;E#Gifnv"
    "ag(Wp9PmSn`bT23e~^+ISffCWpw=0$clRbmhGh@x_In|U%L^{`AtH|fS+OuTuCt_vY7$4>L+umHYteh)s+;B!jT<"
    "jCyd5_|)j@F?T0Yuh4m#g(bB3PAF4a5%Lduweu%aGkxPRKZjG<W^&Ai0mI!?(9K9YslB`R_c+HZRDtXD@xYo2a~X"
    "g@d!4Gf9Oli8f*`75cp$cy{ev~l#-V;=E@SCWVew%N_vpFc6c1^y*fBl9pV{o)LAWBQ#s*?Yafd<!rNuF*XX}kx*"
    "`_0WS3IZaM}PO9i2S|cVDumNv;)71V8`K%mtKx#NZ{{i2g(YkE3q4s|_Xg7ftAxID!a*LUJCjmTC$Fd?AY|?#fM0"
    "reqsBt5w<fAxz-Mv3_cXsdg5KU1I=g$kwQ`Wpu)%H)wVQm1?NP&}I*%GkSuGO5G4rHl#1W5_xvk_RT2EsFsijC%4"
    "W~g`S%=E_+TrMo<aTuz4}Dif6G!4Tv<EXR+f71$55>9HKgn2)&|G?(Z|)&Qo$Knf5+4Ckm_gk)9T!1}OrMp$DTDd"
    "qr5?vEQ(y>43Qn28E!Ag_6tbi~6+>bQzy{5b%2hhebCtbc9{a*-ut1^AKru-C||rXH#&eWNOOl<fJQujwVVxYm9W"
    "$FjmjKXQOc!c5*Y8ZF;C;A%q2iYFFDs;rJow6EPT&>_>GSwi-S9XmDRfi;NttIUa2A@?lnt(ky3d&mBHxU>C~^*p"
    "6^Z(cS|_X{L=P_X(V5xMb4&I3-UOt@aA*+_XUbgGEBbBskS|g=~Sqx;@DM`~_0BvfOcJKr5^TRytYr-E_Gq@)6}}"
    "pks;CEk{!uR`Ux@VW#94C})Lk-AdJ;z)BIxu~g?4O1+&E7y0mg*9G|wO6MO;Dtpj`ILhV#bEm$h!PaT*Vgr-O8(i"
    "rbEinkn)O4jOQT8UYN(nfUEk6@#l=dh3t;=89dWa^g^Zt}EE^S{BO=ekhM(hUiy{=Z4y0&;g#8GB@GBxh2OdGy$)"
    ";M-|HgFnIxH|v%k~zyDbl6*vaDTz5(&9^^y-H($l3Mw}jbs&sUZrI7`FF`T-@d4y?)e1uw6u(Iug$q6VD^E*RQ95"
    "*yZ(K;A=D1QA2eKXWIikPNs=Q+rfs8Fd5HX|eC&em`s8@;mDNh&?xaP;aIK-q3qUPYJz9mo(1cQtUa*gxSG|=O;0"
    "n*=ESD8FIxD|GQel07onUymjI&`uW%#QmjuYU+sB!ve_rGy8qVM)g^@V#|l5KY|rag)WnI^I5%Ih3LiSE`h{qj*v"
    "Wz?ZihCH1DR<(P0FmU1R2+~e?mbK4aU{qUzbSHQGPx?mzUg2J4+VN&P@Of^{-br+Dcp4qN+20RiX~uYBWdle)cf{"
    "WuddQhV?3lRLnXtX_fP|nT8?|^u>=|1#TR(<C!YnCC3uY4xO;96|BBA*xES=!D!ZRuJD9Hc-$n?J1dRFxk9EJfjF"
    "7~Z?S$?aRpDC3dejEI2bIR%o<O-G-IAy({1>Fm-FP_2|b~o%>`&3Muit70<rN|hcJFDXp*Du-};_edWV_b0I;DC~"
    "s>4`6*zn&Z(sAV<10s9H;L0lEHiK@C<Zf4pTE4tCH=0U|h4w-%E^n_&H(X{I<|CX1ajC-c3Pm8+(*y;j_7e?d0d-"
    "B_KiP3VHv=iN=6L1|tJ(OWN>qc(?MH<NnF?JiHwTXU9S=2Y#A|G}GXC%jTAOzN<GD9fP8`ni#%E4PDi;m8s)4icO"
    "V~QO$9H;pW_)ajfB=D76Wx>ys>?_T$z62(|GRBn&jdtw(2KNIPFx-uHh@>_Y+fzUqBu?8b@QvvhzO71xnN3uv5Wa"
    "=&z)gkm^z_~-YYa|bEP)d7a40xpd-$vYTNV@{>x>XhQF-UANpK;fNV>6w%7W=g_);XX&9M-ttMOsF!Wc)(Kn`eJv"
    "+VLxHKq70(S<h{^|ylE6+{W6>@ve(DVM532IJ^xK2<_H9aqKydW)V!DT46D9K<0U$N!#4!oTQNwa6xySU?Ujba*9"
    "u9wDZ0$Ma#5p_Me}!Vq-MR9_*YNqKNnLMULLkt9lh773);B?(CPr60dxp9-nlT?2*#w8KE#j+d<(`K0aseNe!gt6"
    "k#eL+F5=7CCKgxw|5z3C0AvA11N#7G+b_VDH$K_72f3+WG4D^<Ar>)?(OzCQVAE9UQuv5jo;AUyw5b3YJ!5W34b%"
    ";bxTCAs@^~nG#DpgFw%sjQw9gb*PzeXGB4`(~J}-dFz3@yyhjC+e%5BcQAXvc?%vJPp5zw2~Wj>DbG7t&;1Ez0lP"
    "@K6e$BcJ6J@l<2ZX>NrA$3-6TQ--pqm<ZO|V*(HzD5r4G|bv7>N8;hvjtc1`34<iK?}af)}>(3g5Mjj|}k_|rv4Z"
    "pZ^1y0Id5Lr)OM3AvgpQIRuNUCt*(ak`qbWuBpaa_1fbE5JesE-XMJoX*O`aTP8sN!J}|93)t4RJVw2fO*Ff9PiJ"
    "1;shZk7c#x<3xDY2I5w+@UR_7IS}Nxpj|Nkb78YfUZGf@CtTAny8DTn7o79R!%qM!|1co?moA?bKyY+r4!CLQ^al"
    "7)wQ9jg|8gMyzKbhW6<VDZ*hetgYq%=PT-i^2LZVrhKYPlG~f~PhLPF+x_;kGZ(SZ*_&FHRyEUShMCRzg93Khp?>"
    "fM6kQ6I>9>lD#Yw0`AM-fjn$!EoasA@cZQ~Ttk%U&HS>Rj&o;tM~}Yx;i!@t7WXP684NN#rLi#v_ayJLJ0a)hI7h"
    "Wq_KDgWOwJw<oZ6vR;gh@;=`M^l5_^|FUwf)YJ0dtaE$EaXr?V&6RbZ551{SIrA!T{uWw6e!CXjU0jkXcI;{q`T>"
    "{2N~Pj3Yb=3T0!3brzO{>2oB&zI+FzQYVv>m24)MSHv53T7+KhxSbi|2)`!Jvceq-a(#f(rCa#^A&ZZ`j=Xd9pOl"
    "Q^E2D6m0SQg;Vn+8b~xss#F>VBNUM3O3y(2%6gA}p;}W!&hPG$0*T+!f0uPI<TqA38saK}UVy#dN!-hg^Ar1LJ#2"
    "v(}3GAB!;ZHE3X1FlozwsYH{<zT&#c%Mq;c6WS*(gW;!?BhR^LcB-wJ@kG!=};HVl*}S0l}(!$FBv%c6YG92Y`%#"
    "IpVqgus2uPgr#LD=!4;e3Q>%-l_PWU0_F=rkiEo!5j~A2NL&Dy$oW(WVIHV<uM{Tc_mgBvs4h6({*f_YwvUdE4}Z"
    "bV2y=V{5b^?LBh;xW>yha~^p1jJi)sJNgq-fr&Y#U}-Kr+;pz6~E0N33L_KwE&57eFKlS^Lk%OTJX2qlj`Fw-i=b"
    "}P1PnOkbk$wFskSevePzsO;krUkJ-OrRy>bcnFZE*sx{Wu+b=(n;u({P&dRjamQ1TG_bNN9=KqY0Yeuh*A+KWit<"
    "IrN!%RtYc-g<^Cnv9uEH-?>gGGIUVLZR{V^^Q4|QAF)!J%GE}Kh^b98a**6HNtmnBlOiT==5E7b&=dqw|DkK#GVi"
    "MmK8>^=`x}_)@%E`OsQo}(umWe#U$Qqmd2z?y4Zup0~;O^OUHMNZkwXNTniEj>$^3_YCq{VY-PCzdiBobqsFTo{~"
    "wYam|vN41zelJ}JMm1dcD#A?vhSk6dPzksJ1b}1J;8A*KSB1cjS*g=_8rM(hOXbaz#_8aGLZ=2C0X%T~Nj~FsOpz"
    "}i%igN<Dp9pqXi9z~ysI>x;A&t*2mh#gw(*J@M^$FYeP!XiBsg2QcGJ)j5g>i?aRVnD7z{biuX5M~xJJLTS4%`9D"
    "3;2_R78V*t&m^J(kkbASO=YYh^6(c-MwY6a;V|)w*qT5#$IdKRi<`W9nLMj0{1(ZU(%Dl@1iUM|37<g!rr!#C5rw"
    "PwDay0nUIX+>~b@?iYzCZ*pge4C!K5iqaYHJFeV8#LE2G!{NLYMYeNA9Wx1#C_r1&{En+EDojP^)rAhewz?H@WhO"
    "Gpuf&!)-Z5^_bg;yRLSi52EKm#=xB2w$rnJ3)^zar9(by`r#F%e*m%f}!j!hvv+d<ACawj6N1sEd@&8eTDWTNYxZ"
    "x8z49e{rjNJB$dlVP1)$kx_)WkDzYhzbM*JW`WweRZQ853MaB&$C6!_MJG`}&33K|AvOZHD@@0ap(Nc(u#mkx;oi"
    "u}FtTCMlSC?n&{0!|(J9@m$TE@L)<aujp3HK3D?5wUKPfYY{eQSloEqi*kUpL>PT~dwTf~ZlSzYFiuU+s>v$P+nE"
    "~mdW=1pfeNtmu~+2#a(yqc}1X`PdWQq(^>4ZXdt^+lj-B^`xK@6I`^E!3y|z0(8y@R3(PU8|<|OeOW0@PFpKR=s="
    "h6h_`s{=JpJkhY2H#qIE?prtBYs19P9bCNJQ{K;o^!8w+$jv_6R^zh>B_}Sjs;XXHFv7}GBu4wZJgtrxp+N98Kkn"
    "he~qAoU;>24H8zd^f-QKqR87ZWWO(v6Z+;w%fqtNJ5IeYa_i(iPM7`-J)rg;iP|r$)UARg8Z_A)50E{-;IgAb~)G"
    "k5n>^nm1iqeK_x5g1~vb{8GIcOgT720V~U($bePw6FNv-4^qV+F^*A5sM#Q#*al{&yP9n7)7-Sy)b#1iOzn{2erd"
    "I_Z1rlW_Hc_>{Z<{3wzDUyEHoP^aw9!;mv)D$0!;S|qW2mooln)=iUL(NDoA^CD3Db4ua6_GJ?Hvnvw+6#4}6ds-"
    "mSRd9iD0U9_4CNFRXHPd71wSUmLhyTgu%XQILCAQzdyMDrf2wf?5|XeUzvO+Y&2n992Y<(!RGQaDsq|W#gtH`KKI"
    "tiZzmOvo=u$2I2zb5a^+0Eg8Dj&{%~jjdkN{;wIt&GmrsXu3^tn&K!SXNJib!91VQ~ccLU0jOOWlNanfd8TiZ42B"
    "XSHUbiYk<&bFZwHz24kc_h)l}j|8la%HyPf2*!v7zkk?;kurQ^U#0!G8eahmMJ+NfoQ*xR{yrCv0IeU6oXf(#wsD"
    "=<oF|qWY=WHF*R6#q-ZA>y#?0<fS*ywn#b-0ZwNyMa<@ki5e=bix1jeOJ3(#D(ctFsUA9Zb80LcZA2t8z`aXFK$?"
    "JBgrOtkhQ{nIdECsL+ia3=JqBN9`36<OQymS#Z^12p>dD9ICH9VoCoDb6R7E(qh|){E76#lPJtnbt@3w$VM3kN%w"
    "C^apL+9@=YDJlf2$KXCs-V~omRF;Q)bU^GKDCAg+cpvrJo(^#-z0G{&Rh1v=cm&URaQ|oHfA}WCWh<&H3jnM8~=V"
    "NDykD~t^Obep@n*89Qa#qPB>&>>8X1p{C~Xbt7e-5$CwirXcEo2cC~x0z5A>l3xdLzSPm4mM2Af!A+^^?{dAtB7b"
    "V(DIZD$6ZYsTDk5Rrr$#tAwU?u%&LAJ2Gw2_!ey7Uw+?5To9-7r{HVmPPiEk*TF(n>i<&Zy4j0!A;xwrC0`3Gm_U"
    "TDVR*1d$qR{~~2ud~`ay0oHcC>2$UBre3GIo!i@LRrh+v2<)k8OuO<$X5(~tPS}`>Fr>^IvoJm?!n4ss^KWcAh<M"
    "lRP2-Tjn{po8nL~+ItT^{O3;<=wTbGJ2kJ8b{@#Yko_>n?kq{~qQr3*s#bsD>y#&yKZ0##PZaYj+oL~XiijNjHh8"
    "(8=19<v<*b}i+odss1ISJ0vM!HT&tk|6qN%q`?K^o=chkN?=}rEk-ms*{Y;OW4`rb`tC2sljxH>$;X~s!N8^L0kK"
    "KNBaj)pTghWCfkzhG_7B)m&b1Tt-tC`<e+NE2bp2xtZ_YO+*1QU=H^04%qmcnNnYyhiwu+pq_MY{bW-MA7tB>A42"
    "Lw)D8}hHzQxgqHmjF%l7i&A5ZwyPm@`MyTnd;3&=#cGoCb*`h8m_*UlT4w=G37WBu|(HOk^^T)Kdiu#F*DKAzm(A"
    "Ho_`5EGl!)lHIzJGObm<iww(dlw6J}2n)8_o-govi}qw*qn&#EqT=dqF+0N)CRB_ijWZDSepF*6?$wI$@j7~2{k+"
    ";FT;&o6jFJZA=&9AvPkBp(ut-*eEAblzi#N09>2Ou1xIv3V`9jc7(2M+aOmd;}NdA2zPieCBhiXP$HQwT1H;+%FI"
    "m#pd)54;y*1hn8SU!2g0{NXbt6nGt70T7D!&VO}hOk-WyvyT_kGU6fTzhq(>KcAmBi#gWb<sl&F%5qunAE?qL41T"
    "i2uDb45EId@OhWMErD>Fiu6V~b8S0U12T<3%F(#U$T6r}h3$7i_ao|_qrf%VOgq9IZwK84%xDh__=8zaS4?AXzZ<"
    "dN~&B+}!&O?PsV_G*kI{Hp?3?RuTCM0jTO|(RVuG?sx`kAIIyS71q0-JwoIZqJQCN*%7zt)_~YRs@<Dw3Srtq<Xo"
    "Cn!6w@|4b`@5M>~by#_QbCfYc5|()e_^hVX10c%`0w>tJWl>(|H*4~`EyBkf3wH~bwPVHtX7%_+Xa~g`h!1p|VC8"
    "`B?k9`P4cw2_Mm<LJ-rJRLi@5WZQ8e+~yA@DlTY;KLnoRCj+1!z2=!Du!)zUJ)$QcngIb+=YMnwj8e3ad)I=hK{*"
    ";2OLMCjT~@Ve*t4wV;@s4>AJOx$)*6Dkq68r(s+{ova4sRu{Y#t?C@NR^mPBD>a`7fv{po6M4?`IHu4zg&S@u3a)"
    "8S*syCwW!d8wj^0nVaWe6yS8n65g4}34QVU2Ce_509?(_Wk#QCuWu>mkxmA%sB0@HdXYp&YuCWSZo5u-k2GYJ_ac"
    "g5u)U}vPU1&BWF@9>zj*rT&t>`5J-`q5PVY%+E=Jon)PrXAd(bTvjBZ)mytt$VgVhlmPX3zs|X11jc#~heUa^V);"
    "Hi~gz6{!m(U7>M~x@uQ6^M7MUhZhFo8HC!V@HnNkOy-g)wz~#jO{s-OCTibU-Aqy`q}Dpx&mJmdMa6uCEjWkrzhA"
    "@xq8VL=lq$|Oi}5!miOXACJ@a}usIC#FJ;*X%x)mZ`+u||}*QwaV>zbfI*iM1P0N};pXt04h>jQMm`OrjfDTLzE9"
    "$fAn^G1Sf5r>$x>8Iw4LP7(28i!p6wDo<~?kA3a%TR)r5vunV*seXnB7yDmyL!Vss=MoeJL;xe=QQ&xpw#SCP}s4"
    "yE5my)`lp5zyU#SQc*D5`wA@V^{?Mkh9yYI^rCw>hXZe*xH0=Da3(`-z02M@9iQ<Z}QLnbV?-_NBtCTIWS+UF_rS"
    "(qte?54%CvCm((*DT-qVU<?&rc5`f=WljQSgu+{Bm%TJU=;nws-PJ^4q~52@JOX8{jdPaPIaJ3YbmK4*q^-s-$Wp"
    "2}?sdhN+?Y_~6Oji>GI(^miY<n#bvZ%mtIj=nHPzSb`Od7oF?u&kk2zxtJCgot<yH@Xw>#)sZ2@E{$gFtg8o1DZc"
    "_K*IP>R;^^>iFAhxQ*7!oHs$y_UjiUZ=x^{J{@&H!RY=Bmetr9e#$*W0V9G!-iq{U0lJj8L&uVEebc{Z(^6?y!&Y"
    "?x7ggQl{&r?+}mh^l$yxhiUNXJ-zN9v}QY*qIRvV8=(n^^R0@(bY9<u<gXC+gxV<h@vnWG7pbnm}dt^XFYWv25UG"
    "ucmf3EXdiGcm{M*ka5ijQFPQ<#VEt2XSPu~43jPb!wnh|<V__TtwdL{>Euhm8*EK4u2qxz_eC|gZ={Uz4%#&3XVp"
    "cZCaDavrzN47g3hP=RTqE?{Dy+>UEZG01vrSES18{TGiSW2{-S^lSH3w7LCDcRDTP+i!%Gt4twsl3lLlkRo%gr~v"
    "YV=6W>p?5!>pKlFk=7Q$W$g+R14|TQYoW3&Z#(;0X}Yi;Y+S5fO`IOUoMu#;fJmgkrqg?OI%5=|09X=m7J`tgU)7"
    "v+x$B(oW=_S)OufYMQ3*5gEzd4163cZXH&dIbFs9fUX*}-6<Fk>t;`r&RB-`}ZaU6e*YjC`WYvr6ep^ItbNE%spJ"
    "X-0?CzqF?WLCa|V_R8IrM=O^?ezlRB+aL{mRUx5%Ot(61C+>U<@&>FQX`sx$?YvLj;O_m7~e8OjB&~&HjeJn3#Eg"
    "wwr;D<0YSX9T%}=)jD+U0Nk>!*P<!v`8BDi^l<Ylzoa`Szeevu_f(2{&r>Xn6RXZk0zs`so!mstXL6o4s=sS#O9h"
    "vtqIaP2Phf-8HIs65m3ASaV(O4tX!Rgif=YuC$07EPT$4`>=)x2a!J<cP=!C<&w4v!Ks@ty1)o*s1eem*_{D!C<w"
    "GcA7NHj>)vKEfo@8fviPP=j{B(z3O6vRD(SXz?<D1yoXjlYXNj_BR57W~#5oK~Tpt(8qzgB^PtJldv6!i^IjWoNm"
    "Xd1DTT9vq4a9sRIh}TQ7Luv!>Tdrt*P1VyQ{dAP`H#o#@D8DsssZhYb>!V=|O&)Dv3S%(#%n=Z(ZH$43+sRn@Gfv"
    "X6rfrc7RqxMl7;0+$&sUB<XV?EF65`211@?3(?=n%M1m#RUk&AZU%E2VxV7fKzHB2YA01d@aBmn%g?sKubrHn2hD"
    "Ow5?K!_-sKXPTw@nmn}v;6(^(t&K)MqpW2L;NvU#|jCl&iTJ#Tfj1+aDs(vD;z0DAlES*rt%Rtk0B)bs_;B^wEBb"
    "C6zbz-{2=rv36?W9Rx`m5l9g>fUhH@sd!*i%UKDogLh*4T4b6d;hh&kcBFYg?OW{|&^0uhx<m))hR-3}zqSP0NS2"
    ">xHxGfGCKrA8>;arE?arnc2-WTQ=ER8e|3unHXV%<{YE`J8ptyL~ziu51}4Fmhx~4jrxZcju<Ug_LMsocki<|({w"
    "(nu?R$GWNZtVwX_(;=Z7{NhTbOZ>g)B&XQf43Ob_)IXv&DMJJ0R40k^x{9n}Il{NY$lTqAt;E(}8(S(>@GKkDDCJ"
    "?d5c(+>!)696=73xLIzCjTu857%dUTUzc5{xpHbTirT@`gT1+Ei&wZw3+QU!Z$ZE%*9M+*hYJD8@B|<gz{R$x8bV"
    "od@R06t<mEmpc<(O%|k%Bsg@|~<iU`(dXJsHrp!<rHwcTK4rLbF(=ofS#bNb+la#=&tKQb{i1HZ)b=k&j{Jp_ew_"
    ")^G^M+QGPSOH3pzKBS6MI{l-L%yxD)4mLS_K+`6WxwRjhLUYb^!{;7t{Nuvac>=PB|=lPmn@rt<S2OPIa%24t^ip"
    "0i3cV#`1ecDjZTh#yCBA*Ur*n7+!0MWqwQozNi!6L39%M%S;3S-45IsK3`r7xE%ut&sTi7_Y8jusYQBn@b|;hvr{"
    "FUbuB}bzM6;@Cp<;onWcXsacq85vo(2JBs0?u?{$`gS8=n4ci6F4^Lxjz|4GC`T*3$H6SfuWhK42Bz2d31Mn}VQL"
    "WDFo;xx*jBcjJ$kwX5~-D|UA_t5!^mcf+l#U&ncygTyFB;&N`W|2_kkrt6Y+2U<J&VnwQlB9iIl+r9`lAQkSDK=}"
    "xB9cpP1o|j3QNK2|2}VWGafHoXn`joL`Ey}=WS0M!!VJE$zgHI^#f`JlK3(40zn9tU<}#mVYa4E2Nxo@*p;LNJO*"
    "-R!Hfy_U*7wHLc>L=uoi4Adg$bkK^Hgc1^!s;lKXvXFvhzCHmS$b{x)-5q=mMx;ozJ_`Y_VTXj$b^-rqUY0Rta~L"
    "5Mk^eOMSFjC(vV#qDIan?PMW|OxikmadgDlo61K__GUD5GZvDkbs4#_MkSna@JWK{9q4(%dfa$uSGL^`3najb+ky"
    "jN@X2i|=gmO(dUGI2+xFr~lfFFYf76tEFr8m^H^ZMg#PI^X-R-jqmL|zGCEeCc)|%GuZNfgeS4HUQ;co}Y))Gxzi"
    "*&a8zhXsGQ3f!y{qAdYvUV=gF_r@G({-GZ3A;p|h;ZN^wx+W4%zzqUm7J#HUeC)>c{`it^EaJpjio0*Di7#IQ)r3"
    "i;uUPrYy|#-iLZ7)!j{It?9C)!bohr>(4@CU#T#w(%XNzn{?t-jL=NyXv-Ay6ze%y8A|q&ELaF3(MQ#2#BgeEzN{"
    "UNcc*;zNg6#?{Ct8%47BufTbL`>8;}J<3$$#^YgTL=T)xUo~IX-&&M=#kfzTMuIdBP1U@71w`NAv9{XGb`d;->p;"
    "%HS#I=}mcEghdT}Y!+0|XR9?K^o~QIo}cXf@@!9fouVsve4UNoROk>VN)&IRJ;P!pw-^)5un7&DT&7$vup8#_Ip{"
    "#JfAwH-fn`T==}Xw;S@2l^?qzv9A9wif0NM7u=yd%Y!k^(V9MGSB>XZAq)tu0q5<KQt0j`<Vd2GRX-K&U)l1*?BI"
    "WEjOgHGNvi#pGdHE<~tj?9|aq&sfniDh`P^WZhLTrYe0@EmB<%`LWcN4R2bGu>J(sq<Tw-EbW_jBvBk#=MwhyK)>"
    "*RKZ73o{{9?5+hG1^gWANlC6)&8EgGaGPJdDH-~rFToezTeqo?;pP+yoJZ_IDdc2H>scj2+e5=F~J#R{EW+^h7p&"
    "cgTW?A?p$Y&{vkNI?}F+5@#=jie2t;(CYPOi~MW$&ogja2!`m+>Naym4BY3n=4oXo@l0i`wml9|mfPc-ZN#%j%QD"
    "z*%A%5bAlU%jpa0!{w_7@53n$(sHy~<elz^pU_h?Tojn40aqBKYsko_Aq$aZ%K_0)$sDm^u~^;s)e9u>ktA7`UtM"
    "uEGE-sI8na29nv)SmGfhB1zT?-7F`xMhZEMOcOGbLR(`~;VleK6L8-HHE@hIVWV4(s!w3qoJXutbe@_d0Pkx8*&&"
    "5WF3$hBmXO#o9?iwjb&m{5@HHwY-H>M(;dtb?;i7q^ddY)@J&ZaWx0oBug{^}y&Nefd^jXzfy83KJJ;CPN})9rI8"
    "(`7*JIoCC(uQ08GzjMFIg>rT-8wHC?gt2D102G1nId<;qm_tQta)kpQ#=AA87Qb#?V7Bu{+$1=U!6q38E(EVO_ZK"
    "ukQn~^aa*jSsP{qu{R9)<4>$@DHkd;B9-@5IeHinIVw?J3cc|EN@j1^;8?Mppb;aty4+Ci{%jId@vsV#4SVUp5u_"
    "7_<Rn%dQ7OqnPbVa;!Hx@E2f2oya4b79-fK%7lx3QLKy1XqGPCP;vYg9Ry-!!K}1Al(BdO3?=W#1&W==&dvkwc1^"
    "iWHNbI3R0;3TCIQbJZ<g;+%*Z-*UvM701kK>OQ}fIksWEs|rrdF+-V<0fVTn=59MRmUE{{#0Zjhbkh0{0)4XiVe%"
    "-s6DTrk+24Nv&lDs@b8h&y$HHE#~jOoqwdsWc5&V%d`<JA1yUNQjkYscAd)lvorXE)e1GNfk;I_w>r=l#`XEi}AJ"
    "JaFm2y0Q0WaO2m)a_o<pI5N#v8xHs)pU1E7#0@s;P+AdJDbWXf}ftiEE7-8j0t_fIPXCT%6V^yG7HLz0fa)okWI<"
    ";Vv?qjS!k_0^QHYZ)6&;h@$6@;8EOhqZnaY}~1$EQyRw!p;vw!^yS083R+vxUy$)q^iz?)Cpq+W+Tv|A*1w%l_*x"
    "kWT<?I{4LYmjoP|07y8U5K`S{X)>@ch;f#Nvn^873#!1I8DAPTP%M)2h9q+?Q6xYeU#A#v6Lj|NTIX`M7+p@&EAn"
    "9{{l0F0pYL(SeZ8m^_jRaj{+2{!dRZSDz}5I`Y$>JNo&t8s!urb>fTgcr%^!<`jXdz_YJ9tes>8eVw(P@tme_EOq"
    "^2uMDB7cfuk&n)^Mwa6vS}d2DgpUlo-zbs*t6T4Y*^-~Q@om^-vdT5!qo6RZWIWW2e4D8Xu=-8x108Xzm`z)1p*1"
    "4P56!X4h-@2`P+O^P^=T$Q)JjL-z%ixk&}!}<ild#NBwz)25z}_nvI7$+b~tGA9cDcZt&6%aOuUfw0tw%-u~{pSM"
    "#IdD0}yOk-vojuRsu3F0fZPUI1J_o86cTpf!z`r*zncSh)m4T~9y<`yI(Ip7_$N8$fl)CkR0GJFuy2^7HKwXh*u5"
    "E-gTv@CT$Tf}b6(!K$@nK7zzVjp3-%+?E3%+e<iGGuBdIqgQ`=ib|}LhnydeI2Z~4DGtF!!xLb`3qzM?=^QqE5%m"
    "4VcWnhN8ItR1sCB$Yrb=^bO%D|@2q+Zg8|JITk8!D1z9%#^XLKMh#!p-DmVH=bDwo%y0t~JTvJ<2DKcGq&S6>P!z"
    "C^gd_+Qd&LTFIwJ+oS8v4HLsmH=aPHlz~Oy4`Tu>g7b~-~vJ)w2*Q+LPa_Gf0OM3&dGX}kG&7XPxut$XnK4TB5Wk"
    "=(5TzrZf{3)Jq=|T20Ye9eT(r!{ll#wf;+IG(NwF?;H$>ii-;@+jY<cpNCDLvqpz((W6)CUHm1L)UjT)qQ6wn*N%"
    "@-dgmu#m7OQy&%YqPcjhAwI`hCVSDnXNGvjP=aETlq}G00RQu#m!f!-v~<9iM~u^*S~(f517X>%IiSgpc>mIp4j*"
    "u55-bZR{79WO*#eT=q6I)t_D3=k45P`;wP;1e`&VwWirmi72K=K=2X!_2$okm~B?f_q0&?aF6Uar)a+EovtqE55F"
    "1*NSEaGh8f+!q-ct6ON2kG*UbETv6ecXCsWzRuApb&ILnvpSrqxS7q3gg@Btb+fzN80Gw)fzXw+RNne#YL2QO|?T"
    "!fSpH{Rg!YC&3r<Qm|S0lhYx^fmQgD~q7uM<f~Do#=Lxw8k4`-kB7kzgx%wR!TBIW)uRjJ_4aGq5!Su<0;lCEcL7"
    "A(!V6=Ao@5J?lBW`tNE}*_rJ{?JMjKLv$-Bny2dRDR3w`aN_k<?&J|8KaV}U})-LX{I3RHEoQKFB;Ev0AS9C>VY("
    "@Q?X7(y(45?7cGd(0cdW!M-OdhK>C$;^7N%JD_0|nRE9s(vN3Z4RWo^}OeTO>3!yU9yH{Xr-dkFxncR@q9;e~#J6"
    "*hKejy-Ljw!+agL+mQ>^UInaL0|I+8UIGbR`eD2lI+RUkN)4HiEf=W0T5tz_n?!OR836ebe6N0Y<W`VC6?`f%GWa"
    "!ZQTk&bd5s;(U!&<*!F2|g5;N~M7H^c>8PTs!7fU$qd^`H;`>!ic@^`7m<2T!tyYhJ{0a=-aUxXMotXp}+X@n%3K"
    "n8{Cluah$kC<-a-KNActDXqh9lhKu=quv|`{6BNjoWJuBpASVYie2w@vb<$j8klw+_AHU6Ai>WAqC4M$E~S@hJJj"
    "vsZ_E)gc5+DQl-$B^Ni%uH9aDyu<;pl6dRw?6xGI&ifPvPK$y4aJxkK4t-r#nGfBEm27t*g?r}C`c$!-XshUiv!Q"
    "hLG0;$kqX7vuyAVT%8LVQS0a6#pFtF)azqVuZo2$FCZQ)QaiJ#&%aiUXZTq%f4JobOXJpl;Y=Sl~F|xJYCu@Ff#f"
    "L^kvJ#E_~RU=N*H`X~J{+##oI9>{xNikl;-ANM8uLBgwCrNCSj8i1uUhPX~7NXeI9Cf|SiwFz}ub4)ej{v<j4<p|"
    "RV!g`6A>*f`*aO);^>F64C<2n~^(@KpFXiF}%)JM;}85V=3+&c+J<L^q%X85>}lGuD)-#9~RsUPb~B%ewG?dRGMY"
    "Z=baZy*{rtZImb<ih?u)3BSWS!m8xvuQP^Z)iMtl1~6E(m?n}pcud?oKDwlz4>OFj4}!qc<kY}(~DRJVrNAtN`%}"
    "dazN!=f;yFsk$hx-UZ*Q8*!;+m<G9bgk<zRY&+RcCcVC;7wYK|dUI_NHl0X>jt%q~9{HxtL3{D@1lDf}fp~7f7cu"
    "&G_Bk3w$az{`0xTo}=$iM%QfEV<_Ou}xpc53zix<Th(KWL$@5U8xpzo_%@ugbc^P7&{dD+-TF<0gSl=%%r(#=~>d"
    "pBPSRrRpgbN2gX~T=xmRcc|k&(q63Q5D{&lh9SgsmIa{JqIfrmeGJ3Vg73KJJcN;KXRu7#T))Rnu(=^zc#DZwUD`"
    "9jJ&AkOnCGy#I<H{&<4WB!l}DetQmLW-@x@Py!R1i@_~NJfmO)V-5mxyP7=lkwO@$U_D78gm-EQ@!H=w6HMtbqyQ"
    "C0lBqB17`&2i&H7jN!efef^|O=2sw-GmR4Dwf<cL2zT|2}$4iA@(0jFskRyX~nEDqN7EDc<*jUFMxg+fm&6V9mUe"
    "f^A*Li+KSi6`!O9<28yY>+!8^P!fN#4U>8kQxuJt>BqB)jMlZdsxlL5)sW)P_<FOZiy{;nOEy{cr6!)|o=Xs~agA"
    ">5IW!-~F9X!x%*Y}rdrGIk=OtvUfPfVw`=1o!N6jOL60C3W{J^kGHaiwtcrVYS1B0S_g!(y$z#VRcCN(RD@BS!Ji"
    ">;np2d~vpKK3QTgXNejp<rrv&Bsx)$AbQavDFyXTi!1C#vY3V4NS3!ZS*22KS9^=(bK0XyCELFyJ8?6Sfz=u;;Kv"
    "iJfQtl+bxGZQ4)+BtF=q6NamFVZ81Yi=qUF#ADlf})LFY}E!oDbUv0|mtWHl~Qi_tM|t$ej2L`%<C$&}hDqU7x3E"
    "PpNgkN;P3VJsQ8B%9UdtI&?q_Fjd`-DBUEN~$#FlQuQ8a@3Hu2AxO=eI+GM#=n;{^aw*OB>vMpa!E~noyfdUHoK2"
    "L4!H8F%!k@hpcaM4w5L_3BU#rGDKe_)tZ5WcF~3#jV9969HrI^{6F}9JvNtw>iW5wXKT6np2GnKzrZoN#Ec~=I$_"
    "da%?%bSXYZ`d4lIr16%MG!+b3d)T2=6K2Tgvxsx9fkws<}2O@vmdIl6BqdJkqh6zX9QKZuqUogb{MVHhtc!7uPY<"
    "GoP(y#4oXg@b)%sBet-@5GswHnp5MR7Ba1;l~uhKNIZ$l$#+BMhJWs0Yd^V<e_~CnmN8HQ5e%IyZDOI;ULGbdU;q"
    "2|#9}q&<!dij`y*ReIds|-oFs)*iP2w&F%|as<nh4?CTts;1;BlJ`0Vg3iD<E?$>@H0B!t(SPNy#SbTW5veE}4zo"
    "vtfZw;DxmtwLue@ah48WtTFEw(`l=>ksm`E_UnnUlt#to%b|>zQM)gKUU}Eska*W@7KD3{Kp!+Ko-3MQ)w71Qj?z"
    "{S(Bk1HMU7&iJ=*3feDyUO%T7=2PkZ72^%!9Lr_~wxI{4{G(34pK6KkBP(&YGN91(|Ze|5jd1n^Anps3HBT>lbtD"
    "q^cP^SR{q`rT9`6h=aCX{;tWEH46&_43K^aU!4!016rzEn^JGJxMl%xix!td5g>k#zLKUnDzSAThPW-ZP8USqy*b"
    "nT@EhTSK5+#OT(FeQO#}MVnBCji?Yeg#=BaR3eJ3J*Pl}-!D%UMk*sAu>KHl>nS`!Fvc={%3y7<^_dWv&8&FK<y3"
    "DElOMLNzRWcni~0068oMFPul|9C;Eq714AgOo9+J6!lw`sYU$wy%@jCzyb7vdWZupJz$yeX5lTTFHqkRI|3iu-G*"
    "&o%a({T>BZ|O~Gi0vho7QGbZwU@RQN{wLSH@4rj5#+l;JfP#9mRA}ePKmX>^V4u^3th|auuRiW-I|&k?*>6J|G&!"
    "n5&iBbAU!modPH}+b&t}sJibYmBsXgpN9Z_LHHDHNmW<55;)LPB=p=T$fqfrttQuOk+g-CFEsa*E=}U@ap`kNcZ)"
    "%9JtpRM|%}-01^h{Y3ot0CXEi%En8L=yiY-%zm{Cl7dmv{PC-Q@V`*dFjL*@qvacw;1Eg9*vwgVX&U{1i7RV&Xd^"
    "F7H&u<%q?(>MU#YRj$1m7b{(|+>6=09lG9n&~<NZK59gTXl&uyhTJ&K2UN&$`=O(lv-5GDLz8<QVph*d#{Nk%uzx"
    "b?*FX6JO@J!)RcXCKnmkMvIFB#!^w#&<3}tKzpeBxFWNZ~s0=dO+9q)9FuSPTNJCca6qwr|4JGHu0vkv6|CvfW!q"
    "$<*=uqon!wNt4JloLIx5+mnWUTT}kM+0`r=(Wr6s<+RCCbbQPon{%~u2MnuP@+R+Ss(JEl|wYre1z7ez#j7Ob*v7"
    "O0CS`TBdFGdey!V-zwTzrc=dmdPeTQ^wxP2rM<|K&0OZx&$P5^DM-lRt)Kq<^0zc1r5D}UD``)4@_sU0J?|C}N_F"
    "?z!I#)!b^*UBNm}P3tqEHbsq)(EITP;6g1l1PeTfOtZiBm(e$E=s>BK_?W%-Z<1fOQ+D&2?7!7|VE$Wavj_Fl7mY"
    "+kzq_2;6+qi(HM-8M;wRnX1f=o{xvn(5;AQz^_CD5jZC%j7EqB=Gl0<C?l=I63K96J%(XUxKgfeKuXLezEjYwfZg"
    "&ohCEbbu$RLo+Q4LB`cK>pYzM6{WYcLZ5ffGa<M^;rm;MIpzTTkPFd5i}A&jgOBl|{Jwc<%D>BVhep<U?p8X3=ca"
    "VVI8eu?zmf50p`qHWdeoCSuvZ@k3B8P8#8#Ra9R2i^paZ%GT--5}*UWLul)wY6-NKYbrQJljauR7G}<tgI=BHSE@"
    "JdF9Yk8rWd6n%$Hg8LgM3(`oT;G*9P4PE@aPwsD>8m@UVLJX1jv(Ss3|@}t4sQw?gsta#SdY}N->*XFDbQ7QR{(J"
    "EV~q%RQ*=joIkd?lwecJCb}XuaM072ki7s8vT)r%FkI#bAy69;r;lA+a$%iGpOSH##k(y@;*(i=>0KalhT}B@dI&"
    "Kc_;u6+Hd6+jTI2h_u;qiM;GarWEIiZFG>ZQe`EaHBe9ZG(_yIj<RBop__@SUA05yB>=&P_#tF^co%Q?u~zBfEZR"
    "p(TAA`n1uH2tnA%FHM3E#co-8hR_H|45+fBHssp(#>X3{;k+uN?I$XgfiYIvht>3(bP`SX+G|2%lS)fl-@$27fS$"
    "9Fl_riU(<_;UN+4*j3~pTe4L(#SF%N_3aWM5~R}6j)=^nSowUx)jGuQd)rRKabJ+7nWcP%FqAQejl90UZ?Y`^~p4"
    "CEhA2|1|xU?B<hORRaY#ed28Q2TWDwk08buO4^znnLDk-Fif83q?RBdbrvr3uQ9ryjl2>CvKEo=fERSIp@pbw(OO"
    "*am#xS*%OqOZ+CV6*_bw#LpD*QE(^&zCofjc4v-%w-?Kc}HLiwuix*ix_f)veNY!b&byB?zEpsX6*pWccDEv7yD*"
    "x80W^Hw58~B{M7=;WxK0`)Wf>Nnlt;jKfEKi`55hx}?g3Q748$5rbMgIuF(o_dX8!P$}V!-EHue#!ZM;dGL&g_Uj"
    "+IUP)x6x+B{+aJ{Un9R-<Eb7=2oVlH#jlZ=X-nFfWZ2^A#<rs47^02fb#^o!dn$pk`8h^Ixl9Ke7}E_VF3z>+rO^"
    "d=g>>rXd~aPsdae@(X6rby2TAKh47f}R<w>PWn=aIIEx!(%McT%AK>cIYP*{{a)PC{U=q1QezwMVp5=Td_XIBA{J"
    "dhcfqQvbC}FRSFffw6tt$ZE21ybE#xi#Bf$020_LE5D(IHCk!BQI*lSYejL`^<QAKicoZ>~2FLF&-YBuwrF8?Dt%"
    "@NFyOe@&{pfzmYls?tqFY-~ztOP6RDlc)NRw@5Yo(s+U4wJnT}LMFfPLuv{l(5&v9tDw+&NJDqf*J`yt%2{!9+HV"
    "Sor)d8RT`-s52x+J#E#4BEf{Fp=|}>I3{y5WR^iqQ(JG)Zd<b?0i_+{eq2IlEviNoj2IM#v5Q~)lx&dh>4VUezNk"
    "y@{AgR&=F5n4vT)vVY-=o-0*VpY`@+^2PO4Y6P|wFSOlk_R7T+`s^x_md#~nPjHJ~WMx^=rLk;PjSUT8?pcLJiNK"
    "(I75;EE$G+P``bMoM>MD>{2|ax^lpXlgI%yc;ab1nkrW|HQPQcjzji2rCm56nN`2G|L4_q>4n~pQ;59Tc;(Qh;7m"
    "mQS|D;2NzmA?I^tVe&0JhgV#p~e?L1oIof;b2=A3ok<Eoaq?VXFe2HJRUouY|K7M-8*j`WSfMYwPNS(XYJY6oO>m"
    "rH4rh^E1rLRE(TwY|DR5PC>*gHL)_9u%RHJ#@x5dS}}#vQ#fE<o?&ZZL!NVyy4&W2H!H5!<$o9p<x<``O;7-&}(c"
    "i1jl4)OGFhiJ9Fzdxt)uWVIZRNJn}#f42Ac(dik`+0oB`oPp#pG`W8Jre?^{Ed1q5-ycR%G{0j)@#Acqi$MQs{ye"
    "?Kx|80E^bRK8^hi;A-KwabH}tMal+8H!mVX;povn<A1Uhuxwff$k!ORo(Jh{!Hoi8j_^N|$!3pI3Zxdi(7)UV8EO"
    "z;#=H?n7-KD^mJqaO>xoSTw^Z+H*f*EmNG=bkQiEo#Ohq-W`$BdNb#Y6{g?y{Mb5E^XHx0<W!etw|Ct9%}DZt?RC"
    "dyA@9xnMK(x;U#6Z>wf2XQVqasQXwOWhZLyKFzqSOd)^9~vmg`u|J^o@HV-Lc@sr^UBkGBZ2)I>34?5lMATJB7B9"
    "0}aBX8>ZVl6oAf}8zk#$X1YpEOnSs6~HM&1g3<L@LDk=iW@cyntH7C*-an0TMGX8BxxFL@pS(xxquNg%eezEYcZa"
    "6^3C!%irzp9phbo%WnCcdBX*TW>`()WI8|mLKu4p8923%4^Ocn?_+BlTV)I(1@ZuSl0_;f4~NmuX0_y(9+3YkW3`"
    "&c(ec^n@rwg9jT<bpQ=sdYLlzJf%8n{_f-)m<+`l$$8plq3D}JV<BWbK1CB;v4Tn>fM7fINpK6s1e=uLL3HOfVPD"
    "Hi3hgGZbMFD!0nSB`o2GPaj^AF$_U8fvS2(L^XPFav_T^DQv+<l=Uj+0;?m0^TX!vM;Pe(h<@zq_P~f$yPr87$6X"
    "T4qG0!?-sOyPtI2ponu697=N4bM0vy-FC?)(PSzJgkC>!V`&!)QBiq+g(0ZgaAlpgw9Q4-bRW}`h(UZfI1Fn0eK-"
    "3yrd;88AW#I*>7>R=%BWdnN`gLxyd+qit&U6tJWqeDB`35j##5J*6Jrf%t84GZ;TeHJ4_G^7lF-}~=jV(ND(o@&5"
    "Yir%|k)h^MbINqlHNBrKAzj1K8!VJ}u%`xRFg$K|J6QL2p1g2r(<&A5ISV4E;RvA)e)zW+2UOh}L`;-{wWBL_XT<"
    "WRZq4lVNA%?2`P02WIP8p7gyE|Pn4els(E%qRGbVE(-j>xrpCa#c1P3elvB8?xiPLRG>Kcro!|jAiz<zH%**m0Y3"
    "7$&@MdIjS0pF9Ow+4qlV$wWek!wd|i9qsyf069;YRHU8Q>X`<Fqq2AXvg*xp7>Z44YSQrFgmUcgJ}X-AVbyzs~&T"
    "CqY!Tu(d;hVt2WjOZBAe8?;jjI#-N~zz%28NyE^VThX#QnM2+p{{jfrUFNxQdK#c5#S-stkZoRFI+J2Oto$c&8lM"
    "dC3>?$vjpddM-m{u&;JWi<cCM}$Hl^!P39Bl2oY*AA0ygeBt$M)mp3<Lk3-!4HbNiJ7&?gT>tI!M(@ZEm?tZ8ul$"
    "EyW|g!h-4MyGeS1!93D>XgV1r`?QP%yT(onj4(@9H<-VX;!qc+o|2RcDqvU`5!AB++d*eQQt@DY{_Z*-Uw`f_&-9"
    "F=S<t?4X+3J9M$*$&w!{|7sI!jYBfO?5On*U8ID1z}Yd-AC&W#7D3meVKEs-`$ukvv^-Nm<du=BG4A9hXQGT*r3O"
    "NyUgf>@Jc2uC?c&bb#K*33=Oc0B^{yv;_7jI6Yyn*w;#IiU^1+B`l;4lfg~o;pd!X?acUw7GQ_cfiKAC(|5+#MQ&"
    "?ov&&6yuzHi7Q^E-aP}w83d<FYm%PYXL3OOK3OI&bEwj1pt^<&HXjjOnT$;sUcWfh}B%i}4KZi+7@+&O&T5zliBZ"
    "y3Wketah<e@3<NjgSmMs79OFpOKFi5sLza3+9Y7|u*rWj5)YHx4$2mN*HzRN`&|g}urFA@Lbph#nN2p6r(3I4lN`"
    "w*)+6oB?QN=>l``vDqsDKATfB?~Ljk!UJ&3R@BS*-@#6em9w(x<kL4-!2>$Bc5vFrk%%wt8m!><c{UkK&iMRFWx^"
    "U_)?;Eu0f(y_oCPkG5Yq7&unG!a8u?9JR%2|6<l3DTgF{0z3J3_&j<g3H;U}<(B0ko2EYu?gLbZWQo%xs`@Wi*rJ"
    "1$7FtvgV9?etu+6a*yY(0BP}W&M_REkIOW`xPlRv~K}x1G+cqnfPS<%OP28{xUODN8_87Vdn7nU2I-qqI9e+luVA"
    "|`HYprTzu^}chUv;Pur`N$DLJ%?|8S&iyxf5LP9+nV>2ReqiB0_)2VL5&DcNQdwOuXf6zI7@k}DX`5PcbZ+DZuQ#"
    "d@!bjo`|Gk}|mJTv-ubpy0YvsoOC!1gq=7&+*CSlHj;9j{*PISSn9;v-}RaParN{j;ZkB=4Bw=cN#L9c!RRc`qri"
    "u~D&MFK?;mo1k3(X1Xf-xY|CiUStatRf^TsHL*kFa6pD-8s`KX)1|mPr)SPFpfbad<{71^zNP<CMaOey0yzkeKm%"
    "9s5ot7qh{ogwMkGf#OD;DW+7CQ&g(Yh_5Yic+M=(Yn0Yae~B8z-UnZA>7YGSA!7=ZrVV(>KyO;{<n-)C<s00DzWn"
    "OLhyooc`W6fObduLsFVhQt$N(C5kXljO69UmI}q_GtJuO=PRY9N5G`0Fsng9FXZyB?VoFyr(z@u>1KBK;NU&vy&J"
    "5XNSi}yC@eB<3^|NguTKX;F?VcT>4Ea|4UHKmkMi2RGBRD%O&1NhF&hS8|Dq(Aa+r%fWQIiSl|Oo3iCeTVZ!8@G`"
    "X2#9PYVj5C+fAWv`?ivlUS^Y~9&(8Fk=dy?5Sx*4cTF#nkvfA*s>7Q6<&WfY<eJH)OemSlPrX>KO9x$0tu8Cr{vn"
    "?urPxpzf`J`vAafPA7|F{!P|dvdFKlm*`jdNx}Y1<lp36$TF4#6Ow!0Bb&3{IUgjaH|b)50HkRna*_hU80DOc1c8"
    ">(i8VtpL@60T6o3rzk=mL~`T8zxl`ww(=<s`~an0kvPd;ba^b(iN?VnhlR@oCt$<-pu<`i;`@}lU@*dVlY<^;rel"
    "HLmD(rc)30Itb7#fv1rQ!>RxMtTkj@)pA|OqmB1C3c$5N%bTl6dMXLe0Z_TVudv(nV4d8*9ZbCZYA8<b}jNa!AMx"
    "%FqA<Eoz^k)1}O{sc)eM1ak15hYQ+wVo?($xJtO{Qw~$x(`QUOBTeBQ3G__p5dT>-Y0uQbjwZ8W_|AA*L&~`uOd}"
    "yqr6;VueJVy@kCDRZ_z2x)H7g*Es)dQ#4!>;|x2anLi0T=<EouvuM4CS0&bw`~`n}hSxS;FwWT|m;&)5)bT<=58h"
    "4-WN^d-@Y@`hEf!{B7yWs^%5v8oX!r%*(O#1cZ;Jpp!y)<;`jVT%^ocUO?##>vw=iMt36uf}x~mc&5or<0ptQ5ls"
    "{7s4^elCP0LigXA1V#KB<D#qHcaH<ae)CZFC~nE=_HPDu<soT!%ce}I1ov&;NXYUNv!UZ16d<X2<{JYjy?k%3TQr"
    "|FYfCi@2oOK5liw$bnW1IX#;+!z=GJiEqV0>qD#;uL!wNc0Q^7J!MtXPIZDl+(mZw1;32;1%_@j!-@k*@nqWizOA"
    "uvMRG)okl07uATMK#DsDP70KiBc~+3#fGCMpP;Z?B0|cxOC#Af~>>rU*cTO1M>u_}}(BPc;uY9gIv0EU?C07|5A&"
    "IDTF|ZT<dhI!a`31M#IdVeMg3Bls4*=lMt4jE?swBI~%h#2=R&T-wd^O6Y9h<NU8Py)=ib*%dHJlZ<U>vb;xI=U+"
    "D?KWL^G=pGc|A$5u*26f%1~|=1(3-ZY=kL<?NpDp%$Dv3v3|~j5e(T4GYU3BcUL3r32mtAqE&N0efy<Auu``64pr"
    "6^Kw7o-lE)VE@t(i8C7(mKf~HXJ*&Rzp3CiMTf8J=fxIQ&W()s@bYdgKhDzxrZL+li-P0$6gT@tA!6^%)d3r3#~2"
    "}mRHFnq^<+UU^&h4&7GcasRvsRQ<n^n%<q>+Ep@?iHE^c4fjJ{H>5x{AFlfx`!QkFQ4jJh`-mrK2%$0A^?Q2SvEi"
    "`qxPRJ;S2BI!(@?YT7DF;ax>W$^s|O`dUP`)eYK;UZDhk{YALNp2tKYT7m&%;B(g#rUp;sP{|^2OgtLx*h!RBg2Y"
    "Ym2Pu^4CVwEdDSV0CM)w%sL$EZ#7wV0GwZ}9nOfe*-A8qjy6Jp&S2c}Ee3GxmP+Yz3VE@qdB1(n>-hIhQj@U#iyR"
    "WRM6+Z;(wsOAfi43hXS4z5j)hV9Lgu-DLT$&}kb>5Q&BC*uB869@>IeN0d&y?3JWLtS>pJJBX-c>klkPeZdXHQ(e"
    "4|>O5O|nXGYP<1lp4TbP7otIHmI`+%Y4U1`>5ZxL@oD{;45?N%hapmVGiXKP0{>W`C0LeTpXg-1)Ja?OY8RL%F^H"
    "hhRFTb;6Nb@`BBoE-deczSm7hfBW2nd6Y~mNu2$Hg>z74q2g_yP<+=9&G~6ZUe!C0myyvneo#5-uvWjsWxeQ2S_q"
    "H)CS4V1st~-X8Zt9@SI^iT{hZ%#u*DtLeQPM%%;=iwph^!rjm#H`O((6SfSyH$ms~l>1hAggZ<wyt%Y5RC1AAfMi"
    "7fmQw)fA;Hq$2I5yJr1E!YZjim5{2@f_%UPzZ46rnMI2FaR$2g7WV{inyL2al}>W1eVV92+n|YC9|8(h4x83j4cg"
    "KUR2<Zfk-cQXbBPSj>xy0;A2aSQC3sui4Q7>Tx!wViDv=302`s^gk*Hi9#rltSJwksddZUi&^fyQ_*%;x8*}cPpT"
    "+uaF$<qz}$MTU;VIUw??)DgL|+DL`f~0s>+lfiYM#4oU>_@=d?Ed4jIY3_(!^%{QUG_dwVD8lMBE@47OUZotFCl{"
    "JX()Um`1VbQHbkS|Jo0HW-aCTw*l(unWR3UiuIMvNmvOnpdKh%H?F5U-XGWpacQP-^Y5AB3={ayKDAey12SY7wAK"
    "i3;aLuy}>M7rWl)GZrx1NC5ED!3*~Lu9FcK~-HgelMo-<O%WD`}?_%}ex=0IsW;BA!lz%*`i?KlXnh1}R>~d9t-W"
    "7kA62;B4$4oJL$w@J-i{L00SLr<eCwuVpJ?C1@Yg%;1d<%0%4<8?#oYuTR6<6QpB{qcQH-I?fc|jU&Z+fu}=(YqQ"
    "SRvGzV6aXO%%mnIJ9==d`#p$_=bz}vOV88s8(7Lwv3!EvF)BrwHA|LD=g8FE48hT&D3%>OH|0%=Wq}km2S<91)`M"
    "8H>q%EsMcTYLz;Uo%G}(7Mo>#uWkd)4-c5EAGa6>VDqv8#aeL!=C2|)d_picK+^^%>h)x2~&NztRMJHc+BaC<=ak"
    "mqc|j=o6WsSNX%VGLlbaHFsadWfP?vPY+eSLh@J;bTCm5(!e=MT)2Acp!_^34)0Ce(U(REz%v$`bl}*IyyLbe42b"
    "U*!lX4tq<|>dGbK<!gYAV+6D8#&kj#dG5sJM6vsTumRvYW+WrO=iR^=C#TyWEc9XRaKG$=(Yb^}e0VYK<pobPEdv"
    "jg~`4UuUFQ#{#-k_GLkK<v#sFVbTXKX)`z+9q{w5~fRc~7q>$UvcAR63QqV^6Sh)CACLk)>rNA;o^>3;N8XM4#W4"
    "4+Zaka23*Q+U_gPIyM`=tb@p_2VzxBSo3CIooh*4E>g!L>p}#;m8COycQ<T#2Jfh43X6)$W1&He2EqDxv#fapekS"
    "@lmFz=uGAgraG*u`V%e>GxFRU4uyyk5OrH7*fmLNr36o0OP6+h$+!1eig*OSistzNRlwkLYG`=QIZSiDn`iREsgL"
    "o<LD;mO<m9w-E1z=!ud`Ukh{xFto@M%Pl1-EdGW35U5P>LM#HFG)_qQe*jeHC@dH$x{r(w(J>=7zGKZ>2%;HpD=U"
    "T_Pji1E|G5^!IF2CRBKQ;QcMaz#;90i>F0*?_fv;tpWBZI$HN~U-jol7<518<iC)bt(dz$`3^$UToS>Q9pzD`JML"
    "(2n$CE1ZHcZ$uN)V6$7hI8OKw|{#yc=?iLwG1pr};loc7{QSZGQ--w(U%f;z_C3#79oBZf=)c=a+hN<7)%@0W=x2"
    "O??(%mhqmR8Gs`7>eW(yVBt3Bry^V9;zWBL&9ir(XmzR|4{2#qk3lZFT`r6qwak}O;EMLbBr9{u7O<CCWpuA-UmJ"
    "5lwX3=3k3;au#|K7-j=%+EmHDRv2=8$I7V|ru1;|e1-a=_BHH?`YeGU}iLKK4Wp~`V_BT9!SwNA5c<%=HF=#(IVq"
    "YuKr7IF+w?@|PqKo&j`EUlqcAj21&^}*a?AayU`Ue8UR%*v3I;b<;)oXCX$40J5ryTD$Y-^^*64kPDWZ3HU*ax<u"
    ";Die3HiD5~%eo8>~(T}&$oTOp*HJ~eT!yBo@n@%MmC8a=95P0X5FuyF4bT!GTDC*w+{=xIJgU6hgY0YwYUjjCjpp"
    "@4XDdR}q^tp5Cl60P2<nt7hNETR<DLMVyQ=8e6!1F*rmgxe$`O}G$NCIQr(!?^XLnZ8#a;Hoq0+1mj&IMKFHyR4V"
    "jpI$K#(0`jumFySm6&m&7(giLRn)QkI+Khg?A_sN_oO%D#PHT~vBFde4opH%A=Yli3?WM2f$JAV@#YbRDN_1W%7)"
    "y&R7PUoIGJxE2FCVh&AO-LV0bPoU*j=0b5Gq7??uZRX&6DbX;Om?#z9X{wQZ4v!w!>*XYKg6N+u{*+o>iiB>kGS1"
    "XR!)k`Drrmqpa4$G>|%xrzW5TK_<J<y8Q6p(r_6DmI@^YD4kDbzRz7S7Y;92I56RU3llw-C?U6P?-Nvh1oSYpgT{"
    "+2b|D~NMwrHXKIq@2L|v*zi1bz;b;Z#_4zIL@vYb0-5)_wNSsFBF+0S2`Yb;>OfJ%CN+mA?=%^d<u7?7BGreU0_{"
    "Gs#=X1&<nR`y)Urvr+JV&^Y6URUaf5nbNboFyu@>gcb_;Pj8a3Yuvw_c5|!3{TZ#&DyMjmRjbQZ}HDs2vOnKImO<"
    "h^KLvrNSO_U;7)mv;9RfXCrUU=b(^o#0C)@QWpS0vW7<o<Fk~4yg4fU%eX9K=nq|@`7KU`re!TwR3DnNZ8usTzqy"
    "uo3^2hnDy7;?L08o0f(niDT6%K>tQF*$A|l-Lv>L@x&|mY*+fO8sa-)M%SR{YSs5_fT`*fOUFp9h2VWY?(y%mgWE"
    "qTB7eDCyhi=(aRo+MO??GqiJTgX26CMQ5v(S+i~rqfO^Ms#9b9(u>qo6%<fiNO3>a#SS!zDaat$1Gd{VH0i4*<z6"
    "F*_35YUP3_$1Sylj#F0F^I%1k&DPK+?UsBd8q8V8?&`k;YbaAQv;ET-o3yt5_Ns$(1B!EDRT{pXG1mX8~Sm*r@KG"
    "n@AR%T3u#(7@BGlB=oINdi8d~XJ6)=`X4=ngu;*>%`81#<vzFz^N>=CWsc%FO^FfgZ9J0N1b6#iaj+o3C&qV51tG"
    "_xmtjcAik_iZnqh?ljCwhJV?Nm;H&`+3#zX-VG-5a1%F0K@#-KWwI)<8o=+8Glt1ZBx~b@G3lQZ?2AQqwLr&HFDa"
    "*4c4LBaF0$o2^z7r8VF73uqgGA8tXM6f8efYQJE%MR%f+<6pM05|9{zH8bcSW0-jpVf4m6|^Td=^q$LR`Fx5fZ5?"
    "1W6nIL$68D0#fV;75w6O0j*-IJ-zO1_F-Cbv}V@){Y9<tk46t%d28{_az!|d-4Ox0w8$F=LNy0`cKbhQ}Ux?8>F*"
    ">30WydN+idp35u0DAY$?u_hAkyi#dES@&0YL>|d<%sVwtc9zRby>_vtl`cwGARKM^gtFm9N=5hfcn`5^bRI#&3*U"
    "pSSbbf;+YGJ$=bI^`{kW58W;TDcGSz@yOJ1i$L%P?^;FK4~P8B_Oj3Ze~0%t*Q#`NgzGE=(2nrp&Y=X)!LpWKL9m"
    "`EdK;xBcz!`wxHUV}$V!eVS80FZ()=K9172Cu=Z+h4lNHT$Y!rzo`!BdTj~fJ<U_&gh`&Flmg*0y<$S5`n`zcST_"
    "!D(!#Ge0rT#<p%4n+Y21PLQH2|4Lx%)nFB8|$hLgJkDd7#>O-f9mMRzl=40GzelW*3Cx6FQX?Qxs7JW=#CC?NP_S"
    "99Kr?0mko8H}1*!%aBgP6zOTw(ANh)*P=gPIL#BYRQgLkOk8t+;AQX(Ivt2fwg9<r4}e6iE?=-Qp=fvP>SInX`Mk"
    "MV2((+s7^fmU1Iv-YN|h-qZ)}p!j&5CFmeqQ==}l`IG|k|5tubb1Fw$|`gGYDsS>k*?>f;3j1r@Q;x{%&L>guo48"
    "*7y%*kP(=xk+a24832>+V`r{5?K5V4$<$#TST1eF)X=FB0OoW$|t(&yi{JgJW@&8(#S62li_@Z&>eEH~NJRk0N4S"
    "*Lj~WMc)tQ4}#rpLa~?v6)E)B{h%fPdfsgU8r)=fV1#Cd9*orOzYq~9Oo8z`5P=g`B@3o?<`_(|Iys7}I4tmR%pb"
    "=?_MF$mvqw43Cj*bUg2=Aqhtr8OmCp$<#0B#S9bew%^IiTlakT6m$PLOay!Y@nCdPnR0mzSGsagVP@9E)x9uUvQR"
    "CXG6n)RuS&UxY&6+S)NdwKvP-xf^kJ5*qaV+TtB`;^-%aCy*ldEM>7%11vR?46yWr4fS=wMGYplDf+f!DyQa5lAF"
    "7FD{V)04pef@*T$HPKqlEV_cO^_A`+hthEJ{6eP6e^tO(Ugs0!umlq0PE{wFmB&+xqQ8uRv`K9Xz;qu~+XIV>~g`"
    "ZZck!qh-t;ag_M$@|$3j~icKwogD8*%a4rl|wP3hLcJu-De9I~Jih89W4s%(7T5JVwhG!edR4)?`pP-ZE^X$ZvRJ"
    "d6y%)Vo4cuQ`5;?R7)n7%a*n#gAIehz!;9e8EnXg^aG;_NVqOb(HpPkzOBICsiICjKydWUOhN0X9?rPTXR9gE&Up"
    "lfm+;AK$$?t*K46Q)6Y&FRFq7M6pa&7}{xKf+bNV=q1M&jBYThCV#gT?t^MPw9oJ1oR1YyzJBz)Km*6f|fH|&A6D"
    "lk_7#k-LOsIAaGTpLSK4Lcy&YD9at=+K^-t=wB&t#HN9>85rj;Ib~my2m={1P|u)Mw)ZGZw>HXDv!g&`+bB37bhj"
    "YGko;`<DL_G9~S0DfHGBe{V6=@hN-xqDz0F_AaKKr44!1H17mxYR}Y?|D=b0(o!V!$=mkr%$obW7&_@+hdWZO7+u"
    "W$K9KPiBzwY_gI+|q_C=!T<fZ(#0Fdyfd{5b?t_P;Re?5bNP@10;>OUTOA1%HEL=Iqz(^$x5Wp)Sf56?l{oKCOjj"
    ")VU=UOttUQP&Xl@Up>%3$X<2%y0IK&?#2m`>yd)b=5KT0wVa9Qfs4l-lo>WpBXlV{=--)Ui=o7qn(1O3OsmP_gUw"
    "SVNy+l;!Lpc5O^*Mxz*<IZL<^>iBiBq8Z<(p4&47K7u7##JsCgvoi(vCP42vg(Ko{S8ey#xns1;;hau%@})t@Eak"
    "4@5hy#^dMy2t><x<Bf`B2P2|20n$jRq@iO4Uu7^4v;u>^w~A=5p}>0R|OHs#tjgX0YYj(UKum-nuq#*i4XEgDViE"
    "(Hd|ixO35}s<$17vY<?Ptp2PJL2L9ar&dZirWUtqKhr*Zq6(ENj=QruJf4u_eub->QCvitP{q05n>|n3*b{rd>a4"
    ">_87#`Dri+H;c5jEDyjoqMliV4P4oTC4`YrhmO%bL6!AA{9nc$2^OJ(lg5*2<SJZdtEGn=RE@Q&oG-@E9lMkIr!|"
    "Zq$49MMJ{uo3j)M!lFM03BAnx*W=rQx6FQEG-v&GP;-H%-Jk`txp0o%;|Uz87rqZ;gzI3-L(>z{l`4$FwMO5UIIK"
    "3X!uR8#f;$FMMzN~{7C+Dkuk25*GUsA0;A|q@OuFuuTHvlc7@s@eboe;2wviF_+HiDwrEx{&HTmd;YW1X2u`QG6u"
    "#AAGyVI%V?(~qe26W@p?*QFzoG!9PBprlZK+4s~BQ7Evv5g~zJD^_l;HkQ~!#(GiiGREPq)jQBzhUkY>X4d70S*F"
    "lUuI&$;<JwPV0m&43zl-(LuX)rw){W?59S9#0IHMTwj7E+hD=yj*fhabM5B)J)egaq+WxZOaagdkK>|KSa>ML`k?"
    "Sam6Cz>IgFXAL4kbU@j3_|Ued9j8gq1OAp58&^@mdg@_=8cH^XICBR)&Z}hHpB;JSFM10SQ04Mr@$b9H9X@_QI65"
    "zYU<m&$oa|{K1VdQHH3K=<%ji9?`Z^Pet(dTGUV%H5MVkYWGN(%5erFX<j%;JS!>^G}(!8{PA^>kFydbRxud8Zg`"
    "KaKt3=WLy|w7z3vTJqpQL#%b=|vvklfb`KT=n4fqeiuoCNOi+6=_2O4jHY|z9Ucw%B1X)oBJIzp2Bf?l>qWQ5-mm"
    ")chH(zvse>w+rw_9DZ0p6u6X&)Xs|47^Vk)Ecj~`i83mlFzzV<FJOkdQhz&(hWBzzzk<M*n{cqgt6GLrw!{vc=vs"
    "jjs<E2(IH;kMvtNZ(A!d6sx#BA{Si#OM#eG^7%a3dh&D)Q-4Gv*gKJ$^Nk?5*;*Jwv4_>Kmw_+d`uuXBG1btA+tl"
    "-j&Qh63*o2c>fHMzO6NfREJ#YXfkBRj`;-O50od(3*q#N}b=SKYTflylr;9{jNi!q8S?VIsto>xb>_Dhg`%)Qb#5"
    "c-Wkk>g=c<P>E42>~iZ+NR2MJ8{%9OkjM$+)6wxXcfj-Rx<Cs%zK&GAW6F90crM`DtQ!GP9JS4TtGbhGy$x2=lOK"
    "oCcZ|ePlX3*1xgWJ!m#n9Vm@y790|B8k@j{EX>;d0;6d7}i$<|g);Dw>4*J;^wFbH1vN*g_)c_W(1yKL&ykT14l*"
    "T&s9(Cb#9{Q4lakQ%eYik%qR`Pn$_+F+D@d?U2tmE!DS3#U>z*Ns-8I$(bSJ-f_+QQ(zjQE*5aOmT);AX6Zqq=|S"
    "dwG*mYu6>}};IOvs>){|9j~#Wa$reU=l;0f-r=IwMe77Rsv>qwYYUdq5skghPhzXYs^}6Q<?nUEI!^{KEr8S^G5J"
    "yWIwW*X2`qQifQJ_-w)k!H1DR>fVZO5)TE=d!KIpu_6u1_<d9)fb_KGb*<scg(;qQ!04M?M-Ms752>)#MJXr?&w1"
    "*#R7Z4uP`URP%;0$*X&F!r`UFi{(C&gqzf9v8B;<Qb(R?d_tYrNJ{74uXnACBk61R)^g}y)7buW1zWht|4DYLpbh"
    "<U1yU_23i}MtL)ai>fP=vRJK7_5x9dn`Usrc*_7CdzqD2^|Zd-b(7wBA(4i?;12&PFsT6^?3>SrX`6JcFRxazjXw"
    "qNUcUkXkpRZ5(4y~2D>MV|;kxhe6gWNRl|Wr?WpPo!wC9(*8|ez7zkXlzGJ-_e5GTv!W9qP&Z-KYgllTxsaurr9h"
    "&S}Nb(sbOi@73bOB-#aiQH^afEjFb#yPmpecR<*RyZ)j9$o~brnbP5x{+<qO+w%f3GePpf?z0jExf{?MsAdKHSw{"
    "4I5bz_<?>23$t`j&@SwcHldXG%9hNYyrq508Q>UP=IfbD6(`hVUS=EzyJwlIny`E^Oz+;2t;QviImj;fdY#YOS_z"
    "#Z-mSX4n=Kcy|C9L%C{ARp5JM-e^7H<u!Ff8qvfVthbT^$rNy88Io^}&|cbg3vfnG9>YOQo*q6sJOdWna#c=c@9}"
    "C;77IJSVL}DAX|dRw%6^I3Zc+JDy(#ZdeQmB6j1Zz=x6XFY4NT3Z`BiTEag;Az{7H7Pk88)h)T08f_$21&x(L;MZ"
    "QL&u9gJd!1N-2p=hecic~4UtvpI<SXou<O?5dTLI5m+MpZqAm;{G+9v`Egx*TNFw#J{R~;6Et+`C=`{5H6PWH6Vr"
    "0iSS0Pk+3I>sY=fR$Ka%R7Z#n@CCfrz(ua0m`>mV3F(Zq#yf41cpcK#r(8M=k`S_MFt5g+oW~SAJ;P0*v3|KUYC="
    "M1lvPQ?I5qsF|&C0tbWQOtE1*1D1Zx2^D6A7TOWV9p4q-Wm4LJm%4E-4Kp?>26`9LHp6#(LS`c|9Q7U*CoPep*wz"
    "Tr)zz8XC~MoGwNb@5Dc)$2khH9SZ%zi2%Wgey*p`ZNXNnJ<s)E9m1lo&|^WOT5SXL2yh1OoMQJqI@_jCx`Px!r*n"
    "A#MQ*q5I{I8j^68jNP?d~7@73`6IExn!&q{)G?pQ(i|C#>35z{xkx|Y?1bo7(aQcFUeb0>V36_pxM#F$d4IDBP3;"
    "Wu9Hy14vWxH4tYw0gbo;k>^8hW(^2x_4Bf3V2(M;MAjo+O6_xV(gyc#s|S`T+#r1k0>F|z?cLr>@USXY4!BQX;!P"
    "%e$W?G{0ZpM8}h+2OLoEFoZL7T`05KIoYdsUF?%rb?QojTE+#1@J?(;IO1V=wFv%jE2}QPmZnJ%elXdCrx<;aBtL"
    "ZZDqbE;Kof;#xM<L2def8;J%WF(oPO$7csNkEMhFTbT^pT8QK=wZU28IUzg?V6A<w)!4qvC|Xd2juvH>i#5*p(c6"
    "(pqcKDFc2V9k#ns#Z*o@_Ve+s<nwX5m`n1ru^zDpxCBvR*MQCCgKW<o;c!tl=zfQIAS}d=<eP6{JC;+;B~nKC?Y3"
    "}2hqB6~P`jFAH2%VhA!s)-o7&O5Bxl^f%2OxIwd*_L3URzrOuwFwvyvn=a-)wd2z_a+8*#%NA7PW(Cr=Oe&tQaZ^"
    "7uHBHm;`!XO;0-aGkCu*<=vjt6E58E7YrBHW)&T;RDfVBnY&r>BHN4-{8@N1j0&Un4o5l2$D14Nrw$EB19Q|ax*+"
    ">V+|zAH%!|1tEucKNj5$rRlAMpKDKEwMpF{81filZA0W-K`;U(SKdIsnu<pMakOiy?t7iaH19gGELN8N;#|tcLN}"
    "VM*e8lr#VDVQjBG?{IpEF<-%{UsR&IalWZ7pCF+HQG*{zWnw^Np9hUT!&txn%mQ@<I);?Kn|*|ES=0rd|X2OAG^j"
    "oAEKFQVjDesvG!Tzp*8ga|=}k#2&!5LuH6%V-#D+@Px9Fov-s&MJ94yM`?nFKp@1S+YX5J2)hP`7H8{$=z`#jp{D"
    "2E!RGCw<r;IhA;Qk**D(sw+1IcO`<0Mw$Tx?ESC~muM|7rTpKqM&a03>!9un($lOyH2e4zc<4u8AESb^&%2FGV>*"
    "ZWjYsynq3YO2!9I##s_%lQ{jQ?59=2~GVM@j;&0CfqPii<IJ3XphT0D>v&1nK!G-TaEP8zV6=mbQH09z;-ghdM3r"
    "z#A!x^c8rbbNZ(1hfPf<=7^TqzOZVgHvibboh%+B{->O<2sI8x}a-4jO=~Xen&Y+DC)2e1v4ndXd5w?*6T{=qR%t"
    "J;|leSRww=_xPVh*kr7&!IsU!^khQVT(3|D3t^)RHiwNX1^Fl`^~)^eCcn#Ma>Ejoj%aTPFu+Cx2`y3w9Z<^W~7^"
    "R!8+_HuQddv7<8`ny*y{@YY~?Ztc<>wrs9dvC)H`H<pWHf)6o9+^Wg4ZLsXFo2h(dkA=`XGcy$5oo%<y8ta5@NtI"
    "joqb#@DI-g9Vq@9Y1R8hl^7egiX8h>3vvRDG=(SkN@Jhs<0^H7qr0}hAca=?|j^MUQ<0Fd?u$H=UC_D%}v7%hy7("
    "&S6qoL)E`PZJabCb|AtV>sAX#p0I2fSp+|kg&YB^|0322Gt<u*#%Gk_B3Dm1<$3#Wb*so(;ig|#N^92*=<R`-eHO"
    "XMQP6RtA*49xntZI*2LW`i0p7F39R}QSz(<<X>w(j&QnbBZ)m0#u_^A&Z)?qoHii--J7=ls+qVDf!LvOS0-qiJa<"
    "X@JcziV4`Qc83j2>?L<2>BH<2XCPI6HS7=V36;!#j@iRWQz1cO2*IV4Sb-IL<f0IN#iHoNt41zD4wL@POJh@zHpG"
    "viA$%iJYeqo+qpXb@b@feE;MCnbX<c&rc7M!zWbx>)`K)r)Q@Le|BV#n##%A_Y8RP!7m3V$@7!LXL~1qB)=W}A(V"
    "i#2~i|U&JO;5Mx!3Rc={A%gXV8aT5ZP!EXF>P?7cWUK0Jb9o*f*W$tQswc%MpM93B4c#ev+V`rh7s4Wp9Q(6M_~u"
    "-kAIrVLESpIa4BlGd`>@FjY8*A8t{`J#)9eOYcjIX*c!{N)J8>Ns65l9Picz=4nU59|s%R5*Ige5JDICxQl7GWk@"
    "4JBe&-zaZrXV<6SU>W|q99^Tsq#01=e$kV1Ey8ArCs@0|$e}5am<~P*4qwV6>gZ&0Gx6<jS+E!D=dF}9CZOQvvAi"
    "R|2H_K0VCQSX7PMjLt(}!T2CabYSvEQujb$r{ZWc>%Eg@@r7m3@*X_u>8I@xhb57f;WUZS$ntGNpF7J1oa*%CR2z"
    "+@>D)ISr|MyM6|sCa!KW6d0PAYyBLoZ0-muj$qR7Cwmmrz*2oS2N7jV8NHO*Sz<;e>M6#bPIYJ~mQBJWrQy(ZdV%"
    "Hja6qj<4)382<6P8lmMqRmnL2+AYA0NCT7jH%t$QlkPZjDFd?$y$AlPGK%4lAUIN<Ho{O1ECzN+gVAIT$l<CnuDD"
    "M-r7?#bTa=|N}j=i`&JUb3ZyL>8H3R1he3C6Z26$1Q~B!O`PKttSRLapp`+R(uZ=<AEa+Bj_I}W!2OQ+A&lu-(fB"
    "g=$@RDJ4p76>1sCT4}f%*{Ru}C1WoSml=5g7D>x|=tseg$r!aj%HY~L26sbz**gGD`#03mR!iG~=6$Wxe7cAC#29"
    "Vo({5aV^e){6sQF3^yxHrLyN359cVrQuIVu4j9u&yKwqooycDlIM`z&BhZW!#*BFb^mV_ZUTC9{gE8kcI;}vj}N)"
    "jlA_}{Ycu|A~>~5=(e_6jG_H2>dECJg306j3iFhyw-#2m#YS8C{ARV3{#)FAJxeD8&~X|2IAfTS?B`s5cS$wrXgn"
    "&+yVR2eh~w1WDGRm^Q_45&z${D46`F{Px6&4j(x<siEth(40e5<QG~snLKuk{F;nCxRzt;#9Bf&pDS9IF=M#t@Cq"
    "1#hOBERxU<T+sS3RJGQm<F3)U4i;VWk%EF`N2tl|M=(#PA|L%M=QOWrHp@6AoeC(EN^vdSfAtbfaTtN%5b$x7nFF"
    "u;1q98hFunL^nqZC_4W7!kn;J{z5Rppq~IfJ4Sc-cr237e5j(dxC@60pxBnLXUn^=HoF-f-%JiS(&A-m3Q|t|p&("
    "TWB4{EEq%VIh$-r=?V>+}ZkXNOumP+vPn8!W~QsKEVnallkj4(UD&^p=y?(LTw>#?_o7ye9-00S*9{;lUmE=7oTE"
    "7YP_2durJrV0m|4ltdKLDK&oNdeNA|TJR(s9(P?#C(<Kg@8}QddN)W89M*^KzZuq!meK{<+pi_%Ut|E;u_gVwcb4"
    "@3{_DY!zSj}rzvDH=a-3>M2fuTBpt~{*nNN&k7~FTU|LZ_Nz=pV&(;VA)lt}26($Ka}iz^K+fLn0ZW;ex>I_ZMYv"
    "YRa571E`qL}J5C1sOf1fAgt;_oS>rmJ|EoL{(0ariZCL0Lg@SBCtgWHAaN7j9_w|Qh>#URL=u6FVkB@Rg9>`vnrU"
    "ORV1)@ZKH;Ad5K}LWUA?C6zpxCoc0OK8}YsQ#@`!qdz(XAQn^4}Cto6deK9RAdR#Gzteq;dNz(;{HC|r|K=JVeYv"
    "eM4!@>IaAh*a)7F0kQcNL2r;I7gJ5!DAowOa;cVj$>4k>HE05BK_Iw$v_~xcAH_uu{O7q!GF*oCom&8`}g7#OTE#"
    "1F7R2Fs+c?#>QZiI@PtLky^jvxX0x^S%7)v^x(`^)5G`!lURyI@Da5DA1|ep1{~!7hg+j<b$dK}H7~eR942+|I!^"
    "^}dI5^fuiHnr>XGr=kWegu+mh2GT{5jF(k&7r;S)PL=EX$;XAyUp#S^T@<aNLj)}{?pfyq$=Fa7|PcTRzpF}tJ!G"
    "Vm0YnbII;Y%jzw4b~U}c*Kt}F0Hln;U_?!Z8Sn@LArmPg=$mNVodH5@Qp4^)3}6m1iI>efo0VbHM@{@g7pzZXhqK"
    "?7c2lG%hkOvK#(t_zpcorNZnB%L7_CH3&x`39MG!q=+XL@qzY$)92!3j-}$G(Wgs9_O#m(jh?1zx%qe>gv*W0#E>"
    "`v;IRh)%OWiB%)I%Ic#5|{ua}SL$-9d7?T&%`R62Hfy%SkUSq_{<i^cv3q9C5Ux%&?w~l6i4y0FPw7ko2cwEK{OW"
    ">S^J%9J0I*6J9O*loG-B##-ezTTeRP#Cp=KR6ZFs>AKJRc3o?xw#P<@qFWh}iQvzBXZydR7-f_pHsi4>i?R#Wlo2"
    "tDD?0n86|5~Lv=;Rpl6<f<6Ra8aFf%NP!6@r+0b{GLqx?)G*1m-ll}cc9r$Q#KBqAg0MseFpTP2=j;Ox!#;=<wM-"
    "6!jjD!sZ|WLI^Xr9q;y%qE){_u66DebZSFcmcKb?gcb=z4Gn89kYD+qgTxz%0NGI$T&^?-lwUjAV(sB+QN1oTu%#"
    "21i}&6Ro3D+qtY^F%rkJD9t|`?fumfG{D(P0g-d%0=1Yw&l1)j>SaGSCgwBpqQ`5u~HBqUzL-O&2vs_awK?B2-%h"
    "fC!B!^35y2>cg>K%|_T^kR$0!4g6=Nw!3k>dI`PgyC(1Gvbr0NgDK;HE>NtaTvfGhv#n08{O9!_6G70J9_QkZe1&"
    "+C|1K+_;rHE;c;Z$~8xCAKuM>()tX?c(L5t|Mg)1x5TH($zOj24lSq24Onw#ObI|L?*<&e6ts49Z1-^^x8pnxpM("
    "r#izN26yc(^5!viaYy#o?41u8VY-3pSg%~%=?M-wO7fp!3$F)ZZq$?@|5@-<=%ZU~&Qy=Smf$L_<85}dpXM8&R$r"
    "8vG5@V(l@`h$O1(Zt0vLV`g>%8Ur1S+XazzR1L`glAOp{*AF&csw6k-{h<vK;_kZxy$A+F1IUvXs#CN&9!L&lKSc"
    "e-<*L)Dgw~yi=Y1kx4e_iZ%M~BaRli_LB|lD`R#?a)e!-xOM^L<a~;k|wV>Fbu&2V>lix@d;{!&ZA=9~>G>=}3h;"
    "=6$wfbFT3=DsCUJevM1vF}a_9?|kwnjN`AD+@tTw{~vF9+O8u6p*^mI$`9-Hc&CyjYmc**fT`6@|!BR6ip}VR!<B"
    "_$lb9-29TpGsK-6)Zz=UX?Z&3;fos%k_!kd5m!T64LwzqHB>Y)w6!8ADWuxa(|<o{dX1#^sfF#kNnAb`ahBk+5ZJ"
    "+^<K&#te=ryj-plIFLGGBc)dDarT}*R0{u-Heol-kSVX90IWH`YS;j$nmASFzvAOe)?{Y=PiPCDRZ!43eH6PS~5>"
    "ijY=kr;-f##Ce*H)7gL3V4dJkPFT)id)Y9c7s2ejxoFDRDii>Ip^3=(O0>NCKdK(Kb1A#Kf!$a@w6z(b4V4CcX2G"
    "s6zMoD$jZ)iFfY|g{x-)dnwg74()}7lE36o&(Que_KP|;nIWMuB0Ct2VBYAT8?AgKNLpWfj16KmXgYDFOFqi4<2J"
    "56KZvdWgg}tKfS>@KmY++(u2iBZYLc|5yk>}`9p~38&Xs?AV+(^KK)4H9_9xVIen4spdv*;hIJX=n0*`&>_5x7|<"
    "K!&0(Q1%3g614j_+_ih)3pkd7bn>cP;~lo1;i}HT?m4HpD*tRgTwK%<vB@R<Br~;QMXq!t%k_mLdojX8ebLB1`Y+"
    "{QJE6hUvkmw*F>KnBFF9`)zU6mz_qr_*K4%y)ok@oxYJixJ^CdgL7nIIM%?JeB*(Kvp4>kk=#?fe0ZyQnWBEu6d%"
    "@&9*0YM`p6FGJ`fhVI5Go|#OrA#6qDzkJ>{y-XVHD@)EDG<(r(Hmc(^52V#9Cm+k8wfzY&A>I%Io8;SwR1yd8eXd"
    "^(qc>vcJs633>Q&~&yfoK@WzomN>l<}B}?>Jp`0o|2JF>@|9|%4jL-qcI^BD6aQ25dj92JqS<3fWniYKy-M|SK_W"
    ")Xe#ltu$L0a_AFi!amT68`W!i|gxcTUI$`O9vTNx^%Kqakfqpj$5*Ie8X6Qe(+o_nDAE<^!-hpW(Gb#!IUHyM%uM"
    "1D*n8)$u*aOR7qHYik$-sho&c3iS5kvVQ?AleFE>gyPP!6oXOtseH*z>i3QwgS4IB-0-WYMO$KvM%D>?uXw8B<Le"
    "7?cOGRf%P!?hpv%{sCN~v08sByKCcj}`i{q_s;C-xZZn7t2{pYG0Q5XPbzg$p>Js5$L?01v%o7Hjz^dF?h{GZsC)"
    "YRWiRk4CO;9%^WWv+_5Vs$WJ52foAQI@ozTKi}rE*YA1q;M=_AHYx_X+A(ML9ErSvH)ysG0(9VqW1d{s-RIoVoj#"
    "K0iEj#jSZc1w{h_R>w%*-H<GoScMWW~PWtF3pB76Flb8b0hyY0~7w`xc_S;o*M_5ON!<V>qNNq3xAY1g2ke%lMOt"
    "HvxkmI|7>GU-crAXlVEWyx*1*~+ZOR^YBR7J+_>dSm-KfVRz>)o6UD}4#_pnMW!NjOmF{5|362gk#IPMI;-7Be|#"
    "Fa3E>2pbK|h<#1oDI5cL;cZIMHN|C@>_B5Q`e1>aB~O5(&c2iwFWF8Fr4od(Xub2T*<}*lXjc6)z08)k32UwdV;u"
    "dQHe`g*V~HGxvW6f&yOIHV?k!Tlfu#BQ^Mj+uqvNBee~gGgpYBQ_G)hRs0%%zKMf)jBxPzuA!3E}>&J~a3I@iUj="
    ")NnqOXm{Rdmmw){*ijm%F1mP?7Zs;2)^K^oJl=p6m?CQPlvY9=AXg;dflSW)cVa9Io>Zyl_J*Wkvg@dIW3|kCGJw"
    "MH3pbC_*WbBD9@~q6_E$;E9}vkTW#Fq$Iq=c^0DpQ>LYW^<QhVAbY6Ff&8b~eyWpJ9l4thNQ(_hnuhU--;XFR>QT"
    "=wpIs9Xg9Kk`p%H~9D4a-JMz04QoQgb~Ms#4{cCy;|nl$KaO$Vq=GAzzMdjtDf{z|P~xrh87^5rBrW$f$gp7?a!~"
    "W`;u{XDpUH6)HuKt}o<-!p<d`$<jR~veFWRLmnlMnwS=@70%;7S-2#V3vx2h$lg-0*P$$amd$`Cp@6}KWQ^-o3b-"
    "NglIIvNJ4ZG;TP;O4OR?M14Er`v1CERh>6Glri<6i(H!twv7I9Q@Mp>`#s|Q>!?9~GbcM+GA?%UoxsjMFtZX~<<9"
    "RLAG@;154f{0sBUYxxE{B_dG@Xbe7QGST*7E9K^Iwl#8jgOGt!B&Y<7m?G7KscMTQhZKM6*Drdc_fe=<>iQ%c|Pz"
    "|qKA!)#NikbX5sM(k41C?#p~hd^w+(I-*CXn_~M(dzyIput8cQev&-+kpM3Z2&R5^>OvXQ?JL&j`ueZnF{BZHjcV"
    "AzA_ubC7<DKuu-+z5^`NPGx<L`g?>W7EvH(3}hy5mOFH^&HyM%fs*E-PLi?k9%b`-)t*bE^+s|LD2hjf6XMYY48?"
    "$u&h+%DnCrURk%M1`^)9EME_8LJA-A*G|<}o9_uhDzmLcllz_R-B6}7+j_(arrUr%{!H(JL~GW1XH?QStR^H<TDf"
    "+T>e9a_#Bt_Ui!vWSx4WT<Gq-vKF_UXR4V~9rkfL@?cSZ-34$+(;CG4_AWVytR+*_8EonAwp)W=Wnj_k&nUQ;l?&"
    "aW|?U#51qfd0C5Z5q<Q6J)f!n1hh}{jOG)=}$p0o@!BGG&x}pbiU|@^mpjJueVB$5TtHZ;;UkVO<xAY1}JaDQz37"
    "C2XB?P^D(Fq^J0arP#UAl;o>0ku4bC##hNpXgR(S*&5{q@wqI383z@Lc@Jh9DST3I=Z%DI1k1mCpry3R9vz~TS0e"
    "M-x$;g(7_7D8@bv^;8t51oe=xu0xn(XhLav(Q}{o?GA822Q-O3@<2K|j3bu*?;i4vo)Ry5h|M5KFYgX-h74^7#`Z"
    "wE~TdrlK+Yy-45jXlSn)_~jVL2$6!&4SE#z-bE8NfFAzKNpl$DNyelr^42)Xe*Af^m`*Qomvw;uq9{wVB~mWB474"
    "!WHOhEmtO!;@R`TogZAQ*Oa!A_iVzpB+l4ixR6tYgV*^);UeVcE|MlFWHcZ(F>p)5y^RfHMlfKFL+I3m|iqbZnZM"
    "4nLFiWjwh_BDwexS-=XJ3tM7v%OzoEOmj<R4VZ-9Kptw@0NH%#iGk62HU&MGt$#J%sxoDP_(pW$)1heM00%(D&{="
    "XY`1B8JCGB8ZZc;m=Sq&JFp&;?SO_9&nJHTsc=vIFWsY)Amrtv7cO(xtx=#Souve?OR+-}j7Mr<+E+z@B;Zo(aWQ"
    "=%|XjxYLryMc+4wcZwV11>E5%*BmE%#&ctKk+}dkZP?GGE?CH(U^X<l|4J1O+$DtP7(E@<^(&sMwIURlsSh(i2rQ"
    "6uS`nrF~b#AUSuUE(U_0BNJdV;yvdP&XW#xa_(}YM<EjnZ0($*Vdu7oNJ{SUDvGoJk#da)YwTx%h;bLubSjB75*X"
    "l<88T?ea~wNh%ibY1J%`0Pfo|s*?Zxg7rxR<beC3wvM|CCEf(}(_PE)!pX6Q$uC^m|@Kx0t8T(W<@DR$lf&Qw>sh"
    "rpl4{E~kgL~e-b{D@%tCO>`!*nX^|%@wXZ7zur(C?1U9!S2;lIwo>(kn##>9s$?S$UsW*DR89d6><nSA}AwJNtD&"
    "cNySLX@-_474w4hj_>)K(8?em%6cK5~r;6S@XTB82rzc1=#YqxkuEG(GBrQqK9)-b3UKMjB?A(aNoSsdStDGbT`N"
    "Ax{m1rCEThgfx9Q<y5A2+kELSS@AN#dblpQZWqqWCkhg+#wxWXr|vmlModgexfC;QUDI<{|-%*<2JMmBM=rIO;or"
    "VM_?CC;b;+_jh&;lkC&k(u$~kNw$QI8(*uWC=8QD#f-gg3zWbJfE{vff>eU+nZ+5JO4n@?UjiLm_Q&~Rydrx!k0l"
    "r+g;{$POhLTBBqpy*RE!?$dS&73n6l7Io&y?zpn!g3Dkz0U;pZpGXI}xF%trN%dqF#lObZb#I*#M=t1e(-`7T>fv"
    "_0TSa`x-tsgZw}0njB(n{t@|Z<fahUOhmpeUsg?6*`Dk$~874E#F{UZoyR(yRV<5F7y(IOn8pMmio+j`;i|_r-S6"
    "|U6DLKd_olIj9H(9$6DO+6oZZUMGmwfaa>3QK9MiT=WAf+GEwf#vcjl24BWwRn=*rwjC@a~j49Hqc}iwUS^4~y5_"
    "N*8G|>Jy?380rggX2TwDQyslXXM@vuGNN@tqhl!ikF8OO*4hVD|^#Q3BXAbqNH3D$Jqw<aFkCkbf${xbXChLJ?5k"
    "#i5rRa6@0`sva~FXL56*CrOPq0`W)!dD*G0OlH>yiORJUUjp#!5b5MI&EWv76-j8PQf9cHoKr3xQ<ZZg4#05JWLj"
    "Wgg=%Z1L}uW=0+=QebXn`!GU3B84>+r$l2MNc$_8UnC0Ty-JuTlbIJSYD)VJ+i#Usx>M*M+QDz}`cRRJ{n%eu>8E"
    "7KJw|I@CA6=L8Q%WhrN&@17s&YAxnfx2+_VA}!n4dUJc$stE)z}!fSolqdqZcJh#O^29gZqS6rBIOgsT3HJ%(FWL"
    "9c72u2SFmP`ZH2Y<e5PHjSd*dn8LhS*vLj~MT16XMNL3j2E&D*lQi2M0)~AUfGT<!JrA8Vh+zhEim#)NStj6r|G}"
    "wwfK$SjDOVHp{`WhZRA$J<y!;xwfb2Ao1LLg`F04c`2#*UhP<orffihT5@xLHx=6dw1hLQ(<|jk*%KgR}yT1stnN"
    "<|(BBcH?oLVWvZMd)+)<tGJCe<YnJPc-arT$#aYik)U4_I!&n>C(PR(Za@6Czx{px;Sc>4yz@h!qDlIB*_U`rrp0"
    "}<!1V2=_o%)I4X}%@Mz#ddiN&o-lk3$CaFBD?WKLSCiAwHWaE~%4W|%EoB&MYpZp*uy&o8qD;;N>9F@D)i#^}8HD"
    "xLOaksJidblrVw|LGxC_$k;YF^DLJVx8imW#YoerjQ)(m|=mJbh=7KAWUt18PgdKJktQ3f#w77s3RVgM0^oOUp+v"
    "2XM@x*zQ+VclZU!8DG&hM(q+U&nFVvAhF`TT6~3=UP1yD-wMsxlmzvD7Dy(fO76_Z`%Ai2YrN^_KK=xg=O2A=etD"
    "@9n;^n(3c-(_6os>4zos1c5wIgEGtWCz^pxP8P@HZe@Va^M#<*Ic~_Jj;UY@#-WuOaD#L)}{&%7k3=dIz5;!k!R*"
    "c>DDD_b;93NetuQ<2R68910nxOlVJ)MuQ`-76UsFj26{+1RzFd$+P3rvqWXt9B2Y224S!@j)+^V*W65fIwTsKcdP"
    "OLWiiFPfWmAzVHIoUmmqAEddg>n9|^a<N-+DN4JJs)TZ<fPdT>-VWZHTMOhmTi1rNu}j#p77hLH~5@NFM$yo!dgV"
    "jD4x^ov_WEF{%;;@LxMQk*1B$Z0_8f^KNQ<X7>>=;aE;Bz2B%25^vDPG^ndllo86%Wc1?Fs%8i-0qq>@@q^g9-cW"
    "7^xkzhf*uk5xf>0+%f^uKyiwcn#9>)#56(H-2<3i_eB#QL%yp`fscuBC2GWvQvd6W(tidGn#9x-UNyv?aZ&v*=BA"
    "9WS0~wYb3HM=lUD>eMNon-jMuT40x)P%-;-0*@bxWxA?0(86+|A=#w}g7{Z+s06p5W;8@8T712ZuX{4){lVJtk^="
    "b<24xZ@O39b7s2+>F#T`AA@GjY}etJneJM|(wW{JqiWaMgi2I#7AYnQLaObEGlb|QDo>~zW`A-<TA-B(G4GkgOjO"
    "WWJsE%|a8$6S)~|2`5c&PIE2)4%LKrfVOV+LEYiQd6CV+b9oV0Gz@q|NwsyZpV8mKHja7RUlm2tU%Y-#f}%yN|T)"
    "6~#Zfz6KGj8HNwhgK7{bqO%umVLKk1jcuxGD5aINspC+O%iUf%p|}8UEj+=Ao=8+7JHdfI5T-#U010iBa4CLQp|j"
    "kclb!M2+7X$&umeU-G;XmxwAh8!G(ki%9Uk%0$cnQO7LL=r(*7kok0)*YPi-!WueGhU0}J%GV6N97Oqy<6Nl?)jj"
    "!E_p<N`Wq%}8F))!IGK<rJ2WtK|HtJ7ob)tnCzAE>RHVsSO%#f`R-|Nd_hO5xmz<QNFh{!Kl}DO}>i?AnnvfFvD="
    "I6TzXMd<tG<oLz&<mW%^6Tj~L=i$*W$^P++qqENEU2Gw0GTlwR5gbEIL;W9rq?&(bbz42tN363oFWynwEw?3v*Xq"
    "qI-$u+VJK{f~e49Q6jabuf(g}}a@~c=*cDkBaP2mK6YM||g?&#$EZdDCQT4NO-sE*{aQ5g|23#;50%6yK%_KhteT"
    "cW_1Qym_(=o1}MZQq)UG^<K68136s`Bdxc3hv2mCz^_V%H(tFi1K3!2m7XpZhf!@@7zdaf@1MTh+0%gkg2!<9c$D"
    "{%9K*Q$1JtZpnyAEuR|fB$AfoO>!GIcLFfj;Ra;HM#^zyc!42}IJBHxwIZ1{kd>Nw^Ou2-)psgFGKzWTC!@xELw5"
    "s=|)Il-lip2ZJ&z{4nKxjJFBm$236JcE0CIKRig<(b=iG$kdN@)bu{d?BmWcFdp>?k(OLKmc=ze?_>`I{_?3xkUF"
    "N3nwPBb6a&qXM#EaK)-7%Nn|%D%dObHmn@0rJ|?#4W_wZ923{G{zI$Xk<ZK^8Z&otqDq+vW>Z83j9G+?=AAOj7|5"
    "J?Dx}9df^*;qoCaBPwQH%5ZR*BQ{c<wY2t;kH2a%%g+V~=q`D*%Q$WmfKWnxv(M;v19AGHPx2(PB2&zsnuW-*jrY"
    "L@LeM{>RB$Q0tfq<J&3dNKYZCw51{-At@OVwagUiRwHlX6t5;SyFel@UClHBdB}m$z{-8v5*3^IJ8nd*wg${yaa0"
    "Q*tHcz4e@kQ+Cye@uu_K-_B;})#aO6&pMT^A$uDTN$tIThxz%W}W**jeTudlfR9HM{0NI@SqhspY^1A0TeTq6O&>"
    "Ah|7u&m$$l(YB9CfCl0gnBLs_vAACdG<8^c<)Z7<#f?XZ~^br13zy53a`$Z>!NW_wi>loN4U|y~ml>w>pVSp+O1W"
    "Ut#B#uzkP`v@A6GdFLF(j51`&eB)Xl9NgT3@Cj=<{P_R|J$U+uHpS4`3kjYTEWqj?7*i-cY<6A0oqmDkTtMUBk57"
    "K<nSlA!&SjZ{45;`dC9R23!s@xoq={0&8e*$CO3ms#OiL7u(RmD7<EsZ{G0V7Et5fMdc}`NyI8*QXMRpAeXtBndV"
    "8CKTK>RUU=Tf?}K<3t(%1!h`%nHjf0S16tCCT`pwO}L5(=(|!_38n(+=acQWZTOet1@H!OM0bE2Q*Sgigm0zy%s="
    "?XV*Tq{yaD^zbB^ZPqOyd-q$4W#$f$P)-zdmmJN*7nIJP;MO@~!@~iuCE8RE=C#Yt-H+hRyU4Xx#fJ^2e6BtDmd&"
    "*2>pD>li&@j_ALHVM1B&l~XJt4yRB}KfHBG=j;L2l>bM;L@ZNwXOSnNZNtf@##-j3U=;QzZ;;0XclK!pc!t*(_&z"
    "BKa~vQgR8TMutqG1RiZkSIc6yzWOtY=`c7bB`n@Y8X%F}+^IbGqa#4ivp<(3iu4YZQ4~ccWFs6kwvBlIVXLt;Vv@"
    "}f)(!3Kb~LD(L$9Q{c0>0&Am)E+Klp~ePhUm)%UR&!s+$4qJX_H)Tt}(hFUL_jS!EHJO3-O|xgpe>8kf_ZUcK|x)"
    "Q)s#+><%PT4+)<*m#lOEJtDgOl_TAu`768Ztc|tRvV0RGOlP)L95zDIKc1NS<rW5YmhG1BZ}M}uRkyK3zL!^qv!%"
    "PI%77&3Sg}lqtQFgI-JkM))O>SZ(52**~Rk*B4Y<#*=x4i`vy;^0q_0!I;fw|$LV6uhQcbo+<opQe()m}&}*KZ>A"
    ";CiR9fzJWXf2?3uUrO){40iCpi3c4FPOQ3o%(M90bSY0)@Ddr5}-b&VZe*Yw=lkj<F;R7onrCNyqKH%rEk3HEl+k"
    "f>U)in8DLKh`pyQG+WYDYC*Of>#r<w?3v2vl<FEUGESMbaS#~GX)}~Kk%T1`t~N(v3b{lu&-{^%h&MIq(5=P7dsX"
    "C(&{Qk!CKX{f^0<guaS1C%ve!$#NmQn+nW~7S6_NLoOtm8a#-|#g@=Z{UJl;6f7!$cGRBNVM2h~q9)jFs~C2dnoi"
    "2qB~?aJliby%UUv(s}6G%20$JajKVG?%|}FMnk&f9+oW+Fbs|z5I>2{H=TWTXXq4_wsk<^7roL@6F{O+{-_h%RAf"
    "NdbaHif1!4zJKj=v>{@rc#qQYE?s&`HvFqLO7QACuyyGo-$F6zDTl9`y^^Ujf9lP!wZ{a(3<qy4;KeQ`<=&k&rv&"
    "a4>+m%1`R{qeg{Gqq<hj!%;y_G+-D}U&%{2{H}tvFcT07n^_S^{N<iHj{ah?Oa^BGzlK=-?^7hGq>XYh+nS2))%1"
    "IaL=whOe47pQe4L@6fju4KS4UoCnJ~-E)qwwbjC`5R9&sfK*%*6Xa18I6(Rt<!EKId|XV?>txeaw05vv{<&n;&(W"
    "y%j;vs*GcB8|fqbp!G$@t)J0>w$r5#Aazy+Aoe2;*#YXJ+y;N@$vZLKaZsj;4Y1?}Ku6o!@Z-VA8Z@LJhHr5xunJ"
    "X@@??yil*%r7}s0iGIQxrLifH<6JBQ@JqUA_@qp&Z`Ig{;LP>2nynSzRK)Xnc)}1g#EwH_v*pg;kUxk9TZ_gE6?e"
    "KyZ+-5;J{b?SDY`j99`;P+Nb$&ki$QDtS$!?G2z6FFuZ3p>Y&@MN2%Gn$#YE5#?x{Hc<jsp?+*Sy>fVK+jqFMn{V"
    "O%}&9&r+GIl>Ek)vd2jJuiG-~qaOGKMP^QVA-SsuY!EOc?(AUC;fhT>{+hIcIXtv{5~F?dN*0Z(+iAxk~3BCdIPD"
    "KX|=*9Q!h?KJfB6eaE83_%e(Z{|WBcdb=2pVWyofb|hK6<%M)vT@R;}%yQ$rd|X}IS60iLaHX)3dc8n76C_E2>8m"
    "@=ck)jN5uEJPKL}75X~F3RIjlRu3NN#w&fB}L1mSG!-SO@ZZ+8iYqqr=&z})~CKcVhD`1QYow27YQIo7-zB){yw@"
    "!tQ+z@3}g%Zft_XFpaBnI(kUR6AI@vkJ`}oc?(9)2XJTtm5^Tdlm{BJxLjZOItxCeZ&ZxMWdV3L|$b|MMrhhflfs"
    "K*&m6DAnCzRNv>dY{pkUI*IQROqOD&D_DH^{#<=5KhvhxZv{l48e$|98@plM2xtf`H#KCi=8~VR)I?Y#OgfQ=P-|"
    "w9JH$mC?K=j6(#uQ2Aum&<xe1b_%1vO%I<4tJ%2vO<+@+59H#e&$-#wU0c?W3lt$ZM2Wp!ZW!mpU}y;G=Q`U&Hp<"
    "r=U7Y!i+QsYa+eCc#9#5zQ7*-Wx&(e(gQl2$<Ntrom0EedDAJKt4UE4RJX8SHQeGDDM##(a#vhp3P(n0X^eh;WeC"
    ")sC3zTbSYQ%3X^j-|V)#4=7QnO#N#JHY%hTkD;CpO)v+7!EdoEVQ59f#}dYv&gEF;KJCD`PLJuVPEru5{iY)uguv"
    "%GejBXA-l5>hd$x~6C{7pFXNDnDb|ykz4$$Jit1m-z~VyM(=*ZP}X$FpBfzA1>!*ajXS%*uEX_Bge9zbP3va6V8&"
    "ms0}J%R=3Vqs0p1rjj(j|vpi}hik)XzFn)i&Slw^{spHsW8PnjX6CX`(u>7MVZOIIEHglo}P#5v&@I(yfS&jRHsS"
    "1XG4W4jhtR-yq6UuT~-7MfH_4Uan(&OKOW#n^W9t|d4MV5T!zDkbZ4a8FiqZqq9q_E>~RJmi>Z05Dj-@Jw8>%f=Z"
    "6B2GsyYjqC=Z0bvC_K8pbGJi8Y`A|z9CLk|b%~;QsgcJL)XfH?my!fyu=MHJi)WhHFV?bcYzWPFAKcH<^y>rj(sp"
    "vI(}zQrzkA6LM{afhJ8%C<GyRCH<C4?q*_KJ@;bIASZMt9=sGr={G@Tu}*S+(&k1Uk**j)%^&TYh&XKEK-yh#=U6"
    "yP8<7bO>PNt4%3+Wt#1<4vF^ls|6OJ^bH3!cCY)o`6LC>P_<WDX4Q#pOVE-r<3YsB$t(HQWe1faOk#eZWNSO8WBi"
    "WW{?GtHBeNJB85u@a0<3nX5@&h9C^f0=GR$ms;Y5aC+sVw;Zr04=FM|XmeCcwp%SizS7O3!P29?=&NAx(r54?*7K"
    "wcC=<VBsQ-`<E=LlTFf##6|P;gxi4O|c8MUGZ-3Bzp004{u9w8AnbfWscF`<@=psD_e7k!hS`IdRkjq{Ua5Qc-@)"
    "X2k?2%>uUb3+#zSY!FlBszy*e91&%`+3mWGXch&hAmci6taHx!`TpK)J;j?<6Yr?<{QKuMIGS)~bz2tZB*pn>pdB"
    ";FRsOL+bjc<yE@~_+hF%rIwcN~?f`>6|z~c(-e7&6o1jnZ#w(4wIlLt&dn~kX3A}F?f-ZapCM3<vqINNfwDH=SiV"
    "-{Pfd$#H(Ap^FJ{k;3;r~Q+p^Os2C!B#K+mib0E^1x#TxWy;Gu@v{{)0^8oK6>-!``x|2Mi)5HvRMZ#5kl?(-)Qb"
    "i-S0{Z|4oeP^Q^?6Cd+?{`8=N#9tS$W$~Yj0zi*29O_^X<G5H*V$b}6lOdYw&tO7Fk(>s)H4o^Mm+~6Dv6J$bDuu"
    "k5DTM#mfB-ow=qPUK}TPawBbhy^UDDsS<7|RGSJ-#3%^$?w6EPw%w-C&eMiuOug<pQeV43mQ2nKeDBwZ<|*Odwd4"
    ";bS-%8)zn)=)%=3ht6lB63Q`R%*|Ehi^#}kS*_GUrIJ{HaGHN9_(cR2syxZ2XEg+5$BgzFB=xc6w8iN4S!DEv!uo"
    ">uAKG|?9tbV?9;BBxdhpoA{Y?51(2h9W_<}UU8Nj2dMsr0!V_kU5{t82<?(`O0ryVLN{-X5Tf=@lp<D$P13dSh|H"
    "7Uj`u2iyH-kc}@9ZK`4jy=?rHZ{tjq_stA4c$Ud?x4lLpc$e=o#{D4imS7tj9_ie6yw)w(H;3Sq&mXJrHop{LJG}"
    "8i}kAGpJ;~wJG10|5E~vAllYB(Fl{||Sz)jB<I^g{LY|Hr`Z+$2OIoLs^?U(il?Y%V*aKWTj~uhCJV1|z)&)M>>V"
    "LkuqG6I|0m-R3taaqOi6iv<B`)T4%w(GSs-dvs9*dbjAk5YLEe0caD{isa_@Ev%E?V{XX8Pe8Y)V`jy1~1W9}ge;"
    "oH=3q<KJRd46^e7+yXmk!i^DqML|ZIej0I=^jUJV($#d05XKTGREMpRh(_Se#^Wlvwq7|!DqM@;grpWz{zBN2(Un"
    "o+TP34bl-H5e3_!?0!FvnaT2y3r?mF1Iiru1hQaQZ0&OQ=m>ZH;_p?*V(-8H%w0&hfRz5h;VZHylqwk24Y3K{8yX"
    "j5&mG#MQoCVNMRuiqR1y+>E|t0M{W8HH_eaE^c%QrS+KbVm5iuqf+zwI=lG>bk6@bQ_vn!lpJ&bi)qAu#wqHVM2>"
    "WbDF7wU^`UxOJ{CzGm^xr2}q@5vSOfpg5>QR-!LM3OK@DbUA@9ZBgP%<hO4ImdUyJ(Pct834;0El0T4v(Rw{?*JF"
    "4BDZ8=MgL8Nx6lau|^#H93tNCT%TpQQTDcxwjR+j%4GoqJEY{5_<{&{zIC(N=R!+y6|8Yhvnt1DdOCE7dI)1?3hh"
    "+o!YNV}Rr#>`2UNj87~-V&isuK3Z~Y#qH9^=eNa&XJ&oVeHPGoI5_X|XSGK9rj}9^YGGj+AsL5%xc>_%6%TpInYT"
    "y1VW;tduX%GFe`n*4tMkv^ThrFfCBpY_!qGhe+bIwoQ9lBCS{t=$NYsb$4sBypQH3#vEs!QbVJH$Q6{uub_?oM3H"
    "oP^C=Ah$=NYMW>pfZeH54&;HBaUl-+?3Inb5R<Ga`VblORL>bnNVTBt$PpK2CX|lzwB&*1|$pztT(B@XVYvY6p`K"
    "1^N)u2IXoK8bG??rmvGnJYTh<u@A789(=p>4s|UvV=xaf$rP|X5soV9~qpQCdCqL{pwohx+5p;R0UXGvDgjb9)L_"
    "o>Q)iVm|uR)5~>L*)LU-<@+xhY~1VJJv0DY@QqPA(AGMuMP<S1{{~3|$$*{+Q5U3V#?5r)xxt84e|8KOu{u1~uBa"
    "vt#~R#~2y9eH2y7>K~w@9{%C^AKcKwT@M1pz!)-Cxq=oUI+Z$gYA%6O*G~T$;ccyYx90SQ8us<oO?XN20mI9sgj("
    "Yih4-F1f1~tvSuMHQgwHt5i`mJs?L=#?orJec4zm9QbQf;Ie}n7NXSgSS1^3;~!$a=RaNkMy;VJ#oc>>{@JKfaGH"
    "?V?R?dn}}3mv;l4s`x*r7w4BatI%U3+{K>0o=N!DKFgH(r%XX1)^hP0y9_Ypopn$(83splEQZ%xryZ}(nfhjy(;>"
    "G{+8RT8>-PJH+Rb$^Q~YksjWc9&%QD~-RR@Rq}C_PoF!XO;fE!L<wak>NWhtSXCKFZ%tEY*n3hV%G&h~eMc?$HN5"
    "!~8KIz+&XumqmdNI|o+WC^}ze$bgJuC(d1W?M8acIm%ZtKE%Dht(Emc`UE(REPkUHF#M1z;}&o*XQ7{E>tv8p(jc"
    "$Ydk~!ys%r(Lg5C#L`YKHuF1Q;o<E??9Ka>hjf<Q$H;_<rIt&k+?r#EGYz|zIP<vu8d|Dh8t2)nW0+Z2)8YOB^VT"
    "vzwVvWbQgV^d(;Of~Hk;bMcD>L;cZ#49U9WR5L|I;@JqkyEanoV_d~ho-^)U{vgDH@APKZCF33*%92KGA}K~3ipM"
    "|A&M+Cle{jf?2b$<gtv{bK}-bH>>WIg%ke6XR~#P~85<z=kDJv*)4I3!9v8vnJ`^q}|T9@z3T1dIlPHY`GEnpmR!"
    ";+NDzpxX>NOE!gLRMuPYSC%Ve)jdqI9_+%c<f0eVL0}&<1Y}u%LQbe6ggxEr%0vqzVl>aKLuV%$Xj1wThj<`$H>|"
    "(6#Afe;Z8T1s!+Cl?*_Z>yR|5h|!E2y58dt~kznBKg9f8I-i{H=3)uz2o<vIjbb<W~eX&+%<54VaV-EmM=$SJ{hi"
    "zw6vXqtFPnTv2#CX9%RwRp>rYr}QfS#2x5(XPpFkTHQ_rkX)TgmQfio%%r1GRszZvlbi?&?2YBBz?6ZS6CHe~$XO"
    "{&jpxw}ciNKTg3xu^9)r;6uauEMy(A87N7FZ;p^kZ2S&rGW1JebEJovnA;w$&w%nW?Z%06XTp&PJElQxilJui|>-"
    "N5|ieo=Zq#9<BaKh8~F%|WoMZj~nWg}%Z-v}L?#KSQy?EQi8%6%E}1$2hmo<z$!Ivcjz-My<FWjrkr;(v&HDVZef"
    "HSq~6hF;b8czv`=eHlscLWr>XZsA}_PeFgoT*`KCs_R>xLFgf)xWQi3?eN^<h--d)yrQ{y}bDZO)9(<6D2fgG(Ju"
    "FAlU#jJYmIkrx3oN<qFzEL~y?Ea{sVk8mnDsu+|FMRh+;c9SfDYpjt93a<t)_$j=?9PX2vV_HgRV1$gMj+jcRT40"
    "Q&qEu>7L}8P2TYZH&ZcNUVcmlM1<%QDPHJ0R0wV}0lNJ7e&?JG#*3TP6$Ul;-BT;~ai^p<#Zd@Qi66NoKYaza%a6"
    "sfD&NEJb7I`^2QqHV$=N;qk!>YE9=+Y?zuxQ~{vdzt4R_yQ0uuI3Ub5}onUAF`^zn$`w+zS+cF+iMJ5ey%f!Kt@3"
    "Wb*I$$;90KZ0BEr2+h3_yRJT!Ru^R=i!}fL8lm2aIs!-x?2#t2pVkkI<Nac{tfOj;RwH|%s>(r?66O{k$tJ}izS1"
    "CECz{k5`&BXG1bQhuz0fDse2Q2KBo<P8tCi;G{;=6Il7i#XfeT9$c<)0qV1yy(9!h>1x(ifFhKJe>;SFGR?kUsbb"
    "+X=rM7B%w!O@*lUa3HH#?yA3TRlC!Fxoqd1xxVnLj!q3+tDdXH$lnz|e61C|fl?>o`&L!R{BB$#rRI-8RuW44S6e"
    "i$E=O7P+jQ(t?dsXn7z*hHRKirHxd%&T%ph;q2wJYLUAoqx(Yo4Xk%RB==e}2mU0SViw@E@(vwMb14v5g`-miJwg"
    "BDT~p@k)iRq|OhIRtVJZ+?(Af{Z_vXMeK<FrtV!EoNia6*4^kzPF>OX&c?oFKd2JyAq(E43aYji(tIJ?fa5Ld}Qf"
    "WI1#IH`U5oVnyYeCKt@gfk4QDIsCw%djq@v!kHj!9QV7khtQBqi0)YnWAG_R@I8ygRN|y>^3^CXvjRBuPQn^6pdO"
    "5z#uL>*^f9Mpr{vBO>Pv^s=u54G4ig7_HY6;mJ*A#oBV9Ek4utL{HKy<=0@6_99a~V{v)qiHKOfp#}^#FY;Sd22h"
    "M%tq*AO|7@GkOMaNr?G$yN$-5T-(o!FFUL2JqN>nBlP^!FT^jADOLUiM@~op~ekKDn~k+2&)<^7qpgKlD#^t{6y~"
    "#1FFQ1;#|@RHQ%>G}1)&6?t53ZmLtJc9xAlkO84=^P}qQ-g&BrqY*2yXlTKfOd3DkP`qU{%N_aAS1~xQi+2JyBY!"
    "nawW9_ebby%%yFCqWGJq@S1%nQrFp~TFej|?A=l8fu1TRw>j~Wk~7F~zj38{N>@9^zZk#Azt9CrP81e3sgQcf*T5"
    ">H9<1x{J!G-LN0h+hb*D{M}f&-H4${RiV^Wi6PzH=e=7eZ9>XLRR7S9!gV~=;-td-7nn49nJZMzFBju=}NdXd`!#"
    "=wesvZWA^-Ba!n|dd&$;1=v&P-7O>hh0SSnPa(&q+{{Dbxj=lbe29@P_^*II=jp!Lf*s}^5a!-Foo){`V-7=pJoU"
    "wY(tZ~W)w=NyGlQwW_MF7DHEg~$gOycGe3l;~vgBROlnFelCg+Toy@{FwPK>l?<uXQHKT=f3jx7D-|0dxGLyVY(~"
    "6iPqjLz?K{PN<A7nQp?*+N*4jHTDgM&q{l${+s>{nMYzXJZFxaMG1bsbY~0%R!&h6jNnPZR}a7RK&1yXMXt-vSwo"
    "WBghh%$&BzF!Mc8^nw>aC{S6FIRmK+gWEBQ(<|H<Mww=RA|Dj~iu+vGj76=I_EE`5ngWKr))<K%vldKApCgIlplL"
    "o!bdLl3{R;mv3Vo_VU{(~7Rr&_zaQiHHR?Fz@1H7bZnPK$6^N)9fRhONxMc=rS6WbzPd5V{7&r?j^QR%e>&4QC5k"
    "pV{gS((a3`bm})zlaRs0^2Sn3Qo}(}tEI_+q{w>9{Lx?SbqrbIls2STMYmC=+$dV`nLRUbCc9(@)vx>4a$qkEQum"
    "I*01N$-m39&)(KcQ=nVv7bw)(pizj>AUO(s@3JP%;;Eqvhoq&c#LbDd1^rZbSj*6A&0WIzAa`X)Snau=%Y>k7J`~"
    "u5{OvF<_|-3-oYY2fh{8Y?Dd!o$R_HJG#awl^wrIjw!Z%TCJ%7b&imsNoVgl0!^$b&I-$DpeQOOt=#Zzc;o2w$Ng"
    "h%i%B0nO~lHUvC5}t$l-`9(3(c6J!`O6MnEpL4L|;SantVyG2s1V<c?&Nti=ujlvvy@rax?>yFh%&(fy!cBhB(Bo"
    "CR$Jw02<Y7qwp|P~l`TGnDRN91kvEeKVgG<p(aDR7lMgZHuO+lc$%dKM3Tue*)Q<k#(S-e;#$6q7anqkCkn`iLNC"
    "%0ju1C5=aI!D!@X%A1>$u9ih|-nDrGGdVv|5rDpN-d4;LCT$5a(2&S;(uB5u5Njb&%C6lYmB_}jjz~%@LGNAwr*h"
    "2P>j|wZv8{iq>iyb)7S0>-%E6@DJ$tIKaQsY*+5~Fy-V8z!|Co0$tSQ&=fl2i&<gXy{)%YMZ+E~@GSqWKXrL$z4X"
    "7~W{bUA|^uuu7Yed7%hMniySKnpoyCd^D~QL!9<pS(&U5S}JXepd0c}3uH`PghwQgS-PCBU~yq5m$+h7226GlY%v"
    "f%V3QGA60V)ymSO>tU0&WWG*f|cj@_IZ06eWQf0=S(v~hb*##d;1#K)v$0CoqKo9f{37Gd-coPXJ*K&?PDFeP%#0"
    "@A8z0tmkiPjl%l7;$iz1=55iWUcwdE$O1+tl-ue8EQ7tN?y8?#1<-hL>n-}+$LymwJO`o4JxFe8W(GGVuRg<08`j"
    "Y+#*<WkV;aMHI~Sov)3qL9Ql52_oLax9Ki~^Kt3tcFoTWBHP^U2DIquKlR6w3f!BFuF_&;Y35$nwkjCVccv7cE_="
    "4W2MLm?k`;LFO#XC>A2c>r4WepRc4wH40n?Xw2^}xe~jT9eqhmGt%?;2nTq9+_fT$LEIjztj-n{b;#O>9LG;nU(%"
    "GbJd>5<D90Hg<IU>fjJTz+WF=-Go(SLAh#-o;a!<d$EKqbLF;^UBFgx5Rl;X@vN8z?19V+S{SLYTFtx;!S>LK6fi"
    "=c0(78D-4QRMvR&o^K{7b(%fo85`$|7auAmWmCdvF*LAztDb+yE!BIq}aWN@8%plW)8+^}X^sE*A$W0z4aDvj5PU"
    ">#uTkzA1#lL*Er%-|l`0_wJzakj?#frJjXtX3FpIZ~K8cH@jxsqRVM8zbO&Be8nf>s?suR|y9E)vMVJ75jxFjRX{"
    "@agvNgzmW5c(6}BD9*9@Pv>5joG#A+4Whr)hML(iLmmx|mP~5SMz(cFk5%=V<<7|QiiVplIwJk-qz|_1Y0m{L$QL"
    "wuQqBqZG7{rEx1*}8v*4TB=KS~L<JLcnVOC08ZL6AmRxp6$EgO(=8G$~v?$>eO-*93ThvFB9L*_$7#_A9F<gr3Aa"
    "U2ZeI(6z~o==-K^LtBOH*>(;28#vvySxDl8z|`Hgn90!Icv2fZ3qvH}nCC#t$>rDBPOHjq1nu=Wzak;vI;0O%N~e"
    "xiDsB!sx3}o`!1Mes%R4Q+9H$pCRWzVd#?n9{fJ16J$keca_Npso{E9k~qN1`L?lETQBJ!#DgNs~eQ{W<4V1RRps"
    "h&`?E&L>dbCp%OZM#y!?(SOQ$kHw>Z5nAO6n=P;qjb$lM#&;Ajb|Wi^}KT`whhZiDes6&nmR|Q*%lE$`Q9UJH7rP"
    "TQEMv0wBn=?ng)@Zm$}ksoFK>By2)Gsb38P2EFm6b>5tkBLcZ<hYBU&)(G)l1B~t*1+WJvfc}DKn3C3ltKCJ@dJ}"
    "av!-;cZn=WpX%aNmR*QV_EZXSAenkWi;297mFh`zY7X!iY(?z6xvr#}MM^<owv0t^PKR0O1nReSX)32ls~QqJ+Cs"
    "P9yLvV%ZXl#!x;3KJF*j@#F2tqLL#bA(~C6VykCO|92a|?8gOgK12|O*?ioDW9u86b5zAGtSe|=GnY2LsB3KB(<?"
    "2n-L+ib=roMriFHGqV!H+H->@%&!88Da{7y#zg7g+9b<?wyH<nLY+j1=YQPg2~(>r^LRUqfyzYHGIU^sZ3y%@@od"
    "(gsW{pablX@ng=N|!>KA-f(o_bi|q;<HWv4vIK8QFoKyCwkq?Rn6ccoH(gAHHt70A8T7!g!^ni;{Wie414p@)TnN"
    "%pdg%=V;5;5a8tED2lQlBjjI_@XH1KOm9<NQ%j+Efva&&Fse*<Lb+a)k!u{JZ`w#XCE^cF=ID5#qTO!M0YRLYtF+"
    "sc!k{xQo?W>9$GV!QLBD3^pW)n$DJRXBcoCjfI!mO$r-8OB(wj;jfZnHK5Eu5^qq}nzxzvMA@fg>j+T<yf5(G0o$"
    "e4<7FVv!-i9<aTF$#<bXLd7Z;2iV#jK@%OS(=Gmr?ShT@bLEW2q5~<}>9>1k>C+(3{4-xxLyW3%KivGND_G;xu(`"
    "=vAnS13`2L-Vfua7HGjU&+%!9)kgGT3@yXE$3j#kf!tD~)!>oEr7PK2iTUAp%qlB(#OL_BeL8t)r|k!d98k@EGOc"
    "1T=ODk$Etp=GKM9PgxHBM^=oEhe1VJYCQ1x0sMWe$WA>38bI+vdJ*y)z2OUzi#2|{oZp(-)5S%v3t!k_MqNv^ilt"
    "B4IXj+ZjT{Z2ca3wFp^l<2>sCv8R366g^Pq2HiV4?zkl2Ck)AAfdt#8qVX|pK?=%;PY^bZ7p2i3w;DhEKstaXX!L"
    "YO?3|Ju~u5V~j$azk&s}y99Xa_k~XG1X7rxS97jaN?v5KWv86Ge(3X=M&Gc2RvY1!JMr$$Fd{7%FW3x+qJ8IzPa?"
    ">$>WBz}`K!^-*Kok9A31EBkbI(UzJ#74OnhDwe7BUY?>!N_vNh!3cI$VvTxqMv~VwyUbA2?}36o&d`C+cgRV?x6p"
    "Dh&oeT9BCQfEZ_a4~@{m*Mi$rR?%hVxkSXnXbz{jFSt4u{!CA0{fm7Njdf<IfXK?A(3pS{?5@!j^$AGYDYFv@Lde"
    "H-50=EF6gj9@#pzBC~a+ibSIhOd9KtxE}Sb!j~bD;>n8^(<!>ZniQ3|7HX{`(=HNc}e7HOvwL_iD0m4X4lyb!p5A"
    "HRJCuJZ^Ll65!6lRNuWf;<jp?cq8=9uQkX}}e3sp$6vEo+IyT@nZAx~2kZo(SjgaTt5^9@KlH*zVJWZZH{e@h!+V"
    "y#ML#r+8`=80OKx<@KUH|3j(`2NmZcM~z_m7fibk3kNNE-b+7BtPh4iosoep*ipewxjb+)c-7`8TwHD=(xFb?6O$"
    "bpy^Ed>tmBnBGKgLKnW~E(jKiAi~K!^rTEU9rR^Y8&?sb>|n_;(*Yc`)>l>XFk=5T`!$!asMVbE%XLNG9lT2MKU|"
    "`5nWO!TS`({pwZL<Vr=q^8p~Ek^hTAMR>qPsFV>)Va2@`sBuH^x8$ro~ScgTbXhlYy6pv8|i6!Zfm8wQHgeU~LqV"
    "O%^y`^h{xI+4;cyf}o@k06NHPPv|65aRt=$+2CuG<CsZH0dbNCb>A#Cz-+CK!--(Pgd42(DM--X~l^iMe+?(3C$R"
    "F(G*Do-=;ys6k<ZSR&^j>VuCh%vGUe4ECWxq-Z9IWYA<}qlQDZ&GA<`iUK4bGE;&qKZ}$W%0w69RjqvHy*Gv(N;k"
    "m@gkwBRg&dM%@iLfJqay^q-MU+ogiSn7SR-1Lb>q)?RJ?=@MnmxF1aP~i7&&9jXE~R=^Eo$^`IPm~voyV8)DaNHr"
    "XsZhyBUdLn_0T~N1IR$Wz0Q|hC1FxbNoHI1#hFSC$b14#uLy`yLax!S&0AA?4@XsmtP04h0|}V6<0tkAR6GgTnse"
    "ySN`azds-HO_IKV$Cgx|45yS|f<vNE#}?;n?FF$|Ar6uLXG!8ADma?SkV<8xvIf7rInM>)dwS&1p@xZ#mGh_@goG"
    "|)sdEmF6{>Z5Xm1s~e{Ua!&M^9UdU#G9e`puBc+g89-tIm4X53Sz@5`%-W=Un-~gtlT4*`#}rwz$bS1PLGa<Z}xX"
    "j_J^mZZ{S%ooDOU%EMzkH7G%Wz|J>jEiE4}Po}TW%eRtaW^z(0>42Brp??f}uc9{_<PsxoGX<bV;*PIKc8{q##%Q"
    "5;P$7tgg<oEN4yZRYPKU27YYN`)~Uxzm;d?GKoDj4n!bWibbV=Uw3#w!r6yF}kOTqI3KxJkbmNsL8_6$Thh)^GwR"
    "v<0N`JX=w1o&_G2Vw^{+j*1LFC6#<Uyu~l1^qCH&PTXnr;(s%ny91l*pLwbIHXI{Iwyrn1t%s>Aq)WrJgh)0D(c#"
    "Qk`G{=3BuJXtupSB{`M@X33`7QYm2zc!x>sX4*aX#B_~N{RRY6*sP+@nSm-;GO@)C0%BJ&K!IKkgdu2D#MgC0ql#"
    "$(~S7@1XDx=0;?RYh>h`l?tYv<sz%ME51t&H0d5wEL!}SHfULa-DMooNjN<*#h<BlANNXt4MVtmQ*IpDS{0I%#K+"
    "LeI5-L(mBCbCZ{-4aSC!2HWKU-C;nf<?!E7rGwR&(7L1TWv=z-{bMjK02_R_B@D52gjdvcfC--$FcX=Uy#m>agKi"
    "Zg%6I5pAdk#D0fPm5QPltyGhd+!Mev3r3{$_|^Q*1<%B+A^Ba>5`pj({@;3xg$Sr1nP16ONqILdYDh)h@4V;J?|w"
    "q8WwFI+lSth<#}25VAKt>BZzSFzU3^92ZI~rX<0cSV|Wsq?XO91>9KX%_y4E_KoTn8=B#$MZ<u8TomO*v=Z%{ET3"
    "b0a0iVdCRTXOSY#C#ZNa0~J&oey2Bk^&vXnu|B;<{RE{SQn!oqZ!9zz5LS6##HmjM$LBqo|RKGz%ptI%9xRzTy&J"
    "V!j$qMp}o5ROb+o_&;O=LECuL<ypB&glmN8JT0;o&;D?*Ri5!lwf$IqM5n#WvNQbQvX`Y!^Sx2fhmiQ#F^V~$M{n"
    "P4F4g~1>R``^keug*@42H8oaBgoZ76m+3Jb}D7<4*@zxH{^=NXD+A>Ssk$1v1$WApf4nUE~G~rG-azlV5ZD?h)iE"
    "3NKN8#e&&~kaX>N3ZD7m+})3URf35UQMu5a@FK4#OX{K?4S$fbuXd4ahj{uzq}|2GYS4Sjq~PO5MV$!qP*IuhA2$"
    "5PFVbx}Ch_rrjpkN8YKR-D9OIznBXba(OTu-&3iAY`?5HgbsW=qY)bH_O1uR=hSeK;w(+)hm5^38Fmnzh|!)Q?_5"
    "ZC&*1LS1bH~T?7!qu0tW;zfb{l&^&Bd(8Jf>|FlfS3lg%x-P~=_%R@~X#fPKzsz}aUJi(*HiIGx=VA;i56lvwU56"
    "z^&^>sx&eecrxqs9DPnIkdSynzq@ZJDg@zS(9r9P=$3d@07-DKGJ@V$bi9S2TK@z3q{YsT{t$|z9o<xud0+VFaf8"
    "@3CFk8TyGeZFDgyMagsOKy4WX7#tqzmQX3u%9Y<A8IXYR%H1K*%d|j-XYDhAa1*vZCWO!toY?ZUBR>sdCwORn_wt"
    "j>+Gm_*KL<CR~Nwnf!1MCFlD=Qy493^BVFO`jYE|S`MDNJ=bzmlE-StogN9$^h7kvE~#B8yS&aIXW5n56X24b2b&"
    "?sM#(MTZeC(2bp53RJmHevez+fyq;5^#JU4sixskTP&>+o-3reK~)F7)C~4?$PPc^_4hA5hj`%cNDEDJPt~ZMcK*"
    "aHAb^99l=cpW5s6Ksu|%%J#JYSp@`UZrJrCaBl;LELeYpjCqnN5h00*3;GYTGhY*qEgR55k8kmbzH74SDvZ$nvAX"
    "Ct|r$lXBgl|?pGK5dVQ+^`If*irZGtcP!lNAI-9HT{6nuMdrgx-X#L3G8RofLonwjN~tBN58CBZ28Jc4u~hMR`Kv"
    "!SM_Y`s1(9(RZ`|I)~l&W{~4K?P-xlEUYNM<xSqGS5zHz<Kt%YvBBN<*JHKSb$~Xo@uvYcGlBH$IU@f(-?1$P92j"
    "v`$*+K`yx<EH$$*~3PgK`lI9DVdEpT(;jHF5Ijg$XMh5e>oW1zk@Ak=NQ(5Welg00++GFDDc>b?^T{X>tD(26NDY"
    "No5(FdLDk7yc5xbO;?_X)i>@_*O+guw50u7Yp#S&kY&XhQ#pADwPCogNq1_2%gReDl+qc7`<<zUd>~v;&Lk#7Ev1"
    "KJ+m*#SG}40KIzf1G4SO+XE^XCcO%JENTyw4AhQFIqJX9RyxqyfG5Me%N1=1P}U`ggFfn|mkm&^5H^$fHU>H+ka?"
    "O6|#;x>s{kCE`cggEx4M5vfpM>iUhT~oShVL}aFlRTLD|0?0-K5he?W{<78{&oQ#=J6iS17}iEaUaF$D5fRxQCyh"
    "wYEIruubks0I@u*xG<TR$9J~(03ykVT<^FO!xfz`?I^)7ac#Z5aB-3*?%iH?<>lqo2d0WIWM{N@q0Q$L$a!s*rvA"
    "0{e4hBRtp6~w8vKj8lJ&Y#cF)i#RYWXE*fEiRC8~6bwn)SlldXgilMgy}~UF!k4AK1;n%`vqrR=$~aqCS8!gJEO*"
    "a7kHyn>{>El$mf*tYk_i2AZUZR|YG}OEMr(?5k?I+D2h*VvbYCI8jVNQAGB^RMn9py!Qg1RkCovb1k4wmyVU?edy"
    "41-5~_=m@p{AQW98ef?gN}418oFreqQw^b8J($EX(MC)?`FamV8vpB@89z>s{WCIc@>D5Q8|Y7U+1CUr4iQ2ACpP"
    "_<$hAdatsKh0*z$0D=78cK9%MA*UH16#`$J#;6JbIJtJBjX$-iUDAuBCA(Ui21e~q_rUIMV2dsthQUJX2pkGYsHe"
    "3t^i%UsW~*V&uAf>8|xVF$s{{XzCQx0GipZ9A?*v@>FirGf`pt>ktVFC*)uxnqEh6QrCJn0q=J*qw=fERq9el7G6"
    "yw;@?e3xC$A1(zXqmyc<LNk;C<W8`Q~kik~5YQSP};~CK5!3Ihrqn_rmTuDMBkfm_#I$-%8|owf|-xM2?3T-P=Wo"
    "rU3Flq2kxb3PRN*vcmJGg*kSB{+F4Wv0UGzz-W?tT=lY<C<3QX;4%W1;0;7LjtJ4jT3jb@&BqFfDcQKi@{7cpGdN"
    "r~b|fx1o6)1piY1naA{NoA4Ac%ictjo~CsR5XF<eJ&){eP)|3BXy9Phs(5`dJ`s0xybLeYH`#zbebg-VF#z~%<Se"
    "(kI=pXo4KmsZCnYdpdGI04`$0>DHTn$@v6&&Hz9?TRSHsqV{KL>U++Mx)KL1sq7lt|FZbsX-)&0M*G+k9`a#D1*m"
    "Zk$nMY{^*xO6YOYtjuSPKoO+l$5}t8=Wq8X>r^`}R>NQ9`Q#18N?O=69=zI?!qB9AM)kY=tWW#EXE(aZTUoxM$E?"
    "DCe8N|dNx#7lEqg-N&zd~CqMXANo84fXb%}Yk0qJ6I$Qx#tuUQGro0*IhAWtQ2^c{pXbgIa(kdW5T36%*EcQK-7e"
    "SJx!mlbDiCjJ8hG@l4JWx5h|hq|q~{^$QEo;jB0|*EBj8bQ=0t;b`KSfX4NCipb?c!bZn@U5}eHt%S;BSk$_#%_J"
    "yfRMM8h;M%;F2&l-w5e^WEp@<~8PhGDN86_r!cS0e8GQyWJ{4u4Jyz~q;L=iJEKOq0mEaQ@ZF0}w!AjFe|0bCoY4"
    "nLn)<@VStvxq?~6%h?8;h2td39TXYt!S{ok}(P%sJ#@%;oe9p*KoHc*Bvw&Qp3I7#_>$ari;zmx_&({*#nmw!2gM"
    "7YC!*O<IXV;?|q+*wI2yiXlNv1h(aTvhbw&r5qG~TS|QI;30P?K2ECJ>V05<i??(rRPPdRk9KnV5*mz9%&r;Z)em"
    "f$Xr6v}_i5&4W4~@_L*M~-EpN7aqhwv`?_+?qX0*wh<Q^-PLXY=B6$>;3(H%T^`<OZ{8ww`1BDdB;@4gl??SWp#<"
    "uLyLHDIK=R5palh#(JP)+cDvh)VEqeDGz^o^G2l%Y)ccjT0`D6APJN@1XNR+O{jmDsIFa6f*ZC)*D)+l(7=S`DXB"
    "}7?dOgoJO2leOGb}cW40Ds1+X1AC`d@s?oX$%Gxm=6-y%nnXe!RpBrd*6o)a}7d5NQ^zk)7eEKzM^pU@6O>2&!_h"
    "s97&7@vUMLAnuS`j~;Kyy462z;+xEGt~~q%ZwyJvZ-BWV-bXa6b=uMhOWr_{lVd@gTo(Smu@>Ud2}cBHQOOQhHVu"
    ")sC|R2Gj!$<llh&noJfS-*_L02G;G(;@Qs(*^s{^AgMM-G^WOix5koCAGw6iXhE|<mdb@&i=uNrHr6~WsW!)X7@P"
    "AnVlVmbW$yF%>f8HLT)Z{hox?{6o<>x7tn?pwL(>$YzEf3N^H-@BtUyyGl76ya~xuuPzM)^}Tp^-Nyqn_#rM8hxK"
    "*mOU*Sur<#CGA1+)38O4_y6N3Rt!O(lw`ub4Qj^RNSWw+W1A!9*rfBZ$Tldz&mt+YJ^nViw|ymE2vnfThuFY$)(F"
    "&)@n9P^Tg7hKW+k$o!6~H#>VHizbM<0Y)c77MpceUcxTt`;is8)|*A%*wwZE>3_vchybp~W*Bo~FW>KUt%MpZxzW"
    "wl4uCg>Zj05<TuA8*gqU?jSmES)SRI8Tu-XVR*U4)<Nvdy6d(#)JBhChY4y8$*w=2{vJ!jhZ;Nh;FQEcEhyav#Hf"
    "7^z8J?<i+^TRmyAJro@QCCf9quRQD9rBV>j$=cO%@b+Y?*e`9bpvv!%>)pA`kWOX%{6H*%JWl|R@!C6B6bz0QUon"
    "2OpK0+pG>%XE@cUD4)dN`_V$I3}d!f;?Y<LYLC>%*|M2$5v-nP<7tXxUZF(<NqN3IXTC(A-XGB9S6vH-CcVqJ^YN"
    "){di%Qy94B0j;TQ7E))~U0vH_<7$ZHhDMQ6ma2#O|H~V}JROsFf{q2+fLDYM<z@niU7q7qd<PG8ejymoEExA(8<N"
    "RTL2bPkhS>|%+7xTgnH1H>Mjj2};A#Sqpo&>S-P>%N?6Gf@CQYO5yEbZ7lZ#+=$P5^KnMHA<D(gCQMosleW>KCvI"
    "3dMg)K!96qKq)(kk%>ra3ooOT?#BX+Q~e_zP9Cii-<d@y-9YBgy_0%R?QK%$t*Yv`4#121fxWY*{rxMBwPQ2AubC"
    "px=G}bfi{JS+A4<Qm4O&ouPTs%@Yu|5zI@0k^IKMwAK+jRO*yb3I1vpGjQ4Nz9C)6AhMH?TY|S!37wvc&j$<RTQX"
    "PO*$2=O7xX~ulu7ot&bKCOd)a1`$OhsFm1XDt1aF36X@M0lM<bu!3e5*KZl_<)ZnG>O$x2>~r*A}a=8{_{da1d3v"
    "$m3j?$yGe8f?0wZc5B0ueNVfeHAl(OvBq4wEVLxTv5A)i+&bQ#@^EkWbFP(T-fSBWd3C(O2nTnVcQ08~Kxr<>>jK"
    "oNMkq`o-T#BJBd&-afcSo~_z`Q#dpLpGWQGGNRc#h%vdpv8p<i&*`PFoVGJW%`esA_)pC)7tyx(jn(XEGK2^?(f="
    "myK+2J2uu>1?$UK9IcZTE=$kecK~9<l1`_NkBtegCtB%J&%n*&@DS#i~!Mx_eeSnIrZfJ{c49S&~=-3tX)>xO~$h"
    "2aR=9i0hDb>lzGs2x5bfQ?r-2R1k0&Sf8ixJs&;04txW;FWf4q(cbr4ePFSh>hB3CbBNXr*r<mjzCVfV{wN|7?xo"
    "+k<Wmy7UnJ7JW$i~~H5<{TjG7gXky%!+b<qT<lV{-P3FSAFZapq3$__P2W`^q{j@7=r&>JkqW?>;1%$+Q(qXtJag"
    "BW)F75w4n&r4LGbx}Va!`6^Aq+d<Oq6UdEjswR;RWZYKqr<b(mw(t^nBaOFZx@<^!i5yUejYV#@pjJL0Kjv3tTD4"
    "EpZ<>__lxpwv;OG73hK`3NKO%hF;q@53y~(}z7h&104BHVqyyc1X(zd$EMH;r_MbP>tx26+N?pp(S8)nD7zSy3#6"
    "|Hl&5i3iu*~N64jO^nhi?-1K`3MM~6O>UHyUvslkj93mM`22^L{ebBFh*L0Zv+*a73`jkgGOKhQY!Y2-oC@Ix>wJ"
    ">e{;0=SNNC9=MW^MpB#}*v~JN%y>lG$(ifmJ1(9<VnFxuu?rW_nScXMZg#%DYAPGa13b6P*Mj8=iMmDEaAjVfEk1"
    "|;mq|+=gf^=jL83myVFc1l3bE>^lGZPjc1#{M?o>{B23A~XCLv^WOJ2Nz~Nn7n>l8+1W#Og?bgec4V%j%*6i31}>"
    "X#9-H+Q684oZr9wNH?VY*7N;fSG`7a_!bIAm>~hNwKr%@K&yl~I?dqsHqVx0`Z1o_J#+0rC%Wf@lh6To)CivbJ&d"
    "n)jwOz2o-IGPw$>-Kc7oiqu0w06W8Ne~w+|ZPA90HaGA0LpS!iT3*0VwxEmD6A&G#{2Zh;c<{U&CQQZhLk{`AQ)7"
    "q~5ITgubUGfRb;27ali%%fA=UbV>m;6i7xSo@GBhzR89Z0pt0A+WcRjXG3{vSCbIg%#gE-rqa=8TRfgXu>_t&MCa"
    "gmXqxd1bd_~h>>`x%_mBjW<(K2-f8Lx_>7W5k-aV0JYE7*o!rYpyN~JTW0*UIqr^gYoIz|94mQyXXIv3v4N*~s5i"
    "!^2%yC~Zoz)p=<?lmp@VTGSNV}T_DI+{w6K-e}Pr^^2K(tWUk2BK#<xKB-#Jj$#u4~Mdp`add4WLpsYR40GGh|Gi"
    "R_37{4A__Mbdh+zWLJtkr$~*MWYDX*!+pF$YJ|GFSisX1fMYxW<j<KC#;s`-j_-U?t=M~_Bps7W9%h?g;uQQZiAj"
    "Sy=EOeXZSjP`8$6R)1HYjY3ibgv1iBe;!}sxcG%m+W7F3$J^X8H8kP04m7;I>M>=?L~O_Gz)-Yl{a?0v0NVi0PbL"
    "?Uu{j#uQ`Ge==x@fMv(`Ye$+1{)_U8AXiEt5|PrR;C0b^}CYiEB~}=3D`&8)JtaBKMRy;6)A-heJnUHgX8)lHKEn"
    "8G0v$0%t>JmTsz2*jBT8eeR_3;Ah|ah^`8+|U3J0WE5FUbXor?+Zq3M!Y}=61lC7y0Y{AfV)>3UO3O)6xfU;i|So"
    "UiI%)Ve`*B6WM`XW$O5i>xynNnJBC?|p7-Ci+?T~ZvAe?K`oe0F;D_RX`mAnZ<18WY?~PF@E!@+QIiQV?gEekO)A"
    "72VPwRc-!6+KIiB<}mwjR$ORX@30sm-&x(zd6pZy<&6^^M`IaM$2-_UpI#&==B&@i;c=<|7ZSrU#W73PW$}+SlCg"
    "^!`3|LHy<}S7l;a-KEzreBRh<F_#iZ2HD0L|Vm5|{&i|PIK=r!pRYp9YKwVE%6R3dg@J`K$vl=7mpSmx8>(_nhGm"
    "EO{4@Ii830%@D#Ew}>>;05^b!0Czt^vxI0FwROxKcsg&^c~U%V2INRO86b8e`j0Q7iU}Kj_0d^NuDoq3AYYTd^dW"
    "3I$PIQo!}mR5GT2#4`HR+s^b>(veS+O=+SJ;ZPLkR?kM01t$MAcdhERGuHnipR9(#7Gg9l&trZEI?L6E+jXl?MYc"
    "y)XL_!EHSA!S5usWee0uIpRlu{wHoCfT7kIJ06-NOskPPMs1_-w3c>?07+`K<FXo2~Oc4p6&FL7lKwcA_Za`=~t<"
    "%P;0swTnoD6onmsz%npGJ|^oWg<Ot`vRDNt1+)BW#SG>?^k}->!b?<gv(NGlbmQD%Y_!A#wJP^l>%}a$YZX45Etl"
    "ENu<Z?&sPMJ$JwA3+!wDw<7&p(669x?|7S<GV$nrTdS{h7>zx;Bc&0!Y=k{*uQAD(jPj`>Dd-;U~7HCn|XNd<H47"
    "ndgXo_W7>-u`iSz8=zrI*&9$>!)vHga^;&-7Pe9hm}`p2M%%DQjfZKo|~mHv$A|a%#8aHssXv-zSD{K)Qj`Y&)Z|"
    "dv&xI#ZR;fEhIv~pD7{DVTs#<h{=P|zjA!GRWKie#zjgn}M6U3va8kJ+w@@y%iQAelI7K8=gkfKFuwth);pgz!rK"
    "bbqJ!jH0f`^Dq1ym_YNzi!T?MKFfv5j0mkDk<;c~h!4yook-Fwi}j0!fESYPO6e6nPvQ+MqKT>(zTCEjWJUu{_h^"
    "(eU<;Gh@lQsb^P@vjstw*v=S+iBk*zqYs?v|F-P?CudtvddU;M{KS_mr*QLDT89(*q+Q135I`y%;R>2ZpuM6(*h@"
    "~1eu4i=nJY$TH$o>U1%KB%BEL$)Y?mTiDtvHV8*SNPqCH1K=sE1Tr5&nTY8sMJ_mX28E({*Q-qEsgH2tMoe(*rXu"
    "s&YJFLdPJzHOmK-Ho|a&<G19R!$*7*S829Q|lUxD^r~Oa&Y?N(NCwQIFAF}@-xMFn<TefygZ=a<YP7(Aa_SRk$jH"
    "z?UP@wC^#8Eqj>YutmxEiG0RP&Y|kL<yNkZK8BuYyRi)Nx5hci_g66Q&(b2UQdySSk41PfHA2c$Un(#G>kSoHnSY"
    "%}OqhNLTuA<Q!EAoPJe#vn|H5ilKL?YeA<&{s+f`S1(n7XzGt#wAGdjxc^og8?{Brzrua-&6T2-Nd_W8sh~y5<7e"
    "3R17Q-(CY;ALPQh_(TW)1f_sq?LJTrb)&FUY~*Wqwrl`d7#tROw)ItVt33dHewN>PCN&&(&wGBG0PjcUrDAe7YCc"
    "2N5!}b<LhMO`y>g$V3qv(d1g`}Bf$o2myvl3ZBJ_z(PwN*Z{KAk+IBafwcZ{Hwnep}3n?CULJq_uWo`i`c%Fpgt*"
    "xma1%rrUVCeQEb1Crjh&z?JLn=<G@hkG%)t<m@dGnjPfPhNs<KjB|v&=IwdH5;0cSe>K8^gT9EqThb&yz73UDcsO"
    ";c#n>r-B)k-)A@wlg-zL`4tHhnUAJm0H95u5rR(~c4ns~7Hr{%c@X_Dy94S-9?pYQfbkOhs3}{t-@A3Sg#mFJZ3P"
    "7eA8A~W{3Y&jPXjxp;_@xcrN0&Xa*^jxWy-f;)v*1#=6ncxL)h+Xf{qW2ZubO_^lWrFdO%QCMAp>QU3-VyQ5q-+j"
    "c5RQIIn8#P6U?%QH@|$TFMK+jK*vVlM)bA~T4Wpgg_1e{cD9~=uYHR}ip!D>;8%WrJ_(P%JzM?(cpr_X{7NP$aT;"
    "w20E}LSCZ0b14=Fp^fBMw4qoqh6-`XpzSsS!Ak!z1{ZCCdD4K_E@_Vg6ov~iOMH7z;)tIBViFN3WA@HWqzsm$GWi"
    "TnzrrqxBBtvrGJN4?s&DPa@>TM}vzrJ4DKdXpMtDEw1OIs&O7d2u%}Ur<z67ulYYMf)myx+wgT$);6-@Z)EtI!8J"
    "Ap=U!AOnE=5V}#wc@k@tCr~GL_Q7Uh!duLck7{s%h-~v`EGQL~S7$XR;R!e)s%@#xX4aPNFUEL&S>lZtS8G&_#97"
    "H)=D-5}1Z83qkD(`jWOrK63O1TObh-sQsNejbk>J`<&-A>HHNctyNe=3#~Bf)Hw{aY)};={4?Gs%dmt=A(ElqZ*h"
    "EX!b@_|ud>GFe(UM=vGvnU<a0VrF3obWXyGCB+f6XJ;J?11|-{W}elUP?$|H29Cmo=ed*yrC#SidjwXPikgz-&DC"
    "J9vX2E-uiG|mJRFr-D~2pZe;&rvL30$-wI;RQ>^)TJOLC%MfY_8Vq8qDvol$+TgVU4!H?K*}pb|JV6-j3l#yBtV^"
    "PD11nah9d3Lce76E*>Es6ZfxWi~{SN*H59%~Pr%exjFXxk1@e=w`&lg!yGWp!JHvhpq`D7>nC}K%09e@Kx?up3v-"
    "T?Ik)sR9PJLoxi|rpbmD4=tnl=6CVcYZN4B-SF8n1McWXLsG25R-wGxXE3S^^n3D!2RUl6=xwVv)M_()fV39aDaj"
    "S=G2BS+7%pN1(DlER~%S)QS-S1EK5BK(|ACcF2$5Vymif`bY=vIZ!@pQ6yLl>7Chd;^D1b>ZZfX}<6C5_jZTQ%ij"
    "x)dhm3}{#=)Xk(Mv!Hl!eOC%eozqtOjC!iSwis%w5o1%C(&!65!4sQ>bB)k+`XC!}?c>_4KNb+LG5V7rb2#`5>-z"
    "Gp<=Wl&ttIZ|g(PajRK41Nz5COf)8WbK?wkGL?(5V2<KfBv-qGQ!6ObeSxU=)1GKGdjo^V882uz5Drz{!62W=HVr"
    "nQMKZn0wBf4OokPJ~{-Pmq@-Nxs4e?Kc#lo%}aC@o@n1_u<j-`~BV16BUi|^F#cqh8_y2Q+U@cpXNPMSL@|CZ@vn"
    "PHk7&UvmMfZ4VPy~c(V2}LjDzzJ!czj2kjGq2~|^V5JXJpW-#yu7bf4R<x-#M<I%LD0L5A{6r7v4jy%PU)sawd`m"
    "j4Sjo?yx!kK}1ek@TGw10-}#j}e@plIaag1kmsh{kanGzG#RlMKGWYem^a_$+?@u9=O2a(Fm^e!4TgBZLC?>Fz!2"
    "A)7G17*n|qsyUAA4zuCqplWxUd*rU}x*d?Fno~ks-n5LhIM6J(p;;~uZmD%Mi^R%+EWliYVvS*NZiN70Kpd3>w0w"
    "0HOqVsGVq-(I13N(7`W`v!c_#~Jb37~1n;ol^ypUq;4XId)X`0%Cg#}yOMVx9Fi=_5YjlwJ<wb4;oFYxe_m0G|>w"
    "QQh71;8cLW`QHJI4B^FESa-yalWSZ64|hIj?pk6+UB!Z)ROF9H?N5$CpYgNW-Xmuq)sHSr*g~fN9H$OS)sNgz0k#"
    "IuR<8lB5J%YEG}M|uo)yH@7sN2`ZnZX!H$B2H+N49frSa(z}O_#d=t|;MhT1~;;wTgJh;vNz6Pm<t;{xi+nnjNnK"
    "ST^<uM^>8&-mY)9kYO;g!kJ@vHsg<omz5Z@k(++3T4-hQE?G2X7Bfljr*K<8l9jNelLv5XOParGo9^((M6$qxF8a"
    "irwd#iQRE1dGy8RUKvELHMc4Ch|&r-Mu^#ox29DLyZ~lq4n(r8t##8CNT4(Qs!qS*ERJ-8h_ukop9ZlZsIyeWAMa"
    "qbua&No{OgR_TOJ;mo=-0x&|4-MlMA56t0Vww^do@4a~2E!_OaI<u}+)JEUVRGNK}*DM_!a&f>{YfsDC^?eMdEoQ"
    "Q=QGIEyqC@urVw>j^NuSBED(Jbe=?Q;X#{Flz{7TRGq4l%QQ~jf3%30W6*ng&D?rH9^lOel1P*mN^AkW;4*9msO4"
    "FYLlXrB#yH(TV8(bC9~?X<`M#k#ZVApgLITE)oJqAVm8xKiNH9q=Ymx~M)M;AOkPl|J7x8*&?-4g@+m_6n*0&=St"
    "w!#g7K4WWcjgR9rN>?z9^D>_f0&?a87RDX2>DNm}fKl(^Sv4mu~Fe>m`i+B9-D({zaLkhbwk1L5!^23=ga0oQ((_"
    "{{ty5r{4RKV&s;}npptiTSIx?Ie{poArkjLB+6^LkU!8Z!3ciIyuOq~DcN}P2=FB6hf`HLG>_X<{G*e7oS+U$cH9"
    ">Mx}K1X@etLHhFD`zxnEJ*)TPFgAkmzz?B|YHyBnYqI&q+)H(^$cCEaG&)+I|JXR&9IE?4W7s$qlV`A$#kOMNQl>"
    "-mtrqvFAf?_La_fAfuk8a75L@Hx0wL5jiDK;YQ&jjrZJ?j(QG?;?4A{$fY(JjdVOhv|Xg&NqJ;e*3TAHP-28j11A"
    "GBFIBHkwlsdDPkcPXjw@nSB{W2lA=U5wz<%5Ls<}31bx@=PcOWMtAh6q1<ap)LA8VD$)`KL`0g8Vqmf?+t^DLuYq"
    "Tiqh>a=<#TtKijR$2i@j>~s@rX<+KH|-<^WdH5tid>_ywuQ78+V!f`JwCKvIwJQ;YX7Tlm@N1o|hksWyLu_UeEwq"
    "toj@w<VUHoTa3e=(2-oH(uVrhd_X6RLx#t0&$bX(ha-nh{&sS@|MvLk=v026yczC+oIiYhfD(as>D9sU+18yGJK)"
    "C|Gus+C#1}!@LRS>fVUc3lig~uRJ?Afl_AH+VH&Y6Va<N_w!nJ8gZWGuE1E){#4&LoI9|B(BKJ;q;XUwn*o|Anwi"
    "0?0${eiQJ!Ci9d4_;$jT6=9WoWYbOFrwu#gcM7bX>XYcA~<E|RI@OLWJwP;&QsmtkMO)SqXR85>+s7Sh^=#a<-nS"
    "Ukdo6Q3KYt$z*M}WVxP_opYj~;bLLdz5b14qsD_+Ly46b+ImETBHsW>pp+wyNEh#c-jY`sujy<Bi<gfV+{ng#T0E"
    "?0lAA=d^!3W<lv5s?4@;byJ3Im_!pO|`)kNw5L(?RlFda(vsYnh^;)lpjWuU&t?TxE5(S}i*){m>Z#9Rqjg5;_&^"
    "9t2WH$(1tgmyWzAxRPZ4&?!!ncSk1&|H(UYNhUvx#R^-rzAOoq*p_5p1{a@YONvBAIT$ZU0C|S!vDd`=)8HV|KvM"
    "okhLxTi{IL7x_-!w-zlX10zv;$b)d|qI4^9u>g8F{Ezx$>~6aL;|73WSP#Ht4R2s5&CFHGpN!rl)8NijWc2xz*t?"
    "}b-}lnWalJcLEcAtM$Uv?0uJAm1@;@;pvmngyB3J`@b)VhzhZ#556@Kv#C{c2i0oX?nDEUV4T)HYPbpcuL6E!@@`"
    ";UYeAnfq8;YfV62C_<8dC<as0{S{OFAflRvbT0dUBR*2t==3SF=YWgG8ung)LyE+BQ_C!0{iW+MIxeS)y!nKA=)("
    "XT43z#V92xhmIAooo>N`L!J`HJSk$X9ofi0~bsI11V8B{3@5OHOYVd6Qt82_!`hs2}AcfvQJ#?jaGw;n0}6nZTah"
    "U@p}Dr()F+YPmrrFD8Dr+X+zMz!AF5Ve@E$CLI~WJ!|U{#Nh2owwxreP|FMf1wKzft@AOAx6QHl)9GH%^X@S_5F;"
    ")3&}RFDmTE5MmW)RH;P7wozqfB+y*mBz$G2}!PX257N&Xk)->9p^n3>RK1YM!9{V84nHuk%}{cird$?5NY{N3B%o"
    "&1-P6s*5N^A3jM`+$FWtSRMY4zKc0oo_I!!;Z_bPyN|^JvJmXonK>B3+R<yGT-XSPd3J|VN=xAjPZcot?{~AFSi%"
    "7j4~MHfrO#Rdn5=?#B<281#S_L?OFaYpSf7eWTJVE(S5XAQH5Ty_kV0nG2)V}Dy)gI#JGVeIo?-^21C`rQZvTs!w"
    "^a>HZqquA`Cjf5Dk3H1Y~H}DHB>Uye)<?hk8(0%tdZXFu!!`Bs#dOT>4|06RHTvHc1U#zNkKZuFrd1fh-}B-r2=i"
    "A4T94JeChPpgc4`6xk8(k5-R4QBm_5K@xnBq2e9x70;AUyT>t;xDMn79Jh0If^w4DS03=?16|~Iv_1Py_$+O=A%+"
    "Ilpwd!u8^<@MX^Qc#s^H0f93t#2x{~QDAO~rRow=Yd*nLYx`a$o&s#ci8Ok4aRFTfXyhdytuUItZah`XZ0mM)tsN"
    "kj(!aMe!O#-wmStCS=QoVuEQ46#PD@+Od8oEJIj)xMcfn!R@gRA=*o<l)xMWH1Wkq#%mfG!0kL5zOKqU-JRq)|U?"
    "1?gkXxHc5%XAp4tlx-Ana+e79OH%W~9Fb)F7TM?w!fF4Yjobz?Im11rY?to=zR<Q4&aRDt4@wdEuCmNNvZ|~wWVw"
    "-QB5@`nK16yjD1-s{EBG{}Fp9sfXLLX}!n0Q|e{(YpEwh8R0<Z8z{iU;NqT`Zmb<{mlXN8TX(6a4rQ;Cs=&BWyeh"
    "Z*o`#3WB&&->`%C@6S1lFR;MU(1f^IBFr?d-z;cUA$E%QiAHFNcem5MFBZc2Y9|)2tA2!dH1Gc5>!ag6#IeE^ae6"
    "7PwVw`mf8ITKv-|y<{gxIqV3Ml7Lyub?(^`@jIaiXHw0*IxSQKzs{6-wghfIP;b~evL^tO=E$u`AD&`!i;hqn0YT"
    "h!sAB(!@+B(_s`fAyFw-S~_Fxn<)XF&vNN!C3HL7|Tlz0HGzzDa4rqny;bdUp-7mNd3H_vb=n;tUge+qz8{RB$_Z"
    ">{FjDHMr#ymQR_2dvMizh>Jbxx`@FG2hi|qK6l9wQ+Sa4SSUuX>y;Etam)J*K;tWsl&-(yk`NzMZmS2o90=t@6x~"
    "rtwimC`5a`E?k?4e1p(A+R?#(}m;2gLU7$hphaeQ%{zjEmzyUGf4pQJ!IbGr}q}I+y5Rlw*ZqQ=GqPb`jcesoE_U"
    "lf_BNX0dj|P;Oh;$$qWz8V1l!CSLcY7MNB#Np!Hh&<Q2xRR}Rhj&p*?nFs(xqj?9<+Vp|fB=e8$2K{wDUJTj$9>f"
    "@^ACJwjXcDfHw|nozRS#^oC`-<81!}*lKp9J3?7aAPd*{3D=YJrT^-V>!jTV@(P%XECVj)|Ykbj=MXrM@@us_JC-"
    "Aj%zTc(E<Ar=%DKvjw_>M=-ilERG<<5D$J8c8mfS-FxF5bM+-vL^fICG|SYqWyf;^B(9S_zb0uJti`M?KG-zT0Qj"
    "?3r1aJ;}2-@dWf?hM!HAK<GAsMLv;TRhoSr41%IB;GpyEQU{T!MQ8(;`iAtzUxwQ+e_bK#yJ5M#d-QV`wLjgd>sA"
    "|F{O8x~cd6*Xg@`U=TrSmOL?eBJ?9xTs6N*laqPo(TDZJ5Jl_$+A44F|T7Rt?>yU;6yO(qlDB`>)k#{ze)!P+O?E"
    "#cmk55twD^rl!t#KF@a%L%EEm>2@s2`(2Y?&VRYtLrFzphiIp)p$Rx}oCO<oC%A#$`b`>+jqY|XdovE(3AXO=0^#"
    ";BbF<lo!HVi7<S*?${AKvkhPRte5Ime3V7`p{ahU($a|J1)QDNXV?p7e}fr#_Z7=^_t|7Tm4P)8dRxh-2{7sU)do"
    "0Fx#5%J(WN<`<NoL2N8i){R<fNyTEFx`%8g}~j9Zm*u7ZN)xu*N#`GxF|3;>17TR*Bow_sfIgZq!mU;Q~lMpQQkK"
    "dAXME2U5a>hB9vxbN^%I05@bp;dP*W<L@_epCgyWD4b&Z0iR&i9B5wjp_+2i>vdrlYPqPSabP>jDusG^xc3Vy2vr"
    "lI5c?EA(Wicl3op|s;1N4YXP=#BY=lz*T@Gz6j<&;-tMG@6=Z@sLmh)20)o^_9DSfP+>jCWT`zAhKpCAx`oHkv~("
    "SB=SLJH{aBkZhc0zdNy#4l5V6hh7!bo`@rJNv=Z{(33wWU%!Yx)3hVDe9Zv^i$5<sGk8VkosMvqnD;&JBlMTPa2|"
    "iuwY~8KHXiGK)|t}42s}@;ki4^E$7WL01X@hi#aqAE%?{o*_odOKOhZIn?4mua%ab*$?z!zW>8C+O81jXFpVj$3{"
    "pr^!G=rk*<stMTeblv(F~wl}7<{P)S{z~#_Dtx7$Yn8Mtd;A!%H6de$0y(Gu?z&Nc*#LX7RCZ-%7usM<xDs<N5O9"
    "24Zr@wH}?y3IHL~<b~Et@MqW=PE%8JyaFIy8-JE*540P=hwa8^ei0C<O@Bh4icsl&$_~3LuSSm*RhEevTu>^R&%*"
    "XhT&OH0n*|E!&Y&VE>x7!wG5DRU>Ipfj+CSoFT;PHtSRX_p}h#@jI3Qo9ivEzWd1OEG7+UuWt`xG;di;uZ%)!2^o"
    "ckTUq(-ZxE(<Miq*jRJKpNTHH?s(6D&cgox*{xv0bT{pW@zuKgz<YtiV>XJx-|YPHyKd{CsI85?v~h6QK~>OZ1hK"
    "^^TGD0M7kM_1_SjXXem~%kwubLG4<q&Y?%UzFUw^l;ixDa$eY%ef;llL-UC;TI*1#=R9o;THFX~8Ws~^IL4K{13I"
    "$YjRWReXyhn=nn>o-EA3ePA<-cNg7R%}OKe#~c}kOonq<Oq?kx&%PwL}gVdw~A#E0Waot6hgO^Ciu!VEc<gO+)fY"
    "8ASUFx`g`b+iiqE>qIW@Jx7QC}BIm#G8V?Y!@n3iLU`JJ86j_FZbCi{rdFT0e;Ze7{Lp^CsT(#Av2x~jWXng!WYS"
    "cUuB~~g!L<mDwf~>sGF*x&k5w<;*L$RPFkx)-FZqDUe!5SYRTBD_`<L_JSVYjPBzQV=SbwpNq=RH4vj$7PYp6E+w"
    "Tb!%o&dX=XtFC~;@zUh#*W;r2^{5ce3I1Txpk^kf+8@1%1#Zk<aQykfcBJ0>k9O=kTF8m-miJ77yCsAu!j8DgsI;"
    "$S_b5%kqLm-=&?OD><nnS&Wj!886Jm~QG0!EE)Zja{jkF&nf$z{tPu41TeEQOjjejCjHH-TO0V+lnqx$lgJ#nsWg"
    "|jctw#+o(Mf2m%n?lwS3{XQL)qh2RIC2m%g#9YxN~0e%7jT)EFOys_FWA?R8%H;ctXeN{Wa%bL7l;ZN{k;fmW(&#"
    "~Hi6ZDJ$Un`sRl_z+MBBSuwLv(!dVv-%@wnarq>y;PYn(JHUfy?Nw6^;%NKDQ8dszpv~F*12n8h5l54HUdfUt4&@"
    "NVJ)DN~w@b;(8w*C(V8v2oTzV}p14T^9c6?|mxvF1}N^-=VFcZL6CY}fhfYpZ{X5n;W!+c{Np0eXh;vFz`;xmW^u"
    ";)ki^c9DO?f?p`_b2I{BWx|=x=~Tk$T;^Y-?1YyPy3Bt=JRSpCCNF4dpZno^3~Y}{4Y@|>u8xoUul}6A7YmCKa!a"
    "h9P)@Ke46wlW+4etow*Pqkdn8Cy4nQKT+;RZJaaAKY%q*X-G<cAdIF76V{jj*airt~E(C@yN0*4J5a|^xP-JJM_@"
    "L@Mxa*ww~KEKE(m<CcO^EFjG#tO~<Si?S<z`U0=E%TNQAFL>mq(Xt5QYq_yEI~3|&pV%zr^zRhzdjLUId-iJ*ET-"
    "IefA@s-GeG`a6s^<UV_!}l7AKp8oA6DS>rJGol~2VXI?ikm&5r<@i;r_j#oeJo@mgwG*v$qX7VRaXer8eey}MWBh"
    "k*aV9+Nls_9*;^Et4-V(eWcY_ai;ca66zo>cGhM-X1v9`1fx8XbCd7;Ogc9G8136YN#OlEhjXsVV2!9AS{|8+3v|"
    "4i@W+Sur*~lt%Z_otLU}(`>)UrjI5CTTF%qu#<f-q^!QM%pOu2lZ&~cuF`?iWlHh&9_rX3i|bW}%b|VHgnb|--(8"
    "LLwj(8+`Y4Rc-rIaq<$Ov<drq?iu;%oU0y#(&^-*RvAj~`-U52_to}Tb2s-kVWX%^J)vF)ypMf32g*AFUoZT4?3u"
    "7ow+tN1_1E^E5q#Z6;=dEtyiLLzV^{(*?xbS9yBv*W<L!1|g5j_!u4){VF%gVKatsr8f-6NnvWg<f^<(Zj<>pHnJ"
    "ZqpA6N+F?NVomy_)P+TwMRJ1nA#HaT0j~6?e&k;|CJWnj{B8UIKx<<&f=X7dc?3jbYqa{IrgsD|V0}#=G?sj`rn$"
    "BG#Ax#?d?U%lB#uDN9DqnadaXz!oot{ytapzE~LGpy@)RW}s*uN2!Wt=LZx-~ew@IZTU#`e;8G-Y<~Fe+YPb%geA"
    "ox53Y!Xp!wWt??L{$Fki89SPGpZ6M1{I@*Ob@6b<ETqt!cN|Ngl#$a7P2CWXq~JQA^o|$K!u+*~XQ!fFa9n!Q5&A"
    "-KuMs0HefC9^W<I)^DPW`$*de`w;}FL7sOiba>6v6VH3@K^f?>dCsR#>i{O!l!BQ@}kV7>nE-8Vb&afqaYs2=cK="
    "Xvsb+p%ZK*E{ZzEvp-_2qE9MH~gGo62M5z`1)Ho8#-UV*9&I8@XB+%qP;N}oTg2)R}_?Dj&qn<Z1y)<(3*$`k?kK"
    "ogZ&P{v$*r&lQM!{Xl|lmgoq4nY{&M=QiMpi&bHPjvi&hz7-KuKk6FQqXIt@*B}+6-{i`b^mE2IzKiKTx$tS;-zO"
    "u8)eww;2E)M!YH~Euru(Q%Nj!%>Md`STT{p3+!cX*pT;3$=*{epsDh%9^O-x#)oc@c9Q&q>6)10$GaOKz|wJVNOf"
    "dH$cD8+sLg8sm!iL1(q&kJ1*vIi(j~oVSp1y&ECnEin+`Okrc@ch4Dg+!ou}N&l5jIM-%NnOJPYgulScPD}bkG3m"
    "Oy;2L9mm-1n|hk96_YyjKCkA9@J(XGv|#&<u0W_-;3N2JTl!lXN4#uo3_`)oyG{_x60;gO}V?O9bVdYq6wEtd6)v"
    "fIZ6e6$uwcZSYRsFnhZyF>@boAvnSNv+k11Tk$XnLp54HqQz18!x6*<=cFWVKzr}3`cw=TcTeC2De_q(a0Bda#O8"
    "we$Gm<ZmXpg=qE0b;6<XNl`9pE(MFAyHZ1*65<or2Dq(7EjaJ(!9vof@h5%)|JRkqdD&Z~=3?$-HGe9n3re0Muaz"
    "kWkLY1Exbe*<1!KxNDC5IB?dUk=)t&@sj+%Rh08PrqFaDJ-j<P3T}uPE(#o`b%|k^CI^rpWharNw#&osLtDVPLR("
    "kN_Jr{A=W&SYT-lbAv<~gGnd>^Q_$7_`kDqbV6a}`Q$Aw^C+RBWj7WO6{EsXpQNYKWOM`*xqBcFqmW$-HaPIw-FF"
    "8B@qI&9z!~M^a<)IN64kn;V(ZfkxVp>)c2^yLqs6kqYEdJ-$*8ToS1xkQ0uf|A1eH<fa0U$rA>;$ko~+^=3x^}*0"
    "2U&UFt|QXA$3WHE!#%+24lUisbJb<J$94FRnG1)q`6#YO&qV#F@d9j$f8_f0eR%=x&rmgG(u$)vbn_?v~$bJ>i7<"
    "p%=?f_!5HAc6E3untQQQ|k<DNoXa!DzP#n`T;aV(7RWC=3(>UshfmfVlh%BGMB%+^fq@y39Ml-@uk1UcmO*bc&)K"
    "toTlH>Cin2F&}TP=z2xQaBSAM%xCB>-K-Q?ggVN85JcSxL7E(Hf}72tl((e5Tm0lGInwVIm4t+CeqAF<h<F<b(kc"
    "+;R?QrCwZMx=RPgLtPnT?tUrt)Tfw>Fv%~nC71Ce#3H032mxEIFX7ybF01NtmV+(^JxpOLmS`><!OTkn-7s4eSfH"
    "rW9VM$J0|#U)rlmkv>11$|6%_z*LB&1-t0HSXAs)blqon!)Ye&p_4hnLej8MjcLzy8S0e+w)Z5O9lYiFF58Zk@b#"
    "3kWOU8I?lf_xR!GiDpLzx%n8t)#p$^vRb?1i9D#-(y7OwuFuzdb=oyxz!UOj814izsb_sDO|#Q%MIlOhYU3>DJJk"
    "dX2=pWztri=Ck7vw>E4rC*x{ralXo%Wf)u06#cKQ8^!dnu-kW-SL?oqjAGXZS{q@q6gu?V6=5fT(Dx{oT&IDM%0y"
    "qkg-M^yiFRO*Pi=qb9F-v%9uwfuDlw*uL0G`{Xhz?B6y3D-|lqFYdqRX_?86CM=<yZy8DJ1ar=+*ukGVRLHctf1*"
    "?;Y=-4i9(V?$gbqcl(FC2ScFz!@utTO`ahnyWzXzqqpx)tq^*)^*1=jaG)0?MGytSP8C?W4TF^scy*B`^b|TIfOa"
    "N=SukRQ&JZ{UmSKS-Rp;3fh`*W=&$dkALTtsrqN?><P(eoen_A+ARwxSEQayMh#Q-B4gn71gA`jtB2lfLi>=Zzt>"
    "J|Q%IS!Snm1Xh77A)UEus3viP16f{3mA7sduN7a5wXlIEE}^C;>7DE!F6HQG4Rzic~x-~9Q6xV8w+mW27<PFPU|@"
    "6iTf-_NGwnwQI+D6m<C9)5?6;DuhFU9oCBWmi+qah;Oo>PVXNaW`3wjUgi9}ylr99pz@;ib(i+*O(bLWh2S!T~l`"
    "mQApqK%X_e9L7E9TePB!e^IQO10=yipuo+c*m!IA*D)iJe7Wi)f(0U?x2hdWc;y$~>!CEYi_9%wRbQ>iUZCib~J="
    "8ZZTc2@rc{3kVMr6(Zb3rHCEE^te1sDk<e|Mj2f@RHln9A5tU{SwaPGKuRbTrI3912$tRqnFZ!tz>dt&6I`s;nHJ"
    "b_b?+$8VBrf6acxCB7V3p!#wExmcs|BQVB`f2p6glp<EOe*R5pnzR+KY40T(ot?nB1;E6);K-pvB>c`t!i=I{CbH"
    "f-PRDXIhKuIk6Ao&)nX1sOSX(6`fZtUy!=bxH?I{bmpBD;C`T6|6E69G)Dr6&MMMYUS{9_QaJ%R9w@AS!cwjfH<U"
    "Xqa!2F=$5H*yO8@=MO8uYA@P|MnsKIk3%L-T6t7Jl1_+`R<+P&xy_#Zo=j55f!z6kQ?!Rj?#eiha#c1cCqGhl|q&"
    "GTP6DF8NSHXc+jjw}DHYW;eoov4*wFQT~&j;)a+1@EeO`7Qj?Kjbr`xJ#Mt^}$n5YtA9hLGAU8Z0&ul*wtjm`U4G"
    "6O@W8Y;<atx1qv1dSo@zi$JQjRCE;BD=bSz72C2IDUz&6aP18P%0!A8tQ*ZpN5!&kqe2kI>@7tZvdUhX964$p@Qa"
    "ITb(I($QL48AbG#w>CnHWA%!e~LH_%5^#s(y}L<Fi=#*cwRBOQ>GdDe2<3DpU=08V$fcXXCKRXQN~8?<UwbItiBq"
    "AIEZR%V7(BHu13L?CDpCD$pSB!8H~B~f!?R%)@q%}Elh6#he7gm6Gh&iPvuuWyuCMX{?EHSStGtwaS3mQ9UAsVn6"
    "HRHM%*j)ob)W}@)|RdJe?<Bz9=&M8&F3%G(tRA2`;El^l=#%_1MKg1!cJ96j$<gRIk$Wh;tKM!n>J@>u^jw)!d!>"
    "3KwQ1s{Nddb2Ns*Z?}kYL7vQ%uQ?CZsK+@xIi|d)u$`Mw~Zx)(;wI+mE{1{7x%zikP2^$zW<%YWvo<f46;$&=jBU"
    "6xnFtC#E@Ju6oA__&nd3gGsCq8}At_YF@&tSV6Kwb0ML_G%B7t*DBqNO$`^sYebzrB#C3@AT=AVMVyE@2DS!utr<"
    "j0sA`Yw3taNSwZVC+-3U|Qvtnd{Hsu)nl3VL4=Y8_t7V}oU&_anKz+WN|E%{A|-=TH$-OYquYNFrw^jJ!T%fc^M3"
    "I~spY%$Mws<euufT?&z9VG)IQ9FE47%iWAPhX9b1?GA}5;zf-DlkY8f*&>tWMQQ^5p7YzJOMPXrnn!inkzQ)#+JO"
    ">fjrQW(F{1&ntycv@1RU3@_yj;Jw>fDb<7eedw_$pO^d?AqCahq2l_{kS;#=T8a-V$QNw`zj<2c$<XLpeG|~-}2)"
    "#e|lR@OHV~(A7)HP24J2K>#F}$mh8rF>2rmd*t#?LZ>ucT`2N$oQvaoyfE(gQ@?rw0E0<mm7f!2!oYxfLp$`8kTq"
    "ImNSNGxW&A!N=Db%aqJ_^980$WQzqW!Au0$%ejiROm$%O3=39#VeO<25m@Gk+JoY%&>Esd%k?aew@YP+|4ynS8=("
    "nM73kH(Ok&T|-N(zU=9;#?0uY<}+q?VnI?U7UsU4ddq>Z8OyXLEgyJ2Z1So?yqC8kVh8cfXF9h|9P`Iz~u+seLL_"
    ">Jyl;WfP5C=8kSF%6;nFphJ=5F;ljEO~|Puc+E2*BC}6l=MD)gY`ObA%4iM2ulqvD}<yagG@$MsF6(t(mdf{*~Fy"
    "JP<F(Pr;mgw-2C9i0Y{X&Q{(poxwx+p@-Q(h9&0%_R^YOg#!~~2TT^vpEFK4E5er;hXUj>wNrzz}J6|Zxc9wxqS<"
    "y1y5iEZ)S0cWZ1DK#`!!iLh_5hAn(82T(=Rv|D3LgjdoXv(ND_;4h?LRW8dOe?m9MM+m6s^ugk*8WPtnU@{q=NzX"
    "NJ#BFplmc1h-ZBsG^Ph}D&eu{YB6KC@5`m`9b79UZ_-|D<e9q7lEtyMvz0i_%Eh1s<S{1tRg98_Nm$}$LKG1Ho0K"
    "ZaZ{EQ$q_Ow?Z@)Xgi{(N|(rfznswgS1wRWwK>vAZb5Wx%Tc%qW8zW21=)2F`gE9vR3kt%A<SkZa*X(%hlT9v81w"
    "E0ZgC;;~9vq_hI_Qp!0Zu*|skd3H1GM^o3txk}13BW|_b#i@j48Jz~lDH7M4Vzl5iDD!R2p3{EMdsPnh<if3@m|b"
    ">zSZ-b)1oYw0tslVWwWk)Rb68M#SN=wND>*euVD*ygI&#_DIWcvfLHr)Jlgg7j+;to8T$aYL2553DNto#;5?klMS"
    "W(oNQyMTEnKy=tY(UYWKfeGU8$dVej^uw$9ow~myK6t%j!#pMbHuq{LR^sqK33fT;8Y$o4tZ2nYiby1XjB6L}s-A"
    "4S2()D!N;FS0=2Pf=(}{%;Or-e9*_#BgG^Q(+gn&gYyQ_PbDx8FX>><dwv6K8SqtHA_;J?KAd@G-F*<hX+I-iKKU"
    "J=)U32=?k+jGsUmL}s6)XZq>(D_klQjnjaFh3X(vbo5(3Z9Ya|xpXCkW<?mr5GA)cQ$)P)vd?KX;kYj%<uQ40d8x"
    "4)d(Q|C*s4e@j+R)oHL0v>=-CirHQ3{Q34qbUu}w-5}GhN1#yD~Oq32b;${Y%=;*G>O(plAwpIyxInVcGr%2#EV_"
    "Jo4@&RSfA%E0>bKue-Yg;@|k{1+Bnia&Z!=vfpy6IxMN8kTW+FR=Vs6G&u$cOV1pLn5j6$$cJ|NE-v|DFbk;>af4"
    "4z__ze2sOrEL~GHgRM(3R3}d?qAZQ*xfeS0^kM(}BJ<6!mVKs4a!vrdH6D#w^kfcLCQ+MLO?6k-5AF8AsL#aGrOF"
    "U$X)f7h57S-ti?KtiqBnhp5{12{awP;b1Yg+oF2+Te0KGFut(niBj(h3Q^+9=_2Mx{So!&YUp*(5okE3&$0R&$?B"
    "8@73(n5D6+%`k=5kZC-OU~u1m(1A(J!E>&rDd8pRcym8n*1$KI&ZFJ^`0EE|eN%VzKeKU~Uov}a{1<8*M%28bKM@"
    "n$pV2&t&({}3Pd0@_0zxhZF1{{|7KJ;At}1$VuL!o1NExhb_@CetzE8j~OVUr)?r^xtM~{{h(&qe=Z#dFpIyw^R="
    "hE_R&*n+#`I25qrFWTIa-3{l!B3gM&QEsBLcoM>B8s}vI&C58Y))NqL~RKsjUD)kYTF!G#X9bV3XI4polzdye}e3"
    "@i*FMRo}=-V~n9<Rs{!EPxNrhG#rv&SAYisL0fFS1e~2+NX8*GTAq{L(?#yUG?wnZ8YECoXAo<GHNi2Auk*H`V$n"
    "=)PkFMklr3jbia5jLu3EVuXQ#qFDlMgQI3E0kKxHVhe(xXB$&_R*To3qg*D`m$Oof7g4kr>QNkF!<Lw*G<xl1{TC"
    "e8p`2PZ#b-7=8<ixI;W%87jZz7Rc`i7_It#4(5=~H3Rco=FQ3oYv=G_gP_R>VXrAR^ikUQdR%39M`hR2SeN%1jgy"
    "~;ZAKcc~?5*!5UI3v14KMaHj0isI`6)IQ1AZ^2qTKI87!12*dE#TQ0B`F~-7x_5j7$oO1&yid}Y|OHnlI4>LAZ)_"
    "xWmR69h+a}lk;HkWpO)EmZ6j>eVt`051y%|&W-;}_CZGkqJ384v!JSQ;{-p_SgPn(E9=Q*%4o>hnJP8HUbx;3AGa"
    "OG7ZWng=gCbshz+H(lyN11qc3aq^r0<`Vi>koxvNcR1+YGWqnw(UPxH=qpMxZGp3|AyPAw*ZD@QRU~l!lF2BaX=|"
    "fJ!nibC!%)85q(?IP9FY4!F5mQFIVTeV>)?J&v)E!^+dee7snReUhx&x~uHu`>`6_Q-0W#02431yRT|QdBYQ2Uki"
    "#3tw#{MSZEC}XIj$a81}!UR@K=we`)Lw{J6xlXZ6i|p0l%`d&R~|IJt1nKDT<<!y|O>1?pHR&5#EWGKFnU$zD}Xi"
    "_2Fa0risC+~#kG?y(_OI57Yv%qMi`eny~h^aYR1nISkD#l8XO$CfzUSkLT2Wj~J6onXYFF?tqkf#<&PF5)5egkh9"
    "$PEp!<yZfI*p&ze?Z};CG9UttU4Bzb^4}qYZ92^~j?DGeB?BH;?cXW7);Ea1e?jG;%o$eo>U~=XQZX<e=ayz-#hg"
    "9&l#9(L)v??#_zQZ-DMUfk70te7M#e`?XGmDzcknJSEFnic6Q%Q_iq01()<vIt>EEE)_An{e>ghTTqhkcDwm}M*{"
    "!Tu~vuvuZt$ipy}libr}UI3C`&57uI$V(@Y0}HWB3=YDka4};zv#e%Z+^G2vTQwld7{Ft*5fEseI2chO?YNx<40m"
    "EV1KOOfXJjIV)%yzD969JFEQgKqPpf4%042U!UZ8F`l9U)6olmrx!4b#DgGRxDXmzcrw^V&|$wA0F=^xLwgidCH)"
    "Pb$SQudh6{hd<H)Ea{}Ok9H*((5%AQe>+8l}{e=8x*<T$3@e;7GPZ(oy(o2dP`_0lZU7-cR{?AWsYMxo4YCvbPPf"
    "9L(j|uv}72nOLa3a=)ei<mXKcMqk(gV_v4tz6zp?LuR7t(``9zFfIWDhXk2GrltbVOm#eEm{J61oQ6GsOx^6qB3V"
    "-A>A5aSSEFU~icZ?X0GDd4}zV<%+VrQr8#`_%;aeVBwQ-C-HcW#cw|H2b*ku72aeI6NT1FPb->ukfhfhm2?6S~vt"
    "H)Uny7nB{mijh1zPwa4F$&r>2vRNQ{IXXFFyI=(#P?jld0hNNJo(sFmCiGTCg>-_<j`55{Jd6zIoJD2dw1LQVg5s"
    "!TSyIVjN;FGm)tbWpGg>m#PjU%V)g&^irGsra{$APmp3|3#n^H(`G@J+5g0&)@`+K>SxT`X(vNK#<Ein!`>h;8Ai"
    "n_+&TX7Izf#GsZ+?_7e$cr)B!YHd8d6c^)UmoInHLgM@CEfT{?`^?BBSeZPc#baph^G%UQV1Mf0;Oi^l*tXC<wQ9"
    "&^|AxJD!(DnuTDo7BNI88$F5h@(jnf{AtP4~9pp;K--@7UWSx^SrWi{;tzd=%pDM4e1%dY0yb<<cAD2(WFg&%N@@"
    ")_hS3^@!%IzkJVIm_E_$X%O!6XvQfzU9WAlB~0*{af5nP0<k!FR`Q3=4ErJqHG5Q@35qoEl*KVKp7<!{Xv~yDTr*"
    "fOe)#M3atrkFtj_z~=PH_%Ut_EE-Cs-vVVz+?<}G#u;vz!nUFWrjA$&<guRH;Jc*$;CxC12^V=9S@BrR@gp<gW4d"
    "0Gyp=hJ50(Z^l~`w4c#W!VUQUbCL-ih!(bI-hVTn1Dh?3ZZYqb4ogtHUKC}E4Rf$-F^>v&Rm232ROJQ_ZoVR}}nA"
    "yL>h)TB`szCGKjKA>Tr>H+jD$gWu@QY&~gXIpn8*I0p3Zn?pw6xohQG0YH~ed&YNrgEKoWXz<&*#vmP*pnD^^0rU"
    "%#jLua>{a9n>k<?BP--oh?+6R%Db_|VzPqPPpPu?z^=z$8lTp+AqGIhb-xl4zwyZ{ooryq2I|$6kB~dra2xnL~u2"
    "f;OzRHmLIF9U)F|~7diHv9jG)TY#3%XRV&SXz>cT9w<2@R4!DzjDo3QShJD4uGXHQ!X_X9h4-Kr_-x5J0UFiQ@A_"
    "M6ibg4P-V^p0TblMhj2la$So}jO@Z#osJ`YqHNp+7>mG&iSLA^qNbn}1TZ~{^|Feb&6;5cJlyjKwK7zU>u7qw5KH"
    "Y04X#{UtdnA(M?uwvG-DM<A^C!=wLP27P29WoV&`jw-170?9gCWX2oBD3Lp2JZo@Fu`k^`-Rpe_)rb5vfAUzm_Ji"
    "ZbpvHI&?T0as9%X!hE9@`RZ-6G$A)rstdVz$yJAH5a|mry9rkGqaM$(f+LBWaBN<R(vFjZ#9tKBiLIU$H<K_Q`f5"
    "LW%!)<BbA%$^Nrp&W50kqaP}#32y#9q3Kc0{V)Nt9-$K^#7~4Ufj`zU*BU!*2_-L>S5Gbp*P8SDhn292J=ai|VyS"
    "2G-sD*(KWVm@p*tsJeFuW@#Zs4825NZq6Kxbdj@8B8O0r&3(vsM9&7LKYV6r{CO9WN@p*&YGng#+M|?MRoB291zM"
    "6w87R3dGxD6r^7AO}F(W0V4BXQ*^RTFA6A`|Dqz5ZF<p#cXQu$u|%V<M*>e!d_nxSf_JC?fD<)x=hJ>zV?*%O!re"
    "q1cW@x>%Px^rwGK9O-x!T!=_FRFxWv5s<QP-(9D|$ZDiC2?t&%LkrEX+QAcCSZGxV>qq~iG4BxT!62w*Yj;>K}zU"
    "vViMwAZ2I3&j-@%tx~MNW9koF5~PXQ4%HnG$EJVw#0=82{j`>!f0e9+?4~$gt9Kn=az5b*V1r0-J-6ndkrW7+y6*"
    "Q!dABlpwBXg675?J*$INDoHHqZ-gzW~6_<g>5ZDILB>sp_X{{~_xEBS}_aa>kq@EMZL(Oz!wb3HFqCVcNJmE!2Nn"
    "t9p)B2v5BeL!9XD$cy70C)SdD5j3^+nWHwFU9_>g>iq5n$N9*28uV#4ovGQ(U8N9;mZVYQeq1=Vp{XkCwWegz%^E"
    "<)`+u?DTU`HO&ffn!M_X5`rKOe9ebp)kRYHMO!sW1Eu*oN39}DUz!}SqO8iI$rdL~mDMcPR7Z}6IAWlY8q_offN}"
    "eblWJx}GV<|zh|s~%7Oh7~Mq3YsIyu>^(P%3#*lrRk`JEuPjy620=zJ<sBc>WQq!hbwjg8MGe-8T=oRgsm(;vjn%"
    "=_)<=TTG4?bD}PB*yCG(o~Q59KAohxRKE2t7<XSQuTKYrlB=q?7?zEuMN|&=cp;Z7ot~l*vqP6oLNx_0`6j=`Rka"
    "?PwROGLW%F1j8*Z(j3yklqnkuyP9105pwq_dPZYf-=A{w)xsMLSN6Eq538BPo<M*KPq{&Yl8Nv=Df#o4lnN3+!&Z"
    "uf>J%HRvC3Z`p*>`Ydo~N<*W3r1)2y3Wyk*CE0H;`--XDQ+U=t20!1~~D@?E$M?{E98W>cl+`BP$Kn*EzfPmU^vy"
    "jnoZjZF(zpz!*g&7hs~#St(C_x__PQbZuCH^9ISV#Gn(Y$s4LXtd<W<S$dRI!qb|Bi+2}})D>g7`|g!E{``t9WgQ"
    "QCK6%>n4MUuey+-N?H=nA7hy?;pP;Rxm?cwn)3$nEk2pR&Z;pwdi^G-};hz&@*vsS<~_NzM-P*W=1FG0TtpkG?0("
    "ddi9{s#>!R!95a?KTjFV4rN`h4T$TjwZ6CfzT?xe7N<}Hz7}`{xZ-=!JJ`h>fW0Jt|?AQ8kYnS^#v>XJhY6@L-6~"
    ")NPy7QYPA>=ehRf=0(vM!65o1j!q5GPYu86&vUnT5?a%i|J9T0n<BB$w-0pwr`OXg3e<aAb0mvf%AC@56-Ttqg?L"
    "Q8;&wu~_SjT2tNw~)WAN8ZL4fZH)NYix^OBv{J$U!zyi>o9IIJLeSbro55>L+$_Z`-nYd`*nYYl%A6L}F>r5P9#X"
    "^PD#YrUzJYHYf5tDv9SePvB{>p=d^povv0>N<+@|c2om6ihSS<Es?jEY6}t#WdpdGLIM-ncKfOODA&J}7#jC0aLS"
    "q+wHL$Aa8RI}(XsY?ye{z*5jX2DD65En@=ry7LbCO2Ayc9%UXKDm2?ct+V)505&vSfM{OKQ0Pv12&MYP~3Cn9VUs"
    "@AGN5gVVE==h`AESqN<N5?3%;xn-TQ-VSIjOiiB=^Er!Es!f!%WaUK*l92U<pd!g#>851>tGT*4B!H!Jk@SNn4cI"
    "^7BC@3hPbNg6*Q4u+qA-JQV8Chb#DH2{6=t+83&qa)1n~nH8jVdbrNh(2$AT7)w+`=y@4eDrG?7kf{$@W7ooxkAW"
    "Z8HMPX;SUd}9mL>|?D_KYSK2iJf8{F|@8dG_(Sce=gwp|h=jT<7K2-UBZBjS$|Jea$*4FyLW6g_##x9jnsQ3TXoa"
    "09hklWJ?=-3)iUZvTIPG2hhEvH#<81P@DI=q{T#UsWUK`65|$#T!Yxkf6QSQttB`3s<^z8tJx<IQ|~Xlye<WQ7el"
    "l4>ln4aqT(*0U9Zu>L8H&!fI4BRO`Hc#i#C-~Z+Y!y1n^hw83LTLWAZxg4kF~?Gjbmjo;Te$F;6Y`$u-h;bGpn3t"
    "=tjmA70*?509c0d((@-Io|9H^iW9Av|#Yg?3p9OQ~3PFzot9z|DN;De*gJ{b^=ybSmi+%YfVHZDtOY|5W0^AI7N_"
    "**!ROZWBNq8WNgW^t!ISrC6CH^^<upK7xK4FyPo3CG2^SK3C}CnlCs^#CX*XfqNsV3%2tP+YA;%i1*^lSlRvpG_X"
    "liUjILkV<Vo`OnO)%*m|kQzi0|YJoPl$Xfels0oIB_m&NH&Q?57oIfbbzstF#e7jYcqGmWE8tg!9mNewELOJ+Im4"
    "x6|{yUC!e2&I;)oja<ZEk$fHC3|D$?kkEEtg>AM$Wa$gh_8j*x==5j{BDsH$#rf@1BRyve4w(Zn;Tb8hTM=1*=;v"
    "=b%|fqsIg#w28B5sboW1-#$u5FngYVyYhSW#~sDV*4rK_7odR$_Y{2sdh3@wSz9GJDZO%1fRfg{t)ly=j+#IV**%"
    "w#149LT?sC$uEA{M*|IrN~x#hGye}qgWCXl-bt$>5gzbl_Nv+7x8brhMHJugx2hXt?4TuAKu(s;2qz;ue`g{J?))"
    "YLj_a8kU}z1HF_xMbiHJ)76*lh1|5{|5r1$l)0464^f6<rKOXZnCzL6mk?10k$EZl~^XOCYcHi%u%WOU5B^pQc`3"
    "FvRWlUZCeU2=odDQ6k7-6>4juFMC=TQI#^MuU3A_#8XyLIHr&)7Zs>H9YiMh3gRV?=s!u%h$JN=1k;6d**@O+n~k"
    "ySgY<Cc#w|7gnru3NpvN6Av3a__(i1j46dVvWaf1fv7>(NvH6?=?=EXN#mQB%`_!vM302Ij=sXLUp#CC7otUrJr?"
    "@1O|}rM&v3J0b8djfy$2pwAaVuYmWm3t7=v7dTx%9OckpuK?^zFqLsg^gxahhu#?p;jixgnj%=kujVxu$sg*W!Q*"
    "5^?d_G0U6r&4beDM@6;0n-@sA!WHnz`A#nnaB!C)lgZsDZtV#6;yW~w&#51%7GkeWgGH^KS#u4<p+cY`G}1#YSX8"
    "2o>P{yIiD;rM-H$LQfX&MA!K-N8ju&AqZ9gO&k@sm$zSsu`Rf3|H05uU^*poMy#ug50-|$5{v4!J4XE$30fh4?Ha"
    "cv4Jc?K1w>Cjj93xU={waK%<B-{6#Xf^cUZ$U$q}`HwO}7!^R&SlGmizFt%`5ZSzL=A}ANSwx4u9T1mK@^et{w7("
    "JRxl6{S_`14}We*Wf`kdT?T#++^-+4SD46qf(dOMnE?H`CYKS-&>t$q@9v!*{LHnC_)on7A31paw}ZnU=$`py-`m"
    "@NcPfqQ554*Qo1?wI%02m8Z@u0<cq4cCPrb3Xd$_m%M(>$lzOnbDx{eO<)$#7@Q+kp9(l(FxcVGRDF4JG8?<Xe*K"
    "OAb`<%fCar^7=Xwft?H`}tu17ikawe9&a<{kVJh!~V(ec>h0s+CR~!8dpDe?(F338s}$pj`m;<%-8wv2aI{|=<T~"
    "V`{q6Ub?<Vv48w65FuVKz*?Sl6HjeCE^sjWJyOsd224p+uW=-IWJE19P%#2JbBxRjhk}l9Bnq)^HfI$PII12y!t^"
    "KOn_2>pj(VpatGg%e^bXQkb?b`4C?c`Jsd6f^{67Rp-JNQ{<L4V6nFOCk0fZ-4OrP*-$(){5+`FJr)oRNdW{e#zg"
    ")bfj4fr73_KmWXEw&)mtyDp3%Sji7ZuSdswr$_XYL<Ho=lh=F4$Fi>VvcMiz-uXJ<K|fKuwUs|nKGYq>TB3*wRj^"
    "W$+|6P&n_{{L=m9qc0ziXJ0r;EH+X^x*1ttZ{)9MW><`auCKz-QEWyG}iemr_}D*aNhGtr;l<WiNxB~u4$Ecu6A1"
    "6O~ups9PsJZB|=Qq@#!2YH6{-sH7FGhEemxd1C*lwQH~p{$A9Ez!j7ca#E23vdaolei)X`?^4f!(uKqIlxM(2m!N"
    "|u$=9xEN1JgTW))~=3dFRsQ`i@fujr|#*=XpukdGGt!EnM81PAqqCCdk((uRh7I9eF&2LP*p^AhAIT?r2e4k^@H}"
    "i5ePE;eg!j6FlT;c?n`4YIt(Z$=v>SG(#ZRQnXPXU(F0Fje`0!S5*l+HB-u`mO@_=MnTbPd<c2my)>0p2dN66256"
    "9I`h78A5VUF|G9E=q=fk+|bXGyCrbm73dd5^@X$YvNG=o=L7D>D3~;jBoL%}U0o9Cah)G0ecZ}qxlW6L+^!d&2)z"
    "T)C7}Alb!jz*8Z`gG)G<WdfT>~=KLpyU&NkGAh$4er&<%;cW+gEtZ-(i+c$i$7IvzNfdTDlOWy|OdAyO@iK*K3&h"
    "H6fu44HL5FN2~@D1PISrrSxF53o-{_4#6{={L1|iMShQc^C`ha^VWs94#yIN&ILA7(*7cdM>5;l;s%G9|p+Ky;C7"
    "-!fu3rya5co6uo;dF92lTxjwII|0|2el{$jxqb+M=op>1NK2z8>-8~q0>ukf7fr6xJ3R&v$RHHSAb8Y06;Xaa}CJ"
    "YMv>I*<H98<7c)nk(n(_L!bpBs;)RFthRP%mzR#Y+@e8Z~~o9<Ry^^`)D<KOQY}pd{_K<gBY~pL03)m)I;(wXA$("
    "f6ep-iF|tN-L#zD3XP;r62FSppezzabv;H2(0DUlQ+{3o9<levsjHu6Z-CeTVUS%@QS61e>xwxn-c-<ts6iOc*O@"
    "Z{2XZe%afV);C*mm<=3>Ugs<)ZbETn|4+jp_T?(=C`OvmQ$H6RHRLzT)JTMqq{%MH-FCgeGg=QJTwCqCImw(G?;l"
    "x-Py*z1`~P;GeGd-K0~-#x2kIB@b2(lseK1=eZ<#)1feRXHzjsDl@@rY?c+K~s?H;v-k($XGNI_zZ9WDn>UHYvd*"
    "tN0W3T<SJTK%FCDucEUwzl+gJl`7+1^OzaSL{Tw6eh{`P6iIFUUDHOA<WbLvl{UgO(e_CA%T)qdOk7WS?x172!Q!"
    "nE&R8qyMVVcdi+wx8$hM#EDUbxsC_F>N2t|%#IfEbT$<cXM!(o8UuYpUq1o#x^M<Rs4~I12ntSXJJNO;DA_!h{N6"
    "2Ah1|y3!qAzM!B+1iswj=74g`56q!hfIx<@`JCt+{p>A~s8zzRu_+EOh}lsfwEx^-XHw`f&ajtoWSsuMsY}?F5^0"
    "r55nV8_Zd5$5Refk88LJ^k?$Y&jF`q^PsQg^5#}Ls|JyMF*x`KplVD||%Atb^yr6g1vKcAELG5)3AU{_HKf{bx$9"
    "`FnksHS16X$Y)j3-MLR2zj|aLf%8gE*!^6<bh3Fo|n~f-Awk_%m(Pe>ZsFd+K(u8jX-;ADXalmifxFomf=j%|67Z"
    "0j7{IoTT*jJm<6R?AXhUR*g#!Qy)V55222eZ2g_6795N=GOv}9l*_5`X`~h?s*j*_dBUGH^D`_@j6%b}=!Brb-C*"
    "x?tko2y!u}YIL=ozW+l@7)Smy#;SfTeS}ST%EAxzOu=lekX!Nqh<&G5(^g`euqe!d~tihXce@!;m3VNT>6eGgjCk"
    "VM%W<JA;A{LEJ&GlWbXm`E+ciSKc6>tp&Qs^D$J@q}e3>8IXo?2zIHEesM-vmSmgZT?DTO#lN;aIQw^a0D(y2*Ri"
    "&rLrF#4!o+=|{u<q?ELPYL-7@htwWO-eDDaWQ6`-4+gL(N-t1i;Co8*1t-fPww|599DuFA`R&+#EaZxAun9j!H7r"
    "WW}=$L@EaH(*5woGKtDj#T_7;B+C356gdJU|Y)iQG`Bjy_KA6>f|M!4f46kw<Kt(fSgd8GH4%fRUt?^L5vzB(w&c"
    "uH+$vqacao@Ro*fSNP~M`==06|c7ZIF>RY#OeYDF-T9A3Atk-3^!oGX%$9SvtVY?6UBrNGjGVW>~%c)M$Nb;pEz?"
    "i45wolmJ)96k!M`#D#I#7t>4ryd91~x9XD@V)SZ6_-s38}zYQt?_EA+lMPs=a97WjIEU*mk7?>A~WnVY|C*Z~CdK"
    "5u6v<H2WF~@Ci62S|E4W%gNfa{T8tI7lx^Tzp*AYn&TxW{p^L28>?%`7+F{6?8PS#c1D3`4O!gTa?n83aAmzHAQn"
    "^!@uz_aq&qTu9u5-UQj7Hd$^X|8X*T_@v)KYV)BYF5SG%q(DgA1V#vPyTvFN_F4;J-uH?SzXzW2OaERy@)hd97Zn"
    "(FI~(VLG})n(P98=MkI@8%o=!^RH69By_mLkTSKXb8p3HwBh6;Gks8ps50U_K$Ak$itU?|Lynx();!gz3>0E2h0A"
    "iy(Qe%tLk1|Leuqi*@L@#wK)_kq~07m8sfb?zrq|d;Jcu#K;a=GLlD2qVYxwtH7W_<2cuJ&=Ga@{3umjv4MLZrr~"
    "2rWOe}#a1y^sc$9EV)1psI4Jjw6ZW7Y>3L5xF8CD&n?losfCvbZ70!1qt+5CEGY3=#AAC#c6zxBJ5AV6)Bj+=;84"
    "ic82zeo?OQb?vGmv_16~*+JdFw;o44ZmYhLItw?&M<WhcPPAQ!yE-mEZ5eO7E5QuFEz=T&T9<!qJhp8d-lkQDWr@"
    "vmYD&mhtS*Z==j-}4wCV>#3yU?vv0PM_d3N$2uK)mzQ2OY^T*9z~Rb}XrVXW&%DWI*djF~pGA63GJp}LtosW`Ja7"
    "?2hQ=X|%U59m)ZtIp)t*REW*IqM@VrbYH;YkE;m(pNdPMc}NBhPzp~$n4?MG2>-GQ%d~#abW(bv`a2N=3K;vYYG2"
    "c_6hND=s+~yAi6HEGinNBrq!mKtQF38F-Q(BpR689z%g6&4I{CqQ^x6tiiJkm*gj_rvi5o@3EJ|zQAN=!%W5gQTl"
    "o_BmYMRHFm2bjw#z48n7B!PNj<mQ@CND(@k{Vg8e6xWe6d1dw7r=F%pJ<_w={LYq^w}&oDQ7J7qBJ#x8=6c7Z^(5"
    "Z^e_!A34~-pE&I7f6{j=DbL@qhsu5eJ3~G!W^x}%YW^T$Jo)GcsQFv%Y-rga#xM#fZ2Adf><jO8<Xa=B@<>2k7q?"
    "?@5ZiW{XVr8J@z`;}?)2P?jgD=Ut!;AbDzf_sP}1^c!###di(7Z{&K94ckQKij<Hud@gMgL04L&v8WrJ$L3cfu${"
    "`1M}z5UVn_0g+?{e#g-u1x7-b$c=|mi5&__I+AiT#ThfkX<Q?N!e#e>$u{g97Dff`OHJ<`#sctE%pY9@>L3(lRN}"
    "jktzhGA{=uG)UTL-`rIU~6PC)6SQulAq@lX)IpVDI>h;%)>zS^XVH`(jmdEVA<KNo1_Zl?~zVL+n^enE~V*xe{%O"
    "}3z4}H)@$UZ|fWBdyN!>yp}gzwXh8_b87^r>-EC&+`v+~Y5INq|Uz7pFLvaANQ%WD~t>lT31qLCM8_>*f;X;JF9J"
    "_tasapzci`G4#1<a1`j$L7oyd5_OQ$V-Z%3xPH7fXY}|YdwuX?0RN%XC~U0-tx%i?n}(t+Fs!@9KELb~65D{m(CQ"
    "7eT_J4Q8MVCeAR3=4CpU{93W9<)4XGn%G+jh&gML;}X1+3S(E}8ASh;YqAf;_8wu^ZUdV_v;QkD{GG)m+%*RxAHw"
    "e({&Ku@3=(o4x%hYa~|csv1wXff50deypf4$0BMiA)@0%4jgB@%rPmS{b8zg&F*W1&gFOUVPLjB!aIj>RYGrY^od"
    "FMNXW{DI^I2ncw%CL(7n)2Ykhd)+ykIXu8P|G*kl3YP?&o5#S)0UkzvJW43sK$Eg#hn0veft|!*f$F7-mCsh%{qb"
    "@g4Wk8oh4U2Fk`HNDcp`5P+e-7V2q3XpDtx;>S9U40i*&~otRm=ck0v?6<C=0?sS+D5T?_$VWGlZ^J#%T#$u=au?"
    "aiS*W*9Xf*Qmv%8axzh$b@xuHee#<*;&JiXG&E-3qmAIiddo4*7vyY%+@HGXX4*yY)-+u32C)VqQsQd@cwXXB<Mm"
    "YU|4!ECgOAG$0h)2+Lg#+aEmuiZZCl|QAr93XI{IdBvVnf)F6_es=5$2;iW9Aub>t8AJ1wsNcz(0+Dy#IF%blvH%"
    "6zC)AS$``D+7SGzl*A$<+Kg5cTkPge8#+jCADw%Tz^OxPJ661)D3c;eaL25_i5!ba?P*8H5%%WBUo*<h<2BZiHT}"
    "w^3Z(Ys~L^lO%_tgd27x0@;7`>fSH;$@3tN+9M^+ZKzEUtFU<f8E+1Ka87C#HA0vOU`pxB+sV@_$%o<vOO~gU#QN"
    "e-Rk0`FJ98a?Z)!e?mcTgq&d}x#C1C@3<B>TF9vK>m4gIYOQvLXIA_{Cedq4z1clGkL&|3>iaL5}y3{s{-c$7mC~"
    "FEu>Dm44>;5I;79`&<0(G@SbbPps3Ea@hks2=e!X&aIqMr-$|VMOezyv2)MULO$@rICCp^8C70gt!M>IRla6WeJU"
    ")nc@RrXX$w1%eL_NPqkISrG|)TYG|1b!ImQ?FDUdx3-5?$B_uO;u^r=X!8#MtbK1KKI{gS2gi-nt)dd#d1Vg-ii8"
    "abPhG;?wz4-j!NdP%!GY6zRfuHmC>zH}aJR$_!5^WGZ}YQ_|x;NS=+IQn^Inso+luOx6j_t~DfRhvk-9s*Gi@XYp"
    "JVG&&K==JDOt(ylGj#Zo3@Dfq&X*8NHcxNWu;qDrOiezhZb7kp!adPxlgM%2hQIUC$fcw^sMvRCWZF5!yN63seTc"
    "Se?dU3g-pu}rpxJ-2o09kMbyOT_@rL6O^_vT+Al{dSnK6kxL3TWn{!G^#L#TM_FItIl-i)-kH#x^)2t@nD;I%&yS"
    "H?#5rTO}7COD#hw9L4D+w?G30mdxXd%onbg7VbiZF;io|uW!n7uE`i?Q1|^E>|iITu3-PeSlM!OHmlCfvIB=)VJl"
    "~N6^%^8ub@_k%aC!$XvLW;vIhcS>IgJWA);|DJeQkrP!OL55lNo0p!4xAz|absPVwIwmh{Z5hfRj1$}VUTNrlPPS"
    "v3PUDY;ANS8s*5Gce{pK-gD;orrvgK)1yTT2JW-ai9BI7hX$E3`615n`9j!L>B{N_e{0!a`&pb1sncXV(f+xpzt@"
    "S&1zO=Y5<Z8l^Mqihwjf%&j7AC#Q)|oIz`CshN&;3wlQ4rGko{{N%?t+smAi&+>r8PwX}&gF=m4<3>qSC68N6`q%"
    "uSoOQDcaDkONp{pBgk8J*7%&*rQ!Or*2fdf3S+RMB*u(QvhhbU{+iBu>%C;^POaT#6@??=MtZ-#<AxJQ*FIW=F@_"
    "@#yudz5P*kaCmwIL{fK@nuT#AN^|dKzwEtwGdjsSf6O!Z|8De_9dkP>Y4EQe0X{p04O>-rx($<WQrKXOl(I(8yz~"
    "6SU!SI-NBe$&2#cIMp%O5NCf+|`e=72|GvY~TQB+hCc%cX5R1?F)N}H>Jb*2RjnPb%ni!?4gp>Z*TVx=)ao>F=sB"
    "Un>52_!Sob<_f{!)Tr53ISVgu1e2Y-Jn23;+lu3T_NiPf1K#SFeQa63b_L&Rb1}<Jo06TkzVBukvz4{!58rxeZs`"
    "k(~DqC+G*XRcO-H>DpR{lFUdKpBv<FHo5aR{|3n>#XSA=+nvMn@56fGZ;3)(!D}7+x^ZJGqLE|dc{wZnjl0ir}GU"
    "d&;&C2dSj7qzYi(v>3l5?@46RABI<++@dghB+qrU7dttn>uBr%<k8VVHl+EjUS8hQ6b|nK8*lt;tneRISAH8<b*D"
    "uEN1Gs*s$iFbwyb%#}7%t=R{U`YKP2&!N+4lJ5oFs4iJOCAF3UTglK(sIN2|_z)0Qc2ywpP#+<qugOM>4|(EPH7X"
    "*Oo)h0EM6f36)E(RPltuy`CckYy*|aL5!y##cr<uSQyK=~a+P2u9g$wr#!V`2qLlMgE`8lzf$gM$oaISnaQAUG+V"
    "My$dVJ}37PUV}cTka7J0Ld(Kqdq*qKtnGCut58LZj(UzXubX;s7tv131ta?$6{ap3P0R!r7FAGOi?QTatqz)!_M2"
    "{lkp=k5s%~*$AF8ro-Kb1wx2Ij=o22L-Hurtr@|E7piY8<Q`Xnf>ad{+?R*LqF&!N4zk2fm&}GEnr-QUI(h*Bqjd"
    "YM+gVd-Hfs4&w#`wkJ60MbzHafQzM|hiAS*H(r0#%vlPL}CLiB(Nzc_Bu5yYnU?RrTIut1DEuZExrIPXs#Ot&Wsf"
    "5%31&6iF49=a#ytRuKHmE;q)Qn6H7G@ish*JSUPSeF;Mv6A_r78|jg_AXAa=JaDAd>OSDhtOJ4G#B(9Qa+cd&6V("
    "M@iDLnDDx5AKYU94)Y*0QSUHlpf=o-V>_NdfYxrcg>BpYySn>{>Oto6XS_Up<V3R|qAJ(j3ev_`hiH>(w)E-KK-?"
    "H0|GlhLcu{;5=)zdSzr8HG>w_NUSDD8m-fe*{i1GNs$?UzBF4%)55F?^16;LxqNo0)I8PiIn)@8P|w~2AoOI&)BV"
    "q$3b$8en`NTseAN*RgLcj=5U{gpk(8iikO%mG0>r*S8y5MEPyW+dYtLP==O$b&0v2zV9aCBm5?Fu*<|N4;N(KVha"
    "RXlvq5hzKrc?o`wJ=Pi_MI^lEBu^ujoWx6e|I;B8H$w!h*`K%eS&Mo9AMf@UvWw7CsBS<AMA0DP*AeHnI?tfZO^S"
    "4u#&8-V<#IU2%8M!*~JE2SDubNS?x-aLhAt&O;?(`uH3{FLd-ONIzPXP*~<-%Tlk3Dc3C3nL?sX+u`Sk35Kb6<&>"
    "JCaOgbSBUp>J9Y~M4WnXAB(l2NeUAUJRReLpD{I=lMiReM*4Yg0r32&1cxyX7Tm9=u2CT-ZF9`b{_iaO+EYTeR>S"
    "%w;Mr`c&-)R(c-n3TOCbEbyY;tiIU1Lnc|Q|6q8>2Wi;sS^`HCu@+U-e(BIe@t)d?vuwEJBpx*i?7V}_-mMc`VCZ"
    "<J(;*an!4a?4I(vE%8eXMC6R|S_PxxKN@MTy0{r-GY$k2f`<pfXtrpbN?9`T&qux%TQWrY40aA{P)>j2LjkW5u9x"
    "YVULE<`UvAwA<5(mZi4)L+yVAd(Ez167m*bcq1Gq(c6P7_N~(S4Y?m)=#+iDP(niz%$iR@m>EwBB54YNp}}AP!hb"
    "b#<Ig)v1vYuyT}Q7JY{U&bnozBiyU1%%k$jJgAe5rL>XImYb69>bc-qiKWM}A{(<h7}-1WO2PG;WnpP-s&5K!t}C"
    "xpU+QY&NWPgN8$jgMg7H=AoA`1Tg>)r-h%fTio3Hi0i*E_{gzpN!j&Jn0j&Jk7jBir1QR*oBK#wkLLF2gV0^s|u="
    "S#UUDi9tgO%?WBfdw$G5d*>JlJnc>NWESd^ZsmAl@~_A6Q$~=nqz{7Rp-F~aEA>_ABwr(=>8RU@Z$<s#RxeqdS0$"
    "ke?549G>~A{6?INE7JFH1@%o8@0gO<r=7rHj&kH%iMP}5)O*Jnu-D6{n7MQBJzEhYbF6t{|@J@SbFq`UvP<sE$h7"
    "7DTTx@QhLd+hYfVm!+X~LB#^YXK@OmL-%Z9|O(xN_*#E{&#HBgh^4dRpB6u)JezPXJnJ=lgFXF!}rOA&(c`OpO#a"
    "#$x94nMGfClKIm+62w;0syv&1`YhDu3E96N9UKOnXt`(@<LD3`8J;Keel%0!VCL_TUX$#MS(<-k9!2_+49;hA4Gp"
    "4n15t2XhCPb<^{gzHop1Z!ejm#|VvkWk(HYRqaYY@;OP<{Lcdxp>Ar1JEBbR~Bm4^<lM+s`KO74zeo-eiN@7{gLF"
    "_ihCxt%060FK<ns@B#UA*odu)od{bh)R`8pq-?|K62DAYW8^kbJyd5$BbS%>5!789?x98I|1E<{jF03{>Jbu`knP"
    "UnB_Li<p%Q+T)LL+9Djqhb;TIN*(9VJm17`2Wm06JKCi$Q<ql{Qa}B6&4AAasoxZ0s_o_MO=y8Et-FL4GYdSGsN?"
    "zB*?IXV8Wo78qA%F23E+jVsd<LX!&!M7jeu*7fp>stC0<4^dXV%U9EPO-}jrrE>32<U>#U=ZXp{Kl!`sN$TSA@XB"
    "YK1`M{cpZOTL8F~dy4odjnuGI0SoCB3!>xbxpnM0<Dm`{FX=N2Fy+WbHi1>XV@n!%#-jvHU}46rmbAHhCu-s<0%Y"
    "JPqepW-4{ztGE~KQ#0VEHA>cHF?s@vF!<&<X}h=<8D6s=_X>ae?wkwjBNq#g?Q6N%Hyb^xjKQ#C~-mSpt3n038u&"
    "0I%Sa}5JJq2ijqZd5rvLv+rwn-#R_4AvLB9$(<wPr|_3q;GU6y-cj2xF*4zU23Lqi{>9DN`m(l*IW`eI|4lSOecL"
    "3Kxrn#X$}b@;+s{GY=!~kU7TO}nW{!xLI0y#b6I3_Bq@!O@~b#smS@prWhW$im0=qu)&lpk8$SGwm#rRti+-8mx5"
    "(cLFod|1KuF~Q%|x;v8OG$z;lY2r8No)fEAkxN$m-0qKe@+X!{}<pAqLqbts74m%yPa6sa!N#t~DoL=#-Y!kdd#y"
    "h(fGcEGL%0R<lnYl0V3`fRjQ{*~MyU7^)d@SsHbJjt<paGL^kLx`nX@s&i^NidL+6`dMQ(&mnrqEQ(3JfjHl7m_`"
    "iGrL}U5Osl^{8wt^OwgI=aS;Nvx!=aKn2fD<Hvm<9JK3h@f&uj_?lSpU<8iOH`PgNnjZ+Q#hyU7?!vw>eTMbs1I{"
    "1pM{7nwP1X1Bb`n#~-rbf>J!1F2S+Vr=-(!HZGq<s{(_Cb#z1%f=6RotCQ#q9I6R9cnYcp1tC1VUM#H%dKzc=U1!"
    "6e6gwF@YSh|aH#zQXT4!R^Ggf}6SNdSOBNTU?z{{5)#gE)xrAOkLH4rxOjYnDl@7oTgkjO=6EDgSpRH#Ij;qHKDZ"
    "HB~Xn3yp>+>*hZa6praBGF3A&9O|i<AMF2UjatB<`JVJHZ7B=uoq7l$Z6@%IF2l*`~I<t7mx|IqGr-krWx~zocL7"
    "U~U4rshwn;0+te|%DUUWJ^)W?Xbt+afZx!(7D39vPE%5yA{>=mXQmBs<<vDA(q18O26DfkIs`62;4>yUcwUiUR2i"
    "Vs@D@mv6C{AEat0$oHrs;vjXDoiP4dQU%4WV^Y|gJrg5spC04Z!>y5R;c*n<Nz;=IBLiRMqM=?_*tLk;%=q7i`cK"
    "*_?ONFvOICUvb;=A>t$UYTdC)0&AL9qx~EIrzBI%d#meQHx08bO~bO!%kY0u3}kvw*e{}6}AAEGMfjTR!9K+)vRZ"
    "Aam%cc^jrleQ1*?Wc>jd4;h@0}uPkc4hVdH>1IkysBmp_Of{7?9Wbq0+fCdzDs-HyS02v9#DMG-YR5eh~ey~X`1i"
    "S006mS7-z!gBEN*kkVvxaprD@=+(&|SdRrM+h~*6Rhjtt)KoZ3eC(0y#muXre6!0QLgolqpWX4S-%^-ZqZQg;7RM"
    ")6T^eLwU2+AWXyjMQoyL7ceSgT6q=_G;0OWSR<Dyp(mSgKzK6n4IS^TjLty|YbAxKkaqWEl@ZHxo|V**5D}5%(i("
    "UYrruk5hqK|uF%lbKDGlG_+!#pmH`Jow=BZt~5rdzty;;^q&zMy;oPKc;THSXu?-AgpxaqLZ%2fBxURiJ#z7j?6V"
    "&R?e{l%O@tbi0%Q@;toRJCtr&K4cHL4z{c>_|cFg0)IUs3|IH+ezmQ2^ZiPc@AqY+~<6tDJurh^bqotfrv><_AgS"
    "$0Zs?|NvZyd-V!(Y&1-PV9iBq<88R0zBW%(?;^}lfe-MS3@XXhiDbKkBXt^Tof#fSob=g1Q*joC)O8r3CH82%q_|"
    "=TI8vFsz>Y%oZEE$Nk!<lqL!C_>ZuB_0La6yRRR|t)1iE2&N7Bkyc%xEyBw(E_WY&aCOIL)!sDEp;LdLl4DJoZ*F"
    "xSpC2lu??<Tp4KHoTnq)f(!DX*(i!eW4J&quX~zT+y|an!=4y4D9CmJ9r%+47P78tUMr<ZBNG#!d#A}_A<h%JYSe"
    "U;&&H6P(=PD9Yw2`Rea?BXp6$t^A&N9BphO>c7NM6I{5WNv4;~=u=YrMvm-7~fx9|6LwH)SqFH3HYes7yJplyY@`"
    "2hfRys&PR9GU_G_2WXi`%u%>4+_0vPkPZ!w1b%nG653C{+z_>c?x~g0?G9QF7*T^#BLJanf;@me?B-R{U9}zEe)M"
    "hzdT>-Av{_T*v0C428dYXS6ihAGbe|9kSjx}SVxM4!s%-LMhCjTs-%0}4%%uIS^^wR#YjH$PlX`pPENWAA_t)_Js"
    "0nJZd>Iv6){DHOl|oZR@L*OGWVIz95Otbi<=T{tvTy&WSvr=m4ecc*aAbn@1KxlG`fL+D6Ll=yn$HI`zMAq{B={N"
    "E*PX=H*iS#$60l6@$z5iiNWqu^0Kw;Wozk6Tlq^%35*{hX}b4;mAvg$GKg2UQ+xCW<o!P^yTnF>8q!M=adgPJYX="
    "Kz2Mkg{OV5wqcpSf^zov2G6o7#jRBgLAJlRw(5}fuO25^}KgNQe@-+nXp>Q>q=4l*RdjIQ*pce{}r(`g|1sfhEDB"
    "|&3ZYMge{%+JGwoQ6!EZ56I@l`9VXOiPJ&LD9Tnc1c?{bvHUyFWXKdTZw2FieU{;OR<J4w+OeV*=vUn!Ke`AlGu9"
    "*Euure0C{PzP<x9!wNFv^LGSp*Vm8IL$mkl&Gpv|E>=1OjtMX8EOi}83<pG8g2yUk(C}q8Z=*{EoI^|ID5(&j4>b"
    "(v$W~;mXplT$M3W%wSezrsM8pYQeFTF=yrC<ByN5Dyi_W^00wH*eyDgL|m;_}ujM;5HV{2XZMuyIQYjd*y842;dT"
    "wsxk}w1C~_BegP}ST9WII%7Tj$I+h-4zq)we;&O!*gGA?Vd1AtFc7`~=XG5HXe^mgzI9<n+$;fAVZP2NAbPf0hrq"
    "V{M(5&={RkRfL2E?LJn1}4o`SHMkx%&q(=QIZVg?Bnj_Mi5cUnSh$%c$DglJ*&>wsJ{!G(s#oK<~ZEoa8`?e?=oe"
    "i`&bM$<Zm0#+e8XcX#p;}$rG01Zd$hpz$d#kD`JDs)<4H41g921gq_%lSE)Cgml9Xm>-dJc{T})D%yS#Kg-%h;)V"
    "!Is|SrwPf$!ci}rZDsMg)o7t?dJ%1H7s}kL;AvrM6Li`L+J4izs=q?2=UKAPknvdm|@Y(;gx`PK%1Hpy@6%LA-ot"
    "I)mL2)+kbG1f^(oi5WQf*lh3<R2t<NG2kH5W`Zl(ev4OxwKyCy^8lO+5{xVFu)s!D`vjB)!qV)aQ~Y*gwMqrkCY<"
    "*LoRBPVDs-gx-xX#jp2He`>h>7^CV4xMZ^$(~pwgg~L!Swvq6*q8g#9ag>U`X1rwcqfYDXrf&Akk4{QFxhukv!zA"
    "EtS5tf90>TUP<)B~R10p3gx1<PVd8ij6Pu-o1Rjf+KH*@dB2&*@0*VA18&}g>dc-~0K3MT2dpleRh8NIE()m@$_b"
    "v2TodWTG@Dk~RP<=)|6PJe=$>XK;L1s279)}t$Xdaz=Ci_Wv2geU9S!b--lmpa8RZ?BDP`p7V=dMs?vi>@^;7+`g"
    "RJgw)Lic!>1rCwl2x<rl%Fj>NzJT}dG3KzXz)8ODsBk+0`b%dls!1#9iRj-565|Cu{g1Hy3w5=g*OLz>367!_D2o"
    "pJ;3ep;<bJQ85B|!t3saLA5o<vQH#J}k6Kt{db%<|y0p|NdK_m~MO8x~y;)L##GAJP7y|1yO|h4ow33u_GvYbfOe"
    "1N5rw<O$`9csf`Cs?sgX5#N;K=*f$NmoG=hqr+2OsU=&l$HGKO+DlR-P?6_XRVus871th0eW2+bDt_J`m`B?bJf@"
    "zq&b>Guf8CpzWto9nZ?>tcPXMxgSuFKn6Ubb7&Y<?sEr!>>7<o|kWE3OpdJNz^on(|X<K;#?#g^)bgoWWcjtOv9M"
    "#b$DuzUN!){FM87Nj`nE<UVG!vS-Y4sjT_#69%0O>LnvLd9Vk7Mw6@zTNlUv90CM1X5s13NvBwP_>%i9`KciDTet"
    "7+Sov<^a7iNKgpd+Au+XQHgX;SojDvdIKj}9&T1)iv4u?BLS4w{h`NgNm9ZBv9z3fquvVl?MXkU&YJ|O4WD94)E1"
    ">2H(0f&S71fG?%dKTWT^S(b65xqRb)e|#42#gq|1|rBdH1xD&9^baa5QfJ`e0@ljtASShtOP+86u>7nsWe2c$ahN"
    "ZjiAo0WKbnX9&37tS(><XxVcn(IjfKGsz|Y0bLBx3J6n0wPD5RcbIGd{Z}}AgLy&zY_$;q+~xwQ1&A(w<^7|dU%w"
    "gwwgBeTMW%;D>WSZDJ|`>%Y-2bSLCtTdFMWoU@lx`SxgMw^Z>0R295z%aV+^3&PpT2gRA4HIyOy-;Iop_vj~~kty"
    "USVZR1v8CL_`58H&OyPh!M9>N|l6cF?`tgk=7B81o>HC8YW>Zpnjtd5R56lpR$XjDNI>&K#^(D>q^t?j<f~*D$|f"
    "-^Us(vpTW6%28;X51B-1Q6zB_~dnmgadVd&>zo^d3+U2cV3RQ5AEkiQtI-n3to#v%*a0%?`n6?b%pmiHp*)g`}ZL"
    "Lf0?K#N4Jj=6ZeQ1egOliP|?8F4r?Y8WyGNm>|F5{3YlJ|BSYWJhgS5f;a$QQvsl}4^h|0CAG0-GR<KvTP93?jP;"
    "D%XPCYAFqq)%=jrqlhE<4p45KGzf4XI5WzaX5}34fui{_H;tGn@!AV&BL7uf6<S3ZjcEqZM9xycF>Sfc5<6W6k#c"
    "6Evohe!Hi?V8f$OW)VsnY@?1e`Ud`*D$R{Lt=SMzc-%(<AA5+cs1>lT2gqW>lw3zRxK^*lFxQ?o9J;-a*DRPGGl-"
    "Yg-<g<n1#4f6qjyVsb?mkAEklzro7Qbr$KZ}5Q`E?=bW@Vu;$)+v;r^n>n)dHAy#WSW<#s}q}7!a5wU!HR0q7PO8"
    "K3AWgNYBicLN!Kavk=6o)4yj2!RGz1-TC$NU><x1y(FDbl)elEGMX6j|h@%0O)j9PlK1W;}F>E2t(ryD5QRu8*aY"
    "nk{O+9E=F;y#qOc*zBkWCz0c@7Na`6N1)1X2#pcYX1s%MLy_vm4Y6)~f<K=tRqeepYJ41?z|aX7IkWGF(t#eu1`Y"
    "c+e+Kol4J19LDa#epQcoHjulASVX{{=#A}V;c!UWg1u8<;(JLhMzI#$;?)diPrm*lA_jUo>Ne-lh<w%0-WIculDM"
    "T_D%&h!89ERTNGMC@)n*&W0_bKFD+wmH_C|jxdql8PZM48PlN`M%>5=a3z%4J?z!g<I$IJ_M`K)vox};xy)Ws8uG"
    "~YxN3^4B<zh~e&zADVi1;_#y7nBweWp+vRwJbU4e;wEJGW*662zP?(rjupt07E5x8<Jqo%+h@%a4Zqi4EvQ>aDSO"
    "wpY73b!?X2L$bBudjC5yr|CW;J-Sp$PPA`U&g!LRy1C@ONf31C3mjeTqW3=nY?kG8ViOUTAD_3q5b=F!`Mn~g9Zg"
    "}V+Bn$%*RYwB8RYBe&`g5XgNItezTFv7j2j!xW0;g0s?VwS}H;>mQ{!bfpchJyVlx|LdF0>s)oHIW4!eQFRtNmk9"
    "35YbRKR1PWZKxW0u!jHs$-(@SA&UwMW3nr#0|XWykSMUzYb{@!C(Nn>7A;zCr<CtqYy(TP$~vG<irZ}iQW7k!-x#"
    "sNYC7uEfm1v?J~}!@3dKAJDi8=2Tf7^bfComOg8ChsIE`AG68Jsyd6I9FFFhw&j^3QUesh}b|8(%`1$eQj4g!=Ma"
    "Cd%#F5++6wPQK$sL*X}V-i*waPn+tT$hj@UDellG`l50sDc-EL516WPX>cFMW`QWQ>c+afzu^#^0KTE1cK9;8HQ-"
    "X^%9$gN*p7#Lm4B5Vv7Tx2eDd3CC%_Ao`;^15KHPuFS`xl&{+Y#UTI}&ITW?rF^|a|r%H-t%ecV#x|(~hYhK9ms-"
    ")`;>JrKSs5^~d1u<D<;N{?6!u2-$t1`M4<PVbS2rmhSI(T)7e`Bqw?P)zBd|s^gtz#FItO7@$tQ83OGuSt-wrx|a"
    "vL}!*5j9vC-!f&?6<m5_y`MK&1iG$j$kC5XfMbpwk%7KFf~0Mqa4f!Cle$w9cQAm$@j`pZu3w|!qYr5~8p0F=X<4"
    "GzR;-2FdMlq{heH80kU4uC*#h4Z^T#fiZE)?S9~jIQ(aF|J#rZxNYm4HXs>{f3i#ar`FTFa(!qBF{#2zEwAf$y6l"
    "^Hj{8bmB3-bmlbk}{QJEO@_IY^cu@wA#r!``<m^|Gvw`@F}C5nJC8BE~?M*<0Gatw=6Fr*U{c9%>kjjqOh-r6-rQ"
    "Gjhc*8)o`<-O`NFyLla^xC@(f^;rU9PuNQ&65OYH)%cmm8VvgY-Dj60=wg%+vwNlzJ0uvc?7sc7CB59_$7?9iryr"
    "Q^;giEX$4BUsQb?IVIvv_vv%{?a~ZQ_=W^+%E$S(VRWKz*~I66T$tzdHGW>0oNrI~tD^%a|N6BT~%mMR>ANcE(&5"
    "9n7-2uC|!@hBEZLTr56^+go-|SYHJev00sAIbl!IE)rpkHKt(HN~8fJ7Ai-dSG`$%!tyRG5|iy}k{%0j{W)3DkNS"
    "cwH;|d(#Ded{?O|`!*fPWWw%LI9w##@rxSM2@;TI3K$ap+F%uTkFRaRQ9sGao|&$4PBF}Ar9mess1xUl^&Iy4cbO"
    "|%4PKu!iqMzfA#m5Sy8AEs`!4A^E6TC;ENQ4y+a%_BCMgQ(~Fy7RAN;YD`c<63XD6l3So*~xxmp>2pt3GLB$UDEh"
    "{4TWtp-y0k&QuChjO>$37$dN#$$?O^A34p_2%k`t<7YB!X$A1Y&dwK9`gr3*PjN$ThJH<{Y2h8cOV=t{75UMn@qq"
    "rHfe1e+l`f6hohyLn%y((jmuS?$Ib4V<3a=2LUy^!HXP_zb^ShT?lGpJbt^R$EmIkzoCsf8P(pJO$qg4PkaZ3Zhp"
    "FU@1Xds3`pO#wzif3s7Et2u)Kg9jgSHE=@~=jWRhEFj{%BL|N}mMfd~cY*+Un38!s3Z`?|{f9`J+9a+1@^&yF4W3"
    "d#plsgdzM%^N6s3tUg5IjQi5HGL8^HU5{KoCWS6U$kSH|z|;AV<jwRzErfN1oFv~-AKr#BQOlwk%n#C3|xIjETDl"
    ">Q#+Y?S%LqA^#<crN1|8egFEbBT}?bF!oG&Uif0XA!Vz1{E&W&@luttIjL1Np(g(2tq6mH~Q~+WJlSu&1qSmuc|Z"
    "LM@U~gKgZ4JXMaLOJ;93*?vqTY&^FYhnK?}c%Btc<ol)QdUx}4?*UocReJs&{0n@AlkYWhniux}DVIYoDq=Ikeh^"
    "trBR{{|Wc}2ylLJJ;1_{$Z1G^7W!CJs@=QAC-Du~q=3X;oe?2=oBwz&!)Gn1ohJl2qy<VwdE4!0K(pvGCv!0osJs"
    "c)*r1zo5F&W0EVq<inuS;u_xVvLu@?Pz8+=Vf8x7gzI!|^|;gbK<WL}bv@OzHRi@`%3mZBB;hrb<`^i%=9}v-Ds1"
    "g)jAOjW+c8D(S~3qnJ6vvN=;~M_de-a~WLZm{-^?g0M4d1XVo$!1oEt{awLyPerRMQS)&S;BD4^N|&{h|+rzeZ1R"
    "p%N$t3Y@CDsK&GAYr0S%94bko+|DlLjEMhr_k1EQ=akpU!##7`v;U3+9_pUuY*DFW4oKe>d<Srg9;W_D(Kwzuz!G"
    "5NbcN4@6F-i!Qr2p5_<aE?b;~E#yZn4jKy&kHyh&|+yS=*^9sfc!;HnnDnl1hKc;5fw%S=vRh`lV6XH8!03<`^x6"
    "TD*O>m>4vso6aP<&B#N$TzjlZxIRoc?t5<`e{-VGo`sOngb~B#KRSs1_~7p9lbHXl8Ys5=%gA?4s9NDf--ErxL8t"
    "l+e<60^%597)B0qlR)lrFE>yijA)4V0uM*@GH9n#ps%v<j_f`;OQxUTyo#CG((9@QU#B6bNe?Vo^DQhWddJNME+A"
    "ze7*stZ{4LU?qw$chK?a!0i?GRwR9SUft`*#fw0e(ys)Qfuo2#&XHNlh3%bV^Gq<Q*miuq9FZokOtn?fjo-X%L*E"
    "sE((n%g5S0thFp1wsaaVxG!lAUq|`=k*l<^10f)N+Zn{t1|(8!(*JxY$YNU_c~}>cx-dRYRQx5L<5+)sF1DKlS2U"
    "l!JH^4e7AKgR)B!R1AHwl-YU$FK*xjhe8aj?YNlbe9>T(;I<@$CUSs5e%)Jh35ad;5)kJil0ZVqzpAc#3JGmDcoC"
    "uNcGWIC|sGdc^TO+GIXIQ8G?@+hc{~qQarI9?%P<*Y*OGA{1p<|YwC_VCQv&!sTiBUwSa~hlVoIC3)C1H9PVgWX#"
    "AzHc4u(2?U2`%14pad7d7#&3yNf)`wiCDSFFD|s_w%5u8Rdq#E%arETqIl_%sG%5~+sJ)<o7vzX0#jodePYh=W~g"
    "dfe@Egyr7azlEca!htR^lsOT@=kT+D*;CL=N8GX;IeF!4fe@Fa)TU!j$f1f-VcD15H2iv+yXQ{KdsA4ezfLzy8+1"
    "+b<$?oA9U!HDLB{0v7vl=5bJP`<sOIxJYy;wJJKBsAK7x5-g0%a{W9(a#nmB;_S<*NjkBMXC`Z_WxlZRW8%)$I;8"
    "9V*&<cMNEOU=j0^ibf1Qd)fI281=4RXb>na<m+eF+Upa!h1aRl6x~@J6_W>*4q_j-K_R8^c9T){v!2vZ(h>VJiye"
    "1$NxJ-s;jKNivq^aLx(KK*lP6gR}b#i12k~$h!E(E2w`IsS|MXD*TvzKvQ>i{oP<)q+}O}r8?v{$8tKu`yU%SVMz"
    "@!|>k3=26Z&AR|!RULS3icfsWH6lB^oP#*KoRdvzY0rorIEWm<!Xo98uE0BLW(ZFtr{u>ttR&c~I6bMh^dL~tIHr"
    "<ThUY`cW<7x^X$_lV%YmN%`08l?&*tBkdk3#ZFUT>54^31rCF1|0q@CS1<=~67M!~^_!<1{Xs8e|u6)g&eQ&pWE!"
    "=bYaQ5_gi^O1gnVoRaGE)p?mb&xF&4S<YRgvE>4y$EDeP4F9Y_B&hrenS)qvQf1!jIVhEqjPs84|0W%t}DBXI7Y&"
    "CW2^9jUVGb$@^W43*JsiiCrGQ?)JMx2wPEi^=AEfB{Mx7_=@N5>2J>19N;PGO2W1|-;>Kr5Oh|NR8Q!@8B-kI#`#"
    "GSy23$u>bLAM=STJON<S$O<#j?IytlMr*!xuiyP7AZ3TCljf5A@cs-_ge3NQr9%Z-l$cT0<mbjveM=IA+dj1O7BM"
    "Om~gaJT)FUVuSAD;C{?rDz`B8$Og#(#x8|7I#qH&)XG4E!HQ-+@rKr}*d;|K^=)@xLnmyerv^jt>w;x4oZq7p)Fm"
    ")*Zh0{n=%|!jmSm0a92(YdSzx3O0e%4`g!4ltKA6HJmD7|O5dpn*Vw1{;0&~moqOw|2djbvw<sv9j;{_axT9_F^7"
    "louFLf>$^x<ZM_(hh9F)QCpnS%JW<u1P=dXZytlmVgsVXonFyAG-bmVsH}k6YCp<jByhYYZ@S0Idw!qzC)`mvn>R"
    ">n;K3Q2#p#KCcp#N;^^31%ZY{^#u+5AM}q?nPCP!@WBI9L6Y+EBZs?|9OmWh=Bux%89Tn@eIX)EzBIhEJ1{)xE3_"
    "X6nE25{^(JgqMvfe|IM`PNVF@N1Y_2wVxRnouXO!_tm)d1aS0Ye)v5T@U7LCf;(21sc4GG^~eqL|-to)1S<{q1)E"
    "Q(7=>jl6e_KJttAPrj^gmnCmSe>{eW=6HNJ$iCoLca69HkJuky8EDy~r`ey&+p~ocUk9*qs||CglUu`buScH&9l|"
    "{8@aWX-IqK`TZQ`e_&%r7uAAJ3>S}sM32{#92mIN&6>xk~}%!)KWw$g&wJ7rypz+e}l<Fm>)LnZvFLF)p8D~LyhN"
    "C&`JFIQz<+afokV;D67OjPKikyBf&8rLT&C{am{>QW`a7TdYIYIVaCL4{zq?`)c2=TB2l*_)GZ4K@tBz0__|AP$W"
    "(StJV4ivS3uhC3ehj`t2uMlS}s%N`jB5L&~&Dk$6SQza+5VYhUpgEB@jkW3oe4X-b2zYwgvg1m~4=Ao$(J0}N!`t"
    "!l7S6x;Aeolg&8nHl42KhQ%qbQhc#pQZQqOu?%JSb}pVx><@>c8^ehPTnu|Kl+>URiA44&Y~gt|<SCInl<9>VxVy"
    "UYDQO9m8oBh!6e#$!2}g`$J@8J<VP~;>1)Trz$j&8ZidR;R8`}+UE>IR_N+ZB`~|h6?C=2v00;iME+{SlBVUlRIZ"
    "=p3NaG#^D^bAEVE0XXRK={aHE0s>qsclw!=}C7LD~nsL9$$6?czYP+OgPy;v?WVz(`W|4(A~;BhR#m^|Ymb}GA@@"
    "WxPH5^MAV^cS>Rv2iOKS9RDY6iBgUzJ$c$*q;lqHOB!&Vt{9`SW??`!^$U2b(1bIAkK25Fb`OpQ6x-kWT<lov&|V"
    "4k@bcKFX6PwZL#A7BM_Cg=alg#p7q(SHZt*Du@gu-RN)B*4Ty7AVtYwqndBMg1i=#X?$Q`EYBa9Ub@Yuwm+=@@<q"
    "YHRm@>3ZQQw5(rLy7}Wyt16yimnK<R9~dQ8EAveNhc9$cj_H4~31Rl0;yrNY_ceu?>a^{W;1&BOYDeu!y;*h{ORE"
    "vI%FmJ!(K<ogd5xZJ#DQesYW;Ya<-!Q?;0JpTRCkT?z_fe3ys@*aKyeLl}iy&H<YNOEZRNvScrX2G{5uy@rVukU|"
    "FO<lLw1rkaOlX4(y84NvO(@)dTc9m$G}29XL$Z*KI!^38)8b<<;*gA|&~jUdo2Q7yv8Y`0j2L?O*wR;5WPkKaE1B"
    "r|c5nQ5?EzZ%w+tWh|o-kd|TA45N#1(kV7xDl(Lq2o|WX|moHY6>YcaKLp-XeJBUzpq%Kcf?k#o5thj$#}qSeBdm"
    "{Zb%`u{IO1&A5La6=fI?jkoQmyFBu;WG^c4aYUivLTc~5w@AqYkk#n6HF>nsObJn0THyQa)xd=9G5*67+d4t@s#="
    "7npYcd>*1tyn<O|U*$gdzcY;AYpg(PmoQd^W^lV<D8n4HG!4=7qFQ7W<$qDKJ{qDW4aO#)O{>jvAIu^K1~Nvi6W3"
    "r&V1T`>TvG?}C5`FSI5*g7MGZvLz^fUfH&0VquD<W|>yR6U!%b;@S^5@sN!xhzD%o_5-r2Dq|F*tbh&Co4(wvmT}"
    "xi89u7}H)j5FiW}>qVv1M^P^WD&iyMfBsig+hnHDmc%rrk*cfImiww*KHS=={L32catDMe!$#g(CHHf7;JYOWW+v"
    "QkMjnyyGmrRdb4ZF;i*l6VXj5!$mZi>hHlGL`Xsff1mVmK4VDQczb*#6`=#aH%F){r?2USMvziJ`4H%R_S9UkbWq"
    "&X+{qSjMDU|Zc|g92N<+ljoMud+j}|g6sBFzKd3*b;F3kRKPy!u=C;K*+t>NxZTDT*n9OKO`|{eisr}#XTIaphz`"
    "nncy?sL;-(cR+n)fYwf1QG~U)^lZ>oVWkZcZn+Jd*%9#uamE%g5$PY3nN){bSJNwR-dV1@y2`YC(bRs8Gy7Jr(K!"
    "oK}*^bV<4xQE4fm0WM1s2tK>i8JCwMX$^~I8mv!JD!?`Zn}jn5$m1wsb<*P`F+f6IHfbR3vcz1yb?sEddUGa@F>J"
    "6Q7`~l?B}n73f124I9|BV#2_3F;>(PnP1Lo7S#b?aIFYPt`<!v_|Ju46uz-FgO`OQSr`Gn=8le_DN)__Zd<#R={)"
    "M$h>9rGKzEm5fw1;F-C7&OH*-~Kc@9)UFnFMZ8827fD8*jCW!V-uv&>4bfin5~tkg{vgH5#e&dhx4mIoEqHfl4$E"
    "WY%j}ol5oVQInm+Y1$n5kxrRM144JW}Y>?S)>mr&Be0^(3a-MOPKkPB91nbqt3`vC+fd-!T#s-Qny%6mtd%1!f0P"
    "_OWH#SL%j2GuRY@?}NM_NlNj4B)nY9`Dhs$vG1#~sT%Ysgy^J_`9Kni9K7YfY7Kdzt}YsJ(w8-X3T`mgc9Te~z0{"
    "Qmo#-?f%8hI900=7HH^_jnOX$qqm9o!kVr2H#?(l#}9um&d7TFx8i}5uT_^1oh~JgJ^pNI+}6H?He1P`WU>ol;#!"
    "$gqW{Te1vzKK`*f&dajF5|Pe!K_V;MpNxJFqDip85(uUaawh}~jey+3Aqhc8;k`r||T%)v?SPdC@gT30_^FQ(;e3"
    "}1KPPopTY76o09P!}CSH<Cx_6({{=n{~h40xu+YVDE4f`$ilY9T~cz472Z&oC;Bg@UFJLrdvuaS+Q2_hQD}k4po-"
    "eLI@<2-88GowCU?!d7w4RNm`QnPd=0W7E9vPQ;xihN<Rl#{535|4qR4gGi@EZM<FfAMU65NF;I%y2lZ4YYo045ZB"
    "<}WEsGFJ<KAo=6_%aNYsd#xy_lSQe5Mp1Ik@ekV&|m-3YJv65^~dQ+l{c96UmS*HyzAt02>H0%3$7{{f!Lm2cu9h"
    "WIQDI6&qtA6l=;2Zl9K_r;HfO7|6Hr?qmoQ6HyZ>uey6|kmIV@fO+2+M~9=NQKl4@4tA144TT$5;=hfYD<x9Cz74"
    "H<4azcZ?L+${)p-?&f{9=dNx2;xutJALjydMpUpGdr#_%JCIQ-%zLw(cgd_CZb;<>m|=1C_hyU{H=jhEnhRnDfa@"
    "llYonl|ew{U4|p>U7RaE^US*T~y%MoCJXZ^ed9%Jeh(4_A6Sfg>!21q)|+P=N}_zlvs&SfE3S44Mzn$=7FO54>GQ"
    "e)U^R{0fnpF1!>isjYFz{qkLfJxG{|K{SyWsa=)(-vo1M1!f|6=A8uWh8WEv5=Td%~c~f7(B}vx>RDCR9|NFo8v2"
    "ehqc5As#`C$#4AJoT~Z{-GMaM!ldbt~y8j^OUFSZ!_h3H`u*VW>;FKHDV)^h`<&(8nLVkX}5pi{7<9L&!Mw82ekB"
    "JGW&DTl~6P=f=4nFtwSHtetRv{CoFTn%}z*?X)zVw#*HG?e4TkTb{#p=8aF{-+%K}*g-I*y9tAkZtN(`n8se8MXU"
    "NkS7h`UxkB{FXBHLSS$9F6u^|k7!!t57QEkOCTLE^m=}R^dCzYwB($sJ=xk9l8<YgY+;#>5<lmN){1js8TCP4{_E"
    "q=yV3ljYWRlpTU$92!D)BrSbHm*U`+pI$xo9nH&Egod6mLZNb<d4oUy(*~qU#&2{s45*3__(Ngo8@gxV$6dWSc&e"
    "UCS1j*O@*M&iHd2X=0#B<a{2;B&741-Ha>>c<TRNZ=muPnB}aFmacC3)HV-YfbI_JNs<ou(x4=-z4E2UgyX;NmN<"
    "7v|Fi|Cet5kFE37e%ATba1$MCJ^AD#Vq-B`y8FgkbIPpJHvWtqqMtR#YoWGYyR<@M)vppQ^&Z_qxDxyK>6d9aP3D"
    "K}y<+frb%iN_OL}P#EK+%5bCo&Y~!1RLrRRKs0pgii?{QNfC|6Ov$i0H-=5My1RgFn8mI1j>(Fen%#vp!rEB4Fb6"
    "g``ek(d7h;1J6IYj9Mmb~@3(uuA?ABJ7bi%jK1%$5w8CIfCbQUbF^@P@-O>#HqYJ4+yO2xDJEy+eV0w%#Ac*tI+k"
    "}@XtB%NJQk2yG+Z1qGS)I^`;DlsIV8)cc873Qg~Hl<|C?eD#Mm7N@%j$+hc0f;x}9-kJGQXX!NE`^QqW#{HtgS&z"
    "{&|nH^iR)acY`o*W!vV{<iRXPv;6O{qdars~_35G(I^ZRO032c7OIA^wWVo$_4%5#m9D`{Ef=(LK+RNVY_r341oJ"
    "zolp&omn8%0QSzJ(>TChSARE+AJ8(10>IOIx82KAzLlSJdYnSPVh1#z-pUN8$uX4oG7EL6<*Tey%X>ANe#gsYETc"
    "B&^QLOS9_N;9w?~%ffHgsU&g($bf1y%kX)W_ri1SLzs`eDPi$Dz@H@=QsQQ#z`)gDQRk}TMT@ZcpV&7vF*EL?iid"
    "Lkv9`I?MhZfrH&&aj@SNI$&}!WsNCYmmYQ`At89M6KCHHnhNHd8$<F=6EB5}nZk4}F|o^t1alp1Jsj#DNbq6B07x"
    "pMo=ct9A~jfE7yZ0&$2=ZoHAX>~5ggS-t+0NLO4L`W()0<6mpB|@~VZ582<Spu$_jw9r3N6@@)q|12DzmzMl@Axr"
    "(sU($=_vU|Cr(@9(n3`z-r#sr}Crqc4^mtNj`|s5x4;R@4I&J~7COjT6$5)1_8x}l}TtfO0@+PiJ1tOtBFqIn7Ie"
    "H<XfWye71{ad`W(NgNIYB`pF^S*nLT437keNF=1KUNZ=)K1&GAM-fJ*%)+@xmVxXCMHKYq=Ks|5{L>AWX$&(Ul}X"
    "0NWyK0I7RIh)#fEbB<oBk1j_|_S5SQOdOytZu-2nmFtfSwN-YiJWB=ELmAa>w3DN^oXa+u8trHdd}kuspF>5bQ@B"
    "JI3Y4U=LI`i<r@jNw)v5qTC1p%5&NtW8o(Fj{j^F_-5h=RQ^1^<zdPUY<eJx=$6dN-gqchyv>-wt&I8?z3hc7&rb"
    "ch`o0R}2U<?GxxWAksbCCu5A_6&f{#cfF45Fy5n`TDGyj^ROLTA*Uq(W7{$x~pmRX?7$k{{EX|6kTV)g8SJCl#C&"
    "|8Y(4*6+>qxjFXtxhxFw&W><za;jAY@Do4GTTW~7yCkbzW3B$L{nc04%_Ffp<@(ALo%~U~inqFk;^$f?X2}g*8Q6"
    "nB{vn(|F*)4NAAL8v6UPzAyK4ROQ8!=#Mr}})dsaOHq-~!T>wD~B#arJ<PJsBD{>{L&TVJ3mQM%a_^r8>J)csKRJ"
    "dKe)v{T+~4nlP&i|K?`1#+j1*0Gq#|l*8r+B_1a4_D@6E21D!K1wLYtxz0}F;b^$O=-2(?J+7MGxgs%0LL#Vez{~"
    "$aGu+a<8Lq7ApeYfRvT?6EQi_=B(K(NrYjf0p;qadA%adOOaTp=<%$&(*b=$<kI3vFFZOaX~MqJtEPSG|r)^@T=k"
    "kqyvd;i2}0NywVq6TZ1o&g28X~tIe#G)r5<Er!?Kn#?m@{&sr3nzd$flj28M97+^c8^_n|Kxb%b1eqflO7Z4CD3v"
    "V&?bjAM!GeL<^7YB(W}w^X@+X0xCl_`7_+YN^7!ayrw33ripOLqH6@x=Cpcrd&f@4A1=UCwqkU+kpJ{ns)x>HbQU"
    "j_CxV<9=f-PpIk{eDzfEs&rlr`R=B^w{QA#Z-L0M&FjFX4z|#@i=IZ;9}JAw`+~$s@2jmfS;MkmsSM&q7zCPeUb!"
    "gKM3*mW%6QEck<)g8dkVOCPXw-b}M%7xnNRV09R}c}Jn?4}{STvfaoupr_FNFS|69KO!4wPxRD<bX<A+)6uIDX%x"
    "9_Uok6y4~pGq%V5_D;4PyOWA3Ka^fAEQkAS2s+4@URTwX#5-2NKOn7f~ii}ldrPibQH7@Q+gpGpT0$BEwC-ygj`^"
    "-|t2-O!C;WT3rz|EImfKaEbt$D{vvGdeN1eZ`Zbu{@=)Ic0s22=na1Z#b9b@MLs+njIXT9*LFM;p7ceQq@6<&rWW"
    "4AaCH^5-kZ%MK}9p@0EGWtn<e_3;(~HWXPSSngC?Pje+ANv(Veq$SER@(Rf9};R~kdQeiM(x6_oa*Um6fIe6VP1$"
    "=#0&X&}J=ClKRY+qF6<bK-p8=06h>jr990ZE^i7k<$6c6RG7#qGIkUY7^ia9~#48WV0l?O4~3riRS7=iCeApyg_v"
    "HTxnHzw@ou`^98AaKVpJt|L-ph_%G-BgO~y-A53<ezy-s(lI9FeTY0sBVrq5@=ND0hPixsbd3EH5BGi^y<m0EHM0"
    "plFH$UkwcX_TXZRcPAP9hIA5dnbh;r}sfssFpk3nlCf}%z-1df?yg;ricivP6*DJK(>>_+M?houG@Nzz(RaBq!<G"
    "xwzei1K1iwgbQnz;c6!ti{8z)&Jn~?@&p5fc?fvvU;F<YV#pg0@}!Yp%%hfm}^%1Z<};!&xoriBeIr203>)qPz83"
    "GH422?*;O%bNy8+?T-in1tq&S{yHZXki1vwu(^xay;Osd$sLoLCZGq>)O%CZFcP&!+e6}7||7rLY>f}OvJ09j^#("
    "WA7GM*tS4W<(SdLir`{@x>GW)<#t<4$PW69Ujj?~~`6=}|9f9B{xbM=);$%V*e_^jb|&;sS^8I2JNTNI&Em<PM;2"
    "Pzz}syMYV?CbgF1DXZ!`Ak|v8Iw0jb=YZVIDQ&Ly*tsw!VFbd^lgyl8y3vVc70V;~f)v7EM#l#){{kr^NmD0MEQ6"
    "<sf|(w(+Xx~X3<Iw#BN9gt;8;c1j5;=SbE9wcq5@rlLHy1xo|CHhk<J_4NS2R$jm-dfWY;Kxci@AbNo);UlM-N)^"
    "k!Mp%b0aAEqh`4X$l<*U;<+_ylCnllnfFZ#@t6%%y?Pc&KAWqV7gyY{L4`T=r67MC+bmSt0y&G6eVG5mXW+eh<A{"
    "^Ys`+8)kiI>@3zQHio6r_gLPFv&m&k1{#sdpcMYyUFmBZf$5(~+?p@Ln41BM@q?s?f7y=4ub|W-yOq(0RFQ#tU2H"
    ")`4v=|!1yzcezo!f+H>B5Byt?Z%wBJ^>f|7Hl@AG?*&=-qo8>A$Vvs`r%4*OUnN_NK{TjocHwLb*-+b1&a*7^B6u"
    "y{~!OPOUN3Hln{kWt+e~!2tzX`|{6-jsbp3m!wYkRwUDnt^pO&ZI(3z3|exBw=g1)R2zQ(uGQ!G?JidIgG_I?&;6"
    "=ifSp&YlXq@0$Kf!k^PO9Kk$LvFYT5CQz8tt5m;QWBoyQW97V88aW+VQyfr5Yysi*G(w7A2j?ZPKR4mPBhbCRTiu"
    "80ZA)goRR=E1eta&=2>cI1|n2do>Nk@A1g*=fhwL-L%_sfX_(qXygpsD)6-Aq&R)(0GEX^%(l24!@;>ZoMjW2EJ?"
    "BRqoQw&gG^j@N%>&GCS6TDCI&Fhn-gKLCR5}C@Gy}7s^qOWSUX8ZXu&U71>rFs|^qrchDAP11+_S*_Hydd&$yRo("
    "jY0B*dnOK`mjmZrB(BO+~cs=NN7jo?D8lt@+wYMmumlZ?Ij3%1#2>nMk`FmG7E3!MEFo-+|rju6XP~j$i3R7S|w!"
    "9$Dw$XrpJAR*in<`J-@}olG$<<MP)I?$~WP#izZUHN1a9N)XOwbKUta(%F$jggUy7!>kwUVus~R<-F6N?xzZjQeJ"
    "5`;v%=S^=qfyUk`LobA|+^GJ#tY+b~)mjCDH}DgUPD$=U<2lcCRwYG4u9(0SA+IRO$$`6#9WqVuT-Po0+63(Wpg="
    "IdDrt;=17MR6@?^pG?JJ-jG=>T+ZDrmV36IIlv@J+5Z6)^s{t>Os6p;K=w8xxI`Cncte9usGHaiqsTb#$yPU0H(q"
    "zf7Kw$AKJ-t-H&hey+AHO>Q>$35ko?*1@8meaU$b|evg7VsYlD3TDN0I%eC}m*bm?KtnGey_R3{n-1Gh?`{RrGgZ"
    "%JYd4}4U$K@VJ^Co|^_F+G(BZu@Mrv16$USKoWb8dfTxskckvnI1W{+1l<8b#fAf@_Di^xT>L;9;*jz$~ttp&U)D"
    "qY&q)@~T(Yg!#qImTm5(YUvv!Us6=gqRqq1l|u)5G8p!T(A$tV(#oi|gj@7!gImKh=jcMN^rJN?nm`Khj>CV2sc$"
    "sT(|ZLgxc3)rV?TOR7yr9gvKIrKU}l%+#^UrY_ih$+DAcjvKjBzmc5p%nMn}h@DGjq{&${OGi_vlR<6mHw=F1nOl"
    "l|<~!OsV$*>`cSnA(y=nwVL~PJE7kiM=hKidi)URTGtCN@}8)#l$AiPRx=iRO^5RM0vAeT>D<ze7(Rt&vTpgC-)q"
    "^w3*7k*`C_H`Q!a|hSko$)2=|ac)fn?^QSjP<R0@#i)A8u{@>}${SP}m?W$Sw6q$dMn<K?jDTPEIoNR>dVpNRgrB"
    "g1Hn0yVbH$eO>%N6R)qE1t8Ha0$YzxJ~elv1KL1GJOwQ|>0DoA$cu0BVLLrogPE*mjVnlE<+?6q6+au!4l0W19u+"
    "8p5FF4&Vje0o4U=fPUqB$`6Ybkc*lB`zJJeENOTGik$t%q9O2a9|_dG<Q~&evm>*;W@bOAOR$0#!}{<`n<W04Y-D"
    "|4bY0gj?Cd^lEXY9)l3!y}1kv)qTUkd(q<>jr_8Zo0J?TP*Xd`rG-<-BhvWA@1mmn)QI>5BlDh-5rrRO?pVU~^OF"
    "LCZz<#P(E;-CnPu=;3=Q{0%x<+^~b9Nc+;OLl$b2err^RSMYy0o8Jbihm%@)+?vNV@!wRa)D~;3X6RwZ5(LQLZiw"
    "a#ag7g<ixv_VcPxL;+TG_IHt9_g*Mw>vXu<5Frhlz02_1oYjqnTlxxTVZ_33(8Rd73krqtO42Z?1M+b#!^es>g=K"
    "cwBy?T;CZ6@iu)!gv9m2?KD(eXh28hqx(!O7{t;r?mRBb`bf7!o@rp6@Gy10KU$_r;y@Aekag!oSmdOj8>}ga}F0"
    "?K5w)H{1?8{gzBZFw~3*4|;(jZQgANLK{)nlNvXNR6~X4fP1JmeMH^BNvu^h168i69hKisid&kaiC5EDR1v6#<AJ"
    "38A(n@8Q6fPE86(o{By(309@rMqto}uh!F^Qv5G5+1fQ5C2Cd}9*V#-a;uc)WDPAY}|7TD4ZYQ3NvIQ6!;MeiC`k"
    "U<AYNa2E#Y(P}(XQwydTQQS?#v-~tlB)d4Q;jV`AOcQrNp>?Y1^xtm9X4|M<iSD53dCL~MtkYn7FOJ%oUC+o^m!S"
    "t!164M*Jbw8#W{uIEeZ$JXD-ebT<<LX(+Ykdzse7~<yx*&Xs6PKvR<h2DP|*vDHvL^xPsz&zzi=2X7HwlrC7C5+5"
    "Y1dB76YO2jB=mO_!kX6mGx)R5Qc&8YK-&qwxkcyVY4`CbYUGUUH70hy)&jGE2321nE~^BTNyts9I$6&Gi|!j#s)O"
    "9+fp#xO37Ymnq@A%nmpr!E$B`>Ex!OydU^>2%1x42W>1MTu@_83pr#|_u>p{|BW77`uq5ty6M%=zyJ38f9ZYuho1"
    "Rw{nwcpwZ|*h17rqTxt<xUCkxlVp8l$g7kF5T-}(%hSZ>zw!-Cyyd{zi|s97yMv4O(|&Q`vCkOcDKAGLWPrvoZ{h"
    "1KHftV0D8-@2MNJi#!zd5PI3!}}&E@$&8;y*WJXeACVLPO@{U;6v>>ojgQqj<XVn52(3Dt{6%@Uw1e@pJ&hhFTXh"
    "VS+@~%x&uu*VaU6_<^8aU-u+P;OVVWJJ*npFj$IqGljpdN|8MqfLT}Q$0Z@D$m`N;x^<`SZhGqKqSDxM)Mq?EwFx"
    "LbIBR3Yx@xd>~{=$7Z{b`iF8ol__=-9`=Ak>liqt_hJj43m71BQ?m8onX{MbHdy(S~Z#Qi8y_tY1jI-6_CxcP|zq"
    "RBdzWIGZ7|4WY%$j{25`0gV{}J38LE;Ohsf+`%{X6tOX~V%0#XRsA}qiy~5awJ4~cr0qk`o*0L^P%kzs@DwPW3)>"
    "#oQP~R=p?^kUJ=<K*YYQae;I04t<meCu<@yG@C>cG(%#@u`T8I&<1GKmU4IU=rVBm0cbaH^&GOgAEZo-;ZF@w7yU"
    "aav(Bv7FG!WkBW9g$qNJks*i>4*UY%4SYiqHHC#Ns2HzK=E-7L~Q1BRx;L>QRj*|^@(0D7PEGV_7(2EC~<|`Lm;6"
    "_LyZ;*7Z??RxZCR5V-MZKs_rka>|PpVVFIBw0lxT?HNQRUfY+`2?5-)Ax2KHOsB1YwdO`0x4+*CN=(xu$*C|`E3p"
    "@rPZNd1A?&J~m7oKm7uq#@Oy!-DEPY+tINk^7D+f1;gb*|O~Me!M-BDR1xynk|<<O25aQ<g4{kJi$hu>q?(W9Iib"
    "8&>vK_uDhI=6uS$#g?Z%+A3oq;TQjrcez?^zQ8>F8U6a~!`(l`qPO+**<rO$uQXt(97XAd#E%IVzUlqn+H3=MkxT"
    "zq5P<Y?$vu3W*l@9)uV2%b)n~6ABlQZp?>gI51$^nC+K(#541cQ3yg&@rF2pj((9aAHIJ$U?>ZWrqT1_Mw86%I4t"
    "<_WG6FDLzig<7ApvD}wn2te$laUZTvg)7%Cpd1!29{=3JQ#ZUY`+gyGCbVnDJDEzZmRl<!W%q0%+W(np<&Dfo6l="
    "U*?XkqN=~xZC~ax(f?ksS?B6%jOY}=@&ab$rmmA7Chc|#;SyP@m_nrm4=K)IYIePUoGcbCnq-l7yRvlakt~<xWPB"
    "`wHWs63lf=yE5=f3LLeef#+mm7OE`U#*JV&BrSS()clP*~%iNvM8veEhq5X57$ou8D+n<i4iMK>`^>-2p3>QQJ9&"
    "zO<l8qEDR>HeAK6Wg;wK0GJgiJ>}XFz!{zN`_Ct40WNXj2#U5itBu+UAgF+v3p5>rH)1nQ)o{G5&mrU$KqB!9Pb<"
    "JHTY$f6kp#&F{nR`gG0$MbL4|ipFtYMG!0jcUKnigjDaduqxBv~hnBU$I)F(p9Qe)eGSZ!tAc8F-9<NsV#nEzsCp"
    "X;rNjcr&8xU*jw>&-l>(|tJhmehZv9F=ni@C5excX)DzWpf5n*4EevaJRg)rHZhh2=5f6vZgdTeR*pSqW4eity&x"
    "bGiia@0*!+fv^n23DQAJ*RfIZU*5QO3k+zbyr-JZYfe6@}-?pK3Dti-t_smUg!m+VwH5_ME_S0C47G&w3>N9W>WB"
    "P#%V8hbzl7O5Tkg>19I2=evxb|{&0zF0$Gt`&O_lGIesYGkfE{o&E_+nZ&rC0p$73YZ-8FV5>ENQm^zR(j}LF|4d"
    "h4kJE6y0!i29aY6h1`bywhK}TqoOt>bFv!fCv)yuHXVcAQX~b{l<L9<IFRfQvo5E{Z${bi!JmFQjnX&W;0||?oca"
    "Z3k;2Mb(-^#K5Q`#tRIQ!6PlFg}zn{4AmXiKLVhuw%6uj$a2j*l$SQdoOK+o;FKB%ixL*$ezn({d2EUrDYT?MlHg"
    "B$MqzQ*B`ta<Xyi2kiE|L1;oJnD`9_h|pk>F9*-@ZlpYU-BEDxiS8idL*iCQ{T&(8Z@U!5jn8}uxX9_g0flxGGbK"
    "Ac9~a`_`lU8DC+G`1ZrRr=z3~2j8#RtB!{~R-PfCg>P=$wjt=>`7-S7kCG0}iBeIiqxy=4qRA4yBNF~>R(!Z+vwu"
    "F;|r^N=Gc+C0{o4yF9A@ymL3Ir{ti}||r6}bAN2oX*XzQl7(9?R>6-}hi~p|qjCnK1Ols28YO6STdTHHn!~?&Mq-"
    "A44BE^~$)g4Tq(h000)(Z7Mc%03WDB@r72P4!O<(b<h|02kE#6Zt!o48SF2F=li$wOL_UDfLa|VRPBTX3M_-|r$l"
    "~cllxKDT%3=nC9)Ubs`v!JONPX(A^9lQ|2|evBJJ74QnDzwmnT$EN_5*vgZ>Fbpch^-%h|p6dLr(Wu|eTr8soFk-"
    ">QVvRu2A1o)c)EXrZBZS+6;Tz|$Bl_g)z(k73b(Wig83(5O2Q+<bAinBHO%0XO_3qB|?8m|}|kN16t5xv9^2z@`O"
    "zLGuLm3ZVa#ZyHWGxr7>&`h;<GD#nwyiCPwmN8DyJL`ypY#qS2zGFABTl)D?kyFw55l1fCftFoA_uWs42TdM#Nbf"
    "za?2wGl{)T>yv2>D@HQz;dzgRB+WC%^_uz$EO1_v3C;p-VzMhZ1Svg2ZUgIRbj_gD&uE^o^}|U3a=dyM8&XPup(v"
    "`@2;4$xj7CV0Ovcb30Vm<skd=EYF_x(I4QR)V*ux24*)M<W@@Q#l|{D-4&46!uBdiCmv8E9*PKJ2TwXM3Jdl=6=S"
    "W?%Ho)1XCopi(+x_qAcbY?nK_nF9vnaCetJDRChuW60|YT>R#%)~DZh~}1=UV5#uf8ZW5p9rOM*QYCdnjU-Fft>Y"
    "sqG8%^pG6Q}q?<P?W9?9b!-pP!k78$KvcTt@}O&<<vaJ_G5xTFt!@>YPVJ;KBe@t<*2O_L%CUZ3Yw89eD{wIPmd3"
    "Nd;_V_*)gUk>6=xel5jyxQL>eBbq~ZJen$Z7stgx2T0IzM)uZ`i>*{dJ1-QNNJV}&$=*|n&Jg?T?x|}Vf`(Oin9Q"
    "UG;tV*{4wjq+=8Gzj6{p>AQ@4{VqP~aFQ{u{nx_5hgX`3fTNdORu|0E~IisnIWnXv;>c23to}6|+nGqC(HP(O5tT"
    "Q#OTuVI%eb{Kh5|u$?0Pz9pj*wH9<$a&jT6To~-F)*i`^tRi6saR<mO(81|RcJ%g;j8{IUzAGLd?n@k5f0SW8Yvj"
    "HLU1G7RjiHXcHeE_9%Bb4d6*c1Bm;M1#bsXH9B2{={4%oB|;P^vGz(8C3Rno^je9lL5(l!?bxA}Q!S3R$+JMg4!B"
    "i!SyWZOQf)2I=mbF1UXff0uJJgM+XNWtzEUyNQI{4zR*{-Q4K1>UlSIodBV2oVlb%|Dq#T3t#haOt_N4^F5x5uAl"
    "VARf@8kc>J&XRupKrQ|Y@R&xahR#7IiSD~L;iFB9|#9GsuFkm^~I324j^@fxbO4OULtNA8+6KDcesd9$JigKCy{r"
    "&`<4^t|cCD5>QZr+7(2dBqJ0BU|34hbeL5GM56_L#Cd)R>`Y9)nMC4sqMUGU}E|-q!58n3mzShOwP*4EwRYG&#P&"
    "J{VA@7G5Q@z&=^4x+T&7A-h2J74(RQGm4Q*3$l~AbArvVjW10zxN8i)m~lDz9ayI3MXg?IKJLzB6O|JWC54-v#G="
    "MM&G8>6NT8$XO$mMU7Bd%xB=IiN6Wrr?rbuaM9@eRsKu{R2qX;s|?NCKPk}72K&BInM#{qm{kIGA5EL_{Rb-2hF0"
    "hq29OKxhzZ4_CDX^&hn^K@!4IjL<2B~KR+N4DR`DXLI>>NVCmV}t^1tp=eBuCKSPW<953)MIL4&0E%b+38_hFhPb"
    "LS+7b*Bt3{&tKS66E~9N&g+xe|{-j4~FbIv6z-hSU?ABB8*}5=Nbdtg+n1E6bbtq@IKAHksRW<o;^oSm^f@<oFKH"
    "#FURz#Y%1{MRlv!|qCx7|Ah5B79tWeM_c@6+%qsVhZ;Gz+GOt4(Xtbu7SqTMSz}*}%Vv>ZDV<AZ)z2NP=iLGX@vr"
    "@PVoyzf+W$x4UUrX6>aX<U<P^BrU{6ffEtl5RaL&Z|TWx1rVyJ80@!Qx*SPyv9!4u2c0w;3LJt2b>BxWtMs_KJo8"
    "9ymMeZWZ@R6LR@Dk{4jpQjDj>t$*DEV1TIkd^Xw;1;8S%t14#H)?*c5vq9#hc$1bTbndPPbPPG?8#zEa)L`_aPWV"
    "GqE}^+g-Jy5-rPlh(3ta*Cam0UQn@<5L8tJCQ9%zf9m*p~|~+s<0HAATJTNy;bGCjDU~ppZ4{x2`myyH|brC=&>C"
    "zoZH1Pbc{`rNzX2})-Hxg;GL$>(xbGTQ@A|Zp|d+fH;OTHLxD^Z^#!=^588>rm3#{e2bE?@Nsy?7v*}UXqqg^-n<"
    "X_pFSV=2b=_=DUWJRs4=xUTd?qgpemls@S5|ggzrJ08hr~PiSaL9a=a6S2U{|CXGaLSa?h32Y+Dfv=2;((!)H1Cp"
    "bbZG<ZD883j-BmEe@;MXvHkD4K_jp=4PGYTT9Na&TJyn=Z49jQwI>_c9WFH$M~PhrF%rH$=pX+Seyca^AMm~Ca(T"
    "jqxUv8h4P+I1uP`IfT$j**ZG}W44ekSCmgt122df}o)F^kF)T-Kb1LweKF|!``Ejgc@`4*&t0JVs*1T<cwi&1^CS"
    "lz2pDjgHZ#M-7g04=>XyMJCS3#qZ+Q8}<G5|d;^tNrXnIWwjdnSf~0QE8kW$!?>|EfS=CVdI+oJzdb!vS<GiS*yS"
    "mA7IZo$CF6acT<5Ht;ly%Bn)!~bYT~|q`Xl<0Uen&KCrM|mFuf&8J=b`8Rc$ZVf`>WiNt{ds{azUh-^aD369#s%o"
    "B?pvP-+vM2FNy&Uoi_H2oS_CEfKV367nDLXp?mQLgT4JSA{XlZI2Hto^dBcj+A@(WFS@LHgkq1IUu*HJdE;4n_k+"
    "O?Xsr$rJp@8gzFq6p1r3p4B_6jikbH2%hu_@<3Gr3L0{RhA};vmt+ot3*)?2a4FaHq(`-7GT4-lQvQ7Fk>ibiGiY"
    "EU=>e@1x?1O~x6`QJh{cv>()X^RBR#E&=b}Hi31PNXA3D3G^pimbNGb0yTNzPi_vW`B09xY9-LAl8GE~1V*VFQ%*"
    "v!@)cX(X$gb&?3d-tK+ComUQA)-vlDRy^YC}@Vc&=Al&=hXzWJO_gjBn<F3J7qvsb8PF~hqU9w-TQE*<#6Gr3J00"
    "T&%Ves?w)Org9bMXdW%c3H4=<kkwb6q&HPavbWtO5Zg_6v5Iec-%xXax68fhZ%2E`jHj05cxaw9KVvr{J-G}5>(X"
    "d|;fzo@VH^o+sa`k1~lD@>OCzWUKZ%zSLyqV>-SP9Udv#RSYezJRSl%yqXjD@zEAwCj67<rXsD5cYq5?l7P(L74F"
    "GE~K(RA%StT=%h}+|q#|)X*Z427FzvdM@52${_^7=4MCFDfknlPO%#bEx>@aVUE#|sVX8kzY(~H%n>i9jMdPfE;t"
    "2zJUpu^MshRfpm`37_H*%9d~9%XgNB*yaB{Yu9w`;kr4YMhegt|9l>#u{21GEdVuX${c4o9?e>~_cd44R7lp-G7Z"
    "5bhn(?s=0NqPuqffmgp1+7;LCGcF{*$zxzZxsJ2Y37Mj`ZahbrgIgua$6XH(!9h;Od2LpidJMfw?zCY%~1e}7y|n"
    "xS2-$phD8Cp0tX~!{y(mCq=HPpyd+gtv=htqlp@ibfDCY}-u1rwVC@r6nfjWZs!|WoO~UI8Rf$>W{>>)s3!U8#l^"
    "dtEo20i0P2)B9ZdBN3_v_q94d+#+yj@nE{=IAS`G@s`JUY&fN3UP)?T@IHmD>r=!X*jmcbjTx%&2cuHBwRgr!8#&"
    "zoHM4ud+AxIeHRxGO`VYH05N_;numQJYu<Pa|nai%h;1^Q8_%oP*4U2*gX<K4!NZ?Jfs1?Flu@)013Iy_q?g&qEa"
    "3Ia)qU<%}hEB{dEI*d<?STVi!_lD%E1D?3AiC$Lobc2*W(+fj-mViS*0T)KFXM(#++md-SaIO40}t5=yr5kGN^A?"
    "M-a1j4Pz&eTVW#@VhL)&_;0r(zLH(o73qx%OgVO#TvRl!xp)~7>869K1~qp_6D+F$%GhW6Z<UUpU8sv&s{l?cVul"
    "riwxd(FXs-Qeb`+$CUyEzQ#yT<sGH^_d`I5Z{A}@AFztAkf_^tY8*9%9U5$Te%dlsK*-1MDzK4wBAsN`nbu-K&J}"
    "W@B@t(OmilD}ohbM0V4Gc8DU=H^LI!O3wA|3A`Pv~n0>`<+s8~uZa9nsny4Og7C4ZX&NhQ=ePdo6c~9Q9}ca8z;g"
    ";OP93Ev<tf!)A^FXb`|tiWSytLCwe<^jEG6GOcS)3JbSkFCBTRgRFnYyoC?L(cIx8*vTzRNwnyekw$yq5@H6>gBQ"
    "yySdCYvokHUkQgB7B_?WI&!j`4-AV1xT`N*=5Mtg2r_g_)Emja%<5}teBcG@dGmFcrQ5c<tA-MbO`>+h^@r==atI"
    "FFReKIxI%U1Q*zPg?X*Fg(=t+$f1P^bT<XL+#gCLpks&d+>4BO$JtLW{b<WuNPdJfG2Qk)^e-|iim&9B@GyZa`^a"
    "7h3{<-{zg0IP591V^OWt97Tv*GwirE)U=(>Xs)IKxfDC%zv<QUb_2z6=otvi>*Icy7P}UCF9%75LqGs$o^2i|qi7"
    "k|m-qnRZ!3~9(2MdKy!@zA6p`6qkV>o`QYGW)dZzs=c4;FTTp&W!zQ%EWTx4oDax0t3;USceSpMtQ(%T@Ks11`QQ"
    "XG@z;SIv9$Qq*JsAHaPMpwzb(G{hAKCb)pehh|pqT0y?}wIDZ{d0!T*=?pqFBa#cGamtie)#WANid1cln_4+&q`s"
    "rP8?VdH>z&&T?SzMNf1D9xz+mX`9V8$*hJR-}o-*@upV!rTR^AyMIvAu6xYYA0fY@g)mHLb0pZquAH{JmTWdO^O>"
    "eFKu?Fm)f<0H&9hL=34SLe^I19OZH%;(F6ITW{s91Tx+A_oPccVZzE@DNO-cU?dy6!wfzX|#oBdp}2C{CIq^fsQ-"
    "lF;EJjpBhz$5^d~N!6f?SY6V4id<Sbt4c8(UwAThruYA{Ko*ko0LZD6H`A2U~_m6%ajZcsF4o?nF4~`B`IOgr&32"
    "6KJ8w!HcT6^x%*T+Y%k4{D}2HBUuZF+j}%P8=r?(ZG$k6yhRy%0}S;+BEVB=Ua$WpsS-@-GL6f65y!`0>@z{+~xL"
    "ntpz{ckrtDS35VpAUQ=lh2~j`zc_FF9T(<7bM#*@IYYkWe%q0!CIQ1^Hk7eZ67(WeMYJNHG@V#TofpS@FHg;k>=("
    "zQy%&FRgsC3r-G`Uy`}R&w4*qlq^R-{el58DJ?)B#J=I{{c=>fy>t^Qm6axi-PpuzA??@fh@+(r98?Lo2a_;~amZ"
    "$>AlyNny%l3Gl=&&NlvM&rFVr#~GXADsS$x0zUoG~ZCW3qP3Ce{i^e@OtkRoQEeyMj8GnB8KcTf#`z2F9FAFTz<y"
    "H2vFUeyHuu*qyD^S=6Z}nE@uT_M;aSjq4RZlxk62wU?$Nu${x62u%p+b<Gs@(8V<xh-Gkb|*n;rNT;>8p!vh1C4F"
    ")`Uy?1=fEI<v&V2Dlcgq-7m-zt6%_D*(=d%Gv^=cN3FE@yBy517A`!k$$B0O^Vp7p_S5_xK$Q<9YX5Y;>uL83HaQ"
    "05^PryYDh6vA#ycnlJRML?Q9b)veLj_pDGgnidiRH2@>eIR?aUDIEkBVnr>(0L3<y>dQ^ghxbY<o*QLj48@d2st*"
    "CZXR)YDLMVrOZm!@XOb6PI%{J4LL4)bV{7H1({)F*Fj%*GBaco{?X&x~3zt>Jc{DSvGu#tyZLTFQ}7Ck1X#z$Sd<"
    "?<ZabROve`z+o-P9KLXNlH5S!q2_~P5g^IuyN;Q98{#plVkL_8>2h-5Z#k_f^9d(u+$JWF6;YjeM16au<dVP&oeM"
    "3W=fgM5F2-fB^Bi754T$>nexUl4S<8g?!Oa~Ztvwg@?7$VXwL(o`y1^yLdSpQUSneb-e@kZXP|j%Aj(YaZ<rEB-3"
    "%XlgGnuh0E{hZ$|=tm1%*Y&-_!y;vH$Aee%rkj&9M#h*5}Y3l4N65$WB{Gc_UnW1j!vSq%0Vwl=$3I2SPh}xIj~|"
    "C+|0qB;=qfgvFEBK$&8qEjyb)-y_%&yXDVyGr<1zY}>BKasb~?s%0S$<qUA~q=*Cz$z`!xrH;Bi)y~s>UECTCV@{"
    "B;gV6L!>?>uP@R|G)^(QvkEh=&053E_<J>d8%2YQ8FxX^y)1D#BKW%dl`lcTnDPLTOhS^ON6?-c?uKz2Y`?EKT>W"
    "2!*$YfrBYj?977+$-V4>J8LOW@obn=D<hEZEdzb>lO9$@wl2->+#sP`$g?om6t|$Sl#xs{mrUgtSDg>g9_|QDyGE"
    "}8x3G5P&iBbv%+W#NWlBXLjN2(2ENARxAX3wA@~m6b6~{eY*t-@!YEVUI>0$ilZT*5?|w79n7tQ1?0zdRB;SjlcD"
    "@;~Q;P}+WNBCd=7G{UN9KpUgMIt`&Lct|9BQmQ&<THb1H7NVK0f;8;Kk_pMArnQ7G2pyhT?f7@G*y@+9ZhgPiP<D"
    "B7YC`C9uGFg<ffW0j?rm;jO2-WHFl+*9BbCv{ZPpvNVjEF_v}eK-Ul_v{LWRJj!b9!^|!lYAi5<r>D?mae#1F4)<"
    "nGJ?HFDU*HkV7Kdo|X9)HLLeCy=Y>LA0neaV&*0OnZ{t<q-4_h;8>l$`qyVfb>k!M5!IhNXe&NJFfR8nZuwrC?vr"
    "rTK^_NmouPH$t+E~9`I^s_zUZMB&`z}W!)QhjCa7@iW8?*06o{4GXm<in!2e*&%=|5;yc*44~?+?<)oVTl7h3YzX"
    "B_bn~kv|r`a`+=_n7yqm}ldE2vkNnH}c8U4#>^z;{=9!`7_2PUnyDz7P|NWbtK*!wrw{nhaWcG@rgc<s*9QkE%Ob"
    "&x=y;;smVk-T9pPl06=L#rhYEE5c=<2`ll+aK0dTP`pJdN2fb3p|2y_sSBJx-gxreVylG#dT>Z=lkatM#o5FW9(3"
    "B8<S42i|^)4&Zd|-oKyQ&*q7D@1OL+^jLMeyRHzD-+r+y&Z?O)!)#4a@u{e0SiF**Ef(C4h@p+*w*{rnXlS@{aJl"
    "*MC4kfGUoU3GWpY8dT(!jF+SL7|bu|An-r??XZMOYv@i}#cxqWjE#|Y9i0jD^5QS<$-l@xllQkQHXcdY$hi(Q+#%"
    "&h#GF@RD(*+NgIX^-rQy3Gxe@HGX;-4x%nsAVf73f)RM2S)j%bn;`e=?1~;4EWr03~UK!$9+RiC49JS5_#AXgaps"
    "^FKVL)x*{fsc$A%X-Co?h`*O(+R3Mi&QbRmeD^-a2Fjrx{mKeETwV@|Q$_f)$U4BxvkIX`Erg|&O`6sQRag8L(gA"
    "UhUjxSINxE;b}_v1{XPcBKy=27s=Q>$+MaHtmLg)DB=rPsxbtnkz#uA$s*4%T(z&{IoAdt-`owJ*Io{BE9x#&=pI"
    "*_v4K>xkTxG!d&SiX?+%P~zX=A%h4uy)q?f2!~fq!GVQ1QIr%Dz*zJ>XPas^9aol&q)ivIhnQoqQtOdjIOwui)ul"
    "=$$uVZ_bYhQ8c<JpRq*2cni;v^YvJ<R`$XL0_Q}9R^>SI2Ex>5Nhu8JB3S_gzSyPB+k!?9sa{ZUtCzb=cF8FFRhc"
    ";}D*&p*9;|Nf@;q5H-BLhbiDFh=*T^GEp!;uG+Qu9U2~1C8zx@-&XkA*@ylN>Ta-NTGHU@z3T!Y!mq3{$hEXu>*K"
    "HtIjLP6j4rKtDb%;R@jERb|v~|YpA}=X#vuGoM>@T_jy|H;ENBKIe?82t2zFQ6}9q(yINU8W=ViShkKW2-x$tRdt"
    "}h0>4E5TxAUDW3ui!j=<SV(#92*pRl;bZ_w}CM5rc#O&7D#?gxhn{42?bFp5?MFZ4#=nkGuzAdxfFW%KP97^X;8?"
    "JX3zj4|$@G#1NTV_zNxbnB+7v;@}D(!Q}VVb!qMze)nxyqJSn#_^_@pCv*&Uj!s6f7P)?{QyOvfc?nRfT>xAoRr+"
    "z39g5ZPudZ*>`Gcvh9Ko>e7dJllwxE&V8+8@w>2e8u@_#n7Q9qkc{Bt)BKVng-?7G=Ba{^MVId|MX&3XBPZ}?v<_"
    "MQ{#;;-1&)?aCBTE0&>T?wFIS@)d*v746xOdFoboiIBnQ$!%3*c+hyh5N4#D5M3MOe?71c1MdlVH#EHqmjXx0EC?"
    "<<5~~$EJSB%U0#A1M~n{A&Ng$q7%8tpbs>AzPbhQ*JcLkdbWNZIKVWqc^BwS33ejEw&*z3aGO&*qV*IK4VQ5FH?n"
    "3WAxhl>|hpafQj66dKmL@+=G%Nr)NFT&PJJP9t-{OL1mPB+=tIp=$iX8<>FAn24d!R|MUtsiu?ExOUPPpFAG|7p="
    "-50vL3UZLy_+Y~)IhwtQ2eY63vFL4*BhXKLJAH`e!3GWskfw?FW{{hZ|9vnx8ARzJ={FgOKKZv>1BcQ=ahRZ7As9"
    "3;uVN8a_nR;AACCHXa_0C8&l~@4nzc2{?lp6JSNrT!m&ru#Jf1y1SA5^E-bcksRIWVh++ey#PRIm?b0f+g9R%I1i"
    "7vBDhpwhkKUj8^diubBbdCNYNUoto(9|v62k<TqrV+k^J;`>|+X1Pc^<9c{0TN1fKBGK<LL9Bv)I-AP^s8wPq1QH"
    "Z#=QbeDx;Ct7^^G|yI)u5^{y;!H!0z!-o&#+4IpC(T)AVgf;u~xLe{@kEtjO|ntS^AxzqJzwp<hIS<a62l~T69&7"
    "d%)KKxyGWEfx5!wdfX<mm843F|v*ROs9rbmC?aX0faxA8i58X87g(6Hr#M1cc`jmR+xko1y#`{A9-um0f?asv0#D"
    "P!;*PIhtABU+<+p6|)WUEdK5hUkW>BP}gQ2oU6~)w@av`2IBXBfIIjHBch5MYqlg`8efy@Kjek%s_AL(6gsB3jTc"
    "ej*e}HCKo^vQ-OsdAoKZ<6Sg+9LCF}K|N`X8be}^>kbWEA)qe)@+ohAHbiv{3qL+)ibX^%l}4GMT?{v-~LTufy&k"
    "aKapd7LN@eYhYqz&y}iEI-hyyjCbEBp@4luM$5tc(FVmFXT9tz^oYy(Y-AA1GGrrJ3RgA_~`Y){&?^8!T8Ukzj%k"
    "x`~A)7Pvg_0KaUQBOZH#wy?HSj?;n}}93`&m-q*F}6?BSgg4*Ob|3p)jr+)(VxE*VSkYV2Skjc^9z}cB4630vMP!"
    "Q+69$)Mg%)(*X?QXczyH#PBcD#AyYSeDnjM7{U^0m_e-39d={G(C`h=A8-zUx~{9UwBl?dG^_sTP#l$99ipjZ0~R"
    "g-+m!$RvmIN8*|wR5PSW5}UzfNuw7%^IKc?S>5Q~p??BX66d|Ym8(S?`l56!8Txd<?g~*C`t{PhFc$Q5!kq_^Yi@"
    "V@)kh($s?{U{=$}72K)q#HD_kiLt9?Mumj?BQrCDL5EPTlBBgyrt;hfHsu3X-M4jc_6{ARq)L;m;Ow;ve54flcHC"
    "{_oo!8ue5s)#muJzS)&kFZ48?i@BP6B)5RP$3FfF0>8A%I#4ND4aWYTX=N9G;JJSEZlB9G2i}d908`#tIw}WNVN%"
    "v32ee);<Pr>hs8nckyAikoe~>IQvj5<0e5c+f}<AiJ_K_cUx<H?U#5*~bRRt!qr9NTKDGVXQGg}zcs%c<zB5mn$D"
    "xu+2Hwy~`^NMiPNEZ0(w_BmKrX~gD~X(sgkcQZC_qw!l8HZ|b5p|c_eEIXse4AO-ES(bbjPX*wU=K-UFq5&AgC}l"
    "1GBzU)QQQ8rG6^3QB9+_uP$BlW{HyNp^F+Xtn#M`PkRu81y5eO#nA#oiyd4Nd;p$hT#v@~65<#ruL0VVa_41Pae9"
    "OhT2AH6v!n0tyrlPYECibXq5c^DG1TA^ZmUMgl8_y-zg-KF#Ql~N<{rL}#ye?u69yCnZz%=O?~*(|5Bd9tZt{6eo"
    "_ght-S%vee0kfTAA&uyc-8Kf>*YGuP<&qAiKjAz|AZTQGabsnb|FI<G&i@V!p?-;)@XQjn`G64V*J{CAgeCr9rL%"
    "2u4LAdQR_IU=(forNtGWjN_oBLZvN~p*u|UTJ5Bs75_}bXpZm$X1Krcn#q8Qv*UjPV&7j<8EVl&IfV~AimbcwOhF"
    "&I=L*{eN9|`~Wvea&3<pINsFsG$^7wO*WI`*w|dP}5oYlL_8yBPUq+O)Dmj*84YRo^D+c!@VW37g>)FgJBiu;iOF"
    "*)4pY1^c#sameln_P46XXqv@{DB@zZ$wV>VW~zDrgon7l9}yjFY4@Rc+hUI)(Xja|iucos4dsu#Vd;kUukZ>>wL?"
    "YQK1H}6aysqS_R1r4_RDe@Qu2MPPB7Km_Z(tH#c$~|Ok`1Wx4+-riVt%u>gtTht4K8*X{Vz8?Gb#6Pr(28zm*s9A"
    "HnSh*X(jPd?MW{;Q-`_{k?{9y3>7qif+oe_v8n77~c)P|M0t)@i*}@;;te09jYA+yo|h^e3{<d=)(UVUcAJZ34fe"
    "@cjxYkT|?|;^IqDKq|su&W#^Tg97+JN3zi&ioS2QiXX|_<6{O{Mn@AEl0K;2@b!faXbot$TfNjSnJTi&1>F<M_C6"
    "+o!!IEdiN=0;#lFcz1wih@~;`GO!?9dzecXJxHThfC_e{M{lvKqNEyjA(f!@5VvKJgX>?z(2*9497XW;AUw`Q0En"
    "5qrnfFdh~EFPGp~A{ae>jxjr!v$hT2L_#gMSmzSnx}aVqajsqSQ}5?iO5pRSwvdirdIP!K=GWLh9=mOc-x|JwiXx"
    "9--K(Y7`6an^&^``G{{OTc<m1fNN+;!dj9mS`Kuey`)E&(D69kdCI^_1-@A@@k7i3g4hc4XC-!pt+E!77HFLvj*X"
    "<zVme=w0ME9t49tk$0Tv6prOD|aslw1%l}u{mhucF%E|=R{sxt2N|0A+*P7tRKE+%@e-e@UGvz!T$)};Pn;Y+NGS"
    "q_Ltv%!5DpmbWgJ{V396oC3Ww~_&YLI-#@uayNR7UIUGUzj8j^)`8j;PJS%rD+`B>SrTstE#p?*x-)H;%Z{y%a5D"
    "(wOOTagKSk2=(dPfKyRLxn7BTN!$@hCNr+);E@O)tyelSF?5Z?1==6S;F4aVP1|#U*=Gf9{*(lcVE*Fo*8%d7QuJ"
    "asHmi`FkGc<K=NC%Gw&oP;k&YJ9UF0u&r{lGdXpR{~@v}sh+vR`jF9tx-D)By7-<Ricr&fu%qI!k^$Yj{&!0VY+c"
    "{L;pblB?~x7|iT*I>FPY{SvX!q+*<&|{KHxFQq(GClICDJwdF*E$`a8P6=l}ig@%cUf@Av$_-}C?er|17gIL4l*h"
    "!A>v8s%8wt~n^I&Jp6n0+cx;w{;P1?9F^ueJlmcp7CSs!*l9m0gT6JOCJdecb@%pdipw>Jpc6F^Yg1>{hWP};6VE"
    "S+xr^!wv8mszk+e@Ig<`a%Z{DR(BWhii^=Flmb{Xj?40EzArg|XrT`5=TG4v-->=@P>Mx){$&P1s_nv!Zo|qzmMx"
    "(pBzU!^&NkMG>icCUMEK>%dS|s{t)-{&i3>ddIEpC$LOUSh+Md!ob-m6}+_bNGBt><~!(`z+c*&UE71s@orj;B>J"
    "oeB=_bef!Ivve&j8!(HLLZ_i^;$W516N|I_+{}j;<S<L;mDQH$e&54l0mX*6v`>#aI{0OLINtqvf3p4ac>j1p)1C"
    "adeKelDIozX3oUEU2J$YsxK-XWaR#(Hvk9kTLC8^&}o;>^J87<{>I%ps9)xpj<I^5r`v*H{2p+mP129xo>j$ghxJ"
    "~*^D5aFd&cf-i)Cvsz#++>G#?KDf5+44nlx=4%n%v}B5<h9xZ=}5mkLoz*HTmePqMSP@95UNL@oauBMD&Qe^$eB)"
    "G;OdjYN9-rg(KiiM(FHF_&r=*dzdhh-M$>^pd_&|}oHN{2j*bw;7X})xZmLg52X1wuV(%y1IfO5@iL}yPf~fZtU~"
    "24f;jix3%gwFle;<Er)5*%Z6$o;?|I6;-!Tu{E&<5PGQv%R)r2XJo6Cr*XAC5b&2~W};7yC8%^ddXspSIuZ9oy-!"
    "gls+dAXjq-Ka4NugN>gizijXA?o5u3w~yZ(G4|7gTGU9J8pt1B^k{PQ*Q4X{tI6xbgIBMQ{k8jRx#r}aG&wS~q82"
    "c~om1Fo(ugC$mD;P1eHky38~R3ov*gs_h#J99CL-GsoVcQWH0b(e0_=6%4t9+MB5oghu)p_LV0qE$A&w866*5TC&"
    "=TPcFcfh^@yHL$C1ai^XAo;7mv{tiyUES5Op}OV5-iYvc<b{YZh3miLmij_&$vqq3U*g?>117S3Dq$9o;!v~OQ~Y"
    "LSp2D%c$b@2T*Kt;_i$1c=YIl``a{z?WHM59!TdxnyB#DyahEEjrDR23T}UHd9U6?%JXz<I{A~>spVf^S95K~&Mf"
    "z@bRV8vs%DpbqQ}_z#1?bjt3DM8e_=WUF-N0zM^K}5<SCxwo*+gVyhm9KVJjS6p4@K?kfYVIY%SBH_K`%+K@(Gl6"
    "mxAO0c6tmVK;~qAHS$0<RcF29>?idD*c~jzi+u9nv9&TpMg?ZYy&j|t!Bdvu55Lcyb)FIq5%2&D!QtuBSPi?L_j;"
    "h?OT0Zn%&M)mLp^$Bnn6UPS63u};VHQ*eC1&iZ;W5n1i<G<uD&9DoY>R~OlPQgi0C78Qzd69glz$6zB9&^BA%3z9"
    "xGlZsE6Xbr1RvD=VAK+K9o2RH~*!hx-OP7zyivmD8I=KK$5uujKp@NC!p8TH`vXIBz_0y?eFIGXJ9fs_cr<G<Rj*"
    "B``Ovv@gHjl2$75Ge(@&6)8Ea1eQ!_G+n9sXP<%WW?6T#DPv}w;9l`hUiPy5*i)C;7&b4|%+W!MiGt<Z<?lUujYX"
    "YOR<0OcJFEh!T_G%)H%r9in)4o)S86P4~lRQ_r;9+vMD8UqJjV?~h`Hd)M{0NA9J=XM%4$WWcXXj_ew-VZcJkKs!"
    "`J$79@3+*kbV2qgocR2JmnFlS_K(@k162nJ*}4Od#cO@*dU)zv`q5Gz?5%C8DXPH91IBefdGI6Ke91?>^VV^8GFb"
    "{1#x)zoJ2>Dg%uO0z0SF+J5RNzpeZIcDsybM;F3B)xb)9a{Ol_q9+EpLvzkx)QngX8EKoma&*Pm`}HE?AWthu9`B"
    "d!Z$av5zfl(RT&9yXwT5YL#FZgx@TGi>R*w_ESt+C9eKP(68vB&E_)WcC}k$q||rCD%c)*Wk|44ZtEGrb|)_Gj_7"
    "!`oUSw@q~{E70NeD{p!}fhV+mlZF4kA`cK$nOapvBl5ZO;^a~uJ;UX<rNyn>@>CI7mUp==E;&WI-QpB9z%HxOp+^"
    ">@-?>4{?VuT0^;pUlLr<G<r#ir$RiMh`f%4|sjhqZfg{|@z5-hsT%aiacWcu?6HUV9V$xsyYWXFeg$!S4N$DfMR>"
    "K}M^=j^oTL@VHx;*5R!dza}QRxSWXo(eZnl^!$D%z2s4y(b2{}dAc>&;??oo41vPGUA<>*UbGfQ?UkKcmA*OS-QH"
    "v6i^V>y4x)E$H{q#Kw7xdJ(bTDmghVjT?{q3BcsjNEm1jL@F!S>LJV#Ii)h(4M8puwmR+I9*>E6*$$dm!Ls%7EMR"
    "WlOf8a?sOf(E(XigB88JIL~k1Bb#a_&%3?ZFV|_JK9VBlHKShO9|b`9OR*~St@IyYkUbA=%ckTJg-GfGm?d=eH&4"
    "g(672&sFbR1DtdphLoA0JCZUJows*%U>nd9|#t8dY@+u6uzZ5>9(-i&R;}YzY9JnsqZR(L8g?O~zGQKEd*U1Gi!P"
    "gTNQyTHJAsh8tco&OXp?+b*>k!xn?9m(fr>s*~%JCfTBj6^(Au(@md69(PCd~x${Y`Z<A?7+6g{(II9yMo6x7+CZ"
    "fM&b%2}zP1J!UG>#~+XpKr34=Zw8k%nRjioku{1HI6}mBW?gn?&+x)>>APszqQ~ClvK}jl-Ikg~e}0pcj{0)K&X|"
    "chVkXSRlz`LfVuF6D#f%tO733Pku-8~$fhsk6TYqczF^xJBx%F0BeNSD@)wA86Pj3=6|FG(BhtIdR-rb|wg_Ajs7"
    "5V&!AIDCCW;DlgaADWO#5uvGJF6&{>WzYL%4lYH5V?}I@7HS~N$I8`YsLSS7uUB{xtKiWC@o78tq&fspyEuej{d@"
    "nH-Q$*$DTy~oHl**v<pp*7`9|CtygFL@9Z)4A7Cju_6+yJ4?ouFeJHbY9z)k)`oZnP*wbTfq@~6RjD~PYI$o#NGA"
    "nEcWz-|rUaMkEZqojVu@7+X+b2WLA>;%cldHg|0^gVkgt`cH5InddHJa9rKRw}5=neUYZt|z(iOWX}-(yQ&fi9m+"
    ";H{cWB<ekx;Eg5|S})U73(DjXi=gpu`KrTj+`U)u)#fSrG1r|;99*;j`7Alhe}m1&J&%#0?;KCRukB@f*y+D}T&N"
    "}bV?2}f2Do*Pq-&P@C47>f>tXytPPzZ@X!{2_+8ovV*YL0X<NabHWz0I<X%V)VeV=Gm`GU_m!@7EZ?2gcA$w&e&#"
    "Ik8xR&fi=-147icio|oTdk#5yeDc5eU>ItGPDKiRWWEa9p7U3BvtvVWLsmp+k09$Rxs`=*a;oefl<UVqeCH7sAh>"
    "~d!)-CI8vU)Gd0B4Uvf$fzJ)GEb|_qLwjtlLoYQmt(V!PE={BOsY)Kc<^!Rd{?*U3C6DvUkX5g23L{-0Udyz}jME"
    "HF=yEkx)f-VZDlUYVv@KN^vzXJM+3aFo)^E*}2I8Xd9teLJCwZ`nP@eMKTl++)DCFt68!5-y6OTA9tXURF8Y5YXH"
    "M%xc;z;lcb7q956H0y0NL0jd;Y?WMQ;4FD{DaUVGt*^4>hg^zPrz9-+bechMFqdnRIp-%uXF4xu)noaydi-?j>9_"
    "r@@A^;w*vFClW1nB6pI7}VBUVDQ>f;H0?*W(dDY)$P$bChx@xIfYYW9f`K_v-_bRyPB^Vh_bbOAkBJ&m~Fq>w9W_"
    "txHMkQ~$7km`=q7WQ0f8sM^ApI;;?>CxpfJ6|FK!Vk$9(1hO{u1lnFA@Ks)F6PBFqXsrDr>yAjCxyL$+uB{Bu0m&"
    "*a@e?BFX?i_ZgLrd2<3`*>7dAJ8+--Txif{0#j87^`?6^_nU?^7kwlr~8aX53mK`1ZdQuoIxyL<Cm!fo^6i*NvXQ"
    "UTl`{@F#{HA}#-K&(;t}n1sv$QCS9QlEM6Fj7*<O^xTK7DyO-aa1hO!21dGF}ox?Ozl3DzE8&P_Y|jMo;2bmMkOg"
    "0d`9Kt0;Le?YqhIO4x5q6XKV5kDqE?CN0=g4)V{F`FhEAf=pe42shb^hDjG|ZkLJ#S2-;6Y?dQl-b>2GoTEkgvch"
    "2B3=-@0W+DLHUM+H*B74fNFEW;1Oc&rhSTkfOKBK!=<t1JcF_uM|U+Pfi)y;#YFazr#`MI=HJ#TeIfbj8`q<$gya"
    "+#mgCep!s4|+K8GJXa!n9>JM{1mC7&j+ldYZfr8WFQO`%$P_xO1R26kyz)Hr&DCtcq@L+oZFA3XBmCNMI*fQKI%{"
    "LE%t?ZW|{<j|1??Rq`--Et14N)peyO~$eF4%{Qki)D+5EH9V9sS^AbF@pPm;bACpdM=r*UFP2FNkNasTPA9fAzpk"
    "*iVmp#-|D;%_+Aa7otb@Z1vLgdLQX`R<5>#7$XJWznLTpAq!t;CYj)$D?_JsJ);t?YlDF0(Uw4?4ONKUb@ITwxZ("
    "bLsAiECG48_6L1P?|Hf=eG9QdePCKO&pCYwnji&0UaF76x%`Y)g<q@WjB9WuB5FFvrxp9QC4k%gjO1FrYE~|^s)@"
    "w%>C9aEk{-gHMOy)8?N>eNIIdTotw}VMrE_#Z7tPbf7lAIQxlr6$^FANJrsySyB?Qh9(^AW5LAITa>7S%~wUqyrh"
    "<`_kG>E+YYIwOj9w27aG=Tf?krGM=WCU$<#_9XuL$+D@%XGQSGu7&a`0wHvbi}hckK|aB0a-8{jA^Zy`=UU3zBf%"
    "F&|m+&eVlZ-Wx2I4aXj!2^qts1<_}Ivt-3FgX=GmMqAcv#YD&@r{!DY}q?Y6Myd#PF2x8F{iQ1Ww3EhFe#%7&#Qo"
    "uiQed#6E<n^g9(dZj=p>D243XfoT6Xsg~&h&$vWM^k|R3@m28G2pbZE+C(2YUD!t2nY}=<Qp7&kRw%>jku<W4*f<"
    "(Xmw7C9O1|I4B*mN??bwB+UDj8SzmqMJJ4K^I%0~oZ5o2T`X@-W}uNG{~_pYH~g`BFFhggC7opcq-x1Tbuk^Ryf&"
    "lPmys<7OPcq*id_k<7m_YUcDejKO3aJiZ=D183Ky%H#CN+5HvOGBVMpsfCTDygZbp2ybtwHajIwOGx^ey`T}tQNj"
    "E>**DDh=Yj0;Zx;+a~M3lrXCz!Q0zO*rN3i*;d@tcj{$9b%BffFttU*a%b_4(M;pN@%FViB?^dq0R=SuA#TdP}}B"
    "4&YxD&0Z0S!%c=4c8fs#Os-|?Xufe~B2eE#n{DsW}<|8Sd0`_TVKzglkxR={UqRB~-iZN20&cq-Ydb=DVc@o>EmD"
    "NgU2EbZSjkjN(opI5VU~$SoKa+kv#E-hvT$i#Up0i%elPqV2bcsH|p~7i&ZR3ASi%R6aq>e90(Z+i&<#eZR0gbN9"
    ";}@*_;7l4fKP!-c5QSPubR&h<>|tVUoM9k)&J{FUFByU~mE1IF5+C%<SYKfVku`K2NE-`v$#MqzA@-Bu?~lfN<Cn"
    "+m=+Z+?ug>-Y2yM_p5MtuM;t>1B6J<mzX5YkmF!o34%05eH8S<n#tvE*+LbATH#^J)`QJmf+(}~rQ&H0$BN@?DSR"
    "Ug(Rv_MVF#Z$CD-yi=fL-CH7{fE*O$}<sE*YK9m=^M;X2V`ItqM&KTd=_+?l;ulM9nM}kqGtW{)8WA@Ye`j!AFMr"
    "jus8ncIQd@(yZcG(OeNRz<Y1rvI}qVJvEl$e+(ACCjrQxG$A@EFPQUCPCHrso_L76co$+Dv<6kkg@;bNSpzh!-Bg"
    "L9zV<&XWC{I!Mne}n93yns23*Q!A431%@bewy@P1AC{^Eg>!pEmI^VYKpOEt+tu<hvMuyAL(CSldR7!W#MA=C)EM"
    "+uWaIZ^OnyQ|jcsSVgPr%=F@3iGPI)KP%FRaflHD<5s;@B`<B{elBVHf+W;cSA>=CIh(9{%hC3$v9}a2aIgA;crY"
    "-73a*rS!g&CuHq>%XU36a@u%`wWuZ#R|>uiH?_&XdiJrD9lkNEIF^cnsb=q_ZadiSN(x!zPeK^qV27dxlkK;nlWu"
    "2*6NnsqR;j35U(GK70W>>S;n4g_ML_VQ6S!2vK$%<7oM=_L>U2P-Mr!!8Z<P*2Ch+6bS{RNURQ(B_@oCfH*zoo0O"
    "3T*-SMxx}%O4bO2jQ5(TWH4E9D{NOV^-G6p8(OjA=uUAv)O_F>Vt&}qYr_K}}>w#rAQ^mPhiB${Fq3U;Z6fbiw6N"
    "80!4)2Fm=Jv``Me3=5+f+yrnEDhjMGghY<szB-_Bi~#5A{8#p50YVXelOgO6jtxWeMxbXct!XRM>2_e&tQr7tC*h"
    "0!#RknH89b6`crpZw~gyzMiEanyKv41S>P`(t>iay|*q}S$PVp3qEMBK)TFQ)ih;~40$d@{$J!*&=(X^35+){h#y"
    "rbnulXUbj`qzV4PhB?yf&uaOM~i8?mHG2P#)391Z-7l{0%ga?UQ-)k?L&1hDfl2KlM{F;|s?7o<w$uqy(^!;qTl$"
    "9j8zCu9q@lRLPAp<Ct*KLnxGj{UJ`PL8qyqUlbvZa_|)oK?=Q#F=skX+FH0J6e#}6YHdj9j$Y=J&oyaj@nRp3?DT"
    "lrZ*4SGBX@u%FgQ3boM^#J&~Bn?*7s4&e#Ph*vPpey7i5M&QJX}f9yXYDmlx4>*7q1#lfyc*2W=<SqZfwm+R$BlJ"
    "{JLMv?%I(%fDU>0K??;(r!^(^S<AFQ%cY1RZ??1i)&AH$%437+lIDIN2o33@dEbu<LJ6m^SYodH0q`mPW&xcYc#*"
    "u!b|mszsIoK7dP}7iDQb({vGMFBkdjCU_5Z5U_G-?RU5a-f9Xdw_`Q>rYa4-mRPiCuzpelKg3`*O+?i~Y~19Gxb+"
    "(_3QFTKJ)^}b)@QdqH=aj}PasII2;~lPt|Owzn@tFwRKGnA$`-tCGw;VPb4%j}w-RlA5Y%hc+U7vqM<XQdT6bi?&"
    "{PTt+qW(^dGLMOAk@1e5GT)zwTCxk{T8F<5eZh*UU-kCbPMMZi@zG~#XTpsg5JUAX@U|pXZ}^<^P{0MqdG4_YD&-"
    "Ww+8<+nt*1rTo(<l*Guxt%WR&r)!qOg$*qLv7_vVYIAG3PiMNbr!^tP?Ft{xFP^RZ)(qU$2;yZxF0dwqzW5LmKk$"
    "%Wb&X3W1z+xM0kp5Y7cr2ZXcf=nHJl^qOaPbs#wLTa$dhzmqPy}PRGW%crv8>6eZeaGluv-G*R4%#@-;v3uhcPD^"
    "Mb@6vrH5O?2??GW78lyjWeR%_NnNljt-L&Wz$y@(o6C%MBdFS5DISgRJ`PFKgb<r7B=_l(bSNxQ{X*!S*by(kD9l"
    "VV;G&C_Dl!eKim;2i88zC(J8H~pRQo%2$C^t}%&7V&kP{;pyP(F%9O{AlE;c#>9`0HR%|y~9%Y}Cp{)$}Wf+Oxy_"
    "gzcFl-l+7tCn2NdQ8QeROHv_AVzx{X_T>3*Vf?5I2Uce^2>I7O3gIXAS4`^9G(0O9;(p~kB4&VBWEYx%`>$p*JAZ"
    "wULZQTzzR|UOce2kvERMM@<;|S%fR8iK+O0{h(^+GqN<HH>szdTkKQzB(VM!VZQ-ew9T$|Tx_*SE@i^7r@3Pc;ey"
    "q%&%rJ<6>%@j^X=o9AJWhDHE@}}5U;jK)Hmn|)7fA`_{~DQanm6lgY86JlTRkuOXN&y&Vx^9C#D5@ZPH!N2AfnzN"
    "$@Du$3Zg%)uV8xkW`BQo|K}+lE>A+JTr<LW?dGB<h%i*+rS9Z3vF-)%&0wFKq$nNUe^&{NoR>C*%;E3CQK8`o#PI"
    "ryEaghsd18GJefTXdiCyTMXc87Lmq|b4Bon&J1eftVb*!FdHbja$vKRW&8Wa-rk`HA*uSl{MYs3=QODjIv->@`%s"
    "$k9e5;~8T-imm?w%!^R`&QvFz``m#r!Nm)z1|zc8{JE$Kkgm8{0o1?xYM7uclY3?W`Bu+ywtoV*_`urhQtu{)C#D"
    "jKw6r;6Gvfz$We(9(=4!`LVv~cZp0mdJW=OM6UrKm>RBHH-Q!3ZNDD`F?t?n&d_8Dg=TvG_)ygH@95?;O^B+h_dg"
    "Hl;qp-wK7_=hwb8*bfS{Sh-9K*nlt=}J->!)Y_ORS&C3HXOCoLdU$9SkF>)cS%INwPbXi^LYFR_dP?sx;P;>4JHN"
    "<p~><hQ=bgWeZh>rlHRAM~Wjo1)?c?YI_sg7ccP7;MOW7HAy<UUyZ$@g&8!yXzt2}3~4w%Z^CjtO&Cb$^CcL`92w"
    "^e^l;?d0ZGAR{)_w-<lfC~DFd07&-zckKp`KLNH)@3c3UPmQuyyYTddO73^wGD_$7Kf`u7(|cvD}hc?(utCW;*$y"
    "#eAvuP!nnCNdc&MR|R%l)6_^0@O1fm@P2$<9PdM43rbI8XY=XYZOmBc}~0iYWrU&hvVbJ-7zo_zwxvl#0Dn3hm#c"
    "@zM?Zf9oiMD6%HspEYoY-gp%)ZDF_@txGZTian@mQ7j(*#sZl1bZl)Xwuv&%0Y)u+3LbgT*{<^z=WD`JaAm2Q9fB"
    "dTf1WH^c$cnUGtRD($m{DJ`s>{ULgRKv@fPm=d#cZ*j8<U(7dhz&wTURU7)~I4bmL7VZaU4zF6-aoTK5@<mRNbx^"
    "Sbo=Fi@G>+Q|E@iKA^r8e&u{lVxwd<39MMZye7M?zb?##Z9y8HuMWSvB%;eu$kt^A3L<8x7n?U>pN;McXPd~{0(R"
    "0yXUoOhM!P-N051SP)0>~~TSn^2f)z?{odHm(n(}oaOC3=Vq=^&J-reB}+V}C{ak9IAd?3RKVOAZsG7Vpxq^oZ7%"
    "l6(IqUg>;<w!!v4|~ZEVV{eRp0Q4MkXNPT(R8{lBw<|QShbPPb+AZjt2u~ia@n%wN-qkW-Zk(UzQQvsDi|~Vsp|1"
    "p*-{OLC0W&1Iok_jO^DY){{m`~3kd45*Yju&z{q^>ts@!w!eD^?Q=NysQRMj}`Nx+QW-255$-m2ZN914c7ff=5|M"
    "bEty=>(3sR12YzwFL#<j8JMxFIZGh+sSmwc|Ut>8GnTuo_%tq(viR!E8Reimwir3ZFdC$}PXmg2%jRyFB!<K_yxn"
    "&{Xod2SsLm3A$GJ$^LO0Y5@l2%8|u&mR$+KRyFdVr5Jh=E+mjhX_qbg3e#c0Om4(lGagfIe0RCl9(Pg@@IoyC9BJ"
    "`s8aB&}DTtq#J+bcGP8TYbJJY^iC0QIB&%5?LdA&o5t%Z0?j>gAopN^7;L^gD!MMm+2@a}K+_IgRZU6e^d(rVX9m"
    "7(abmcP(5BhokCp!5X>?M4QbP^q2I%f0R0SIB)pSX4$3#$6FD11Ri=$QZb`^MjXt<+c@3({4iI`OP6)z<E<C?-X!"
    "#c&y9a?R^ZA6B-#8!WB^Uah3}rUd1+irxAFqWi!<B9NZYVz#FmN`K0|o$)eE-xC+me8?1JEyr;Mmlo!PEYe$ELdy"
    "qrHheO$r1%?aHp4Gvyux5mAkco8fD7&BBXUSOzDA;f&9wDjSeUe~Gl7_Zw^2<!B9z~BN)VvCK-e;%j@#54LSKr6;"
    "bv0Hy6i^yThYym~F;clI*vG1nF?bD(t3kp*x(GcL^L~}BdTJ7;^FBZ_rO&ce8lggoTEl6oihw;;CNdZdlG=&I;!Z"
    "`s;mnGZ-PXoDbR|YeMoYG7rs;A4`Zg`yIJCr!)9Bw?DuRW3%}5J9&yBP!xz;K{w!lyeA`R^TrJf5DX++3F`IB!Eh"
    "YtTuI$ur)XKvH}ZkQF%wKiOIEcx-#BQdhTA(3V>c=U)DND|1%;B^{i!YJDPS-zaDF+EkP)yYIQrxkd{ffI;trA?k"
    "?c#@SXk35eYQ$;BIIF(YI&+Ed*^|Y!4h{0TTgI%)vB(den>6OVvI=5P^>dGbq2QjRbu^0Rd9KB9Ggfw5|)O4mXzD"
    "*op+GXdr52mLj8Qz?R5@$=3NAsN1$cxn@H7g{<$kOzJ!QBz6!*P@T8Y#oD=(a9$;J^C{wy4(&B&_~su4=0V{<c3y"
    "t=lqPa8dBHzOk`T`s-3OK@{S$G&&>JiOH9t0NCcTE>h8Mc{4~-(RPV5_#r!M=nR(HELX<jAD=zsRE+DP7=QtV@iK"
    "AcrrBGiUL4#<p!7$|kR6RaBJz5aI9sSA@k-AmVKWd8%3KStSrQ@aYf?OXn$e71w&ARS6uvLcsp$cXRD0-EYQr2k^"
    "XzoJ>YGWOak`IjOOw`LT2s~V6YdM6cBv&3p%N^ov7n3#k$umK?TEb3eB})4>(nULFfK9glmyK9=+CwHpoKxsoAzw"
    "9Yr}kRk#S%^7DeeVy^>sLRsXK&zDy4hy8C51EWO;+Ky7ro)5p^}x;xWfD*$V#M_-r;rqA*9+%zz-PV69=D7_Y$Do"
    "|^_`Yx(~FKax^+VnaSaNE=p@Z^DVFq6R7Y?y~ef8yAx*oQ9h1$7e4tJn?JN2C3ISP_X__lDc&5GPsK@uD3N8+Kex"
    "cms3Lt%6~_PM!F}$no6!t;+=~XW5*=%+0;GCP7kUYij4^Jby$A9~;lVZLZPv1)2|TYdy0>7*O)QKGhYg9B|4!gPt"
    "gS&*CXa|BMaID(ib73k+rNuGW<=`R$lBu`$Dq73~{t+^B-HF0Ha?x5QiBdx=ob>TH#zc1t&1?8&c&OXAKu{*<uWk"
    ">9WB4fhOICr0@`7L>`?3HuyB-6}M0mYC}?NY~+dNrONXRJ{_1WURPxuCfzGF3ky|8=k=q7g7YwPKJa6p**}>%ULP"
    "U_%Bf+SvS=z7Ft#D+Ubq{c&E#>AmxMC{yK3wgQgX+tdR|+(}FiuH(f40U6=z01FB|>on#7y&$2E)z~@EhlbkIt#m"
    "2hXey`1K?2x+M{X?&5@$$D#i&tgS#O(JE_p{eGe2AIy)?kW_-mvh0=z3<qdj<DPFlqj})gQpwlw!wq0!stNediOr"
    "1>I&--dI<1V**CtQ|)+xkDz%Wzg6!aJGK+#kss@Bjdm2*TIW}B(bAdc5CbIOlTYpxY*Yi~oV+(Uy>0z_S#TI7K~3"
    "2qu~^9{kfRCTdL>;=g6PBGF2KEws)UkVny;jh94odw{>@-xKVRPk6J8Z@$ViN{?&Q=csFg<aizS!(DaVie;Wh%)r"
    "_X5Da_*}Sh7J!ib?3P-s_LgWFBrc)c$cAvXy5AEc$F^Sn;1)GMQyTkuumd{b~Zn|F@YJ$I045Fban@=;4*Dm6aj<"
    "Lp!8~JFkjsaH7Cf*9QOBmnJAa(JX5&1tPiuT|DBEEN&-zh<YOz5ku%bF4xDzQlqT3W*I$v<B#IntQtN4eLqsnjv4"
    "duATv=0OR{0Vqh@`@j@SO%5Gqtg+!|}ho**zQ+|9zj`kg+rGLuPXRziHBm7oHI#AW_Puk>>rVqHFAJHA3I#SH=hu"
    "lbP3*Jw~URD<=k#q;}|jLXcmUT(2{;O@lSf=4`#t?F7z`P;54L;5Z_K9~uv#dM6Ys(3tE#E}*I2SErrbaHl6p6fa"
    "wXr&eC6h*^DG0Y4}5MDubRyyY=hf}g1+^z6+*bWjY!0_|{ZA<Bvr45RI!B+39FCLJA37ssZd3BfPOoUP0uVh|thp"
    "N8iGCkF-PY<!xUS#L+&76_oB96XQCQFuyb-i+pyAyx}~vWlDivLeMn4Foz0Mj<ke0%I?W@@3Ng0>w)}>F2!Rf+S#"
    ")!#EKvh>}XjADXCIci<76^D^hti_>`HrDG~-Dg8_EdtOTFUEOfL(^9-I55@a^ylonlKI{2(vl6vB()d-SX~9KzmL"
    ")rzGG<o4Qie2($l2KSZDzNauMAreGvYqQR{3K3G$g{q&ktDf?e|heYw9|G7i!|WN3|k7RyciW@8o-M=igNw3mp;e"
    "bcjL)iVEc>^q`M!(tlXqwy9X`g#gcIvY^Au#+ei{l!k$;c~(mtsjs&T=%N=j>&!9<JjJmh`1%SCt`1s8e5t*j=<T"
    "F<BMlsI!iM3&OD$Ivryj&kd`N5lDxN{eg}#J3iV_qHzh@YeroGJE3vp-6$N)1U?Q@{-)O5%Rzxlnj+E9<hyR|5>w"
    "TR;H*FlZG>3c(?D(BEVtW}LX<ZN5p?2u-ZBQJ_+><Sx~bARKV@%B!Vo+0VTOX#wC<1f%Rb9IrZ)CA0gS)7^{<up%"
    "U!%&B{r)KDuLW|()O6!)M=hZB~0!lWw4)<KdLqos@YodRdUD96LxlFZO%M^lH$*dHG;#78zt0o~8(GE6-WG2sVMV"
    "cP87&*|KCnCjSg4R=49hvv3!@6_yFMBx`Yoj%7A3%#q#&s#H8+x4TCB%w}HLL_Irf^XVlLn+3J7W4%2~4nl)vs9u"
    "%4N8*mj$&Euab6hs?1VJ`C3?r1}r!x`pQ<c_awmJ=d@Rn4Co-O8OMVN;X_`o0YM~zNmC*xNvo_B1KK3H%;$YhE1N"
    "01Nez0C^8_2TD3w~oEw3+bSfwdHL_PhM46Pd|N{tccHl2uxNR#h<Y9BUTt^T!NljLc2#wPV_q~?{iER(>Q+UiJcy"
    "Rhy73$N1Bd&j&qq-Fau0GRlm?*h)SVPua*x$(7z-yp+FTfsXsrh3M-Lc!F3qDJ+@m-S3qHSCA`C|UPWs9HT?8?xk"
    "EDDlp#;!e7zUhgf9MN-%>c<`FBB1A=jXdfZ!^sDiNf27mz%z6Djx;c{aC^#wQ90F8~MRn_mKvFpsSG8F7;E46*i1"
    "~6zc&XH(w<%qm5gZYb*3BRo*Ra(Y5gLl2;FEl4ol@EG76{G+OI^<W;1^=n84ZhmmOjWiNr65n{R>txKjjxz0wp`R"
    "HAT~k2>(a#>$Y^wwmON5-V7PXz|sxMB^a$DCw8^!Um;AzHFDg~2ta_EEo7U6E)yOASQZ!D*Fix>eqYIi{!v!;<1("
    "`!s|<j`<<kAAI0*L7^{HAJ_ABHloaGBQ2r^UcAM15K-_&qPJWMxNYz`<kaB+iKf0Y~quY@T3_3lpcL|a|3f+24p-"
    "BOZm&&qm3S3;@b$_qg`k%kCgiC2i!QX+aWJ>*<MMSa0QX!us1q{T-oRhR94aBfER9u*3cmk0aD+q?VY!^!Uc?y&("
    "TePCC&haL(&T?_3pOP>d}0D^e%*U}8sYpZ~^DYJ`wfy>KkaE40vF1I_-Z%2vg!%II`*$RmHB%0rvek=NIKJR?{{P"
    "{P}f^XN&Zy{Yj@bg`#r(W{5{>90I{;n81{U7Du$9X}Ek8WP)%k+|nTw1<d(=4;*i++zbbvo}e0JU6z^ljTln#gsU"
    "udZl*{F*&{^XvY>pSSmac5WzhqsP|ivI(}XvxN(E7#WrhAOFNXkWEiUET2fXCnm2SQ)Q%>&emWH-`X*sJjFt}v*w"
    "}K@Hy=mFkB!9NT5IsBr@`bpkn>|QP1?&V1v(HSdnr~Oi>#I>CNQ~h4(n{k(6$8BiGMXu9@s}lLM6gmS3((HPFsOI"
    "VY4p`F8Tnch7oSSR8GKtRV}QotnU=c=N`2%L$EWb287D%p-1IAWff>ZvlJ<mmG)6%B!5!eV329!|z1vLQ+gD7G8>"
    "kvHis*cyJ`$uTzo+bdQh7$#u9kuMAzM%!qUz-y<btL9(wDOfG3}i|nf9X-=|sc_X~jQFFz(`D%_FNavOuS$!+zHr"
    "PJTqSMnaEl}aP&*oLU%9SXG8>#()Tcwp%PSVh{GhUL$M5ZL0j-KwZJbCao?f?6B|9@}w|M<>*96TOQ`tKh7b9IqE"
    "eg5r`emKJgAHRKei}T0J7s<|OwswAfaJ2icVkENbYLU<KRVagWGg21$g5BWvyY2)Xcj)|lZ^#b;>TkBZ(<ra{<mq"
    "?#B_u_B^X>D=^KZTlu1C)oIY72uIk$gdg1y%Z<^}s_5|z1<IO%%0%BVF%20PpnU;Ly<3&l~gGqA6Y+35h7mCS$hb"
    "j&p5sGz(Qi0-F1z=FZdyor6tm!*&u_n&{gef($WAK=RJn|hn%2j+>yfFMM>t`EJ>E&_!ZJK_B}VD~}=YmA0m7x{0"
    ")$B!S=Y@$kX@JcIWuCQZ5-^~I7r$B^A7v_9d_-7Tg&-LE&?a6~36sI3f{(NwBOyUArer|&Hbf0psF{@^viq|7S8b"
    "xkW-y0tU`6%gM_IR+g>WjT2SA6_9*qz_rhRvmB=@sivLZ<Z|erVg&_)gjvkDm8}_Z)fe8kQ>M+W&AM`XlJ6K}6aM"
    "7F~S(LrF(O7dI=;;Y?c>w&mdquh`tyA$J4Ga(62^>B>FYD*tI~t6OgdX%&1!0)1j=U$aU#z<-`~x`T`CH_y;Fy5Q"
    "g#5o5659V0C|q?euLzl~skl#4#2tK`d(&Xk=3R<ih{jDrLvfg4BJE4timi+}X39PaeiLy@##2Fl%o^pxZSya?TU("
    "7PdXotacUn(Rjr2DE4-FZ$(`bPf{8X=u$p?d!8;3(P7%e_pGY!D?Z}d$xx-4d1Ez3m*82Rxhcu%!YsA0a%>2AnO?"
    ">7`aRvtIDV~oi;jref@ZGMrM~D+t+DcrH#<Q3adyTZ+qG7g4U|DRX%(6%<tSSDVo6It9xK9KWK!{)<xvoet&rx{h"
    "FuU$PY~~v)Hv_wT%jiDKsxwl;?f+J}=v^;Sv@oThU*MG0_O+)o6Qful0g-J|7*8kIC%0>yn>aE+JXIrj5A)G1FA("
    "&93`bG;!KTiEuP;Aw9k7vosrh^E7&exGP5AK8vsP>}p+&o<x6ItCVc~RFohs8glsUv-q7At@YM$EBfP#*i!6e9@7"
    "uQ<#n$p!At3%ir8AO%KtqZeHWkmrT7VIznl?QySd6nr*Je>%h~8-gY(?>=GAH>c2f^q-@5a3FqSvm9$?hcsZ|7A<"
    "IdgKa6iUgTdX(#^f&R5Rm<~F|GS{uBgYy8S^XN&ew{k#6QA711LlyS{=K`e$IXkf<<h_C=y>Pg&2jUxyzsBu8UMm"
    "Hpy6e48SRxMFIN+kMOQQgvGqthUk}v;<=*CT@-CiUD;n>J)xLDOOmF-$OViv164qk(_%h6RCE^|f-IYzqc`S*oCu"
    "guDKKUd4Lj3RM=V1hjPJbeG<s4YA%TD;BJ#XysFXR2=$>Df=ryGHlV`l)CaTrG;Q#I8-P(F0Xa_DUJ+&c6d>(CJ6"
    "5$ru);N#s_<H_FctKH+6FVvQ(({X$s|9N(?F5Zia0%x|i6ayA_B}zk|ZT<1v?#9>Tp@yF|3w3poG77u^9;p3Uuv)"
    "ayG8rv3kHGzwjn3HXH^-l~aIk~TNHr3VJiJM~j&01f^)IhC96KGOc{70www~NUBdse`)4=abK;~Aw*psKv4EoOT!"
    "NJ}{khWrLr4U0Ndpefa7)v6_uyxDtpyIQf>m!Ga^*7JCDp3L40z<fY^@-fkVdCoC68sf|>IuHy(eBTG+1=agw!Z9"
    "Zxd{;R>#K1DhS6z!RaI}=9@+5c-S|a%3c0woQ@f_sglsV-St~R)<BW|;D1YywEzRq~D=u;3GOYdDm}PlC=~a98ua"
    "YU$+$m=hy2cwH;NjtDJg3g?ip7j>gg|D}y6Ve^6}!U+bvRZudxRTc^vO$mRhgwFg0-uhFII8^DNk7G(f|2~e{TPW"
    "w-3JUzE3|^b@TmxX8a~|{);a@1Ayvcc_zP9w43jhase20)rm>7x?8|lrL#qr7V9ew9%{Y|sRU>F^3rnvXsGI=k3t"
    "|`aP~22BS8OGSvmYFCYt>Hb{Gnfciq99JGcv+8TxNC`WOyOHyBS{GsA|mO1mxAlQg|<m`7^XPcBVF4v&agO33Eq#"
    "gL1<MbL|GH-zZ??!+!Ycg!K9pf50x<A)Z*%w+#b9%@gET9@-aH{`*sNV>7@Ro!Du3O?FX7~R=r18W7F!dz*1*<8l"
    "GRsolsNM=2^Uhi-9zwMi@5y+wMNy+u|A*VIECPA*T#VlO|+mo>fP?B4mbFWFo^fYrHdWnR3uO>j4lxUg@+36$?kN"
    "x(~d1ZhtWgJ->(_xKeC2EkfOyA<e6Ru7F&g)FwKPfcQ+qWx8Z)O`G!h`NQWk@eF<dtzV8||=-PSz4KK%3`;pN@Uv"
    "%yLPKM~g4%Bdgy!ED70$lOmQ@zsTn2+~mzm=Pqu%=)ANsw9TC*5-X*rOg)fxD=p2D95<%v7v`&=f&&SE2LD!DqiD"
    "*efgr^Rt`vV<K==fReUbiqj!FmIg7Q|4L2{@{VZ5~^g@s;I5c7#7+46I^eeu&las)UOOZJyx8aB7QKneK%_}3{-W"
    "m#$;s~Zo{q+>D%U2Al3c1aK;f|&}epdhw%T~`?1MDsFNrix0Aw&HGdrDgP?5KGY7NSb*y;W&|ct8W8J*UjZYMTfI"
    "Z!$}x9RCalNGl(hCKu+cwDXp5SAkDZHI~xhOQ`rU&w^ke5aA!Rc#Q=MywNU)io=vo5xI`p#U^}!fgeXCxbDZeOK)"
    "Gjf?2u-ngo=KOG==Ia)s8g^)tX8tZE2Lf?;T7@HuQK^*xw6-^=xejg7B#TR|-*SKMZZXsDS~CE$vW%k+!ig)f^0X"
    "Q6TIGG~0Zp0lA|L8ui%3snETVq0q&n8+KhLF2ZMCTf40?ZpOp8nyZrYS=I%%SdNW<_Gl(I2B(-;j08=-I4Ruu0NR"
    "WK`_qJ9AY~PRL=IU-PoWaN96SLcLO=xN9g3b&K#B1SeXd=j0qx>Af5AuqOv9Pq*;fm~kW5jr;Fq7dEwiaf7EC`cA"
    "(W{>`&)w0EWopLc*I-!?PKTw6Hl)E28%w*vgUN!)&<@H;9iY}zb~@0l`K2I#%FZ2jwPwol23$@-M^`I95&Njugve"
    "i0WhfJVtS9o%bh5*VhE?6Y`gjH?}IYBxpB8VxtAQ3*9t%Snccgiw<#Yp3pf9OeixIfb>=Ks%;i3=SIhO5YbG``N^"
    "&DTcFK84T9Q>~|Ds!{KEzy4&LMGIAp0<^elOub9gV_r@){*#oN8@O(wtN*G7GZw63rfsFaqCYjfIYBwZgx0oG=j7"
    "s4fdOJ%G?&r9m1QcndDW1iL^sVrGC}l*J>~XL8DlUk?rupWQn+l7uEtkjY6t-hKI($?pE~`0$tQy~)w|<-z_AKx3"
    "Z{w(ixF`Oq*qM>)f`p3(ZfFr0c*Zy|d7<vkvKWTjiAixl^JXPj5?LxZr+TyCC|-u-&1z^#(3@^WwYg*t)tOWKpF)"
    "`eSj24APL&8Ip0=3Z^56_Ji!Hh?kx%#o~+@o=&2P?62KcW-PTw$i+KIN@6P%M86FhPIKwOKvS;;ZrssR~}|nwo)C"
    "_sAUFx%p{i5F<|(#xI*e`sW&N}e3Xs$$@p(pp=G%)@SCQ;Aa}Eoyd|~sxC&;;lU2RT>A&1Ih^NK}^1`;Y*CjJ0wq"
    "bW3)cGXDr`GUy9=&-@hj91k;E<aDHOjJ`Dzg&WUfd|Kk8p`~A&_HS?IE37w7!k^)>Mi}7a?o;@lwSwo3u+`9XgSB"
    "xf68Kz)Jb3>@|qS1{+{fXt0YKNR&s-eoa;|+V6piIwO<XDMcTHagiWSf{0TSxlYD80h)m~fr?RSK2hc-_%sau225"
    "z~3mjSv$UB9e+AKftiXpkLF4q|qs#LIfY3$bOLOe#y_ZOwWPocr?-56HJan!hQxA;JuFXtXIVxjV*BufL9t+Z0Ru"
    ")ZoG!R9l)TyJpNB>?bQrHU2lDz&vXXjMq8kmN{9r~@s#&MdXO$dWOP-^FRILJwDu$Yhw`NnNl8Kgvj-L{Hm=udSQ"
    "TpvZ-uS$HuT`@iod(`tITx3Urw+Q0(UoPii?Z^^S+)|_}9#UmoHE`7w)AR;B!mga7^gOxw~eg_tV3L&<*vj&X<$Z"
    "bT}Tb0%v6<+$)R5j9xx72|WH{ImbuoxT+TflxO$?xI6#mi8f0UlV2NXrlToK1d{(0x%}ixVJSx<*4TSRl~|17pAz"
    "@Og>Geh!h6$6s=?q4uJIqHg+#FPN<J%Hk-pLjdcizE3Ts1Y5Mt19?WNF7rG<;5d0ef>2OAPZ`cnH)hx2oz&nfhGu"
    "6NXfZf^>ZlQ;Tg`UC=v3o*(qQ~k>p&B9X&<yBsPU}47VpMNXVZc$2kXW&npac8OzC)+SCxOXv{zAC_>My9R_GBxn"
    "{|(IezoS3G0ZBf>#*e6{)sFFZ9DC~j#NBV>KQT+k*GNu+JaMUZEw2TOU0@0rRbURTf>hy$Jz91G7~dqlDrz=CZyu"
    "sl=PpPK1Rk(mDXHm=1TR`vDs+K@b$ssa1k)L<c9P;^}|K=Bt6!{c79W9H*4Dn70KP(+>xfw%JN41Uq86yk?W|bik"
    "X{rv_VXL`h<o=&4x}An%q`3TfLu!v8n>U7ml%7svarShf;qY>#1m)^&wN&Y%WfH0|!gTWtq;j65JFP)CXhf$%@fA"
    "E=I6H0By-2Skr#?L+any)NUe5gHt{1+IkD;D-I4pQg!B_@d8VLk#Voo)Ca$t!x9P5-PR^dXz{qQU6X=rj4TPn!QT"
    "w;AK|t5noj+W?!XEN&`a%Ckh8tFGMN+&jlm3Ia<b5Ea}+5|wb!j-|BXK0#*(|ForIf-wy8{4jXbfrTuS@V9EWry6"
    "eKbjLo*GqE`B|aO3S5cvyS2gspEzs@zJ~hpinv??uq>^sF%F$Mhih{bf*BgTjS-zx7HIZ)fUDdEJw5ry2P?KbHrV"
    "UK@OJc5e~EjjU|m0V}FSczBxA8UN_8J3~GVzkm#uzy?qx?^YeYHIG9nVYe)a^?sMWt`;e71Po%34tj<lE$79J*uK"
    "1&ODDB+=yz5uVuhImWgT*<TFqdUdpeXA73w1^hH6oM)wZbZQnjSLIB94jTiEw4q^zl@HnWv9mk`j*)(>KbCN!BcZ"
    "m6k}!aT-i*s!!yEbEq0*?I6-@(Zk$|U@at+WkVh}{!pH}D>!!2rBMb-S1$VdZ-TrgYNMqUe<&l8MVp~zQ#6g0(X)"
    "tdNRvP-L#7#DzS#*6xLTq@(xgPQy`S=GQr%oG^5T6XgClUnckC^_q)9E!-iQrJs>x+OP68{GkRawB6*_Mv%_BQ`@"
    "UD@snpdVrbf+-_x2WreGRzF^IodnMvELk_>%9ItVgE>gdx#7SJNC3<L8vV^kUl;n0G2s%o`26+%E=<K8E!}qaSZ="
    "l)?3fs8(JcOm+)OMK$vh)5WS>Tt=ASbEqtqq%C%vfAEK4<Ev<pQ3b(ZG8@Us$qorB!CT+o9V<l3oBk-NcQulyvkO"
    "RF}YDR|WawR)C;hKf|@UXJ{gdv_?sN-Q38%!XUS6M$@qS^s6G_b6?feFIOBrBM)v8>K1tw<1C=W<I<d=ys0jG2E%"
    ")vERmiO%aRG4VkGj<#Z_=bj^BTM0o+7(4oo#(Z<$1lxSn+t)}kT|=c!G#^1y+NASZgI+uCY&F_odNcyXVEppn7di"
    "$zLBW)#J9}(Qd7B4}($K#@;XuQCS3Ajxe<I`O%{y5-!P?=8==GcFegpch<9tyl4ev?D^Aer3MZ2i&gr5gkmp!-tf"
    "Js|oY?*aSSM$J#X}(tLZ8m<$_iW-+5&R$jp1S~V^!AzsfDzh_08UbDUvo{<oG@v+li|L;L(01g>*Af&(u|ZQIJzf"
    "#op*hJwZHBAbPDd+ezr3BYli0r{wlV_T}INNt}KQ@^{L4T65mW#a00d$MpG^8vAAkNb&(zF>i_$vB&!f8;Y0Dlq|"
    "k}`vgttb$BFrp0yok^+IdF3NZ#v7y5GpNZ8>VtFpj-CDsi1DoT{h&8=u;lK|Xdp9E`oZGDWr%TNmr=9F2wc6;T;J"
    "w07iN4&<2E4tT^>6$7H*;A-$<`6FK(xM=>!DRZr7q81*o%)|x>ut%LwjskEDU}KKJ;kw{BFtCk5L1Y$iFZ3W;81)"
    "Pc*u#(_42j4Qy7EeZl$<NjIPuhJJdz;1x1<6MZyhEM+dbk5^m7cNKER+JT~gW16E_C!V`J5G3G7Il!!tBg%Q(D~^"
    "|kj_BKDS}aQ*z=qLFNg5r5f>5W7Fck^R=y?|ZN>F`|0Z<D{Npa$1(8pmA7>bQA~763v*2woXRa2R?8h`V@0N8}sG"
    "fZo)O82^WQ$u4m<WKI@_D(;DC_=U3^{>-;vwv^WE1in)8O@Iv}!j;26dv<;QXHLl41S4B%)3rKM>hBL_xQd2L})i"
    "VE$TQ>I;%@!$}40ypPO~^UaW8fO#(JzwKmt4?y3vpgK_rfZcfKM-~B4Xgb9m94^`E+3h=226#3sVbJVpVy1Te_6f"
    "Jrk2EVch_H-*PQQ{hg!TpK0Hc$H~jpa?yXuc&pRDWH+Z}x}5KtOQ#aqots`>5)*}4`(FBVl^l7&vTFLhBz%FDo2Z"
    "IJJlI4wy{cj1Nar!@;vnur6akn49`VXV@xl@VIu;qQdQ4R1A_c(b(sVFGTe#UuAjR_eoR&mDd6b}sp@-ZL&;W|cg"
    "ls@VDMO&LV+OuA_RExug>Oh{1wfGgYrC+;a)5>SxP}0AGL;3aK?_F7lICpSq=Pa78xIIlf`Qpu7X9;*h@%?`!SHU"
    "35~dw<SU5|7$uXN$=#04PfJWDs44~Abw<IlOsA&Owcx%ypdZHkqV06FDF+U5Z@Wi%D5X4n|9CQUOs*`^@uEdV0uU"
    "DIv0j&z^_7o0nWF?;`EhksVT${)x9r-7W<S_fHmwGWr_K<y~l9y|B>6L1idk|XY<TZ`_$bBsZq?=yn9<LNmUh5GX"
    "$9C9uL$j!I<M!0I(@#5z%yZZfb9HLsn>^WfM&_D3;8>Dj@(~rjGGJ5#=-!6i?K@Si0TS@ky)j}Elg98e-tQVADo$"
    "yLSW-KUWk%|{FsHR6&~0|ZphF*l082$P@pX}pHd`i8MQ@Xi4WPIytUh|%OT09RE_)@~D#upYA_El51USk?)&zJw)"
    "(afJ)+>j$LW3rrzGo;NEp$jPPxJFN5t7CC{vsfc%ez4t8_<hARPV0JS-EISpy#E9iMe9a!rVK_iodP1b@qi4>OVC"
    "}{$jPdny?khci0u6IRRY?|C3DMAInw8yrS{jr!)3F+H_(g6;LPDk*+7mrz}lFAc6P`k@O@}vA7u`Y1V(BA&8+TZ6"
    ">6EttJy4Tbl}*OthMGQT%`aLJq)jK!p2x=s@Ep{dZr(B`ysa?glzC&}o)n9h_fcYH$6J0Q4`5i|j#j+aGlm&98%s"
    "s2eGBL(J+)_wClZ-@UfMs>p4;I5}!vACRs6<_al*nxR(_(I}?x$%EUwSA-Ph_0<)3KCd6+oPFk-U^kW-QC?9YVxm"
    "+Rbwq+rtKs9v=~X^3yaWXik59`L!)y8L{58N@_*;3`i2n%7HL8KzIV`)#4HsNWP>nvKw*<p5^^b3^GAW|v{GA!U?"
    "qi7)+-|6q{^t2TN}**{28#Cihf(r$YwI2b&vyOi*$fa5^OlX!rH7+OLH>X{U3Kh(Ps{mDb6!j|&Q&HO^qB0rQmTz"
    "FH0QH?PmeSnm=-{j2swS)E2u*QkL5<}_z*bwRGI1M8uh$~^`PSDCu}Bv>^4(xUhgJv4)-dHV`keG3^W^&B$6aB;!"
    "!XG_;R!}Dd%}s4Pr+qvS)e#v}wf))0@>g>(Q8utpg816idvUU7+x-9>v)`6C+Hkj{P)Dq|l?vR&Pn7zN_<2-DP_{"
    "_iZX}IzVdbu)<{b6ga7+>C^W;P-4&QpSiQVsyo0atvn~&AzPBwYE=L(^+wKDh_Pv2lA$`n^kiZOL&&OLviXJUWfJ"
    "Y{)ph+JfSEdC526uG;4$<_!^6Ff=5Ug=BmUhJ+Fx8(_d@nR8t;u?9w(2IpAHXRB~I7Sc^i+Jc14pfM+n>`9sR54`"
    "wIsQJWf>62@M~a5onM<y*24D6=F_*5~~LP>BJ@JYZ$qllw>yFmhG3^Mh<0MVoslyE2Cfp$NnB`vAcgXK0Hpy);MT"
    "g3H#R44?y3doBXo9_hx*Qbbjd3|6={24)&9m2m3$m?Y=xVV|0_918qFVQ8FHK#GSV0*?bT@!F_EPt`0D0W+vnG{6"
    "!^ld$tw9^RL^s%oX7bUvR&JIoFk(2FIe8gwN@=nR1^(u%8fG+!+Jl=+nrRBmc6xA?XIM5cJ73W~EyS)uld_ZS<T$"
    "BIOQvqN=`8mSU49Xzezxi9NjK*%>OIT9!n|yT=J`o?ab!q?00@(&Z3HTtqGTwJ3ssVI#K2%BDZjKvi@<kWwU1V3-"
    "vC1VvI&35Us(tu6em3J|_|%3mmy$7F5B90CN#bP9Oz?zRC5TT?*v#vU(z*I*;-w~UD#XifJ~=jG<)6b6GsbY^jTm"
    "LrYdWg35CmzJeu5Z|AfKl%LA!Jn1G{ON~J;xPexX_EsC%_gsOPe!i{UNXo{<8Pm_U*zW((0)dSTKj(EA!ru5HKbb"
    "T4iE2AP~|#RQ17RHwh@H*F&i4F{3^j_`K;Qn{h@ISO_P-=dqrKLa$|m<Y<=3S%@Q%au5KF7{wmoiB|AH>W<a|^#!"
    "b3h5M`6Gba@I9d?wwN8SMpGsEmos1vbJk>Lb$Ruhtqr_JKAe1R}YSDCE@cNYMy)NC1GU)4zIdHEOToI(G)^Zn;O#"
    "wc~6j5WV{MQHYsI3yD<H!O%78l8JJSPHdL53l4&XSL4fjzLPx9v&w3}W2oQv!um$yq+7MUw+ZXZ)%qTh^8kFY#{1"
    "&S*vX0L5;e;tFIJHdJ8}~mhDrQ7ypx#)oA_M;Wdy)4XII5<<7s&J@|$j+QT=M$n>v2Ez<&CRSxwmg(T!j4?KzgSp"
    "3V|Xo_;sDYVj)|Sotc~a^U+>&~u;-?OMWx2*sc#{^Y?&6+XAakM?Ze#z7!+?k#c8{L?^o61modCwpB*WXfz*d6GY"
    "Rk=Kq(KUYCwbOO|@s2jJ_hRf-4{PcOXmEdjXHWji}o<R0Abs!82rAKBPnc|t|gY!DJ;mJDP#w4-0ufC+=CDOwgUM"
    "%GUi|cdkEgs1;xKtNyReB&_jGAhs&ezG4Znw3f)>|aMPbo(oCLoZ{b3?dRHTU&He2UJwl+vircOGb#Y*f)SA(zp^"
    "*W>-2-Tj{*#y0P8XMC9a_*Wc9hGI>gMEd%*J5+l2QbxJWdK9l!cQ0%cX(9vVHG2#dpn}YeL0--2Y@*R03yM{zc}>"
    "?i&fUiTp3K_jT7#h7+U|_^cgH&qZMLoQESVRlGWNO~oyocau(vUizeQgq+nPmTG7|xq#bu^klxm`DJ?ZPi%a;84`"
    "1rN7(3667+oSZJEaIN;ePn3relH$%gQ#91x4@qr_=Oeo85hA&{N|4h|GM6ff8Gwi`EHAhW<(ZFPKvjRQ@C2L3*@Z"
    "MUs&I5ikklAiVKQ3Ap0&%E>xGhv#!H%3-{ZDTA;bNEkwxHG;2shp5JYFAZn|=$!lM)9T7Dq@Z-BC%|@Y7TcvTY^~"
    "3wOykpmJdozrP!pWv1GE<ZuY|(xYSu>5$z`yp7U5a3<zu$kdclVq9zw96Ux^D*f6b>u;%lwm?uiioWKlJovM?ZMP"
    "Ut_uEc-*H+s=R3=9_59<K89O1Y}v`%u4JXi(z%u2*!$E!viEIF3Ifx$wIwdY7RFZFKASpIpXBh9((*T?<n!hYOd>"
    ";-MLJt-Ov*PXzOpd*YZ6->uO*ALW3;zyzFH^dF1KH7dV*yx!-!yPz3)CmbN}pp1BfJ*7VWttWx1Hl^XiHn69Eu0@"
    "vb@d|8!s?Lw0p&noN~w7qj(>xU|92yTXZ^1B9aZP)Zp0_a&0>Ei*X0m?NN}$;{mMlDNKrr``$4=#ov`kFQcrVtl<"
    "Uf4kuT^oCxb(m(WOo&sRg{PrU$sIO4n#+(t*KgLPdnz<syaH0pxAjh=Cc#(U7$c=3mLEJWU{jxl8I$Hf18K1IY4#"
    "#eJzBXuU^saC1+*OWi{@Bni>f5)-dywo0^}-{XO`@5jrIK+ROJuA_@4ieqv7)`*V;mw}PQuD216ZVlCMb~mX&7S<"
    "g6NJPIh!HUMM--%xy7;<d*4OxL9?w(Br#X^7Q4E-!JyrVE<`;A!auRMewxmXp2KVzOVkNdUHo=%v=5cyd4%-g^NU"
    "B`XN?v3W)y-g-W7s|GU0kbpCk;i5?w$<y^A&n6(G6aCc?v{;Z~y)DD$dFt`RRMzY^NW*G_2XBR3=#f@j{u^@R9<T"
    "msy6+imEjZpbdOKUh`ezwXg`5JTf}m7sH+0;)HsGY~dfJFE4b3|^gmJCm(v-%XzX)3^6eisDDBk{fB?=yIN9Gkfx"
    "vzj3JPF+cj>*9K_JxA_Xq<6XJ)&4d>o^2jNF%R6U|1n--4oin>van_T!Sp%BtWd-LL))!@1xXOFwAejsD=6$B=Tr"
    "dZQma}f=LMvfo-gBX<V+5wNaGPbaVjr{5)?8*xS$}?UE$gFX^9+S?!un7MvSA|hHld*=I>gC?+f5l?mhEuH8?A4h"
    "4_QPyo~Zqmy<fB-M!3`xd}_k1k;PN{od*NND+<$0Vh`|V-1Ok}tg;y`K_i3b9;>29Vq(V&mPNN^5QryzV*NyHbZu"
    ";$g8(Y1va`CGxGb8DI~1MVFj}{kN&EBWj7`;Dv{P1iIq!4L5ap6(Lm(>VJ9a6%b?li)yILNh*{`<ore3?+pqp-#&"
    "BWp*_t*y=f?j5_pqI#k^^XlAO-4^1NaZ$t!5?m$S#5qYn@wT!^V({GSh`C4D9K-HC3ipf*c(Wn+s*2F6{zVyqE^c"
    "&>Q*}+BtPZFoXei%7pZ|cU#2*WSN1(n{KKestIt)^m-kb?+n~U0*KECI*L?9>D{XDo3E6h@id{@DP1z_e!_el*Q0"
    ";))^f|mj+TogWl}$`w0Oq=s7G3wyx<6IdexKd&bYmFFo%=m<$84oOSCu>&@0e@-FrfKyl`bZnm#vzX@!_dYO_cQ@"
    "C7fj=-BzA&$_jUyCF_&X(j(T6Mk?T07B-Z!rmv9ivg#PNOuqSII+;mg7DZAkO-04vFMHUf(IUM(ou>&jjUoS6WPa"
    "DzEX=9WMWF|wM-A^Ot7JyCea@>>-#A}CgeJ(huu8af_p3T{TW&M&Ysv|qu$X(Y6}<U%T8JM_&kRr4FHlO0_Z9$92"
    "#Sl$c!z!TKa(qBH8i)RwW-rAmk=@EB#(GMbqfE}9$P~i^`3Zg<JY71ySx+Ddh^qk$DTX@=GDKIxhVVtb1oaL`SLR"
    "vEE>bdYME$WhcA<p(QT~h00h=)&{e08f8TDA(ZnB~ylbq*8JFA|I<=O~#Hq*fgttQ`gm=w)GfQ_J9;?R-A~$t|Ka"
    "xKQF<Bb_XE=%SndJ97vNwJmnT}}FrfPp!XVgsk9-DN)0s}h^Q;tUpjaOJPk7|>B4))5Ie+~k4k)LI=o7o~88r%Uh"
    "TLb7Ifbs&ZfFk6XU|OGIy;AL&wav)n4tSw6b(m>rS6e;)-p5D#)8lDZP|^YMkUC6S^rc#r3vR8&eZkOJX;qo_H8c"
    "m)nc+uP-wb{PSD%_0F@jinQot`N$g@Sgn@?7J$QGzbH64L05QC|4g`b<N_cMWxJoZ{d2(CMBvyb#~+hO8(eKJ_R7"
    "o<xAzd>?6ovnI0&~!TrAu;7}=XoYr@*e)Kpuy9~z?17qiMEU+`A&@ei$b+R^72!yv=vlhpyzbb#KVtcsn9hYcuoq"
    "Oe2zh&cK146FqAxfX3Tl)2FfD1rwvfyv}f8f%S_Ek&d_x^k|JU6$)b4`(TfGG0j^J%nFkhv%Jm`k&1>g@`ggZA0@"
    "AEK$Sb~`>wYTf#fJ7ks93->(*%VZy^{hPoTal1R~&6XaxwB5V{YRuZ;*91C-ci<?{j}rIA%${EN(u(GtNa?T`clb"
    "*Bs|!&0UM`>wLv_(*5S=*YszrX4uls#x*|y`0{lr%KQe(1LE@qox?%0t%i`<{}uO+`wHqJSBap(5pRIsUyUK3QT="
    "_EN@=;lOHK{^lEo|Fp7UR6fqNoamO|>+8%HhIJXS{{76IA{DRmY#E9ci`yaReFvf_k$uop<%I4#jVovCw?-k>m^y"
    "Ci7ZAr8rT0n9oz(J|gNsD^CUjPa|v!xr~+YC&@W#~8nQeQ>ya_}9tK?&0|5@xkF=wRR(l{_=W!c(i+TJl>ZcE?0m"
    "Wt5zKSkPa_LueV>0C$A6oc3<v}(Y?jGAv+%<&mzOij(FP@8aPz=WYZH2P`=E_s%&T=92myYbUT~p=Ljid=V$5C99"
    "Z5kEr}IOiL~&r(@A3XEMwKC+U^j8tSWsN2bBr|ViVqFS&PS?IGN=_u-q8e+caEcfDE)z5YCE`ru3axcRaQD7@ZdA"
    "%wA%!GkZx5Jd>OKD8NN`Ah@VlKGKiQmYGTb&V>SGXHLAY;w2%?3pPcfh&_sBiP3zyfOf44XG+8}d3`uO8Xx{L-kB"
    "WiPiPC#8U=QbMNJ$?Szj*+zZnqUeER%bR%N|Ghz!Dkd=%3@r@(vtUx@QhSl<IfA2N#pz7JHD51BI@@qK`O^1mP{r"
    "sl#a9Vj#v#30k2U_1$i$c@+1L=GUSr^N2rf;Pw}7=19AkoA?#^AD5BRE`;AO{Ax!IIUMiGXN;<U$C2s`FzD#;fMn"
    "%2myMUvW(Y8Ye4I8$fYt<5s_eW0thuD)$Wv`Dn8^2<uM%$fJ1VwZ9JqYoUnsF35B!#Hx=rNXDi4S%W{2wVQ{ZdxT"
    "gJK^54KrGK4@FTuNHbki0AxIDt&xvTpX^U_a2Xl>TO4q=RBnIu`nKze57%+~o!d+?*aOr;HyW!rHK1p1KwmR0d1l"
    "UWxAL)eSKp5k5k~dXX%F-L-9RZ#+WampenpEUYST8VKS%UCtMP2Id};EQggK%$xBHG}Sw37;42g_jqEIMG01Us+Y"
    ")q9jfHATm~eDcMmX*jMPE|aZcLFC6<ins}gq(kK_{?QwTZ=8Tgx(bUyIl#rl?<_--2IG&@(-Nhyvw3#(6sNMB9xx"
    "Ii}3%J1y6V3sS&3-6RuG8HV1SR#xmY{a#8cF=c~_e0k~w{hOkSKOC|UCq9bHX91POAm`u1lst9z&$<^i!E!{m*=b"
    "-`E2rcCFvnZc-YiP<>bNm@@i%9d?VdbVrwo(7?yoV1<e7FvWg|`HU>T-?GeQS3_X+a9D?yT_Cmo004@M1Xu62?LZ"
    "OucQ|p*9yMP`x6xQk9zs(Fr0RV;!gHAKTKc#6!h|>HS3wWB@*H*wGa|-DAG*c=$7mvHHIexAuH=tn01i0Nrb08vP"
    "CdX9f0-fBHY8bpCP<-V`p2uwt<Pm#idSm4NacC!sfzCXvI?{A0fv9G43a={tNA(p0eC*#zD2DS+`29vnI5Bg`4Wl"
    "0=9fc-$gjKsZjHuXK2j+&!s_X<0?FOl9&YK1So7JLO|GC6kqqK%S+KYDE0+I9ecqM_kT(uBIuq^)>G$B^I#5sHqz"
    "1>F7zYk#I3DbF?yFCI%4d<f!*$Vj0D?aBx!<u{E69b(diAI|SCwSn{8>QC#M2@o!P767oFUnI=XCEPjv+3n!c1!F"
    "LLpMsdo503wtKD??K_qNU;yt$4e$ERSm9#Ea;?8P6>gT&3#;<F9nHVz67UQ2~Prgci6-{yaqk4vpiu9_wfMWO9eJ"
    "RPiQ|HuSA0%Mf*zzT2sRo0BYB6*vwAB)VT0K<0EMK5qAz9hPBBvj6_!!};R-nC89_li$1PvRClQY&pzu7!Wz|y?="
    "JS)gnT98h5U8))(%}b4f892L`F|lKYTQ8B3h}X-D4|$ctHd>XM<P8N>3_FY_P*PCF0_)$p+!Q+z4d_?e!-+3>y+j"
    "f`iS2J_Vc0D7TGanj-!T&F<{VtkhwW`^)O)F`Zvk8%S#Rw8k?TmXvOQoWUEL8O5#D+UzZ8p8UHuAPWX#+5jrxl>n"
    "4dnHCLO1r(#l<H_lVOQ=Xrs;qkR)}>K=%%2yolC!}jpf{em@bo9pmsoLa(4TzT!kn3Ro~edBLVVV`5w$Qs^uMjic"
    "%CS}-0Pyb{y-+7H<bb_${lHU?79~5s;paR*2<15r+xxi;^uNA|Ou7@&vW+y@_>yHO-_IKb7IeK~Udd$|Z|8!>U6W"
    "K06Lpyg8ZO=rogpGZCN5HXu%medy+xSle9A40tXYbX*vWo*%DxgMBgqW6di~`B8>#cG9&9O$jdUH8vku~qRj$}+n"
    "tEC_^MxEm`Zv5{1xd#F9+RlA7686r5oVBBW*#liEN;4X?Mrd52FSQ%9Pvt%13)Cssnx>0WyK)9h<7m@3=Sv>wuEW"
    "8mXT2L)BRJq6cH`Ez&Y(KOFu`X3aJ>kl2lB>BLYOV0ES2ekrw0ug?rM%5(WXs9BA;rH09_=+7o>CiTzn+r#Z5hlV"
    "${mz^2S*Bu!Yz$?)a>Buom6RblN3@VNsSG7f4q%6VV{5g77OJ4-6>>Hh6ZWjN1jyAXmNgxG=}Esfsw~rzD@s^Zcg"
    "9x||cU%D|+Y_)3!9VEKX{+j5@GtpX7~iRtG`5P@IT$EOJU%-9odN2!-l*hfY?5Y-MH=~Q26r2aM3(JwYyb#d7?+H"
    "HWW(Lyk^oF8wSilwjjnh9BzcN9_Vlo5|iG)CQ~tj!I$+Z$(BkrfD>B)QtJ@zd#HV&)!aNtqdRyO^(B8fb`hs_N))"
    "qmXRxDQnsqp-Z@M9td5>#4MIF+RN)aT<c6}bdM$GXk*S?=mV7eo$>Mb<?(n&cm#k`5mx2M5UEqCf&EM_=`)h<91^"
    "k>bdMmw;VEX}G<{@eI$tkEg~<KNhn~8o46>@mh9&x!&c#Q6!Cj(~gQNa}W<**5iSE<#H``(~MVia(0{{A>S+@l@3"
    "-`UD@(lx1SU+3PFBOo?jC#G8yYWcNg;Vjg6ptB88_nfp{CxAy;N@Vo(vQKmHkSYhpDXPW8)fU3ZBG%hv>a`U$<5a"
    "@qy$yE@J;n!W9OCEUe&BWbSv)iX;pGDHL|Q{jA&K0DhZAKjm#C)AkP&=g_rVv#srV13%>KCb#r^cUImw4I?bqca!"
    "yrHc_(^lWAz$#1B=8eDze<dN!FLaTsvniCd^1vJ!3v^{Y`>vZND;6aA1WPMmT4$xZB_Bj_I^o=Ox<JqxnN1&jOxd"
    "w!hiDKn?a<mqv~_2VNeJ70bK^EWVTfDU*@E8p%R>so|wp>^b<uNbd>0Zn((0-jdfd)4XRU{a`|_S%Gvdt=$;Rz3<"
    "Q6+-Qh`3uenj$MV>vPNg())Ok0`^0h018E0fZ^vo`d^xvKgIO6~I%%CGTxYtd)tv!g*Y)rb53EZ8!6eG1v*zY|x;"
    "eL+HyOSs_9aB!bnE_JAbzhMKWW~mzMpvyT*I0$ue7CmXB`z2VdQa8P<^UPe?#*G941um6pT0B9YWpO53bPsw?!DB"
    "zSr2r_!qkSvATUQVkAqH5*O#s`9EiS8T-m;+4KTa7va)Bo<kAaeE7Oyvjl9IPhAt&w8uU1}{hr0xA~}h;e3?rOba"
    "qjel{SseCDeqLI<gAu!i0Pn=aV6k5Yr2iAT-CKHQR$*bo)U;c#1RzD&Zx`)sZB63GJT(x*#)+dQTaoi4iW?kHdmY"
    "@?-|fOW3G}liSV+L-VkHGHrGwI?cp*F|(i-KOFz`=4ia*#UthPgJuo4VTwz+R7F9?nRxy2t03j*R2n>ZO0R9#iL`"
    "&tB9@LC&W5~4Zrlqubu*m!WJJkAtV>(dF%2>+CUdr0<*PMYd<nm`ScK`SHYMrvR%jMXB^sDSu1z==dz7t20Bf#Dx"
    "-i4ISK;_Np2XCKMT3Ng7F!G2s<vk9K+I6(2kqKKKOzT<mh^ogSQ?s+rd(hk@doV;pC(fcKTNGbA&oL5^N9ZFmsei"
    "OnF#^I+a%#w+e+N{U@Z--3(hu@hA4<AWy%qA@CU{~8W_^swiNbAI=zVM^N6Ve8+1))yw3y>WVNje+(bJC&DcCyG9"
    "cuqPM7$`^+i6r2-u(yvnooqNyk}<5);mF(xGxBVn82Vq1a>9<31kn{t`o|9W3`4gz7477f*DN1%p|F!~wPhz*K~A"
    "5Gk%&$T4OAV&Op%`|n8me7<JXw_02u=NtY9=6MwyC7r3?vuEJ#kDuVv?bR{~pc7xE=bSNN+nxCyA$y_%k!B8rNCm"
    "J3^>s+1ll4+TxUQrg$<fSZTCp?49v5Y<W~)FpbQYC*WX0l<b>+2M7S~Msx_=n%O@Wi-yR~)fkMi#ymbdZfn_lL5-"
    "<@e>rGiG@Ij-i?{hE<CFH<hdnPWZFgZB_C^6-6@T|ETFu%yMh<FLeHcwwCyNws_iMLrh7v#(rtUYtq%j*~IYhRMf"
    "=z2u=Jx~oFiT<~u9c4HuKEz3NqrzE!I1DLjPBQHCz>D<!da4$Uedm0-@2fh2Q&E)lK5XVhEhU<=~FY{>~IYw#qWJ"
    "KmUl@&!k!wG}N@vMY~yW0rhN~0Y<+fwEu*-co;=6FtdVD=)s1xYzpOQ_38%b=TsIr*rJ*l^v=Vv^<KC(`)IgWZC}"
    "#VThOctOIn*~yy)BNCqMJE|^3q_Vux*0a(gaRcew{qcxj(yn3|4r&WH=mTR&8o_CdPmeN*MboBr>hWT}V_<5V<Ef"
    "KoV7ZLWTWfOUo#9`9?$Q&gF_#T3#gTOz72v&w^sbBTp!vIu#lZjaBhnXH{}givGF}5(O@f!$VK|hfI?Kve<$888s"
    "8_b){ftuLm47b$m*-uoS2y`;Mu#sA?yu@~L$OD>evz^TC&h!?{|mDkJgE"
)
course_bytes = zlib.decompress(base64.b85decode(COURSE_ARCHIVE))
if (
    hashlib.sha256(course_bytes).hexdigest()
    != "47689b4acd3ba6effee7aae4586f93a9ee2f943bdd77104d1cf4c09e738a686d"
):
    raise ValueError("Embedded course files failed their integrity check")
course_files = json.loads(course_bytes)
if "COURSE_START_DIRECTORY" not in globals():
    COURSE_START_DIRECTORY = Path.cwd().resolve()
if "COURSE_RUNTIME_DIRECTORY" not in globals():
    COURSE_RUNTIME_DIRECTORY = tempfile.TemporaryDirectory(prefix="lucy-practical-runtime-")
COURSE_ROOT = Path(COURSE_RUNTIME_DIRECTORY.name)
for course_relative, course_content in course_files.items():
    course_target = COURSE_ROOT / course_relative
    if not course_target.resolve().is_relative_to(COURSE_ROOT.resolve()):
        raise ValueError("Invalid embedded relative path")
    course_target.parent.mkdir(parents=True, exist_ok=True)
    course_target.write_text(course_content, encoding="utf-8")
for course_import_path in (COURSE_ROOT, COURSE_ROOT / "src"):
    if str(course_import_path) not in sys.path:
        sys.path.insert(0, str(course_import_path))
COURSE_WORK = COURSE_START_DIRECTORY / "practical-work" / "ch05-b"
COURSE_WORK.mkdir(parents=True, exist_ok=True)
os.chdir(COURSE_WORK)
ROOT = COURSE_ROOT
print("Python", sys.version.split()[0], "Pydantic", pydantic.__version__)
print("Offline teaching files ready:", len(course_files))
print("Save your work here:", COURSE_WORK)

</details>


## Commit to a prediction before the examples


In [ ]:
prediction_notes = {
    "prediction": "Write the expected behavior before running the worked example.",
    "reason": "Name the input and rule behind that prediction.",
    "falsifier": "Name an observation that would prove the explanation wrong.",
    "revision": "After execution, explain what changed in your understanding.",
}

### Reading the Python vocabulary used in this notebook

You need basic assignments, `if`, loops, functions, lists and dictionaries. The less familiar
features used by the supplied code are introduced here. A **library** is reusable code that
Python can import. The **standard library** ships with Python; Pydantic is an additional package.
An import makes a name available, but does not mean that you have completed the exercise.

**JSON** is text for exchanging structured values. A Python dictionary is an in-memory object;
the JSON representation is a string. Use `json.dumps` to encode and `json.loads` to decode.
Decoding proves that text has valid JSON syntax, not that its fields match our business contract.
Predict which of the following two decoded objects could describe a stock count.

In [ ]:
import json

intro_data = {"sku": "MANGO", "count": 3}
intro_text = json.dumps(intro_data, sort_keys=True)
print(type(intro_data).__name__, type(intro_text).__name__, intro_text)
print(json.loads(intro_text))
print("Also valid JSON:", json.loads('["not", "a", "stock", "record"]'))
assert json.loads(intro_text) == intro_data

The first result is a dictionary; the second is a list. Before indexing a decoded object,
check the shape that your function promises to accept. An **exception** interrupts the normal
path. `raise ValueError(...)` refuses an invalid value; `try`/`except` lets a caller inspect that
expected refusal. Catch the expected class, rather than turning every programming error into
apparent success. `finally` runs cleanup even when an earlier operation raises.

An **annotation**, such as `count: int`, documents the expected type. It does not by itself
enforce the type at runtime. A **class** defines a kind of object; an instance holds one object's
data. `@dataclass` asks Python to generate routine construction and comparison methods from
annotated fields. `frozen=True` prevents ordinary reassignment of the instance's fields; it does
not make every object nested inside those fields immutable. A **method** is a function attached
to a class; `self` refers to the instance receiving the call.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class IntroObservation:
    operation: str
    count: int


intro_observation = IntroObservation("count-mango", 3)
print(intro_observation.operation, intro_observation.count)
assert intro_observation == IntroObservation("count-mango", 3)

A **callback** is a function passed to another function. This is how the classroom harness
invokes *your* implementation. The argument `candidate` below is a function object; parentheses
perform the call. Predict the two answers before execution, then trace the result to the callback.

In [ ]:
def intro_apply(candidate, value):
    return {"input": value, "observed": candidate(value)}


def intro_double(value):
    return value * 2


print(intro_apply(intro_double, 3))
print(intro_apply(lambda value: value + 2, 3))
assert intro_apply(intro_double, 3)["observed"] == 6

`lambda value: value + 2` is a small anonymous function. A **closure** is a function that retains
access to values from its surrounding scope. It can bind a tool to a shop snapshot. A shallow
copy duplicates only the outer container; `copy.deepcopy` also copies nested containers used
in these fixtures. A **set** stores distinct values; `required <= allowed` asks whether every
required item is allowed. `frozenset` is the corresponding immutable set. A tuple groups ordered
values; `(value,)` is a one-item tuple, including the comma.

**Paths and cleanup.** `Path` represents a filesystem location. `path / "file.json"` constructs
a child path; `read_text` and `write_text` read and write text. A context manager, used with
`with`, manages entry and exit. A temporary-directory context removes its contents on exit.
Save your submission outside temporary runtime directories. Reopening a file is different from
reusing a Python variable: the former tests retained bytes, while the latter only tests this kernel.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as intro_folder:
    intro_path = Path(intro_folder) / "observation.json"
    intro_path.write_text(json.dumps(intro_data), encoding="utf-8")
    intro_reopened = json.loads(intro_path.read_text(encoding="utf-8"))
    assert intro_reopened == intro_data
    print("Read from a file:", intro_reopened)

**Retrieval check:** explain JSON versus a dictionary, annotation versus validation, class versus
instance, and defining a callback versus invoking it. Change the callback above so an incorrect
implementation visibly changes the observed output. This distinction will matter when grading
your connected work. Reference: Python's [JSON](https://docs.python.org/3/library/json.html),
[dataclasses](https://docs.python.org/3/library/dataclasses.html), and
[pathlib](https://docs.python.org/3/library/pathlib.html) documentation.

### Pydantic: turn an input dictionary into a checked object

Pydantic is an additional Python library for validating data. Its **model** is a class describing
fields, not a neural network. Inherit from `BaseModel`, declare annotated fields, then call
`model_validate` on incoming data. A field without a default is required. A field with a default
can be omitted. The result is an instance whose values you read with dot notation.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class IntroCourseRequest(BaseModel):
    model_config = ConfigDict(strict=True, extra="forbid")
    name: str = Field(min_length=1)
    quantity: int = Field(gt=0, le=1000)
    note: str = ""


intro_request = IntroCourseRequest.model_validate({"name": "mango", "quantity": 4})
print(intro_request.name, intro_request.quantity, repr(intro_request.note))
assert intro_request.note == ""

The annotation says the field's type. `Field` supplies constraints: `gt=0` means greater than
zero, `le=1000` means at most 1000, and `min_length=1` excludes an empty name. `ConfigDict` sets
model-wide behavior. `strict=True` rejects conversions for this integer field, including `"4"`,
`4.0` and `True`; `extra="forbid"` rejects undeclared keys. Pydantic can otherwise convert some
compatible inputs, so choose this boundary deliberately rather than assuming every accepted
input arrived in the expected type.

Predict which rule refuses each payload. `ValidationError` reports a failed contract. Its
`errors()` entries contain `loc`, the field location, and `type`, the failure category. Catching
that expected exception lets the notebook inspect the failure and continue.

In [ ]:
intro_bad_requests = [
    {"name": "mango", "quantity": "4"},
    {"name": "mango", "quantity": True},
    {"name": "", "quantity": 4},
    {"name": "mango", "quantity": 0},
    {"name": "mango", "quantity": 4, "approved": True},
    {"quantity": 4},
]
for intro_bad_request in intro_bad_requests:
    try:
        IntroCourseRequest.model_validate(intro_bad_request)
    except ValidationError as intro_error:
        print([(item["loc"], item["type"]) for item in intro_error.errors(include_input=False)])
    else:
        raise AssertionError("An invalid input crossed the declared contract")

Use `model_dump()` for a Python dictionary, `model_dump_json()` for JSON text, and
`model_validate_json()` to parse and validate JSON. `model_json_schema()` describes the contract;
it is neither an instance's current values nor an invocation of the business handler.

In [ ]:
intro_serialized = intro_request.model_dump_json()
intro_schema = IntroCourseRequest.model_json_schema()
assert IntroCourseRequest.model_validate_json(intro_serialized) == intro_request
print("Actual values:", intro_request.model_dump())
print("Quantity contract:", intro_schema["properties"]["quantity"])
assert intro_schema["properties"]["quantity"]["exclusiveMinimum"] == 0

Four is valid input to this schema even if the shop needs six. Pydantic checks the declared
shape and constraints; the handler still needs authoritative stock, price and permission.
Ordinary assignments to an existing instance are not automatically revalidated unless configured
for assignment validation. This lesson validates new input at the boundary and uses the resulting
values. Explain these limits before relying on a model object in a transaction or tool call.

Chapter 2's full introduction expands this pattern with a separate data-repair checkpoint.
This notebook contains the required pattern here so prior Pydantic experience is not needed.
References: [models](https://docs.pydantic.dev/latest/concepts/models/),
[fields](https://docs.pydantic.dev/latest/concepts/fields/), and
[strict mode](https://docs.pydantic.dev/latest/concepts/strict_mode/).

### SQLite from the first row to an atomic change

A dictionary disappears when its process ends. A database can retain records so a later process
can resume from evidence. **SQLite** is an embedded database: Python's `sqlite3` library opens
a local database file without starting a separate database server. **SQL** is the language used
to define, select and change its records. A **table** has named columns and rows. A **primary
key** identifies a row; a **query** asks for rows satisfying a condition.

Start with a deliberately small preference table. Read the SQL as instructions: create the
table, insert one named value, then select the value for one session. `?` is a parameter
placeholder; the values are passed separately so they are data, not SQL instructions.
`fetchone()` returns one row or `None`; it does not guarantee that a matching row exists.

In [ ]:
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as intro_sql_folder:
    intro_db_path = Path(intro_sql_folder) / "example.sqlite"
    intro_db = sqlite3.connect(intro_db_path, autocommit=True)
    intro_db.execute("CREATE TABLE preference (id INTEGER PRIMARY KEY, session TEXT, value TEXT)")
    intro_db.execute("INSERT INTO preference (session,value) VALUES (?,?)", ("lucy", "09:00"))
    intro_row = intro_db.execute(
        "SELECT value FROM preference WHERE session=?", ("lucy",)
    ).fetchone()
    print("Matching row:", intro_row)
    assert intro_row == ("09:00",)
    assert (
        intro_db.execute(
            "SELECT value FROM preference WHERE session=?", ("another-session",)
        ).fetchone()
        is None
    )
    intro_db.close()
    intro_reopen = sqlite3.connect(intro_db_path, autocommit=True)
    assert intro_reopen.execute("SELECT count(*) FROM preference").fetchone()[0] == 1
    intro_reopen.close()
print("The row survived closing and reopening its connection.")

Index `[0]` selects the first column of a returned tuple. `sqlite3.Row` is an alternative row
factory that also permits named-column access. `dict(row)` then produces an ordinary dictionary.
The book's `Database` wrapper supplies that configuration and its schema; the wrapper is course
code, while `sqlite3` is the standard library. You will use the public connection and transaction
methods explained at the exercise boundary, rather than needing to reconstruct the wrapper.

Now consider a budget. Moving five pence from reserved to spent requires two values to change
together. A **transaction** makes a group of local changes commit together or roll back together.
The example explicitly controls SQL transactions with `autocommit=True` and SQL statements.
`BEGIN IMMEDIATE` starts a write transaction; `COMMIT` keeps its changes; `ROLLBACK` discards them.
Predict the row after the deliberately raised exception. Catching an error alone would not undo
the first update; the rollback is the operation that restores the prior state.

In [ ]:
intro_ledger = sqlite3.connect(":memory:", autocommit=True)
intro_ledger.execute(
    "CREATE TABLE budget (id INTEGER PRIMARY KEY, reserved INTEGER, spent INTEGER)"
)
intro_ledger.execute("INSERT INTO budget VALUES (1,5,0)")
intro_ledger.execute("BEGIN IMMEDIATE")
try:
    intro_ledger.execute("UPDATE budget SET reserved=0 WHERE id=1")
    raise ValueError("injected failure before the matching spend update")
except ValueError:
    intro_ledger.execute("ROLLBACK")
assert intro_ledger.execute("SELECT reserved,spent FROM budget").fetchone() == (5, 0)
intro_ledger.execute("BEGIN IMMEDIATE")
intro_ledger.execute("UPDATE budget SET reserved=0,spent=5 WHERE id=1")
intro_ledger.execute("COMMIT")
print(
    "After a complete change:", intro_ledger.execute("SELECT reserved,spent FROM budget").fetchone()
)
intro_ledger.close()

The literal `:memory:` creates a temporary database inside this connection; it is useful for the
small experiment, but the earlier file example establishes persistence. Neither example proves
that a remote supplier rolls back when the local transaction rolls back. An external operation
has its own state and evidence.

**SQL you will meet later.** `UPDATE ... SET ... WHERE ...` changes selected rows. `AND` combines
conditions. `ORDER BY` makes an ordering explicit; absent that clause, do not rely on row order.
`count(*)` counts rows; `sum(amount)` totals a column and can be `NULL` on an empty input;
`coalesce(sum(amount),0)` uses zero for that empty aggregate. A `UNIQUE` constraint rejects
duplicate identities. `GROUP BY status` computes one aggregate per status.

An **invariant** is a condition that must remain true across operations, such as nonnegative
reserved money. A **snapshot** is a consistent view at one point; two separate reads can describe
different moments unless their transaction contract binds them. In the book, `with db.immediate()`
groups related writes. It is a course-defined context manager with the commit/rollback purpose
you just observed. Do not assume that an arbitrary `with connection` has identical behavior under
every SQLite autocommit setting.

**Your prediction:** two workers both read ten remaining pence outside a transaction and each
approve seven. Why can both believe the next order fits? Explain what must be checked together
with the write. Then change the example's initial reserved amount and repeat the failure.
Reference: Python's [SQLite tutorial and transaction control](https://docs.python.org/3/library/sqlite3.html).

## Memory means selecting retained evidence for a new request

Lucy says deliveries should arrive at nine, then corrects herself to ten. A useful assistant
must retain the correction after a restart and avoid presenting both times as current guidance.
**Persistence** means the record survives the process. **Retrieval** means choosing which retained
records to include now. They are separate mechanisms: a database can preserve every revision
while retrieval returns only the currently active revision for this session.

A **session** identifies the conversation or work context that owns a memory. **Provenance**
records where a value came from. A **revision** is a new version of an earlier value. Here a
correction creates current guidance while old evidence can remain available for audit. Forgetting
excludes a value from future context; it does not erase already sent provider requests or backups.

### Derive a retrieval rule on visible rows

Our small table has two sessions and two revisions. First filter by session and active status.
Then rank the remaining rows for this query. Reversing those operations can waste a limited
context budget on a foreign or superseded row. Predict the returned identities for Lucy.

In [ ]:
intro_memories = [
    {"id": 1, "session": "lucy", "active": False, "name": "delivery", "value": "09:00"},
    {"id": 2, "session": "lucy", "active": True, "name": "delivery", "value": "10:00"},
    {"id": 3, "session": "another-shop", "active": True, "name": "delivery", "value": "06:00"},
    {"id": 4, "session": "lucy", "active": True, "name": "invoice", "value": "email"},
]
intro_eligible = [row for row in intro_memories if row["session"] == "lucy" and row["active"]]
print([(row["id"], row["value"]) for row in intro_eligible])
assert [row["id"] for row in intro_eligible] == [2, 4]

The database exercise performs the same filtering with a parameterized `WHERE` clause. The
bounded result must retain identity, name, value, source and creation evidence. Returning only
a friendly sentence would discard the provenance needed to investigate a bad answer.

### Understand the deliberately simple relevance score

We lowercase using `casefold`, split into whitespace-separated words, and count the intersection
with query words. A set intersection keeps words present in both sets. This lexical score is
easy to inspect; it is not a semantic embedding or a claim that similar meanings always match.
An **embedding** would represent text numerically for another similarity method; we do not need
that additional model or library for this lesson's explicit rule.

In [ ]:
intro_query_words = set("DELIVERY time".casefold().split())
intro_ranked = []
for intro_row in intro_eligible:
    intro_words = set((intro_row["name"] + " " + intro_row["value"]).casefold().split())
    intro_ranked.append((len(intro_query_words & intro_words), intro_row["id"], intro_row["value"]))
intro_ranked.sort(key=lambda item: (-item[0], -item[1]))
print(intro_ranked)
assert intro_ranked[0] == (1, 2, "10:00")

The negative signs turn ascending Python sorting into descending score and descending identity.
Identity breaks ties deterministically in favor of the newer row. Slicing `[:maximum]` then
limits the output. Validate the maximum first; accepting negative slicing would quietly turn
an invalid requested budget into a different selection rule.

### Follow the context all the way to the model seam

The course's `Database` supplies durable rows. `preferences` retrieves them. The context builder
places selected values in the next request. The replay model records the actual messages it
received. A successful insert proves only persistence; a successful query proves retrieval;
the recorded message proves that the retrieved data was connected to the model input.

In Unit A you implement retrieval, close and reopen the database, then inspect the context.
In Unit B a query loses its `active=1` condition. The old value can reappear even if the new value
ranks first. The repair must exclude stale guidance rather than merely move it lower in the list.

**Before the main task:** explain which records survive an empty query, why another session's
matching word cannot grant eligibility, and what happens when two rows have the same score.
For transfer, correct twice, forget one preference, and vary the retrieval limit. Keep actual
row identities beside the final context text so a plausible-looking answer cannot conceal the
wrong revision. Authoritative current stock still comes from the shop tool, not remembered prose.

## Choose an explicit starting point for this independent notebook

This Unit B runs without Unit A. By default it prepares a **supplied reference starting point**
and labels its provenance. It is not evidence that you built Unit A. To investigate your own
successful implementation, set `LEARNER_HANDOFF` to its saved path before running the cell.
An invalid selected file refuses; it is never silently replaced with the reference.

`SourceTask` supplies copied-source execution and handoff validation; `RuntimeLab` supplies the
controlled failure experiment. Their public operations are introduced beside the main exercise.
The artifact stores identity and observations; no variables from another kernel are required.


In [ ]:
LEARNER_HANDOFF = None

<details><summary>Prepare and validate the supplied starting artifact</summary>


In [ ]:
import json
import runpy
import shutil
import textwrap
from pathlib import Path

COURSE_INPUT = COURSE_WORK / "ch05-unit-a-handoff-v1.json"
if LEARNER_HANDOFF is not None:
    learner_input = Path(LEARNER_HANDOFF).expanduser().resolve()
    if not learner_input.is_file():
        raise FileNotFoundError("The selected learner handoff does not exist")
    if learner_input != COURSE_INPUT.resolve():
        shutil.copy2(learner_input, COURSE_INPUT)
    HANDOFF_ORIGIN = "LEARNER_SELECTED"
else:
    source_task_class = runpy.run_path(
        str(COURSE_ROOT / "book/always_on/exercises/source_tasks_v1.py")
    )["SourceTask"]
    reference_task = source_task_class(COURSE_ROOT, 4)
    try:
        reference_task.install(textwrap.dedent(reference_task.fragment))
        reference_observation = reference_task.visible("SUPPLIED_REFERENCE_START")
        if reference_observation["status"] != "PASS":
            raise RuntimeError("The supplied starting point did not pass its connection check")
        reference_task.save(COURSE_INPUT, reference_observation)
    finally:
        reference_task.close()
    HANDOFF_ORIGIN = "SUPPLIED_REFERENCE"
print("Starting evidence:", HANDOFF_ORIGIN)
print("The core task below validates the selected artifact before using it.")

</details>


## Understand the supplied execution interface

The course runtime is provided so your implementation can be connected to real callers and
storage. `SourceTask(ROOT, chapter)` makes a private copy. `install(source)` replaces only the
declared function; `visible()` invokes the real chapter probe; `save(path, result)` retains a
successful implementation and its evidence. `load(path)` checks the saved identities and hashes.
`inject_failure()` changes the declared boundary; `repair(fragment)` replaces that broken fragment.
`close()` removes the scratch copy after you retain evidence. These methods are supplied harness
operations, not additional packages you must discover or install.

`RuntimeLab` provides the same copied-source failure experiment without the complete-function
construction layer. Its `run` method records exit status, observations and the compared expectation.
A subprocess log from an unfinished learner implementation is feedback about that implementation;
it is not a successful connection. A syntax error in the notebook cell itself is a separate issue
to fix. The task below names which interface it uses.

For direct-function units, the visible driver calls your callback without installing a source
string. In either case, trace where your code is invoked. Supplied fixtures, database wrappers and
replay models are labelled infrastructure; your own implementation and changed-case explanation
are the evidence of learning.


## Main practical: construct, connect and challenge



Putting the new preference first does not remove contradictory old guidance. This time you begin with your Unit A implementation and its saved evidence. Lucy corrects delivery time, but the next request again contains the superseded preference.

## Verify the handoff

The starting-point cell has selected the Unit A artifact explicitly. A selected learner handoff must validate; the default reference start is labelled separately. Run the setup and keep the runtime and implementation hashes in your submission.

In [ ]:
import json
import os
import runpy
from pathlib import Path

ROOT = COURSE_ROOT

SourceTask = runpy.run_path(str(ROOT / "book/always_on/exercises/source_tasks_v1.py"))["SourceTask"]
REFERENCE_LESSON = 4
HANDOFF = Path("ch05-unit-a-handoff-v1.json")
handoff_status = "MISSING"
if HANDOFF.is_file():
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        handoff = task.load(HANDOFF)
        handoff_status = "VERIFIED"
        print("IMPLEMENTATION", handoff["implementation_sha256"])
    finally:
        task.close()
print("UNIT_A_HANDOFF", handoff_status)

## Reproduce and diagnose

Predict the consequence of this injected boundary before executing it:

```text
"WHERE session=?",
            (session,),
```

The controlled mutation changes the same implementation you submitted. It refuses if the declared mutation boundary no longer occurs exactly once; inspect an alternative implementation with the instructor before adapting the experiment.

In [ ]:
baseline = broken = None
if handoff_status == "VERIFIED":
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        task.load(HANDOFF)
        baseline = task.visible("YOUR_BASELINE")
        if baseline["status"] != "PASS":
            raise ValueError("Saved Unit A code no longer satisfies the visible contract")
        task.inject_failure()
        broken = task.run("INJECTED_FAILURE", expected=task.spec["expected_broken"])
        print("BEFORE", baseline["observation"])
        print("AFTER", broken["observation"])
    finally:
        task.close()
else:
    print("HANDOFF_REQUIRED: complete Unit A before performing Unit B")

State a diagnosis using those two observations. Name a test that would prove your diagnosis wrong. Remember, correct, close and reopen SQLite; invoke context and inspect the actual system message seen by the model.

## Repair the boundary

Return the complete replacement for the injected fragment. Do not edit the oracle or print a desired observation. Repair the actual source. The starter keeps the defect so the learner outcome remains incomplete.

In [ ]:
def repair_fragment():
    return '"WHERE session=?",\n            (session,),'

<details><summary>Hint 1 — the consequence</summary>

Lucy corrects delivery time, but the next request again contains the superseded preference.

</details>

<details><summary>Hint 2 — the evidence</summary>

Compare the two observations, then trace the changed field to `preferences` in `src/sovereign_agent/assistant_context.py`. Distinguish a schema refusal from a business-rule or authority refusal.

</details>

<details><summary>Hint 3 — the design</summary>

Use a parameterized query, set intersection for relevance, deterministic ties and a validated maximum.

</details>

In [ ]:
def connect_repair(fragment):
    task = SourceTask(ROOT, REFERENCE_LESSON)
    try:
        task.load(HANDOFF)
        task.inject_failure()
        task.repair(fragment)
        return task.visible("YOUR_REPAIR")
    finally:
        task.close()


repair_result = None
if handoff_status == "VERIFIED":
    repair_result = connect_repair(repair_fragment())
    print("REPAIR", repair_result["status"], repair_result["observation"])
else:
    print("REPAIR_NOT_ATTEMPTED: missing Unit A evidence")

## Transfer under a changed constraint

Correct twice, forget one key and introduce another session with the same preference name. Prove no stale or foreign value reaches context.

Create a fresh task, load your handoff, inject the defect and apply your repair. Then change only the copied probe to exercise the new condition. Keep the actual observation and a prediction written beforehand. Explain why a visible-case lookup or a blanket refusal could pass the original example but fail this transfer.

The instructor's holdout applies your repair to a new copied runtime and checks both the positive case and the missing protection. An exact exception or changed state must cause a failure; no broad error is accepted as successful refusal.

## Exit ticket

Submit the original handoff, baseline and broken observations, repair, transfer probe and results. State what Lucy would experience before and after the fix. Identify the guarantee that still requires separate evidence: Forgetting future context is not secure erasure of backups or past provider requests.

In [ ]:
passed = repair_result is not None and repair_result["status"] == "PASS"
exercise_report = {
    "unit": "ch05-b",
    "attempted": int(repair_result is not None),
    "completed": int(passed),
    "failed": int(repair_result is not None and not passed),
    "skipped": int(repair_result is None),
    "connection": "PASS" if passed else "NOT_READY",
    "handoff": handoff_status,
}
print("EXERCISE_REPORT=" + json.dumps(exercise_report, sort_keys=True))

## Changed-constraint construction: Retrieve the newest eligible memory under a limit

**Allow twenty minutes.** Spend three minutes predicting, ten implementing and tracing, five
on a new case of your own, and two explaining the surviving limitation. This is dedicated work,
not an invitation to run a supplied answer. Both units revisit the same invariant after different
core experiences; in Unit B, attempt this task from memory before consulting Unit A.

Implement transfer_check(rows, session, limit). Each row has unique integer id, session and boolean active. Return eligible row IDs newest first, at most limit. Require an exact integer limit in 1..100; otherwise raise ValueError. Empty eligible input returns an empty list. Do not mutate rows. This deliberately isolates eligibility and tie order from lexical scoring.

Write your expected values before running the table. Keep one accepted case and one refusal.
Your function is passed directly into the driver below. The driver copies inputs and checks
they remain unchanged; it does not replace your implementation with the reference answer.

<details><summary>Hint 1 — identify the authoritative inputs</summary>
Name the source field for each output value. Which input changes while the rule remains the same?
</details>
<details><summary>Hint 2 — choose the boundary cases</summary>
Start with exact empty, exact equality and one value on each side of the boundary where valid.
Do not add a special case for a visible product name or operation identity.
</details>


In [ ]:
def transfer_check(rows, session, limit):
    raise NotImplementedError("Filter before ordering and limiting")

In [ ]:
import copy
import json

TRANSFER_CASES = [
    (
        "current own row",
        [
            [
                {"id": 1, "session": "lucy", "active": False},
                {"id": 2, "session": "lucy", "active": True},
                {"id": 3, "session": "other", "active": True},
            ],
            "lucy",
            2,
        ],
        [2],
    ),
    (
        "newest first",
        [
            [
                {"id": 8, "session": "lucy", "active": True},
                {"id": 9, "session": "lucy", "active": True},
            ],
            "lucy",
            1,
        ],
        [9],
    ),
    ("empty", [[], "lucy", 3], []),
    ("zero limit", [[], "lucy", 0], {"raises": "ValueError"}),
]


def same_transfer_value(actual, expected):
    if type(actual) is not type(expected):
        return False
    if isinstance(expected, dict):
        return actual.keys() == expected.keys() and all(
            same_transfer_value(actual[key], value) for key, value in expected.items()
        )
    if isinstance(expected, list):
        return len(actual) == len(expected) and all(
            same_transfer_value(a, e) for a, e in zip(actual, expected, strict=True)
        )
    return actual == expected


def run_transfer(candidate, cases):
    observations = []
    for label, arguments, expected in cases:
        supplied = copy.deepcopy(arguments)
        before = copy.deepcopy(supplied)
        raised = None
        try:
            actual = candidate(*supplied)
        except NotImplementedError:
            raised = "NotImplementedError"
            actual = {"unfinished": True}
        except Exception as error:
            raised = type(error).__name__
            actual = {"raises": raised}
        expects_error = isinstance(expected, dict) and set(expected) == {"raises"}
        correct = (
            raised == expected["raises"]
            if expects_error
            else (raised is None and same_transfer_value(actual, expected))
        )
        passed = correct and same_transfer_value(supplied, before)
        observations.append(
            {"case": label, "expected": expected, "observed": actual, "passed": passed}
        )
        print("PASS" if passed else "NEEDS_WORK", label, "expected", expected, "observed", actual)
    return observations


transfer_observations = run_transfer(transfer_check, TRANSFER_CASES)
TRANSFER_PASSED = all(row["passed"] for row in transfer_observations)
print("TRANSFER_STATUS", "PASS" if TRANSFER_PASSED else "NEEDS_WORK")

### Design a counterexample and retrieve the mechanism

Add one new case with an independently calculated expected outcome to `TRANSFER_CASES` and rerun
the driver. Change one condition at a time. Then deliberately replace your candidate with a
constant answer in a temporary copy and show a case that rejects it. Restore your implementation.
Explain why that counterexample is stronger than repeating the original example with a new name.

Without viewing the worked example, write the invariant in words and trace one observed value
back to its input. Identify which part is a local fixture result and which claim would need a
live provider, host or external-system observation. Keep a first attempt even if you used a hint.


## Save your evidence and explain the result

Fill the prediction notes and your explanation before saving. Include the exact observed value,
the input or retained row that caused it, your code's invocation point, one failed hypothesis,
and the strongest claim the evidence still cannot support. A completed code cell alone does not
earn explanation credit. Do not label reference-start behavior as your own Unit A construction.

Keep this edited notebook, the Markdown if used for notes, saved handoff files, and the JSON record
below. Your work folder survives scratch cleanup and can be reopened in a new kernel. An instructor
can ask for an unseen case after the visible checks; keep your implementation general.


In [ ]:
explanation_notes = {
    "causal_trace": "Explain the input, learner invocation and observed result.",
    "failed_hypothesis": "Describe a prediction the evidence changed.",
    "remaining_limit": "Name the guarantee not established by this experiment.",
}
course_submission = {
    "unit": "ch05-b",
    "planned_minutes": 90,
    "starting_evidence": globals().get("HANDOFF_ORIGIN", "INDEPENDENT_UNIT_A"),
    "prediction": prediction_notes,
    "explanation": explanation_notes,
    "core_report": exercise_report,
    "transfer": transfer_observations,
    "explanation_review": "HUMAN_REVIEW_REQUIRED",
}
submission_path = COURSE_WORK / "ch05-b-submission-v1.json"
submission_path.write_text(
    json.dumps(course_submission, indent=2, sort_keys=True), encoding="utf-8"
)
print("Saved evidence:", submission_path)
print(
    "COURSE_REPORT="
    + json.dumps(
        {
            "unit": "ch05-b",
            "transfer_passed": TRANSFER_PASSED,
            "starting_evidence": course_submission["starting_evidence"],
            "edition": "student",
        },
        sort_keys=True,
    )
)

## Extension: let the model manage Lucy's memory

Everything above kept one rule fixed: *you* decided what the shop remembers, and your code
wrote it. The remaining question in this chapter is what changes when the **model** decides.
Lucy tells the assistant that mango sold out again on Saturday. Nobody calls `remember`.
The model reads the message, judges whether that fact will still matter next week, and asks
to store it.

That is an **agent loop**: send the conversation, read the reply, run any tool the model asked
for, append each result to the conversation, and go round again until the model stops asking.
A **tool** is a name, a description and a Pydantic argument class like the ones above; its
`model_json_schema()` becomes the contract the model reads. A **tool call** is the model's
request to run one; a **tool result** is what we send back. The model never touches SQLite
directly. It proposes; the shop's own rules dispose.

Two modules in the embedded runtime implement this without a framework. `memory_tools` wires
`remember_fact` and `recall_facts` to the same `Dispatcher` the book uses for every tool.
Writing is marked *consequential*, so the dispatcher refuses to run it unless a `before_write`
gate is attached, and that gate can refuse a write. `memory_agent` is the loop plus two
interchangeable models: `ScriptedModel` replays a recorded transcript and runs everywhere;
`OpenAIModel` calls the real API when a key is present. Every record is labelled `scripted`
or `live`, so a replay is never mistaken for a live run.

**Prediction:** the next cell seeds one memory and prints the two tool contracts. Before
running it, write which fields the model must supply to store a fact, and which shop rule
would refuse a correctly shaped fact.


In [ ]:
import json
from pathlib import Path

from sovereign_agent.database import Database
from sovereign_agent.memory import remember
from sovereign_agent.memory_tools import toolbox
from sovereign_agent.model_turn import ToolCall

LLM_MEMORY_DB = COURSE_WORK / "ch05-llm-memory.sqlite"
SEED_MEMORY = "Deliveries from the dairy supplier arrive at ten on weekdays."


def fresh_memory_database(path):
    """Start from an empty shop memory so reruns and replays observe the same rows."""
    for stale in (path, Path(f"{path}-wal"), Path(f"{path}-shm"), path.with_suffix(".authority")):
        stale.unlink(missing_ok=True)
    return Database(path)


memory_db = fresh_memory_database(LLM_MEMORY_DB)
remember(memory_db, "lucy-seed-001", SEED_MEMORY, importance=0.8)
memory_box = toolbox(memory_db, actor_id="lucy")
memory_dispatcher = memory_box.dispatcher()
for tool_schema in memory_dispatcher.schemas():
    function = tool_schema["function"]
    print(function["name"], "requires", function["parameters"]["required"])
    print("   ", function["description"])
print("Writes allowed per session:", memory_box.policy.max_writes)
print("Phrases the shop never stores:", memory_box.policy.banned_phrases)

The schema the model reads is the `model_json_schema()` you inspected earlier, wrapped in the
`{"type": "function", ...}` envelope the API expects. `required` lists the fields without
defaults; `strict=True` and `extra="forbid"` still apply, so a call with `importance` as the
string `"0.9"` is refused before any handler runs, exactly like the bad `IntroCourseRequest`
payloads above.

Hand-build three tool calls and dispatch them yourself, with no model involved. Predict each
result: one valid write, one write that names a card number, and one with a string where a
float belongs. Then read the `memories` table directly to see which rows SQLite actually
holds. `invoke_with_reason` adds the shop rule that refused a write. A Pydantic refusal stays
`invalid_arguments`, because the dispatcher never copies raw arguments into an error message.


In [ ]:
manual_calls = [
    ToolCall(
        id="manual-1",
        name="remember_fact",
        arguments={
            "content": "Mango sells out most Saturday afternoons; order extra on Friday.",
            "importance": 0.7,
            "reason": "recurring pattern",
        },
    ),
    ToolCall(
        id="manual-2",
        name="remember_fact",
        arguments={
            "content": "Lucy's card number ends in 4421 for the dairy account.",
            "importance": 0.3,
            "reason": "billing detail",
        },
    ),
    ToolCall(
        id="manual-3",
        name="remember_fact",
        arguments={
            "content": "Pistachio is the slowest seller in winter.",
            "importance": "0.9",
            "reason": "seasonal pattern",
        },
    ),
]
manual_results = [memory_box.invoke_with_reason(memory_dispatcher, call) for call in manual_calls]
for call, result in zip(manual_calls, manual_results, strict=True):
    print(call.id, result)
assert manual_results[0] == {"ok": True, "value": {"stored": True, "memory_id": "lucy-llm-001"}}
assert manual_results[1]["error"] == "content_not_permitted"
assert manual_results[2] == {"ok": False, "error": "invalid_arguments"}
stored_rows = memory_db.connection.execute(
    "SELECT id, content, importance FROM memories ORDER BY id"
).fetchall()
for stored_row in stored_rows:
    print(dict(stored_row))
assert [row["id"] for row in stored_rows] == ["lucy-llm-001", "lucy-seed-001"]

### Run the loop against a recorded transcript

`ScriptedModel` replays what a model replied at each step. It is not a stand-in pretending to
be live: the record says `source: scripted`, while the tool calls, the refusals and the SQLite
rows are real. Read the script as a conversation. Turn one searches memory before storing
anything. Turn two proposes two facts, one of which the shop refuses. Turn three adapts and
stores the arrangement without the secret. Turn four asks for no tools, which ends the loop.

The cell starts from a fresh shop memory holding only the seed row, so your manual write above
is gone. **Prediction:** how many rows will `memories` hold after this run, and which turn
produces the refusal?


In [ ]:
from sovereign_agent.memory_agent import ScriptedModel, run_memory_session

LUCY_REQUEST = (
    "Mango sold out again on Saturday afternoon, so we should order extra on Fridays. "
    "Also remember the dairy account is paid by card, number ending 4421."
)
RECORDED_TURNS = [
    {"calls": [{"name": "recall_facts", "arguments": {"query": "mango saturday dairy"}}]},
    {
        "calls": [
            {
                "name": "remember_fact",
                "arguments": {
                    "content": "Mango sells out most Saturday afternoons; order extra on Friday.",
                    "importance": 0.7,
                    "reason": "recurring stock pattern",
                },
            },
            {
                "name": "remember_fact",
                "arguments": {
                    "content": "The dairy account is paid by card number ending 4421.",
                    "importance": 0.3,
                    "reason": "billing arrangement",
                },
            },
        ]
    },
    {
        "calls": [
            {
                "name": "remember_fact",
                "arguments": {
                    "content": "The dairy account is paid by card; the details stay with Lucy.",
                    "importance": 0.3,
                    "reason": "billing arrangement without the secret",
                },
            }
        ]
    },
    {
        "content": "Stored the Saturday mango pattern and the dairy billing arrangement. "
        "The shop refused the card number, so I kept only that a card is used."
    },
]

memory_db.connection.close()
scripted_box = toolbox(fresh_memory_database(LLM_MEMORY_DB), actor_id="lucy")
remember(scripted_box.db, "lucy-seed-001", SEED_MEMORY, importance=0.8)
scripted_record = run_memory_session(ScriptedModel(RECORDED_TURNS), scripted_box, LUCY_REQUEST)
print(scripted_record.summary())
for step in scripted_record.steps:
    outcomes = [result.get("error", "ok") for result in step.results]
    print("turn", step.turn, [call.name for call in step.calls], outcomes)
print("Model:", scripted_record.final_message)
assert scripted_record.source == "scripted"
assert scripted_record.written == ("lucy-llm-001", "lucy-llm-002")
assert [reason for _, reason in scripted_record.refused] == ["content_not_permitted"]
scripted_rows = scripted_box.db.connection.execute(
    "SELECT id, content FROM memories ORDER BY id"
).fetchall()
for scripted_row in scripted_rows:
    print(dict(scripted_row))
assert len(scripted_rows) == 3

### Run the same loop against a live model

The loop, the tools and the shop rules do not change. Only the model does. `build_model` looks
for an `OPENAI_API_KEY` in two places: Colab's **Secrets** pane (the key icon in the left
sidebar: add a secret named `OPENAI_API_KEY` and switch on notebook access for it), then the
process environment for a local kernel. If neither holds a key, it returns the scripted model
again and the record says so. A missing key is a normal condition, never a silent substitution.

The live model is `gpt-5.1` through the `openai` library, which Colab ships preinstalled; on a
local kernel run `%pip install openai` once. One run costs a few thousand tokens. The model
receives the same two tool schemas and the system prompt in `memory_agent.SYSTEM_PROMPT`, and
the shop policy still decides every write. Compare the live record with the scripted one: what
did the model search for first, what did it store, what did it decline to store on its own,
and what did the shop refuse for it?

**Prediction:** write down which facts from Lucy's message a careful assistant should keep.
Then run the cell and compare that list with the model's judgment. When the run is scripted,
compare against the recorded transcript instead; the printed `source` says which you are reading.


In [ ]:
from sovereign_agent.memory_agent import DEFAULT_MODEL, build_model, resolve_api_key

live_box = toolbox(fresh_memory_database(LLM_MEMORY_DB), actor_id="lucy")
remember(live_box.db, "lucy-seed-001", SEED_MEMORY, importance=0.8)
chosen_model = build_model(model=DEFAULT_MODEL, script=RECORDED_TURNS)
print("Model source:", chosen_model.source, "| key found:", resolve_api_key() is not None)
session_record = run_memory_session(chosen_model, live_box, LUCY_REQUEST, max_turns=6)
print(session_record.summary())
for step in session_record.steps:
    for call, result in zip(step.calls, step.results, strict=True):
        arguments = json.dumps(call.arguments)[:80]
        print("turn", step.turn, call.name, arguments, "->", result.get("error", "ok"))
print("Model:", session_record.final_message)
session_rows = live_box.db.connection.execute(
    "SELECT id, content, importance FROM memories ORDER BY id"
).fetchall()
for session_row in session_rows:
    print(dict(session_row))
assert session_record.source in {"scripted", "live"}
assert all(row["id"].startswith("lucy-") for row in session_rows)

### Challenge: tighten the shop's rules and save the evidence

The policy is data you can change. `toolbox(db, max_writes=1)` lets one write through;
`banned_phrases=("card number", "delivery")` refuses more. The next cell reruns the recorded
transcript under a one-write budget. **Predict before running:** the card-number write is still
refused, but is the reason still `content_not_permitted`? Read `MemoryWritePolicy.check` in the
embedded `memory_tools.py` and note the order of its rules. With a live key, rerun the live cell
under the same budget and explain what the model did after its second write was refused; a
model that keeps retrying the same write is a finding about the model, not a bug in your policy.

Explain in your notes which part of this run is a local fixture result and which claim needs
the live model: the scripted transcript proves the loop, the gate and the SQLite rows; only a
live run proves the model's judgment. The saved record keeps the two sources separate.


In [ ]:
tight_box = toolbox(fresh_memory_database(LLM_MEMORY_DB), actor_id="lucy", max_writes=1)
tight_record = run_memory_session(ScriptedModel(RECORDED_TURNS), tight_box, LUCY_REQUEST)
print(tight_record.summary())
assert tight_record.written == ("lucy-llm-001",)
assert [reason for _, reason in tight_record.refused] == [
    "write_budget_exhausted",
    "write_budget_exhausted",
]

llm_memory_report = {
    "unit": "ch05-b",
    "scripted": {
        "written": list(scripted_record.written),
        "refused": [reason for _, reason in scripted_record.refused],
    },
    "session": {
        "source": session_record.source,
        "model": DEFAULT_MODEL if session_record.source == "live" else None,
        "turns": len(session_record.steps),
        "written": list(session_record.written),
        "refused": [reason for _, reason in session_record.refused],
        "final_message": session_record.final_message,
    },
    "tightened": {"max_writes": 1, "refused": [reason for _, reason in tight_record.refused]},
}
llm_report_path = COURSE_WORK / "ch05-b-llm-memory-report-v1.json"
llm_report_path.write_text(
    json.dumps(llm_memory_report, indent=2, sort_keys=True), encoding="utf-8"
)
print("Saved evidence:", llm_report_path)
print("LLM_MEMORY_REPORT=" + json.dumps(llm_memory_report["session"], sort_keys=True))
for open_box in (scripted_box, live_box, tight_box):
    open_box.db.connection.close()

## Keep building with Prof Rod

Found this material through a colleague, classroom or shared download? [Get the complete book at profrod.ai/book](https://profrod.ai/book) and [join the Prof Rod learner community](https://profrod.ai/community). Bring one result, one question or one failure you learned from. Share this resource with another learner and keep its source links with it so they can find the full course and future updates.
